# Options order flow and intraday volatility
## Quick verification

This notebook checks the headline results in Tables 7, 11, 12 and 13 against the public aggregate files. Select **Runtime → Restart session and run all**. Allow 10 minutes to read the six steps; the calculations themselves take about a minute. You will see the four development contrasts, the final-window decision, matched reference errors, a Claims ledger and an execution receipt.

Inputs are 13 CSV files and one JSON file from the public repository release cited in the report. Step 1 checks every fingerprint. The [Extended walkthrough](https://colab.research.google.com/drive/1XhEt3VMt_wj7br8PRf3j88DTXFQhJ_rw) adds coverage, session distributions, subset diagnostics, availability cutoffs, placebo results and a second implementation. It regenerates Figures 5 to 8 and covers Tables 3, 6 to 14, 17 and 19.

### Terms used below

B0 contains price history; B1 adds **option state**; B2 adds the **mixed flow block**. Their cumulative predictor counts are 29, 69 and 138. H1 compares B1 with B0; H2 compares B2 with B1, within the same model family. Linear denotes the winsorised ridge model on log variance; Trees denotes LightGBM.

RV15 is realised variance over the next 15 minutes; RV30 and RV5 use 30 and 5 minutes. QLIKE is dimensionless: for observed variance y > 0 and forecast f > 0, L = y/f − log(y/f) − 1. Lower loss is better. A paired difference is baseline loss minus expanded-set loss; reduction (%) = 100 × mean difference / mean baseline loss. Loss reductions are forecast-error reductions, not returns.

A **session** is a market day; an **origin** is an eligible forecast time for an asset. The development window contains 419 sessions and 160,832 origins; the final window contains 25 sessions. In source columns, `primary` means development and `confirmation` means final; `log_ridge_harq` means Linear and `lightgbm_qlike` means Trees. `B1_over_B0` and `B2_over_B1` identify comparisons, not loss ratios. These definitions follow Table 4 and Table 22.


## Step 1: Verify inputs

The embedded package supplies every input. The checks validate the archive, the unique file names and every file fingerprint before any calculation.


In [1]:
# @title Step 1: Verify inputs
# Start an empty completion record for this run.
EXECUTED_CELLS = []
import base64, hashlib, io, json, sys, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import html
# Pin the public input bytes and the second implementation's expected digest.
SOURCE_COMMIT = '9b63302e2a9d1c8c43fb8755628f0e3dfc48ec03'
PUBLIC_SOURCE_COMMIT = '9b63302e2a9d1c8c43fb8755628f0e3dfc48ec03'
PUBLIC_HASHES = {'artifacts/rp4_v4_b4/primary_statistics.csv': '5c215fc38344839ecedea7b27f69efcb85fbefea84f1367d06229cb7407902d4', 'artifacts/rp4_v4_b2_rv15/session_losses.csv': '28174eb5a4f032d0b31b6a7b65e76e05710168278419d87bf31a92ebec2d9ac1', 'artifacts/rp4_v4_b4/coverage.csv': '67941d02647ae329fcc704a0b109949ab4312cb6915006302186b21770e853ef', 'artifacts/rp4_v4_b4/robustness.csv': '09925f9cefe4fcf2c19f4a915ad2994c125bbafdb94dfd332cc821ce1f180acb', 'artifacts/rp4_robustness_public_v1/reference_qlike.csv': 'af976a11074c144f024703c3d149a71103a6fde6bee11f7e43da64aeb19aacda', 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv': '3e59dcbab5760464b0abf41ca2f748016e7d42a51542c195c3cee6687a2e862e', 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv': '66ba437b3878bb4b2c5241863236b0eaf59712eaf3be8a63bbe40a1baca56f88', 'artifacts/rp4_robustness_public_v1/pit_60_contrasts.csv': '65cce2e2c23602249ce2eb1f1f00917729580689d14900134a59ded89bc0d698', 'artifacts/rp4_robustness_public_v1/pit_300_contrasts.csv': '976c95b896f9f2a836ee53a985e8462f7a6cfe2139690d66e264367f1ab78df4', 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv15_draws.csv': '3d5a8decdf511c04b225e49894b23f54996ea0ebfc6476f8d3ed4357469098c1', 'artifacts/rp4_robustness_public_v1/placebo_log_ridge_harq_rv30_draws.csv': '3e8b96c33e63d772f786555dfb948a170c970a49e3c490b22794570d0ac9ac8c', 'artifacts/rp4_v4_b4/volatility_regime_secondary.csv': '58a8ab77312256ab7b198f1b685b576e800d7619d6138476d16ba4c1e4b36494', 'artifacts/rp4_v4_b2_rv15/summary.json': '451e53869bb5f5058a635302bb08ad920ce11b8771fd2c6780545255b1ac25f4', 'artifacts/rp4_v4_b4/regime_secondary.csv': 'eb1fd165dd03f5c24efb717afdae7a30e04898dcd6fc53cb3172c1a2d81b03c7'}
BUNDLE_SHA256 = 'f3f82f184e585574b950ad4c65301fe57ae7e9853271275fbb875f083911b263'
CROSSCHECK_EXPECTED_DIGEST = 'db0fad77a1ce7b2bf2cfd33d4ffb47cef119219834ab76cd2ac02d5eea8c4cf2'
BUNDLE_BASE64 = ''.join('''
UEsDBBQAAAAIAAAALl3gSj2Few8AABs7
AAAqAAAAYXJ0aWZhY3RzL3JwNF92NF9i
NC9wcmltYXJ5X3N0YXRpc3RpY3MuY3N2
3ZtNc1tHeoX3/hXcZDVXrP7+qJksbFlO
ORVLjiRXZVYoiLwikQEBDgBZ4/n1eU4D
NGXcxojMOFWhuGATL4B7+3afPue8bzev
15vF39er2c1i9WE3boePi9Xl+uPwcjbf
bsfdbDtut4v1akuAD14t2l+/BufL2+s5
v3fjZjXfLX4eh3fz7bhcrMbZcr3d3r/6
63LxF95cr3fb3WZ+O3u3XF/8ZbYcV1e7
60/Cm/F23C127dr30e04Xg4Xi9n14upa
7ZL+XaxXvLfdDZc3s+v5xWw5v9re/X07
231cz7aLS752CG13dG+7W1wM4+rydr1Y
7YaRlzfz3TiMf7udr/jsvsu/vtp3+f38
ZrH8Zbj62C5zuZhfrda6zuzyfSd424nd
35r39OLDdrj+5Xa9ux63i+1dZLF6P27G
1cU426yXIy8Z0p/ny2G1ZmLmS56pxVcf
lsv2a3YxXy7ebeYaq31g/NsF4zTnEtvh
dvZ+vZldjhcLzRMvN/OPw+24uRhXOwb5
8sOFvje7GeeroT3nJ8HDx4bN+N/jxY4h
3Iw3658ZkHF+cT3bzRfL4f6RDr1nQhdL
xnJDj29n1+vlzX3gK2+G2w1DvfllcNGG
wSZTvBuCrYM5N3G42oz6JC9syN6kHExK
PpcQeqE4VH4GZ1wy1WRdwsQYTLY1mRRT
qVxbQaKZXy67nHLy3OgbO+NBNrNvDFfR
rUuoyUdrXHQ21zTY81JtjTXVWJK3Lh4G
p42Truli4Aax2pyN4ylaB0OOlZ7lwHM5
348t11ezzeLyagQdm78O+x5G73O2JlXr
QkoaGXMeEldPJYaSvbPD12/+/MOPb1+9
/f757Nvvv/63l6/e8Ofw+sW/v3j+9sW3
w4+vv//h69d/HsD7s4b3s8PsLZbjWY3/
8sez1Xp3tljx1ALC2fr9md4FaZfPBJqz
2+GnNy++nX336vXsu+//i7/evPjPn168
fP5iuByXu/nZn/71zACA7fzmdsnVf534
M9HF9mz9bgtMP33jj2d/sGcX681mbGga
gi961uDrXWPPsys1h8D88KCp5F7o7ebD
OJihjfzzVz/8+JMeV1dIuQ2wsY/C1UPm
qIcsJtrH4FPxltlKtgzPGrZ4FXwAJ7lG
Y51zwzfuAC7bwGWJJuDI3JZoKqNgz5Mv
PCM3CzbYFN0xukwxzpcKAngcm0vrpece
jIspwrb13VgHXXQpuOpCcVY/JgyeOySF
HNgNPOQJcL189Xb2hAAWXWjT7eJ9Ax+4
apwtnmFklHqh7+bLbQ9hTJwGWSh4FMKq
t8DBB0Y8FBu6sS53hZIMaPA1OZftAXaG
L8N5NacENunRMXcZumJhsFyN8UG9cuc5
QCjFFfDoizMT9oLR6EdkGOhU9nt8JVsD
iyqG6PjDdWNLhHd39e7moImNvXRzn210
CTDlknm2c1NZuhlC9ZE75afPXvGOqO8a
xzNatCLHZNGHNuzHkVPUhYSpcSk8ClgP
maAesLgikphydqiWULGnLpsa72WTayDI
SjhirmzRIs87vhTnoGK+5iGOUgysgfDV
8Btk7QnReZ4LsIdcqrHV1X0/IRtrQ2y9
uev7UawDruCsQf4Q7ST596kpRC0NXoX+
Qf9fBnvhagwPTGPvGsbT4gFYQghBwQbE
fuwUgWUX6qERzjDK7xebm71RtNEMINUM
B7b8FGjMOHNn4cYQ90CchHow8zXkFGET
W7P3KNMeEK446AhTJiHD5UzcVzJ8BUG0
MjrRatqjLRXmslZQQ6cm+shC89Vh9VzG
25VGsgXSgXIgQgCTGn9OYx2BzCXCzMhn
FBS9liarBHR5wMXKkAZ/GRjzzmtUaMp9
g2sxtTicNQNafeyFTiEsAQA1EXB88vM4
tD1k2jp4w6wjk8mlKufj7vwYvsqLgwA9
vh+bdkRqLhoMtj6B4csOPrGhIIV8Htmt
NU7F0ifZO9/MOS6vkVfOCRloV4LkQi/U
gVqEONFw0pDshWDu7vFgnlUMZWLH/D/y
Ym9fvPmdcPby1ezH169+ePX262/+43dD
l0X2Bg1EshpWzEzkWb0vOBp4qhc6hSuP
KvyzuMryTQxpYVC9ybEb6/FYDs6JeGrO
uVb3q1qWKM9sIF0wmcoxjaHcLpWWC+Dq
uQgPHAz2Hgyz7kKsoUyyyIq9j8DXk1+4
mFof1RkMRMoA1cdeqCOUTkroqzcgGHYk
hXUSEcAulSTFqekLITFSmJb7KJM5NCRU
KZaWolnl7Cn1QqfA5mz1/zTYPj9px1Bz
56FaSZysEhaspvGZ2WsmT4hUkgaX4DD3
dO84p/Q8nCMZRVkr0jU8A2ulJADgleMB
Nz9xZiBN5Q2gQF4gOKiTkCtfC9ahs1is
bqwDN+8d3bMFU8ZqwFG6c1vJbY2cGp1j
aT1ZHqtYK/EYbRqeMU2YE5wsyz/wp2xN
J3YKXKxHrWpfS/gK6/NQr1/QHGaRbEK1
rtQLdS1YiF71CHwYUuXK3iwx67UglHgw
VG9qwNDJotzS4N8CxKgKBQYLpSNrNR7g
ZH/MXJA8FpUEGk50COPegYEYclXsGUbR
Hbp9HOsVwAATyJStJ6kgPYpDbkG+SuYL
e7IEnnwKCWL0rP7+NyJjVEdEinizTdck
dLL8Fdt1oJBH4eohM9QtfwEd5jyWFL1R
nWsPLZwLUspEgTFgZCY5pCpNaKaNrBII
Ar1TecIHiSlgDhiribvHGXkyUxIcQxYA
OA+WEF6DxbxKrrYX6iLLQf+lSaKKdTmB
rATnhoSxd2LELwBYdr/43X2jChJWp6II
2JjSiZyuTezHLTwOVxlhdLEgQFCJLd1Y
D1cBG4+ZwlC5Vvvewyop/6iaJEw5Ujdh
LCaOtKxBriB2ocmpw1wGyVGGs9C3Y7Ol
IohlTZEfxCQeVCcjyR9jgmiWCGt2Y72q
F5chN3KFdchStUaUZXVtIiQfENfTBxYZ
cBtsH+8asqVUVXyxUiNIuxM5hSwb9pX/
kM2nP4/TxYdMV5e/bLPioUIZjVsOxQm6
A+5YENw5ouPHJTAgg50nDeYBYTt9y0Wy
TnKDgHFP3QoY9gpeQqrRYCzgAWgZRxpY
By4haKUb6wCtQpIkTCxcVgduog0J3VJh
AgCSTp5E2hMz9gxqbURVDr9V69KuX/UJ
P4uribkbO+W9ignl0AhjDzXzzleNdtMV
ba/0Qj2IuZKLNiBDwUHLrh0g5iSr2Skp
0xxO7FfWPkTGzCOt2qBspZkaK7dCW8kd
wNqkAAaYHZoLp8HpMqKtlyUp5cFK7au7
3VhHJWsxQneqmEtlE/InpCXJmYTzQjDw
Gl8ExPx+rmjKfWOzduLghJBhiWhcN3ay
yNqSTfM4gD1knnolL5UVtNHSJsvFA4mJ
+IQvi3tW9WGyBen2xX9VXz2Zm9U+qx7L
JdLF6rFguKcjhJELIGu+7RhwUdv6qBpY
QUNjtMpdu7Fe2Qu9xdhHbQyQVyph11dE
wyEpX4T6nmy6aEVBDTKxJU6VpRxiIsku
LS/qhE6Wvf4XUFKVwzC/KoPGVsiYxnpQ
SlrzXjlhqtoCPuwIkf9BLchhUqHz2HYl
eoGVBhSxZLRWNT0ZoYCCyeyV6VZ21sZz
0GSTsqS2f+aQx5pRL3AMqbZt22mso4W2
KtPgJmQr2hjVVrZqdWQR2kAo0Z7aaXxi
PBXb2QYT91ysxmOzMAmZkUT+HbZjGjlZ
g9hfhwTAPQpbD5qmrhKqdgBOREaihYMS
1nY0J2LRuQiScsRSOLGsdM3DYm2L9JnY
FYtdfAA4XhnjxGzpoaAqIKBSmDuoNZpJ
hq3zHL60zcZJqIeuUlUghvAz1AqmGWIV
ZpO2t2P24s+nS1MZkyyaog0aOoywttgQ
g6LUOHVjpxAVvfONqh7s3Mn4sWyyYoFk
PDVPPI11wcTg69ANuaAKRPGwp42TCtoE
hVCr9oDC9GAEWafkG9L1xsekPfukrJS8
sariaieiZz0UKIDrAJLZn//wwMsBnYgN
SPpOJ9Q91IWTCsBTrKr9o9xOdWWXpdZe
izZ8tvbw5sXzVy+//f/NVXVffajuvsGx
astejjm0vYhO6GT9oR7qY+kx4Pr8FHWT
wlRROh3swuI67QaSwzcLxRdLIcPY1+SP
a1qNoaxOg2F+xNXunNQTY4VA6VCV7Rzo
Iu+00j/r8GBpD3/Sw6raJgRsa+mFOsAC
PrztQXVoxG+HcE6GCE+qB8CcMfoCYGVz
wxVcfN9gI72S5lx1VtK3WZzGTiGLbLEN
n/vNNk99BMwQISNY4ZggiKaY01i3yuUz
d9cZwJyiC4fiKS9CrqURAhpXphRGRzKz
zZQWlLf5rayzPLVwX4BHajahsFJBkLBP
r0Lcw0qnCEVHoegUWTfUFUQdLtSWjlLU
4qWHcpU6ZmZ16X+0yfOkwJb285RaPnho
GOAsj6vU2LRi+3HkpN/KrSLL8LrHgOvz
c9QvzMeUWom16Nr+UIMiOwxMv44Q15zs
ZAfRk/PHUJVFip1km1BhqwMU7cB07XAY
Kw13VIIvOnbTdqt9hPh0DEf1KL4ZurHu
BmJW1RAx0DENkORUR9TOFX4uaW/tAV7r
d4HW/01SiCdpSSGWdGhnA5hDK3rHrbQj
RpPQKTTpENKh+erh5l3HCpgw5eRGp0d7
oa7biiHolEPMtZmkuzopFoxJCaT6OMM8
qWGRmKUiT+OcTnPtNx5dbYUqFFUH8Mtv
D9H786RCf+Gq+DpWDEqb7nrpTARntu1D
d2M9XeTO2seEGnF98Cm66Dy3xst7Qf1h
ByCeAl3FPSRo4n2jZZzaDOMynBb9qegp
pDEP4dA8FmmfnbBuNQsZISuJOrwMEdxR
l3I+h77gj6sqVJNqlrYQ2jFbvhj3uxJO
QALaAZzq1Pwxd8FsULevoQLF0qqlQX4h
MCRYM+Sy9EIdmGm/AAl0OvUMz7XDJ7j5
pINFIfu20fWUiYtnGvYuU8Uo5wr5EHlh
0FH5dtRmEjp5fr62bJxJdI+Ck+wO3MHv
WFLqhbqnAUsKXrVTla/SYSORxA+zEiCE
Vuzy04oWmT7cg9Vn3nXE2Z3r/DL38ZK5
SHYwgRKSFYQ7MAJNNtwY3JGY01Yf3N5M
TGO9XUTkl3QhF6MT3MglnJWBL8awaGfK
nuSsJ8VXzrVj827/DxpqrCorOOGsxJhn
D74XOuXkXXPy6NOjgPWgSeqXs6Cwov/H
KjrncNjYgb4cLsrhcwBEDWFisSAdtBQ1
VGXLNnqr3A1zFQsJTYGTOoe0EhlN+wc2
0Utt8CetjNb6Yppipm6sg65YtWup46g4
KvRXFZvYKmFt57KoJPGFKGI7V27OadJd
84yndfo/QP0rWUT5SLt6sZO+q7TtSNBU
vvofUEsDBBQAAAAIAAAALl0em9Sh9ssA
AJnSAQArAAAAYXJ0aWZhY3RzL3JwNF92
NF9iMl9ydjE1L3Nlc3Npb25fbG9zc2Vz
LmNzdnzd27L1SHIk5ns9S1UZTpkAbvUi
bdRMi0MTJdqwOe+v+DzXRmJtieSx+i/8
WEAiMw4eHh7/+Ps//vEv//b//O2//9N/
/P2P//Pf/v3v/+2f/vEff/vbv/7bP//t
3//lv//z3//2P/7p3//n3/72vy9//Ou/
/eMf/7//4r/4W+t/9rfW/+pvbf/Z39re
f+tf/vl//Mc//x//99/+57/+y//19/cT
/n//xX/xt9b/7G+t/9Xf2v6zv7X98U//
7T/+1z/96/+2Ldvx57r8uV1/tL/2vi1r
W5d+7evaj7//ufQ/lr/WvS/bfl3b2c61
n9teVx77srelHWvb7nVdf6487mU5ruNY
rnadbe915XZt936u+7Us59H358rl7G29
r+Nc7rZvrlzrhm057/tu/er99ev7vdQD
bfVMd/fr293athxt2Y+t7vFc2ta627Yt
7aor1+bSevT6g3Xbjvu4r58L65b9Wu5j
P9u9X/fxx/HXeW/XutaN+9n3pblyLs5d
F+zn0bZah3rd7XmN9drb2mttlmvpy33V
dfVa27ocfa333s/zeY3l7muty33eS93B
hftx1FMdvR7v2p413Oplt/NYe+t1j9yx
fvKuq676z3u7nwvPep76kXqNpZau55br
ftSD1P+tdXyecj+usx9X2+ujnsueC/t6
1GNctTxXm1/6vq7rXu/t6Md6nHVhq7c6
+7ad+/7zkM+y7IsFbks/2nm1rT58fcuf
W131REe777zIeR5/9L+Wa6s3qy1z1hJd
z4/W7x+1Ndb12GtxXFjrZCfdtbX29ZgX
7ketaF3Yrn2/sw/3u2/HVt+ybrH058fP
867fPGs7HUdtrlxZ2/I6t7prPee+vH59
PWuznkddsKyHK7fr3u7jrK3T6tXnhdde
u6g+dB2Deimn4Kj9cO/HnUe6vjfNvv5x
/9Xv+6zn22tT1y8/X672d73X0T3zetV1
9WpnvUK7Wqt7Px+u1rT+/rlv9THqnVpd
2ep+9WJXnZ+1/ue5cvF+h4NWm3o7/7j+
OnodvGXrvc7IujxX1hapQ3qs13H1uy1X
XdnrjrU4rR70es7Ivl8e6bzqq9Sv1WUO
QD3NVhbirp+fG7ZOcz/rENRKHJv73ddW
B3BnRpaxDT/Lsv65rB6tDm0tcF3fttae
M1LH91zP+pd1pNpyr3+cf5Up2e3AZTtr
Uz6/WW9RH6QO0soCHVddWf94nnUQlnrf
WphnL5TJupetPrvPX5+pLr22wy9ttdeX
8/nAdRx8970urx/sd23EWtDrWM9al/qE
63zjOket7FqtRqsf311ZL1S/38v01IId
/fXr9Q3rol7fvw5i/Xp9/LuMZ9mfXhu3
fa9Otn/fW3252r+1ifvcNWUKy5JasFbn
oPZf7ZZjL9tW59qxeH7zqKcrQ1ErXAfu
sqfPeodzt3Pq4Nft5zuf7uc71Xld3fNm
sevUlsmoB38Zm7P+pOzuWub4yHE+73IY
9dn94NXe2+Esg1iHhZPY6p6192vz3hb9
vPqzw8r6tDrS3UEYv10nz5+ddz3ntX2v
TGOkuZH6mKzpery+R+3+ulWdojqya2xm
fcT6o6XVom37PT+HM1v2rR6jbnwMU1gv
fNcXrRO5zJVpZY/KG9S3ZpHz63Xr2rFL
vUM98bNvtsbCea6tzNIVO1zna6v3q21Y
Kz+PXj1hrUQ/6so66HVhWY9er3rVM9Sm
eR6zFq7nUJY7u+N9ymas7S7zVXZ5Pfdf
u6YzNfUS9W3qHeqWc5HL4pa9O2tly/gx
NbWTrVVZDJtiPttWD3046XXUalfUlWV4
lrJJtc2zGeaVZWJqE+w2fNn02tInM1um
odfOON7upNcHWPIiZRFs/nJZtQ+2cv91
+PZlevnaW2Uz6tlPFtqB5ozK6dWf1t9+
rU19wVrY1c8vdUrrOJdJZLzrkC/1tN9L
c9ZOrQNS55JTvbii+Xjih7r1mX3hGNfP
30xZeanjfi2O963o5bAf6pnc86rz5JzW
tzxe3syXvPKJKxq6Fjdd8vdqZ9bv1K2f
aGWpn7grXqnXPfrWXFlPUWe3bEUdtP1Z
8fou9VD1i7VR7nbFRpSVKldw77bZMsOq
2uzl51t94bKCF8dXnrvHTCzO1fm9Ota5
vnKd9tpmVwVe+/POjEfdpyyEbRfLddVX
rKNcbrIfr3NsTcuLrkfFZBVnsLB3s03r
xepF9hmf1T4pm1LrsFfA0LkeZqk2WONr
6sGfW7rhVlum3rk+nCsXRnttZUbWa51R
UIWE5R+3Mgi1tWqbe6GrFpuxrO9yzc/N
AJbBrKhq6QzmH2udqtrL5ZpPwVmZ57q0
PYuzrnVFGZT7do7vveKEw8+2EUH2xmlu
Fq3ccV3ahJWC4nq5imeOn0vrk5Z5KTNQ
V9a2ON213UIAUW3FMbHcn0vL8dQDcTp8
al1a9qe+Z8Wl12mvPJfWcalXqUco33ju
eZnbLiob1WvjlX95rqzPUCvC7h0WM+9d
Zj/HlxGIr8yl/BkjX2tRzrW2z1qLWYao
lnKxUiNUfC3R5mblqh2WCoPL75/7z82c
lbuW+hBD1s7Oy5x1M/+zduHsXM36BGUp
aoPXadz3vE35pXid2iz3vLTW8Ob+dqd0
Hy/eE5kIX2qzXs8DnLUi9UUqQ1g2CYcH
KKdUsYU91+aTnmec/i6O7Xlrl9fWE8Lv
3uG5stUOrAN3nxXh12dyy3LjZdXLinMh
568Fys0qWimTWZ+h7rm0Zw+xiRU2xjqV
Nc7NyiiUJ6kHFoX359L6r/rKlRlVgFP+
ectaeulKX+qF5CzPXe+4CGGBTOAsQ14P
2qR2h4zhfBlyO6MSl9NWyMaoiKryhjJs
tehHvdnPTeuEbS4pg1Wn8mrxN3UomlMn
cH9ZykpoajtUqlNnwHPWP5XBYMWd0ut7
fVxRoULSvlPUtaw/P1repIl2K9FzU1fW
ialNUQ/B6NUG/VxZIUgd4o19LitUf1T3
rBfzduVR6hGvn8NYIVXtpzKbZUPK4PT8
eoUlZYe54vo458+V9dlrL9Vy8+ljccr+
rWLaWqD6aD9X1otuTNVedqXi1DVXyisv
u2X1BD9X1rNUwlKmqX6rjLfVkQ7KIYWe
d5zXa3W8CcdfC1AWWqL1nGo/UJu9woSK
uespXVn/qU543a3seOVF01TUwpZ1vnz9
fVxZ26wijtptlUls57O9K4iptKJcbH2J
fudNylLZabXcdfy3ec+bwZTA1iut+TZl
Civ49hHq185pfljyCmMq/K7/jflZkuSX
d5ExHtMM1HWXb8MMVJJfl1auItiTvNcR
GJZqrg+/cNaPVoRTUUHfHt/qYFWqVgeO
B9lFM+UoBTK1JSr+WOeV9SmlmvVNyjfV
fqwrd9HnYq+363XL+p3KWuTiZc7qly9f
rNb1WMp13Y+fEVDXj4pnKijauoSonMIh
0a2vWCfzFTlK+eSyQsORY5XVavI9keO8
Z0XJnuaQ19enGZ6z7JavL5LqX359FRbV
Odnq8DG1demMz+sAVdK5sWRlsDlMxrA8
Y23UygSOaR8EnRXv5NvXb9aF9dfKIO5g
hHsuzS6KKt9WXsJdKzwpc7jJk+vj1T9M
IGLxerxtZUDlrUR5tVk58XrDun4udpn3
Oj8sZPms5nXKygnv68SUYT7m+/Ra/+4I
HF6o16/XlnSkKwmqjVv//bU22yJEqPyh
b51VLNM9M5K77s/RL3sd4V2mylPWvVcB
dJsBYRno+oU9uWpZWlfWG/kmXeLxumH9
0SJXrQD7OqV/tS1rv9WS1U0nDlGurJKb
umNc4eLKio/rA9TxlE6/8vcKM04ImaD6
FpeV0avXadxd/c+MqSupu2oJmXfuNhFc
nXAPXm9zr9/rknAHBldm7xTNjU0zAhNJ
Hudzrl1GK9ypo1v/VKteZ2Cd0U75wDoe
m4vbEvtedrYyXYBbPeHe5027sOQQ91fM
2+KfT4l2rXQrw1af77F29aWuhAPlqctA
xVUm0q0Uzy49jmnDyvnUCa2ItrxWIrP6
h1vwWWlV+fJzGsYj2e4FpysD5VHLEVQ8
ddWhlpJ8B4QbM1IvXCa9ktQ6B88BSKZd
i71BLvf4n3Iv9VADkjymd664gseuxajD
27f4HwlPben6uevl8mXLlZPXeySq4scX
eUH9Y32B2vCPGdmb8P2C9tR+bhCYBmJj
/Mo0rS9MR+rF65cf2m3aMiG1rvVVy7Ts
rxSvs1cOVO3dOib122wikKgC83r3b3Oz
ueCG4tUyXgCXmQNWlLAmHK/dWB92Z1/L
k9X3rWNwjGjtk3BXhHJJgMop9bxv/drO
btYhKpM1r6yTe8kuErVkYWpLyBPr4YTA
r9+urVCRjK9aH4bN5sAhRMzsBJ5qX8qe
68MCj6TTK6t3VCAOndgmLFD23ycpe31A
eq11GbrabyLL+jL398rEUt/yp4oOylMe
r1ytTs6eULuOWlkP2XHF3juDKwWY2VBt
haCwzlOtRDLu7iRLsGoxr2cr1DNtkpDa
5/XvuiPvP5TFrK9S8cv0e5Uv1xNVWA48
GekdLwDZrCyVpX4+4RF875aabhXi5FLA
bqszA9x84f/yh4rQ4Xvlx4Og9V5OtKyd
M73+MsWsnPXlXmrZWfJnoYXEy1KLWg9f
XoOrKBtR69I5kOOFxt31yItQbUkuzsbe
DCKMdK+jPOHUMkK1m4QBPEl+vLG4iVnq
D1/WuB52KV9wAg/X4It1aqSOixj0nkYW
1F/fqxLfFV5kcSpyLvcFdSkj8fYZC4C1
9vRVjyelrQi87F0drbrrOKVzbW7AWaoL
m8i/ssDn21UcVLvBMlcSWGsBOquvWwtY
W3OZX265fdyNdazDuQYHF1W0hI6w2XVe
Wa8h9DsTzwddlDtJO5jNx0fy3afIr6vV
tC0Ah4xFGLb77POcLk7jYtErsM6F0ogO
E5DnPitTv5GAQo2obII3r6O3fGL38ibv
Q7X9ucAUy/TXia4jVV+53S8PLrBcrXD5
vqCPIIeKJisYq017zODrDkRwQfPhe03Q
UstyS5Fqaad53cAy5YIchH6kplUmpQ6L
o7tMZIpjaMLvw152w0rqNhEUrzijjIqS
KjhVxDolZPl6cjbuP9bzhYrVO0KOKrY9
jjwkO1WHpcl913Fc5sKMkkg55bJJm6Rx
/dkKm9yKd6rtUHGGp2sSgzKRh8PwWNe6
sNUaxibVcvkWlSIs9WMVqS6sxs+FV1B5
0Vt5i+1IPaQ2wc7J9Cd03YD5DG7lOeXP
UjYpF1h2ilGqAPWcFzZFwTr3FXj3FA8F
m3FOOYRtXglhPpUqKzq0V3f+e6D/Ysfv
VfFozmYtXK1Muc5rfrIzEHuDFnc1tAbx
AQmvYKq2THtZlqfcV8VkoLW8Rp0ENZZ6
+TLPz3ZhIyuGKEdfe2hPnahec4MbACyv
/VWFrMPie5ziqDM7q+61KoSC66bHq+Vk
PytHkJvc3viQf8fqCaBfcFhtVcCHRL42
NNg6APNdf/X+5GWvtfF4zuwue6rzURvw
eeP6gXqG2jIX/OdKYW7JbqmVqNPV54bu
sKFKlVcBigs3iUqSq9qp05OUKYG+e8Ry
eb4w4OWqjVTn8JioHvRTAqF8VSeAJ6kn
rIizYray6/s8IhVJHLCNirJrGS4/vvOS
YBER3PZCH+sJm7jrFNcoeNdOH6hqPcz5
+zD1HKZRAzjZxGPeq14E+pFEo7ZKiufl
Gms7V5Av9pu++xpnvT6yolau3NUaTjCJ
QtC8abkdppU3UIuyc/L6FerW59znZlQ1
u4BNZQSvFM+ZprK0AAqR63PhmhLqqa7d
UjHdUpjeVM0UY5/fBibXPpTneDFXKpM6
oDvftn4vzl3u64bG2lS11ZcZIh0CW8BA
ZQr1o2qWFUiqBNTn2c9ZmatL6j/Vvqw1
tj5Cw6XcY9ngVX7SXjux/uz8rLnojG9o
4KcKSa7pceokbC2FjnqI+5TM195kg32F
2pBtnqhyB3Dd8ntycBGfCCnRte80f7xx
KmWt6gNKI4TsdTJATpWR7kqEryxh+3Nd
amlUiuvLlNcuqzM9RINBQynLmQhu/SrA
rhZGHN4nXaDWq9wxSELKfwhd69Xrz/ay
gxUOPx+OL6/IrIKBJmWWiCrtqOzygzPG
vewDBWFF5zrSnwpxg/dVoLLNU1qJ96km
Vj9VDq3WpvIEoGJtoA4Jn7loh+n1hHwX
jMJTuhmgqw74eVxfu2YVxd3QqQpKy8NA
tGYSk9pk2QEP1AZqwh2XD1tyv+eNK6QZ
lI7tHAWMep1KGS4I6/0mjti3kl1FRj8N
sSzbLth6h1P7KkopC3ztqscjhFP83GTP
9TavE1rxcVmWVeGgnJUIsv51LdQZdH6G
7LvMSdTjTfFbyiwp+iMC1Mmvk/O9MqLW
lk11wxBFYzOKEwuXWZFWQwNraUQv0qUO
f+rzSg69bMat+F5f0HKf654McoHOzSiu
/lOFgLWDFTMTC+MQlBWtD7OrpbXnUsl7
wt6K5q7gQBV4qjGUp0qePavdkKJTabVu
sudSdQaO7oAhT4+GonI7G3g3Kb3VP1j+
MsOC/u17eVJT6iyKK75K2IkZpLr1XSqL
tTpl16BRLWX3GfEt9oM3T0zqQrUaNCI5
/TrpIxU8ySnUphpKhR8/HFS5UXmDieCV
vS0zeahb1I2OMBZU+3Jqy6nNumD9ZEUB
rdxs+fx6shiJDSKv7vG22YDtOgswhVr5
lQG1yLWXkJiOjxeaa5MUEz8KGlTP/Mow
T4USyfI1YAH7vBxEbPEM/ldlxBMXqNxJ
/QVGRI25dnjFBO9d3eA19aYn15oMBjgo
1lEt2p9lKT9UTvmsbKnuuwd4utywBUs7
JnekjLVSOdpSveB1pM5nlx3iMiZl3lI4
sPTwwOohksvLlUXMYtZfq5IyG/is0pdD
WehTehiHsx3hJtUrchfBqI5RH6x4SrY1
gS8r7ocPIJyCQe3OcvUdDBU20Ly0boZ2
Vvns6ZysqERS56YOWd9/3pVt4vSA/6Ng
gXLF97YKbpdl4vy1gcscAq0BZcuootVl
ZdOslK36gGSnKi8WQsUie561D6BK6L8J
B789FWxsv5J23YrgdT5+AP7yiyvWX0z8
kSsPh6VSh/LADvkPau+DYsz54vX/AvgN
74Ys2JG2PlciM5WzqJcXKR2pCO5i8qBG
LHh7rtz96u1kgRjPXLmItSqwCO/k+XUY
ulLOrs4Kw9xrScpsg8l34erPPcuHLSxO
7SSUsT+2v3pwMMlrmf/+y43fdcEC/+lh
x8gFHuCwfhTlrfZQ0MPNpeW/ygIptO3X
oC+Oyt2htuOLdwlDbsqRL6LO2omz+AKH
Ke8ZnBK2W69Sp35n8JZR6Pu5cuPp6saV
j5Xr8s7qtdCd2gL1L5/nTCmhsvdy7h2M
5sqOhacKVkZonwir+p6yWtkz9aazHrR8
3CYs3mpDjpM4l2dbAgHXl4NMnbUR+yxs
iq7ws1TjUPZsCRwsiRw+2mt7K8QfgS0u
znEbwHJ9pEqjK0pd9mNe2nBJjmXnjdr4
1Gd5rF0d+DolG3PRy9Ie9klLGjYK+cqs
gPsKSp+SU0rUso2L1+nDaFiXispOIcB8
KRWDXc3lTI0qP1950Jkb7h9s7bVCAanL
sotkpcqVFszauAJArTemCaeuKGbpwdT+
XV+ebVEvIkrbgjLen9IvpKGOrJpWfxmA
AKoCtB1oIrysZ6uzr25dCzyLQE31rxKx
IGZK3nhB6CyjNv9i94XxoqSFBpfaZm25
Om9JDmXdz48LJhhkRZWWurNl9YXr70qB
vgz0Fg+IUMLbn7hAj5up6EV06MvAWVOG
USSLn+nTU17y7DpdsBBxTJiFNnKlCr7n
9rpyBeaVMdlxjHvCHaUWbrPMwuRm1rlW
gCzPVyZ7AcouCiJA7JTWZxia3HEPN6ni
CAFcIlvFbEhBm5GowFmlFm5f33swfco9
lsE64cLLr6UZNSw0Xq5E0P0qGVY2XAZd
iWZLqh1WxiamV3KbsZi4YoEWLypLfrMj
QDpYo5b5hKq1c/G3fFNEDnjUGkor/KI+
wazKQbwgnnsgQLnsDT5zqjEojhdJ68QA
gG8lIXXl4SwuiqyIMBOGOwGPh81c90SM
vlN4vAGXlYn9Whv7qqfspoBQy/PkTAMM
2ZVSMGZStQDmq5+nMvhk2otzW2kGGusG
Nlbf4NErkOQqJ+dYqhHHXmFRk5jatGX8
hXYdDvdsHJW3HS/5EDBnL9a2RuW+wh19
9qKK94bVCnyvqFrOVFvT38Zw2h5oY4GJ
M2TNXlxS9O04H8sWy92/k6t9yb9XUqso
6UoS+HyOGy1QBtDwAHqw53r4QMCi5FeG
UzH2Diott7ontoRMqvSj/8+NuMuWoB+H
fGUdCUT5ji1knLu9ahJlms+gVYCX5GFw
acStMhDHrAClHCIkg4pVLDaC6Rvl48IP
fx2oBGXqlIkwA/mj8olL6tRuy6+lEao2
3Kgd4+F8o8Dh1Zcbhue25M8VVYGQaw3L
JL326YVxYTMlQU6MfDBvda1t/SqbJySq
JWgYXqn5VsCaLaTa8LLCmwSjPsoeyDHF
YWBLIpKzTTIq/uSeBowKV/040qOgCy88
4cJc7HKDsC81zC0IyCqC2PF0bmW9n5Vp
fy7rn+GNNPviEqsiNkzOz+4Lh39T2V6C
VNEIyLseU73tsf0CQJ+k/ri+deKMsjKh
z8iP4VKPm6gnPFFRwtkNqwdcwDktsor5
+w0laBNQlT874+6dwTonksDzxR67QCe1
1vVYlVq5MGh/rVUQ+P25pcfsac2oM5ja
dMcwSni0CBAffslYnT1FyDonbEVZ2Bck
BWmKZfXDewwIFLK2x2GvPJuhHFeIFLXf
YTwQqTVNJGFjvQCD2rT1pXZ0Fwk0s2CZ
gUNlXfeZnZZvAxOoEV6qFfGjKk3lRNEl
nh2LqX36oeBhIfqvLfatDETbJ0a5QW6l
YuVKyxI7obo5Kvhh6rfr/N4zALjkAYeu
huaEvuKKSmBSDRcIp9p7xrvXaaoFnkC1
ZoXap/mJupMFPBOHSTLU659b1t9MxQP0
dIRqAUdE+CmHUeb/ehma40gfTvPnoejU
7bxT7StF2Pn1wOYgws1/nQP9c0Ut9/kq
D1zSFGwARJb0XwBvcV9On3b/tTJnzoid
30MIQACbSafToOFGGTxsv7I6l624KO/d
6zYvdY0+Au47+7RhlKW8K+bbnkCSVTjQ
lSusWc6QWjeVH8GdUnibh5SHL4vT9YjI
KkNz3DHhRoX9ft1VqUGp/+oyzHNcyvDp
2ELZf65EU+hOqWN5h21RiVz99QpRkUOu
X0dqxM09rQusem2XSY44RVCAXi1kCXVr
Q/gjWd7NpDyXugHcAlIwrqyQBW4Czat/
mDfVY4LE0OVW26AkVk5Wr31Y9X1asZau
o1BKagMOgjAmWr2Hwz82xifSrg+0jhYn
fWMhtVlxQKy/MhNA/XTcBkj/yg7ZFbAG
pxFPev9aoDWJ1WHl7h4K7UyLVzVkNZWr
DAP6UoCLCmGkedCRUW3+XGpTwDGXFOU+
1rs2wWhjeFEJMdr2FLF3NI2xhy4Z1AKG
v5ZrZrPNb9RBZlyOgOEILpIi4KiI4bn0
1DBWD9xU6s7BFK6MrFKCbqlam2mqos6J
3bgDjsL7Y/IGLQDdZfteoz0g/JVia1mH
Tb74uZnoK0Gl+A4wKF6sILqcFgd63xPX
1K4UOlNl92eypb7V3XYtSCgT80rE10VS
CSJKASKFCphVwv9ZyV7DTUteXtYqpYpN
PaGctAx1BjsVxCX1KZ9U90qSU9Y0HW5x
TrOeUjeoGItjuFJUUVDZ8OOPQKez4WGs
zTFMwLaEkodFd84dfofMrvpar7/eg5Jd
EarC+KgBzgOmHUZcDgwdJuC2sAdHDVfr
r8OQUqdEZhk5/AI7T+gwELvHAW/Isw0C
v6NXuBQLtH4CwWyd0EUvj3T01DGUBrZB
TlaGDyv4HlXecWnXxol0kYLFIM1qLtIr
wC2383v7tATnCmggCTDxz1qHCARnVfer
/CVxo3j7BMGtrzaUNJ/gnoBT6qddiVre
YTuL8Pi58vLMlaFxdBUEBulHfdNyADuZ
wWgZ3QuIj9FQljVIe7pyFH5RR2fYune4
fJ0Qpbd01jD2lVYu8eGvxi2MRi2IB+Jf
IPkK3cCP4NFhUl/bJxF/C0auFXTf91eI
gj3AinVhXksxZ0Myr989lxepFmd+g0Ag
MPbg5/qeOrKw4tmrIwqtXYXZ/wuxqZa+
DIfgfj3eXHUIgi7bVUSVoJqfrZN2pdls
ln0qKWgVUgrXj/Bx0ES40bLE7cWDFUlL
Up2viv789snYq1NcPy1Uc2HSUqn3q/7a
rv43y7HO0+kDB/8NaKCaJQTq51fdWOuB
XKM2f22V8xq1lB0j4nRej0kdTd5Yhrm2
tM/qPRp6k5ac5E8vbMOnvQQyCWbwyCoS
dsREG5NFrJ0qLWw9DIttUH8F303w3mdV
/YR38/0ym+NMJHX5LCKIq32HPYPy7o2U
N9JN9xzjS83KeqZ8f6XXtQnaFijAOskQ
UJc6PPgewNbcUcysF6728DFtCKy6l1HQ
z5IIMx2BqWVVVPBsrRF27xpl6vVaqija
j7qE7ZzsyUuu4IuWy1nSDnsiGNQOOhCq
ZiifALZjOeJVpXsp3e+1yxEqPz1gc1VS
0rtUvbULCNxmMWMT47FXtRN0aGqhg0VU
tIN8NNPsjoaLXJaepjPUZR5rMMD2V8eU
AHAL4qPTXjpejt5pqZjHKXs+LV6dmoK+
+1ozBiH9+UfZ/A7le67cRU7lLtRwyjy7
p1yw1mUDzM3TWc8kbuMe0dvgWz09QWU9
Lpa2fS9NeEqHEqp8EGQ4Tcex6yexl5Qo
+yBwgXRABtusS7LsAq4G8dtGoymISS93
nbjlzebrd2jeZXA5kCv8MpWHO397eXXK
N226m5jzaEBeNIiKq+wPe3+WvuXxWiMr
pblVciJLYF01GzERr+49Z25NO+356U5V
zblwYOQl6/fiHCmGhjCtswiI/rkVyjNi
xYkQWLsCpoext+uIWpbjnBeCiG41H8wy
15Ulqq+t7+6aZtW7aZSp58d9vTbA41Ln
Bm/x3AWAzx0xJhoUQ1aeh4xP0eGKZnw/
CeomjNf+fwZFzD3xrjB0YK0PH3MD4igA
o3QSQ0Ac0py/wEWFMO+V2dCMVx2zqBg3
CvS9Lq9Cgj5w/Nl0xudSiSf+ff2fe5+Q
v+Skvq3eMEoRrtRYXSF+mRpE3iey2EIO
ZAWwBgLTR6xAVgBp6k91yXmsAEhwBElM
nN/VbrgdfPsZWNV5Txux81ZXuFAOqKqt
0DO7ARGuwbehEOJkbn+hlSxBWX761+bq
pHiBkYkCk1bqNhMHBUgouRhuT20yfM0K
4Uc73MyXkKqXoNLp/BMjakbNosPz73ml
6Aq8EiZC+Ps++tJHOttfjYXDvkAv4RlN
3Ke7eA/TX5PKLEStaaZZ03NY3i+XXgB0
nJ5DG9m8qzhNezdbD1xZlfTCc2nDI36v
kBV0Y7lgmYjwGn/utehNQqrVJFsn2KU3
3ona6KJ48lzJHin0ob/WtvFZCCxca5p3
t5lY1e3VlWqBVfpHRbFS29Fzp9I+30Qz
hMNx6OgUwqdJQMmnvPAywoJPlbBO622b
ic3vbN6GOLFHH+WYbV/8OVhPhRJ4kiJl
be8mwdb11vv38lzZYKRG0DtU/icYV15f
XUzXmijHzdBS9cgpvpelftZH+NY1RVVu
BjqvSyt5qfse6O31xxOR0OeuyLhoIbqz
5rCvSo3Yq3sW9MAciKVe816zjjvBkIq/
y670uTxw4rs+BdZc+V4bPRzi+iySVn3R
89JdL1K5wwuV3ndsaVNvev430cprfXZo
4EbO5pKQVDS6H7PLDcdX0q7ecBMUyVLq
YLqTvsE6nkvPUDzUnxgKl9YNrZdAuvc2
0ZX7yAtqFENTdCV8HcVXdfXn62y69EQr
SP9q0n49vYr19/ceLsVzy3JBOpTKj+iz
y66odWhBjnEfniJlneolrdW1llDNupJP
xFPTND0Q/tfyHH/slczpClBVFIBvz2eJ
ExW/C+spy+yeRGlICZ4PefY3QY17lznU
w9d1qoM7V0RsYH7ohbgEbE7Ykdeoba46
SNzhXRSlIIC5jTyhbONKSOK6hvNyrf1V
Lq9vcEGGiSlsbqqYVFEVIMXqzGOoKasC
o1pia1nPyfCmGR1NapAXX4vjXgjrp8IU
CZsXhodicYUCsfB9flXLYz2DHPqFXpdB
uniUIx4yb4K4qnx3IKQ9hofozRoak2+b
1QF9NbDO6clf6Fh944Zpj1+ac02C5nZS
6qX1qk5EAE1fLejSi+K06tnHMbjErOsL
ENhVLOIHgVNuGnoqHAQQuv7aOjmmxDLg
EFLoNqP9Opo3dQ3ZW9kf+1WOpdt09Ei+
cYB+j1ZpaZgDXZH6kq6cOuMjbRkP2NsR
nRPHWWvC9lfYXeWa6XaQJHl+H3O+p7Ko
ju/rMFbl7nB8SNw8e6KDmrWMh+fkrUP1
BzWW5TrO63XlDsW3xmLgnFftBarhK1mc
q38vUBYb531B10sKOU3jpU4gI8KkstYU
Zkjo2PczQkFVXVMbauon+C9yA2RQMPEA
sz/GXsVIaRRtP99vjy1Z0N1Fvc+VGsKk
9LcAZrfkgKNTE1989owryjiGS4tKJUl3
Kbh3VfrRvNvmpbtwUWJa/1umx4mAI6SR
aQFKfp+udfEyLVXPTRlkmUdBVfBAD8e8
qz1YV9ZrLOJmvUDLK/BIzQ3KixB8xZpt
cqIgTku7pl/Hk+mfl1dO3UgeCQy4qHrQ
/nbCda5UhgMZx07dhKRQBpE9fjb6EqAH
BKZVTEvpniQladWRKPl4LgW5667JASh/
uEcXAV8WPlJ3+d4+K6mZejsd8qqjmlef
9VmOULDTU1fhcF157INeHQ7lMZtA00hc
BrZyllN9gzyYXRwcHvVgbkoUCGyoihcb
cn/seAV6JxSgTNCrszPMegBMEKO6kBJW
j1aIFPEdDacRmV/H9cgdSQGsoue2T5PG
sB9CXsIYUdnRmEJJJ13U6/fa5PDB/kM1
VimaJ1quv9yjHCzZtGdvyKraNZmtWbOx
CDuHXXZki+2Rd0saFooI+6xcNIpaNE/I
Wu0JKtA8on6lI/VZmT1CaXB9xdEzcQL5
ABomHatv2vtLgVin4xHA3XOGgZUOcS52
/nqtuFZv53BLyFUbSLqAG6q+tH8vj+db
krCcFWnpOniQdkoeWWgKBLVEWcg7Rfy7
wsL78ZZDKUqayh5x68hlmm6076yy12dv
wzoHPFYrVw6MZ7jUnspGXCCTe15KaUUc
WPle5VQxuMDldFUtJBn6fFQ0fLz5ymFa
rMC6pmZVoQXe3vtRD2JB65X4Iq8vdlHQ
Un27+q+zFbw9Wl0EDs70ijwHNf1LlQoL
nff2qX0hWp0MxjY6N8elV3h51hRi6sI6
UxAeWh7rNusbrGJYgUE0Rgmm3EtZIuSc
7Xy4kVL72n93bQvp5qCO1Ypfo/Z8res0
PpFZTHkD1T4lIOi2tOXUf3nO36+FYIwQ
fmnRjSrZzXouF2rwcnwv0NBd0V3R5Pn4
bNez2qhhXZMBEZIjv9sIJWBV7sv52hdI
vgpfqPc9KhYYoRGSwyyegf2SnKc8pPxE
ySDVS8pWmxxJNf31tRXlo22C/POpfKHn
qxctE18Y2oV7GXleYgu+oJOOXKOKoYLI
cyneWv2rm+bk/imH1HfEl9c71r7zikGu
RaIVLwV9miVR9RL4tAi4YtKk+DoPpD61
k0aZYVzpvITfQ8vwWoNVlNO6IYv9ehFN
N4Jc6qYXZZ60q6PZqGNR8Kq/M38eMRak
1XZwXpicgivlL7pzx+Q0ryTwboAeNlwb
Mkc7Elt95DLtS3tdCri7cNXPEJuQqkNo
kQ5QIPl1xrz3aHOHu+PSPen4zuKS1KSE
pCNrfBjQQVlqTIsHOd6ZCl5/TyIyPndZ
XGyZHprlcxyR+ICBRHQwwz9KTLs2xIqD
Exj8pEnEmvY17FmtBMMcJPzVG1l74+f3
0fk35fNLy80yStHy9l0o2NLS+HOp4kXY
ymLEJRTahOkiI6qW6/m1QqHjlrcmY6Z5
bH9F41h3RIailbiFH5NeowEu1h7YXtBG
ZYEbphzwJwI1oAKSfgJm5YjnUkI8uhpI
4bTs4S2Zt8J1WdoXkYguwqo9XwGiDdEk
SxA1mhS03w+gSq+2iZifZT/8TLDV2i0z
2lcPuoQUFV2tH+YBVyQXAZsdx68lSpVC
EaqnEwQXbboyubIWphTrlkF4aGnGWcln
7NNirQTEiPEwfjk8SMTkssRRGiSfS1UQ
FMfry+537NDCLKEJHejiz8ssH7Z8xQDa
n8KKAvxGhAYGdLZ5acUUTACMpbbY51KG
EaX3sKzPpXbXHvdOPTRSUHY2Xocu3y/6
yj4ouR07gQaKys8sTVLWUkzSgE/GJsI7
N2bj5tO8Skc+rSDgiphQumwgW6hcRzr7
ngt1z+FLSX1DqFO1rdOocMhwPheqc18j
XNfs6ZaBx5Ej775Pkh6tGfkXivaV1zkj
DBjW4znJR8JdHht97r5ShsN8YJ8PBaLr
/l6YtFz1cG8lqLOKmLr1FawsbLUWIRiV
v55iztT1Vc2oc3hKA1QNQiB0HM/09qP5
fq6UL4hG0R7L2KVIp7tJkapsx9PRtyVj
ZqH1aZ+WhfTJeQ7S3PLUgHVWO+1pidh2
b3vFXMePExD6uZCjoQ1ZES8OaH77TPMX
Ind9hut7XaKfhV+jxQFFfemvJEU/zJIm
vopNhlCbYCSpM9DuubRWBcGCdAwywmAm
EYMUiflMj7VHLwtZhREVlP4RqS/AqZpm
2M3PA8B3j66FxB7IoyoHoMmjeM4opNYa
DBRPtaaxAHtlgfLfWgYf+0R19N6iFMw8
XiG14O+HRZV21O/gJxYMNyZSs/CmeZJX
GCydA3pMdb8RBsS27BEBGVX8T8YZ/kiF
XpsKR8iQwMG0mTny88ot9b9bnymmVjhU
qbSe8bn3+Qo5b4LRV5h/LT5uVz27dMLr
iX+Raq6InhGX8NC5tIfHQT4r7TfPpWfE
f9BEEWEtURQKK91Qi1Zr/F6ivHfl7hE9
BYSfs1MLSToqzFQBPgUY+L3dIVe6JnCI
jAVF1zNOTDLE1ivxTOSOrldbTHY/tKCR
8EpYjhaX6pjE85w0KtiEUjqYoo5HvpFy
AG1Spe19VqD0SZ2ALnDqOshZqx6SJs/5
iE5/HkAxY4tmrYLvKIfg65wg1n37lWTE
d6STlbpzVDhnBr+kKXVJq6yefyWB2uIY
FcCd81xe3+ZOeQxhgcKg5AqEiRmMVzhz
K+3WB+ZIGRmLkt+X/qzWsh0vFUM6JPCU
rl0h8c8ZSVFOZOkfxtUH4hBYa7ge55Gn
1vKCk1SxyTVNB4ZldGS0oowcEOyMOibq
394R9PFnzgOu5RbJ7/SOz12uUZycE1wq
TD3fQoMZXZLrfu1cUNqpNrMGFHFTASQl
RK1n7WWw0hgW0+C4udI3CAh3f2S9xpWw
chy9uhRZPB8H8b3v4fC1db71LvKnvNUj
K5sl1+eB1kt08Xwt0EmvleSZ0sqeM0ZH
vbzfDShdfi3QqAiUV4EGNzyyGeqizZ44
xbXryjCDGZCmd+uhvePnS8cj8MO3yK3C
4tEiFoZxsMzjMS3aTvN6tQX00kii9aLe
yOBlax4jDT9d6LjAXQZie7KxqvPR9Hwe
kqQ/VbAj3iZli46QtWa9avP+hJtYtqoA
0ugO3oX41Z/A54Ci99X378WR45/pkaio
CHzzQi4ulALJ8UkaeF1TZ0A5DPSPxTLT
vqgbMKRpWfDSKClKt5Us1crec1PUnl/1
KGL0721UVuoBzsRdywTS6qbRWdNninbl
SgdIKwUY86lq86Gk6qQz6UpLNaurukke
Qeqvd9ok/JTh9IAEtKVfz9rSvv8uDB7Q
5wUIEuaVusSuVR8bZflLVXYI0bIhz5Xy
G1oxPR0Em58t+0l/ttwNueP7+LmnSlYP
NiQn8cdX2h9J9OlgaukDSSUC106L2TgH
SNGLAE7Ip+gUipQYrkdA4th+rgvLA9qv
VEzlpf+Fj5+OFhIt7XNdBdE9iY5esXv9
vMzmSBLFTKFmm8tyRdt5ier0Hu3qB6tk
GtJ4fIO7IxC/4JPIpvbjRQ7e0k2xYO3t
1Gw7FgkzL6ssMzB19bYQSWQoZzIEoxfY
CUFhV7x4NSKjidLM0cUb+l0Uj1cGIx1l
P1dqA90191CQjVQ2/4AnS0FsmRTLFuEg
8E2oy2HQpLFHfbs+yHkf31vmxqZNRyqH
iBW9/SwyhhIcDu6ojXPsg4ywkKkQ+6zv
FPddvj4g1kE68+dCvXhlmHYCW3c4IbwH
wFEQ5pw+t4z+NE9HLCo1WfJuAEYBgbLh
uFLeKZsNShZni/dMC8qBONrPQ65HeuQz
TYPLdCGtBBopyiC1rz9X3pr4hXuC+zL9
z8pEHmTVigW3FtXNCAIUiW8SBuKd1jDw
M7iDgN2r7pK+PD3hPBYQEOfP8yvTY35M
SWkeWL8yraVyUIPlfOnPG7LLr82wX2Eb
2GLrPfrs9pAaYdVzfxGR1OaxWMsjui1p
R0t7U631S1hZkK5VA15puc+RkBGypt7S
169dkzoFebc9/vKQXz4hE0RARav+PlJs
7cDlTH1cL5Guo2dtJB7yJeniSjyf5B/j
hlECRproltYkRkZFP2rpkdBgtZa8zPwy
UoQ1iuQYhVFN0tmrOxAOOoXBjzQHXlHA
a3eocCmzLSlUP9+FzHXmvqRE0pV7Pqgc
emYfOkyvpRllYeQsLU3QhknrDps4czBU
0lOjVI04uaHyCuc8x9HWoaaz7IFKUsNt
oYWcoSlP1OZYRhpCOo3MbcoPGJRXaMuv
Doc9re9pyouwYzrrtXdwibhk+7uggkso
jcChdWWP0ACKz/KuS+OaXjiAfFc67FXG
roBfea3v5UmAargNXqPDvM24nC6xhmI0
Q+JZZVkComzo9sSnZtnqiIabLVAJfPpu
uMU7lLjKvJ4Vp+CGPgnzKRM8VE6HqMMR
QHi+8iJfpThwyHwH9apjwthm9D0mY2eN
Ev/BMCD1/rQIkR27aXOtL0LVlqCNfJQS
0ZCwoHYvOIEotG+DnMiz3JQeRK+8k/H4
ARsx2AlX6TI/Ph4z5OP6BCkE/vxsvpGe
qDXrFl0G+RH0UqvK+WSsZpJQxUn4fY5X
EVULwbR7TOyNwcWOX9JgOXK8TgpAvrqF
Xj4v1SqlB9/Rb+klSm2e2n+bdRdRaZis
PfSNkQtqMkTFJvrx3XRzjBoFj69nQBZw
T6NMFk3tv7Y+5bhcCRRGG8awmgIFytN6
WGlX8Od5Ea1UUQlY2sz8D+KWFIIuEhKB
6pVVTTdS8p0m+cRZxT3YMCzCKbwi83ul
yDlpPc7EpqwcGaTz0xm0RwQDdHTPimuC
JoCaXHXsMlM/9C0f0ZP4wpaPAZxqgV4Z
zPoby72/MsrUTKVv1CeOUUu4InKUFGfu
bmkzHxsG4qjsRf0jUp+XcHRiRymc9SRK
iPBBTs26kKo0p/mFcZIE7eGGjTx1B3ir
OaB5TJwej1OhOfAioWuXSmjTpzRTXwdW
UYKdM5Vli7YtKht5GHXtN8/7+PNTVktr
6qICsl9Pxrsr/6uzgeAHXARfI6WqDfCa
vshb4C1CaJTNg8WMMQ2yreNp40vnBfQJ
//j8jIjYHGi1lhB/fq7EYYqVRz8eJasE
TXsoZCqqz69f9hRz3YnBpi/Hftg10MA3
53wBEaxRSqp5SvEpVUYIRxa4AM5+bZ+w
aw5zgiiIcUxTXgdn0UgTbTQiusGwi7DV
rWdv4lrmmtw9ucXl5UcwZ2Gp2fkwz5Vd
GYZ5pTV+hFWI3tkiirtB0F5bEgYqN7gp
FeZJaY/31IXaBKLK4AU33PEXl5Ap14h1
3WnLWpdX453BNEDrJKJnGIgtJxuiuG3f
yVW8zJIyfbgjaIovFEZzSFMjEtSMskRA
Sq0ZRnHNegx9eGz4cxDdA2Ie4bubeVBh
4guGQeKOuCrqrNYI8PAVIRbJ/CNVIBwH
sR/RiwrYChtcFD2xNreZ0rZoWPMuqtau
xPdbMrYko82eK7UEaThPU+SWtuSgsdQ0
2KRfx6t9NlgLKNSv7XHXH0EOVFNa1anF
VKxwoh41/n19vh/3lk6jHqnCFm+4Revj
qIQxeoGPfQS3k+kmkhvBpV2oRtCjMQyP
a9c6jBbQMotnGYdW49lNN0jU92SEYZTS
nKrXS5vrON/1ofDANmntpJKaI6Z2pA11
H92/5jCR2aqjef7Kzj+4cWWmZmhEWOJ8
YRJ3hsAlodxiLY5E6uLTPSymZ1ewNmda
PSxnQpo1mnB3BqjtL3BQ9P+R218HfKnS
TuCMInV52GnJZauhCxDhGvJVoz+3RzNs
O97BF2Aw5ORrYNxo02I6XISzv4qPCCe4
EYgVA5XVt6WSFFT2+o4Ot3EaxFs3yVYt
KU+KEtYoNG81S8OFBjvt3k+D2LxSzJxA
5SLskHEkGPWnpr7IwD5W7YysNHdOujFh
yuBhnmvKyi86LlujlUFlWFEuRwx86TB2
UcNr9kxw00wSbGuKs7BXNIyWrvXJzDeE
YElPN5B+HEeGqfdBPfvOLfYUj1Xn8HZC
BX0sNGFIVJHjGHMhY+5pmlA/SS/RY4I2
7Sz6BEx7uc5RPIbXeDzO4YkN9cNo2SFd
q2Uji47NgNkBVLtewaFaYFS/e+h7CCKh
QeLxzKQ0/bg6KscEvkSmGEYZPiMvn9Ni
IAzoZhdiabmYYWAWcli2JCn9N72wDXjZ
qLAz2veXLrVX4DX0+u473aYfH8sFUFI2
e+TJ0lYkFUMDh9ReKkM7w22AYuSxnkvt
YlmemHAQWM41TE99TVCJmaXBc7Vf3uF/
ufTKGFcCQengfsJTbew+RxSgc6XoSzaP
o/qicgjTHbwjffJt1OO0/dFLOsXP6/cC
JQSC1QRH3sNOf712IpZTh/mRTMVUQoM3
kxDsL9py+RADgO5IY+8fA3BjtIce87rS
AjYkDUydJFWoT4MLeKQl9vn9I4yljiXC
1acOaZQKHShGax5GKAk53zXqBTGBawRX
zAGI5PEExKw4dg5lkLFCPoSiHDP9IQzM
FWrRx2lRKiyvrZj995/+YPK3GDU0kO/M
0wvU13GRcabnaBncTuVsknSQeZf2zEnp
gJDrfE121Ni5aEqiwQDZS3aoYR7d8+9P
xXx0+rQ7s/tAmncEyNYwXYQfP5cq5yoO
3ld6njPmxZyNM8UQZLznt63ZNpqGxCBj
BKSqGIrauk2txrE4kRRdwiIjbnDNiSyD
65U460iBJUCXXjYbvH0SznGl+WcKvnUH
o+Pck8K4Yja+0Gx6g7srlWQE65igckTR
70YS1Nb0c+VKHWlTE+xRKnWlbE4twSs+
7YBRiVJ0V6SwhY0raJkyrHdiaU8zIvKV
/gMvdWioSUO2bq5o7ytEt+/VyYCLw3yY
oQn6ECMiLaNihTrLOVEG0NhmrjJC0xRh
31BYojahIxu2p1I0Mi9EkvW13qAcvBdC
84kN1S+9nwLo3GAAG6pVGWlUexheiCiw
pWn2rbF04w9pioRtVDDgSmUzWivpCHtu
WX9v1enTBJeJdqH0W2YDEFXZvg/VlTkd
e/gqR3g7cwseOijpHe/JaocStHJRekve
cz7NZkoQcPMn40JIzKVtYJsDgE3swSyL
2OEe3R3dQucaKBUG8lxpKLAuCH0/e5gZ
qsuKRRQu9ikws6adCIUqBOAo2AEHFoXv
tb2HntzddzCEKAro0Y9l+Gkd7dd3i20b
At2+RabwSHxfg27VdPcYLxX5MUAFx/QU
xupRmVdaFxWVBeSSe57bGCVmitaUhjM6
bCUxo5+iouzoUK+pPt+JcI7X4txpbBsM
rLE4ZlshDJ1YiM+VJgPhSQtq68T49YiN
n6MD86XidRCK1MqdFncaBmcOghg78tFf
i5OxLSrmJ0wdj/Y1sQkHZssh5wXS5Z0h
e7bStmxTQtxwLX1BCQt4i0pkenTse5Q9
XiNxdfVFYIMCfYQTzqjhRU/PuZ+7cV9C
3fF75cVT+grT7Uy36tRswu7WjxFa0p4W
ctHPnnnXQsLXOGCSZWCp/c646RMNjRMX
nL/GY421ySwr2nrGVgizXnN2K7eqV01b
ow6wKCIcsRaS+pfY+BYkOAGYdpU7I4lS
7gFRe8LXZtRSbk9HnSYCe5cK5xFizxSy
wN0ZKSDI/Bja1lqFrjTevpQPNVekI6ls
XRTRN+1IqkMZCD4PH2S5TqNqYTaiId5r
VOpwUb691JpHKzckw0wj6myUV5smxZJB
z5nj1j9F19rae3tfSWgxeviZh5ffzPA0
wbppjD9X4v3Zc3e4RRmA2sMVk1y8HDhS
0aWTGD63RAvasJ892JUc67lQ/b+NVi9w
6fjMplHpA1xf9kG1WaiCjKIDNsQyh1ZY
8mkee62L7axRZxmTvNpzQhSiESKb3GTJ
TPKMJUHPN4fhGTUBjDZPQrYj8xnKIpkF
SsLAp/y5siX5UmMHPg3VjtXQmCu5w/nM
4eDaOwKwXs11KCJAnVNx1sPxvK/SOMsx
2Nc9OgcUeG45lXrgHP9kdhbCjBYoyMGQ
11NIvLPfzuN7aVLPbdHPu3jm10Q/MYSk
kZwtkLEuBFEcOIEI0NN6yHXCoe1Gw6xm
YZiA7Q87TzUlSQez88r4QYY4xadEkSkS
31MQkpUHLEVG8zOkO8RmzaHMynvWEI0Z
zKiLj3TPzDq9M7ZjXV8Pqq5zK89tyhBG
x2iSJ+Rk6u5nMtBcnXvooAq5jA/WnTE9
T2b33WeK0WPqDtUP8UOPyNxrpPcSheMM
3umZzIKJGEofNll/uXoV0yE+tS6ZcWGu
HfS06X14heQYVcAUkfB2ZlhOhnCGyDVH
pAC0dZGMIUltzYVk3aPPt9yTP5tg5wLy
b8ixkTndfD2NK0QVtq+12ZYopipzCOW0
bz/hK0IyVXFgolHeVBNWIA2haXd8VhFj
HVFHmRJtfuhyLDTJLeUcXbTFWmfodcS4
xnD5kC1QRO456s4gLxN9aIsx09k4tAtg
NLS952NiTtB+HAIza668Q1kktrlOZuxq
G5XHID4vIc0wFeFu5m9dff8+VVvUPHH+
k9dd6ySw4kOcbHhG3+V4YtPIiw79kI8C
YMuMFspPhh2cYxCHiDmMz5dG5xHxuS0j
cnAIo76FKaN7Qav3E4QYR7d7LwQ+jBhh
rmEOGJkaVK73hekuSFi8jutOFEga00ef
v01Xk/A4Fz4gOX3I1DVybD/9yHNloujT
lyiYnlG2nt8NdG3M0aZdvo/47Iy4V4ZO
zu92BMddU1a8E9aIQDLeu6c1/8kV7lhy
SlTUFBKrrNG8BcH0mZplAHr0jIY8zEn4
OeoLYdtNs6R4MpLUM3tvTJoT/uxrVNqe
O0I7tUmu62fiqAkHJL7NV/3IdM112aOb
455NUfs92r2WmPoqBuKCEJUVJLepuCKj
fkmrOuxnyBA2dqYa6InlFIiJzVlR+BoJ
E/BPeug/tHFbQsB1BpuNljKcRzP8do8x
c5qdx3ykz7Sm8ePqVzDSu4+hreiEV8iU
BrJer982sVhsStsepwfuCFBe6NQsvw7T
aUJU1NtATueyTXuu3xMeeIpyBaMt+rJb
oiWa5i85soTC2mLQsM+PbI4V3QH3U20p
6gk7X6T3o3fsDCSSzMXzdq8BgCp6ZmPR
KalvXBZTiWIzqtmQt2mtDdkBuUuiCRW0
vzLHBXOnm68+7xniOSYzRftKzPa/7oiI
qRPREv4O+TZaPZgWNjrafp8ySpkZFRCR
LGYGbWUa+6FDTIjwPN6dKXC+J35ei/4P
esEVlsG5TQeVsXMGQnImWqgOgi4XFg4n
ek1ZKB01ET/cUxp2JTlpaoBUVF+/fkWl
zvAYUu7Wu0WoWmVV8DsndapwlhuAkpFg
sDak8XsyquNjv+ba3Mk4DigYGeRzecbk
bJhXcA/DLUiTOCliMWi9nPJxoIac6S1z
xvWDRABuh+xEauCew+G0t5okSiRjC5oG
vjjGuD8l++nzVPEFuH6uDWuj64DQ+y2n
mH4H2QqBXjk94E6DCrEkxEznHRGPbrOu
ZbhDbA/LW/OriRTL+bUy+xK9u2iKGZf5
GlZ2R1EGd8FkgyBPKhI3xulG03B9WUIk
mWsIuQYlMmpCfgn3eo2/xBDNCBuyENJQ
cW5q59JIomHPGo5u2VUvoC8d0UDLl475
ZX9ydFVZOBBOj7a8Nu6pi4QtuV+KVMFU
IvlykQLe0iiDMtkpw3zl3f3PUDT3dHWf
JjNUODtVwrFnaD7ZJvXkLJzxgpgNq7ht
jiQYGiNm4V4YU+fQy4JsmQ59tieywFUP
FhPlXdFK/0tD+2ig39r5sq/QZa2ozvkW
aS1/A5DbQjeabUsJQpW/yeiGsrkPdhGB
BT2wz6/zTy3SmIq2PXRRyGrEa5atb9+r
s49BhhQOqDNvZ3995N3yelvk7DbmFIob
M7R7mdm+QUg9sywwz8cUReWa22DESs62
lwWrd5FmQ0mvqKIhkeH876b+vH59ibAu
JdF6qcxwVEpYBjN6eW3bE54MD16Uy+/h
ByQYGl/6a5IWnI90sNFLiDm8kNIEUSZI
71eS2YnL1DJHevwaIP0Td1H4k0tRRtFI
4BtjVghoEe/2eaFqA0DrND74Hm653gJ7
tyNt/3xjM8vp6Wg5TOk8QwV1kGw4D8f9
5I57y3uSDtfLc0YWLYxvP67X7okk9+hz
UMg97HRX4mTGE0gCHoQYttB98HT1HUm4
Ii1y5px+iS/2P0PXUncFZOkoEuD8/cPV
uBPmSntB3ukFWow4IEERmzf59XbTla4m
xf7RthRKCWqJprzn0u0KtUfvRhtKjdj6
JsH08IFnEwAkC+MmXyfkF60iidgwxh8W
EWQrUnwx0mO0CnnlxKxBEeZzpj2fYZBI
pI9/E9qoVGc+yJvY1S0owUlumt6qeObx
d3Jl7VDENOv7hWMUbUcNF7j+L7XnFnUL
/E3aDrQpteMp7x10zmZdijy+khBVqyu/
ThaEvDxgcB4+wj4GZ6nqg/YjlK6xu8wV
vYhjBqjaUDctfZl0mZmGps3tgTKOV1cj
kSPafBjlpkTAq6403x6mpPw2xndG5xk6
WAtPmeGpkBzk01Ult5Bd+piovEeRQsfi
kyibVOVzkpDTUZu5BNG/o8DKqPzsafpb
bc+oFkDmkZCAiGE0Nv3rB2LRKKJMpC/l
HldSLFntujabPdlgE02M9zvGUGyTpA6U
Gby3B0dOu15Af637ZwR4eWUUC/Mqru91
WRf+ZKOLYiiNNoJHsBA3TjlxyVAlpiYF
rvSmCnqeCEhbF/RPCxWmb255hAWsTfR8
9C5pIGIVK83AS49RgDujrUax4InPyBYS
5f1JFfeoP8JJRd6GvUxdRQVUpTvtSPAL
6ZGOrDXT2voTjpP3BLnL30UMHhOW1QzC
xIr5tsKZ34jkUd9ySI48cFtmBwIzMntv
F6/wnI4YkRYspie04UdUbZXCl3xgyaJW
m9x46qRuQwkyFW+8l0we77LSiHa2dZlB
9t0yJJsiq/JvbLteC2PXTx5mPqiBnoeI
FtgxEiRwWhfS7e9XEjxgk4HVyAuaR7IP
yIeuy5yJOlZni6m+IvYWAbsZ2Z9mQKZ1
FIabhNWM3i3Hhi15DrFJm3emDbG+Kcgq
LtHNWaPzNktXV8b8okTtqTdl7i6HtQwS
0oxuKDTJ2Optyr4CEA/T77CklkdCNqqP
UZlYsZt74MOMRYaWRI/tuXId4iiRE16T
4CoqD0aT8uavlYnmNeXGJR+aHMDMV4nv
AQHTHpMSq0HwYA6lq+WVfxNBJ0ygOXjg
DWdmBUDOzxmhRRcdsw9lRcKlvguKwU0j
T/FcKbBfMg1D3XkUlxWj6GDABF9DKGoP
DQ1pMgEujEkmPJGxRc9TMt2C0YVyxz6q
QqdikIKreT7f7nsVoLGxPB0/uU1VXfNQ
I0/So6WWecDKNns6ZV4jhcwDpnprL9WP
jevE2R2EtEzZcOr9d9obVtpFoq6U6rfM
SDCwcq62Abb7FqFx4bObyiJNXNCT83I7
HCM0mGLUmsdcuJMr0hq4DU9ATJI/GXLG
1iRnZpPxkWtDXb8PVNLa8ux7mP86kGbO
n5mDZC72EJzhCJHcN/aFKu0LHYDs6q5L
JfiMHpQjDW/f1/bCh9OJcSTkF8T69UbK
BEe4na+aMaEHPftxE5fIecl4bbiLseEv
wkYm96xnChHHHtj37OpCurxeiHymPtZq
qDWqsEaHS46IOZHuw++1ASPcLRxFNEUY
+xTgTYtVJ/VpqK0v4vGQ0jiE+0WaGOIU
ZptZ5cgyhzskztpm9S2QNrQZu6asfHZt
UOMeaY7jeqXLgqeuL5RewpiijfUgrQ6R
c17o+ylVmQ1vgylzmowXbPn5aUkLpE8+
HUALbCGBy3zU/rUu2xJ7DiTYo/ZzTgFv
3PYtzA/kw1M+llnR0ZlSBp5lWGCFNjP9
RekXvEISVRukoDu/rwFVwTX1usUnGzyw
ZtQSad1ZADiikd1DrYpLJrHaM30ZnXAW
hpSSjkzFpsl1/Ix4M5PwUlGd9UuwrJZ5
6iV1WN0zw74jjdq+KQEdCnorOoyxrL7x
nNFF5SEyOpuycg9FA74dE1zpycQl5EV5
8C2SUeEOU7cmD0/SfXpkT7Ce6XHHns6A
C1Cf5leKr9MjO3hCK4Mxj0zjSTdgJDQQ
MZ+1QVoVshFG0XYYUXcNL5sUZznmt9Y1
d3LrFHw+rC9RgOZd/+Kj/jAXJ3lW5PZT
fn4rphPTH829PZ2CrkRq2aFXOMHTYis7
YO3RKd/jQLUt9ODS56ywGudxZk6JXrHF
BqNfkJpxrPbMk8l6mfPY03uV6hYcDAQN
6Zsl9+QnRF0r4jPj249LLYCw0TSf3kLQ
HxoLKYuWckuEo+Gcx4cMNFfGBT0yAkSu
NElPJAYyC4czd2MU/FL4gl4T3Zvwiprr
oW4YrZOUMGFElDmQEt53XHQUa6/8KILj
u/SQ+5Zl+hK9Ky00UiWqa1QwG0mKCgDT
ZTKhHS3xmsBAafuZxS6LhJ0puj9fs95V
tpAKU6ZfIikv8W4UIDzrL2MTdfrOQPsB
VmDOhDAKRUKC2XdynxUEqDW2c4xvnDAM
DpUOOiSRMyr29FDAAGfrc8rE2YcGe4zI
zlZnVqwATXH4ha0wu1daATKlxi01aCzp
Uf40jY8rz8yVyhQblAtgUUU996gXbusL
2EkDsAzKs7dYa1p3JpDsIYv9MjYpmoYy
TJwxoNbLMpDxahoJTD3gS85AUGn6e83y
SAdu08MuQx3+iVLpTcmNyMeMcAHxYcdC
el2XGt0V1dt9Rq6k8lJGB2OMQuee3C2k
qG0C/F7e3qbCNYRVG04BwhsJGaKgz5Wk
OG29m9b8KJ5eofCYCnJ+IrpnafZFCBLl
OMfJqOQn960cGj9i13NbP+VCHerRmsUz
PZ8rMwcSyJqxW0eigY8iUERe13lPojIa
2CPXmCZXI7SpBByw4x8fb0ah+Nawwvrc
LQ6ZjkKCKs3mPxdyvxfDmrmVS66ELdJs
NNJ9azOb14FO6dLkwwQDOuw3waK+3esN
gZ6Y1heth7gJUMw1uWiZFjdoQunnxHTB
p9mS3awzqD8zaEQLGFmKnrGZd+K4oAjH
nFJqJLNkVmSja23AF2C7KwTTSRVKlENy
PSpB6EwYEZwjPK+9LHuLCteR/pFtTImM
ZgpW3PYa76IfZFvHQPerpay8Kg6oz7Zj
qJK8FkZuqZX+2D9ij+31uioZeEtdNm2C
w61F12y++00AihvmI3i5hD9MDKJbM7Fh
nzmo6gSl9vobindBL847PR+ZHziHU0Gr
dKMT6TG+cSAIejNECvs2QyW1D2QgdI0g
QHVlMlxzK/Zjmfkb1nHasKAcY8xFPLIu
L0y89dfahF/8yUT36IdMGqh+L24LCDem
4FIIM9ehpWn8NaWJsD6uSwby5UpCupVu
KvbsczqxgvzA2FRpz4RdJncfdGllydsr
o2gpMgmbBM1eWZRzJVF4DRNWV1+SbDVs
C1n/mamiBmi19X5FnFeiYwo8mc4TxQgV
eNNLkQq/+Hwn+rApvXfapp3wY+JE0LST
INc9cF8uJYM9McT0Nc8rYXc96vgIYCF0
Z8CTejiwad5SP4wpO7pEMr5FHYjnp4Cx
PfHhJo0H4cFQSRG5JdDJJHCKSxOjokEf
rmt08BMskZK6ojGEMfbcUk+IyPJAjThH
3HKJVRni9f5lbJK3abzIHJKEmj+QIem1
c4n2VPBb5jWziGiPVT75GGIV/koVEWTr
Lscokctyd8N9MO5+LjQ5ttNmS8nWwohO
TxOl9hBLfy68I9wlsxoqIJmI3NMYI4U4
tnnPK9i+Pjqxwp33kScLyeBc854EZJoO
Z6lOz4fGT46D6/34XhiaCfpqM5JVXvCs
8DHIhXoxaWjyTxRCo10qRpiwpnYVjXGZ
khhM4DI7yaQDx3/ekaoGKUReYxkZMkAS
H6KFgvLsgzUyqOg13i95PJeomzrw4XPL
PTpQRsVSIEp2nn+6QnF6eBoc6xHRnx7h
Xm+tCk5Q8sQz6F/LssZvmyIgojQAsT+e
Gw+fpIZCX9TcXHpGjbji4ZSUZ4xJ9aNv
kYRf8hoaAZoG0LZMBlx0z9YgFPKVFlhF
0WboYLynIQNE9zGrd0v4UXfuein1mO+v
UkJPcccJjtY2uQvWboO/kG6dITM663Vk
3gCB1mPoFrOySonf++UjL157S6CuGj8d
Z8v+0nSn7G8vL1FKGR5i5rIgbzJZ5CuA
wy6UWhkRjMT+yl66wtCJXYyI7EJKdQFV
ltkPQmGpjeGoRAdTYaS1cR4ZrvkR3/zg
bEvGE1Vgp0TTBunDhAsdiZU8vQY2S5xS
Z1Ld/2P/q4fbranAhIHl146hUh4mNSjQ
BNEXahbVo8CcND7Jf0A1uYz9a4zvoXCq
tGasWgWLbnlEKDNjhbfJMQz5UyAQRcjs
wYAJWA1WaPr3LdieMCoay3UheIUYBYLk
8eIrWz6GQ/9JeXlLU2djjVJeCCdPPKPC
bcwWKWSDaGttGjUy2qV6YL6d9mqV+xgX
pigeXaHHx/UMEScPe34mZlF+kRLpUH8h
imoNKsv6RfSY5kqwiGZ22tcv7o+C4WIW
BL3a2Dj90kd4u1C8SdePlGQZGyKke55U
ZMA0bLrJZiuDTngUnuhWDg4QezI0CvqL
KEoFHgvzVjJsQffQGwFFBpnsXzWocxCH
BWuELpdR6XvgMHJyKbpv9DxcSKtyoQXk
9k9asusIZ0CFCz040gJduqNxPzt5NND4
gf2K0EACGrGuEr2EqF0vYFFt4eYB0lMg
00k9SBxAk37e8ya/r7flVBca4RRq96K9
bpls/W0P54y2HD5T6jFlQffoQfva9/m9
NA6A6UctAzt6pk3MMMUn8RzaeuzBprR2
xe3Vfz2fjjMxQY0MtEnzxreg1xGjVcx7
3RJPpJyOFj3jFRzUikW7qT4n2GNGHxHH
4EigMKeb6nIUwUHQrhkkdRQl8t2JTnNh
sCmRpVz+YSiZPKH0fivft4gEZcIq82fO
7X59r80VQht+L8Rc6jt5IWA4nIiI/LVc
uEUEIKOyXsF/xoHogMDJ2mNuMrHXbMn+
jv1DT8iAFa1suSV5AwREqMqcUClZjGqW
T7y6JazEsCQaHa9hpWcKp3RYdD4eod9Z
aOkAUZx5oDLMmc7pEZGKnsXOWLwN1W0d
gmRzaTZDMpo7IxgRIV3OV8NIBjlucjQi
aS6NuKrOEN7o1QKDvnxABoSkMXNIr5K9
PQpT73axkFs0batu2DhkwSD8jYrOC/My
L5CwWHZGtpjyXV+Sra33qyKUwahgNGOZ
Paj31ZJQ5uWapPgmY9XBqT69hOgYqQyh
Dumn7TsU3hKg4c+kpZJC/BufovHn0y3E
SxLikiEg2JeW++dKxKU2mj7OM+jZkm9s
lBfKzmsQ6b6i75oZnRkGoiVIh/ZZRLYZ
WJ2R4DRJQ8NDAnH2XuZB5XjyRSHQhyTV
dByaznittJOidnCsL55jD5/sjlCTOkts
Aek3HNLrO3/KmBMh7BLRR+0gM340H4Tx
DvuHF/X8wG3ycY9vpG414jGqSFu8aFpS
bPEyk/dsDs2Y2xCKiReFARpGNY4Gqzjv
yT0dSDcEzTLlBFFVFpx+oqdStg3ZQnEv
3PC21qDNLRjJRsR9+gHQ1i3cwmnJ0Wdo
8PD8yS9rk2wH9+JSKTLHaIZUGxH4vgTR
/4zRRG6h05nR57OrK7h4hDlr9wT9rBDr
0HgLWZw2RFHpykwgFLs1MNYiEVN8wTZ7
eXo6rxuRKMvbEmKP0RyrzqYXSLsm/dnS
bb5tSZWXzODIOIbltbu2DAi8Q2zVEmwV
OUL4qTzq13nKabfI+kibnvgZru/xqYon
cAirLG8lkHMHHZgGllXZFJdpzd9xZ8yo
5jaw8ldhFY9cp/w+gmwIZ1giy2dS9if3
QEEY8yEkFp8IEUi5Kkytr5rHER0eY3zo
nNhfi9nUZpCFnj7zlDs9ONTCoFMJebWp
MkLbT7FsLs01YmLtIipLrc9bZfZf4GWh
box6544pBezwgvl8Z0i6m6w+0xr3THmm
tsTbjlkTn3vumZYOijqGcSUXHjwQOXbu
MBOPFYkzL2yLwSYCJXbQ47Bus6wbKYBI
LXbU0ETvl/YX88ZeE7NlvF3ZCCNcIXYP
yOGfo4f4nS5sYlLj4w5iXLH9Mz/ajTGN
+JFExfYzndv4Vd3D00Fh9FDkNY7uvsYX
luCZ1CTze11oJxuevY5pcgcR7I4HY/TC
ZGik//bObFusd1tf8+dOPwF9buaX2h4S
Xd4KLN7GMMmdhmtCiunHegYbBeAI+qnY
HS79EcfyNfj1hJjXV9NJajxnJ5q5vGO+
Uw7Vohcfj3xTg6MBsKT1+LGE1+jHING7
hn0AS3VGzpBTn/gMUzQ0c9IPPfkRTf4r
diKUtWkySZukoHxuI8XcQpnHr58+Jxi8
9dPf0sJWkMBhIGOH9TlPIF2hhESYtjXf
D2omf0OwHnLDr5VZ07ydwII/qZR1BiCK
OyrmmHjlh9M1JSciFCw6n8MdIgmlix9p
+Igk06lo2TKXfX211kLJlngydPKg9Vtk
mVNCfo2yNzIk44Dq8KSnJtDtnlGbk7gC
XRJij/GQd4bPq1VEwoUyyhwAD+jRF6+J
YlV3rsAhk4b5yf2LFXv9+VEPIolk32bC
wCOks2DhQBVD64i6yh741FYA8U+d7Ta6
jyKXNwSwKMAY3yeSNNnguTQ6D12Tsggc
R5Rs+A0yyRyEKbRE/xktz0ioOzN9kHrI
zYlnj7dCvzFd9MJNxR5a+otxVULkAeVN
IT6nMSyQJV0MLr0FmpkEbbromxl7DUY1
rgcPuWJhvoix0QMH22auQpoBM4IdJ7q9
CijpLMTIw/Du4XAENMTP1eD6bokimk+I
QMCUcdl6Hk7R5ZJwf2JPIydgrb0JFDcC
GRZyeZsb+QAWKPxYeBVRVsN60Bb6F5mp
7Df2CkesuCEbvT6sP0b1i4l/WVEkE0Uc
b4iB8upMJba8Jk+QsgbhDl8NlG2e5nNY
zCSi5b9mZrxo6crsgrRfsPA/F46xEipo
+J15EcpFRwo4GOOPGyCzR1McfLUOCE+c
s4RKjP84+3aXO2Oucf/hvBZH3WiJjEFr
8wyadgV7vB338bWjYec76dm7vtcm+Ps9
dLrChJpuUfOA2nvzhZfBf0jXK2wJK+vV
REHtfkuTW0X2nyuRJrRJ6e6btkmDLHeh
AJodRiDIsDyw2wwkYVQq6Bm5FGii7+ko
6MY6zg7W1UChjEnDkU1fas9AZWrKwqxp
wwKn6uXU/9bT3ZvZPc76Fd3B76U5R2ej
vyMPWT/2YdxMQ5AyTHmGNa24Rmbq4Tui
ATtfBBFkDDGM2KnqV9qg7ev93t6yEtAU
U1PgSGGwIPZn2iGQ7pW1ZoZOBLENg75C
HcbFDN9k2V6psO5409KTSVzpUoCJ6DWp
V3sFxDY1RB1AeKacl3G1mIhqTf17aa7R
/ks+0YyM69W+Z5wpSlBChXtIK0uMYEqo
hjOaYrJ7aiJdpJvNj2h1ZurNi4ZDgkz5
4oQB5ohiYofSGCHAGdtEdCcU4SEOFWND
ApQnPLf99ZTRb0NQAUDnQgOuxdxJNWbk
jOOIKoAAdo5+9CMCsj385PtrZVa3gtVF
1kC2ME87PM4u2BUkCcpW8isdMHlKke71
gVUltvRi4f6GDgkWMDBj+cIvVFQzeokg
xJ3iSLoY8kOkgV96T3z4ohHPFNh7kEXr
MBJeiP7/a3MrKawa8xskPSWXK5FWJNhf
skFE4oiEsqKo4qO6lVZVeP5yfJvikKrX
PcKb4ov9eGULVLZrjfm3PQst4MB0ZAjO
l+xG1yQHLDbQMM7nCnVSlEojY96zZRJb
O5Ke3dkPkOWmh8I4thdUtaAiqwGnpJPM
VX3V+pftnAWf0GQyzQSJJWxMOyHSXpSE
9vmYpmasH8XGcWFLdJ/BZ19l7+vPpMCo
xfQ9FQ7a48GNSYSdEsE9P2LpZDT0WzAO
y9OHEm3gVVF+jWJmWDtAb1AniauHmbWj
NbQjwuw69z5Mjiv8YA1iT63xDAHE0Kme
WqbTx9BvyCoo7LPKGm3bu0e4suXHiY4c
0QIA0c17XlFANc+GGDiIuLKFXRKsmvq9
MEco1UymUjZu0qzgkBKjsL5Hzy8oAo4g
6pw07UUoIslmKhBVoWNcyQVlEojq8ovO
1PPRPE3IcumatZPVHNdJms8OQkaCVKUz
z4SGFeWAWuCr8UCfi2btIxPtck9NFOQZ
ztSf5z0zM4Egt/6rHtCL2qEiB673t4ca
VZfM0FqATLUQL+aKSNEWVLTtAcbCVT4M
1D5nd6iq+xayBW2uIz3FQBtMR7PL1onE
MMSSd7FUBChMDOaF0MWjLD/zW+aHhXeP
sJ+z50H0ayaGPLHcPbQELw2b+qFUZ47I
Yu7R8nix3VWCK3CP3EmSLnLb5bbpP67X
L0MMmOh4ePsP++jVATkGYOkBCNQHIL4I
LJys7Lr3F+9PEUKMcaAVZzK8C3FfDqIi
s32jkg68/8jpX0F39CTB7heinturCmc6
EkkF43Y8Zhn1e0j+h1TxXBjNs1TPEepS
Kkk3g9hBM+AMibtZbHjQRw57ywxptAo7
nnDa/svcOPFx3C0TOTJPfkIEZzQ+MhNh
HBZqSLqVlZLvGTag82uNBHAvMUwHDveY
h9heYjKVo4UEnEmhvA9LTK3TuoM85ybT
nr6FdMK8uCU6tCS/9uf1qggriy6pJit3
D1t3Hske6aT07fVC6t1rBlhU4jn4qkdm
TmKwjqHvc3G2haJeh5zo/ktX+nR4oboc
4GZlLdRqu2QnE0AP8eUaSaaAEuqIrhnX
eFIivgMenFPBcB30WrVn2yRTB2MZl8hU
tt5eIUTU6y/bGsIS4TNEs0jjkPj79uHU
vrGd7zELktR29CbfZF790SlNYtSa+1z3
1PiFSIHP+ytj2NZQJ5clsaEce/LOoVBy
JlM7dVklxE7PFzGo9pZ408JEvCb693F+
m77K5vVQkGbcssXOGuqHw5JWBSPgewbH
Ymc/v16bhjeDA6SVG49xRxvIWN+3ZJ7U
fKPOfWQOYtg74CR1nOPTSDuuvMJyvzNX
pIUacUXd+owodLu+gYpIgKHFHBFCM4Bg
dr+pfcDLNQ7ddNkQoWChR+YEPw7oir6E
IWdkbnPhaYDW6Vt8KTGa/g7DElQO6X9d
mpgycdaPwKr27woV9bKaFJ0JCri5R7Tk
hxbecymINMo7XXQ+BkJrxORRdv0rU7B7
zYx12l2kJwlACROVm64WXYrvxQkVY6Wh
zEoBwt9KalIVx6IMcjtSfaTYsgv/BFfT
0iEUGBumWSjSS1qaZT/poH+J/xFjqM8K
nOHwBgulAlj6+VtGn0wIOO3lGl9FMHeq
nxlEU2us2L+9HGtvQW2i1twCUNapvQh/
3frqXtFnpCApZmXMg+IiZZMoXGZA0bdN
3tyLYZI36Cfu96vGFnY2FpNZULyGYp3S
v3mr98Qc0zGQHptT94df3UKhvRPa7HNv
j4h1owrnThwmhfI1+szSvtkmgp4TxZRO
TT0XAvW6YQLX+0I0tCVUn5S58pgYIxEM
oYj3+nXjugmLbEGYeCwj4CIMK3z7Tjlz
gXqCQR1pj53NElrGg/2sKftf3pmWFHYx
OSPChz+XitTNzNJ/lBlGe9gN5UzgHKLX
CUXD6I3Wo/aYav0ayWaVSppDk/uICnvR
8W4+yPh1PZzXHnrgc90eXg21dtSa8bn7
QOKs3Dof0qvssZImSBisgzWWWdOBJb/X
Jt1jt7tQcbveKmmpWdvyAY4yP+l2JJSX
WoRgnitV40AFYrqecqB28PIdfD71wOfK
HqkZevmi+xZ+ISEAwXd/O96QxdNWJU9J
7a75wmfoPPuL0LCmrQhdL/3fo5kw89/O
MZNwVvnAy3ceM/0fB1K8tpUzWornr7W5
0xAD8SXQRhR+FjiCwQ8Epz5+GPI5Oizo
UOX+HOQLHy+aeXrc4lgULggUkxiZFiet
T7qvjk+TgUYxmYdS25uAblaAmTccY0rI
V5J5GcTB073sXWySv21mW4STMoYrUyNQ
JF72RjCvI1fZKo+J/2XezB5putfK3H+m
on+FF76ncaO9OoJTf6Sd1qIVFmwmY+fv
hGzXk5xuS0bAWDeKRmncyUyZDMGTSj0X
ItcjU+GYJK6j7Kk8kKh3dmoANOgcUd5I
N0KEyyXnpudck5UEIxX0GuIF30yGjauw
ZNbU9hL9YWsgFbW3yD2mCeoeFA8itJ+E
ZK7N6J3BitbKLAP4Oe0LZqydoJrQR2OF
fZC5iasseXuuRHLQPCwwOaJlmKaBC/KN
uzXvGeFE4x6J7mV1Wtg5FwxmbU8NMnNn
0ttHvLN/avF6iUAH22dU6OdKbOJL1w4k
Oipeapwp7Qpsnx8HDUVrncm69w9oQB7L
Fri+FKzvPwfJFTmRaU1v9IyOjdbV/5ku
rXPs66b4DtLTdzXlJbFRjdnZwhaNZMyh
hnajS8ij5pW7dhw17caihpnR0h7gkL3U
0hxHer20UAi0jSu1fxIPNWt09mUitq5A
g5O4RUIT3U0UOCNV+rzTwmYQvroy0yWF
RniIrnTl3w/cOtcnwt0Z+WA4Irxoxtpr
8kMxquRCVB7RwT4IzverezmTtqRSxu21
XEl5PAMUlndhcCXmRvaHvgQFl0gcm5xC
q+qt+F6fUgPQ/mEqnKjKV8bIKjDP6vSR
vPpOiLbeo+DHCSazmkjYnolUSvcxd5pe
6l/3OxNQtFrf3+syODyRROp29YuhvKss
lvcx3avFppurlLgfKjjps+Q20UPxRk3k
jAAaivaaBPOelDKyHtqj4TgAiqGpxldv
iRPmQPqojQOqlB57+Kl7psiQUDAccn6+
239rndRE4kItDHgrES575VQnTVpTdqJn
eod0pP1NEabVI23fS+OCbQlpOr5jn1qe
FM0iEayNo7Wx/Zj+lK/3KYcnKmp30Cjl
OtHuCg1VxNnay9psGZOnRIIUsgW0NbTt
jvKywbkzXCK04h1GO++1jROFT7GNYQFP
DIbtpamToPK2DhIyZyQ73mY7UHR0aLqo
HeAK+3FTLz2ibPf+9lLrEt+zE+TKULT1
La99GrOwq95QK+bPsuWZPYH6q8lnF5ba
r0fGnSq4HWOmaj9e2X3Owchsyy8HMKYJ
1EAielveObvRlqwQAdpYbGqqLUOEam1m
Wrqjph5Atgw/CdRqQiXGnY66Gdn0xAMm
Oyhh5H3K+8TWkVPp6/fSJBstSy1vZmrW
7dULlwGnskgM7isVBBLSHPVXwxmebyBG
3YHtck9p573fGdlcSdFEewz9uTLR0i6x
FaloUQfCX3uVkxA4AJmUQ+vbuFCtEkFA
g+tsEKb2QBYkgvV9dOZkXlM4EBj/8zlx
65e0atq97hnx/QXvx7f5Xpzg7ZqUcDz0
os7xCfp01iREgPJoqh2arDP7EfvkOVHC
0UxNwUDnv1vkLwEdGYn5XOgtIyhGgiTI
t0KSOqewdMrVyGb1Zmve1B4RjHy/h5CQ
F5un2Q/VFom+fhRwFrgKPJMCxjl5MDrL
gGXSu2vU24xF5Efvs/9altTCr3SWm6BM
qezFAO7aVMKdk49HxToqq8orbWrermm1
xoVIJ9eHjW7jmrUY+t9zJcFs4qlLuElK
lcaANuNNwJMvTZ2mI02dUj1WUb+pdTWI
2dJeWtsZd6h6wx+m+p+JzrznmobLCSr0
5C60qk23PP/K8E8KYVGQ//bb6bDA1VEx
ajopXq4uyA19whiGMTA5bUPbGLgx3U7G
JRo/bmBmyiXaiVWt7dw2K8MmGlwh9Gmz
60kWztQ5Dky9cxL11z1ob2aS7AiN6b9q
oUVVcndq9H8u5bQRzvYj9rqNAGZLKsyL
vC9MW4TYfYiTnhl9m/rdN+H6/jNN/hrA
NBApbU7FmJ52GCrGREWODKeh7U1RUPD4
sPzU/c/oeKbrNJMLwu8dE4qhKj9Xoh4o
t2Z6QxTL9AwBtqH995PCmdsjzmR59auJ
adAk0UUJKj8XprytLpZqWPZCHPbFO4Zw
+nOhiXhNhbfOx5GJKXGW2B+Ryv5elth9
gJ35ExGdmlEUML5HsUHD9yCMIOBs+sL0
ij/WMmjtgda4D5kCcL/G2Qw1mkFr9pYq
SJps+lBuFLYrSBsRPm+JoBVYZKWg48KQ
0Q4NZ9erFVrTqOalNUl1gv9dDwvJH55r
noGGMtwzfTfjpZLNLCnCCqHO9Zd3EtMc
/NKZKY0vJo1hI+RYdA8eg/7Yov7QMZvO
l/bBpnssPVpmdCemkWgp+EQv+xXcwmd9
TVpIZ+pI+AEmHJjkuU06gTYMDRyIQG0U
p+wUUrtGar2auoe+cx94XFrO8GW30KhA
sq+BJWlCAjrDxQcjXJCNXaKf/dsIJ0sm
BqMueIW0+WSqCoVrINlTV0EOcLT2Aslv
r9EJh6CXJuyRYg5krF2RudKs/nPZEXaT
wLhsR/3Y8GGCNBiO+vTzy1BNUqwqXsga
gz2BYd0yw/gBRNK2gENuHCXQJWFShI7V
N3HmfpwTkiUE1YhK5T/wwBmVa+quqkrf
KzOq4MAxUlM2+0uGOPpZJipAOAYmQSGn
vjfJxydUQAzPVK51zG4LZ0PBD4og23rc
NlIW76L2tl2jwziT+oie4yc9dxzkR3A5
KDRum+7WHdWVdryF2U2gFYrpJEtHVdhJ
i8Ld+uRYFG/E6hrlmOHQsBDOIk2DfX9+
L0waNcLCpKQjHHl3o8l3G0V0o14Cg2Bj
8BWm1b4mGdVXAiuvBlVExqSb05IhqyeK
8eyANTkWosOR3PsYpn5mgq0xiK+QmTAb
IVmIxj74L2Nmm7rz+cL4iRKo8CCjlufL
leAlk9IOYxzu1z3HYJFO4ez6NJTsgS2j
j7J8e6eER8j5EduQq88IU8i9RBcxM+qE
CyQFSBhipbcJ87lyjfS5yJcq3GJcTnah
JPeVL4YNo5VDU1xIVTBuTNHY08deL5nm
c12D7dgjEhv2NT7Z8WIClqPIcFdj1+/o
Samzpr+ptvAr+DHow2hN8w38CywxnIPj
phTb9/adWW4RDsKEo8tlZswLYzCPCeeP
imCFc6NTkHMwoK0O2Xsik5vjh+OU51eh
uksyOPj6Gxymq525w6iqKR5rUTDkbY2O
9ASLzFKJOAp4JonRnkY42ifQkAeCOdV8
BHcUC4buCFXrBUdnewta1Xcy5zB9cGTp
c2r4gzsDlz51xbk84ein+1ERgkrVTIyg
BRrH+bZ6sLC8gSCQfj0es7ysKYonHKLm
wQUQ4DLyCnQ1ve2pl/Hkla5665DvIyC2
pSZ1tdeVKtOZAoj8uQ8BuUT+dfqtxHxO
weqmaQmimkgussRHztmry6JdwT+FBZJm
dIJ0dSrXE0/5duD7Ms78mspI/1KnAuGz
a8kxMkkAaIEnkeFALwWmDdW9h5kQMmoq
8xl0ltLHhJzWyFepSF5D5zqUCB1YGHCv
fpskl35Np12YWpSaqaFmLPQ8z7q7j1Rs
DuDOELlDMwVva4F+oUra0iN4Rp40zDyy
v8jIPfpVc2XW5c9lmC6DC44x7WWClX1P
d2iGfRoil1cWyG6bedevOob9bDirSpsu
zTBXFabLyOmffeuLqeStmZZlJEwyheji
90CYsxU8jc0QCTWgyjs9ZyRwEE1J8b72
Qr8zgZvTHhogRwY2QxOj2jV3N6cH0TjT
oJKvLV0z/xGI3r4XJ6rwmgHv7Mn9LXF6
nJl9ZLrNlWlP4AiMfv2Z60szxtewxw5h
9RVxUFr3Mhg1wpcw1wnhpTVbLv4AVa7g
uOh8ayn6uc7gLUzPNdPSkwteIX/6NPrD
ngthcBHeK3NRmzdyozJ+nhA/tb2uXBCa
VblOSfA1mEUpCmXGyffCZHYBEWS1s57m
jCdq4ILJwSNW3WvkdjVw0s3QtLTNQoah
3oHfGkn/DMwKaQ+oZnbG83Tm1w/ZWmLN
+SrcHV1NYMw+5wpB7I4MfL9Z1ww33fEF
9Bnt6xSDxN8StC9iDsXnTNCkvUm9C0V5
Eu7UcBg/jIUK0Kg0R9gj3qEiivV7cYYO
TCMZaU4j3HIGfmHGkIKROmbsUcTXk5Bs
2ytClE025bzlo8q4al/WUL+ZwTCvpAQh
OeWZxuwoUlBlVrXsbu94DpB8LZnhtY9p
RsK0xEX69NcZ0RnVfZhaSQMqnbUK7BHK
TSH5xQRZt4zRFLi2IFU9za5nZsz0dfte
Gy9CrqFnBOMGpZ54kmLUmaFTTmeuxLGA
iaIqzF4kUPipgxbsG47+mXG8KmJauqeG
C2ENNRMCe2HsnhFaVK9s22zw29K2mAxG
+3Lih0pH+DEVWa2nz5X4KHeG5bR8wYuF
WhL9on+9fhqJFNtAbwQyjd5Z82KRptr+
vTAB9aP0Y+PiHr1wGBgd3ksizCEvo5yH
vaHHZMZomX1+ipbrvCXU9P1qBc7wGF6i
MODPPWqeltiPG6OnHXmh9PQquodXagJi
hZNiDLU8zYt3mnheGJCsn6JYI4Gw5X26
ukE7E5JMXAfBEzdkpw40ihQLzwjvweL4
Xpm0klC8IOYOuXoMv8pfZJ4wvkJ9QvXT
o7mH3vcaKJGudCDiRiYqlV28EFIsxvK+
+r/VrqnXA5EDfxq6pVx0A20ntzayBTqs
W5RBBXz+iAqf8sd7gBmaqIlB5QiSl4lx
TbjfoHnTdNG5v9Jxxm8G/aQCmUrY8Wnu
fpZlNRqdKIs2WTySAYQgM2WuIORdl+ud
3jC6KERRgDeggedK3Q2w5/yk1ixCgZQe
1LOX82fQQOsR3tVqpuRm0AAwQ562RCXq
+CFddT1wElYAU88wb+uEcYoOWWfvp92M
RFZaBiLD0bY8JgiBuZdsPbMVakOqEKsU
qav1zFbIjI67BwDtr2lj1iZlvuak+cHb
sJtn80Utlzi/w7NG/jt0WE9rjvnEVjKj
euzhWhBWP9mgEBat8zX/k/XV5oHAfcZw
CW+5TijOjFTKeoo/1mC4fQCR8kcsT93j
s/tcFIEehMcLsxhX2rD++vKKBgTrGcbW
FAMaWNrwH3XUTHpYvg1NhkiukYeltUf6
6zUj5TIoi5C8cluuxCzVSXyk1e+5cs1g
EFoBvp0rTU1WuB8o2UvI07qeGf5kHlOG
0lzAJHoXgduesydUylisjHiOsOCRtrDI
eLzHUem12WPbEEkiSWMGHvhge7EKVTwP
CgDkpXOdPiwh9AqEOr4d95gXugS9VPeF
2s7jnsnJGchhGh+dxUWrZtlKsej+oudv
axpGMkK64TyGXCiy1k0wMyg05ObwWtqM
BR86hlvmYiyvpu6LpPii3/Xax4zeT0ts
ZtLN68htKIL5VyplxiiRnT1Hd+OUVx1c
V5FSR5ANgVPBQoKpo+Z7XTJXhF1d74jv
fQKA9oFSzfBY06jfM28+vNxlzZQaY2Wf
SzX+0ydHxr5il3DNQgXVWX/0HyOy0uYR
HFyRsQ/h2Dcxo8Um769CngRZC6qEIXwF
ekcqGnjWr6Zy/fJQRRT9YOFXxCPuiGQu
r2yCWDzfivBoSpcmXDmC86id5PxlbPJw
jZYObr5m2J+HWzIjMsOk0zKc4ebKEjwC
13E/F0JG+pibgABmlAqW92UiRaaWPVdm
VkzUFBVYjzEVMPFCyIAP+SjdIJqVpcpk
EjO/WGtQ1PIfKR/g7BpSktrAnYG+mZgT
haD7nhFamAoUyPuR/oIQhDUF30O2fFm/
Tc22DIYeFUad0wTynq0qaDJqXop8J9NH
ciR4rgX8fLXIAv2591vvbbujBUm1h8qF
mtwcxRZk8/AvolOcBq4Dp8JgC7nU8+tG
7kTfC73kDs6ABSEbM4a1vUdZZHM7em0w
ZqIVovXEgNv9FWOQO4/+FaL4uJIydR8U
8v07rImon6Ad46SlrWLWDyNIhSxyYhee
g3rUQQOoaS9tM0G6kMkgADO4UkXfwyHG
nnxJj5CGJM5cUUitm5DrCMs4CqDL+j5P
dBGhzZkqF05iJMPITPdlGteOvEJoPNo6
y5gLoHC6Z5zHBAcqoO1h7d2ZKB31ClKY
mqkS838vTOauN8mhVreDetfnVnTm1Wno
KtRbOMQg59q1zN22zclbGiAB3CB9uqTR
Ilii/YE8ORVw9hwQwL60PbOnb1826jfi
safLbLuR76NBh5rrwmiYO2OawuY0LamA
vBCF5EgHhJbsVdngfOX7+EVqRwQyzVwb
9lBHr1qwMX9J8qal2UbWfY4W9iuzLZ9F
NsgMKVAvT2hOxyi0GrukEW9uLg8MdVtS
es9YE+NRqCdkXMA8o7hFGaWwRycww3rO
daBsb3GscoijA2ILyiT+UazGhjOn5iVr
jKCXfi7qWGn07wiDl1hwu99SXyRntzX1
iLRrE+wkcgTPXX8bmhB8BVB6A7VXTGIs
pcYri3ymXgjyuWhDK6Ii48wr8bx0iuj/
Pof4NzuFOorY/Bp/IlgjMQKkWD+UEEV7
VLCfWdvjiKTdqYdu16JwlCljXYAdgs5z
Zdj+MKJsr9hDdAPyPHgYLwavlbIDFwPu
cyUmh7RR/9gXVc3axLj1cAfZCJMEn25M
kSQ5s3T87ClOmHgis1G4ryMzK2oRjCAx
cbVPctkizU7jVJr9QNyZgJhG7T0FvSSX
CpCIUpdE8tmMZYn2iGoOrmWAabOz9BQi
q8yKy6bfJ9wIdcHl0/KbubBLUKUpshph
RuHbEfmgrPmprHTaf8v9yw4nImWjaBlp
GJ2cmUsUP0wf7YDQBjRQKzhdM2TI9FVc
vkzIGNPN03ArXLkzyO15DUm47lQVt9CJ
Klp3UneC9FSkn5U5U0bqeltqxwdnkKMc
EX2+X6+7hnCuWxc22xKKQwIy19v+mQ0D
KH4Z/SmRGTOroH46Fv/fru4wS1YcWdbo
hF71QggkGNCd/xQe2xSJxPnTq7uaiswk
QHK5m31GBPM9Ye5g4jtQ7MnoAwb6rq/x
zhzGzzTVQZ630IxorZU9+1wNo5dBVsyT
bHHXs8tQ1HFuLrBqV2nMlzCRI0HQ0sPC
sgoQ+F04aW6y0epdaa0I4dRucuB7v78a
uBBA1fO1nfG3ZYh1hOjTzmkM1uLpIInU
ns9bRCmRphxoQd8/0UjPnanO3kIpdDUK
5v+9vZF8u/QpblOnK47DQs54Rxjbw5x4
rySxIPEmVjnRVqzBLfcqAVNveB9GIVz3
GT1iV4+TyVJPRMBe3sBCpwKpfGYjuDfO
34EunXFoHsPkP670iuYcSnGyBwwT5Ars
pTHbNj/0ov2vgipCsFAS28uo1ofE7bNR
1UGuaSNCC+1o8C3H39JNQ3gInD/OkGu2
EgyUzeIe+Trj0jjxcRQwXxLoSIrA0t2i
hCjvgYGNm2bM2iRKOseQK2kVe8RL5f0F
kFYcLRrj9/Nu+7LP0PY1Ga5lyNv91XnW
eOQG40aygJcBn/y9P1HGEi+S4e+Js7kC
4hFqYxq1HjTLfwmnMd/lTEun9O3ghZ4D
FuwtLTVvS62JBPHnzQYeUU/Xp6YWvlO3
xAW4J7N3mXcyFyW0kwNhxOBSwAkuISGa
hcGo4XVanT5rThZma5AZipp7fqQKRAkB
xKKFYgKVdBNF9JSIh6/lBJ5YkJwyncW7
wlnk8/m9K0diIc8fd33/80GPxonlO+3Y
RsfnowLdMkt//sEc8QwlMK/4kZy2xMkj
oY181cmYt6KLLBdQvAn6cf6G8TXrin/y
fQ5KGizll/6UyOFEwR8uXHhkXj7FX0/t
ZzDjZcSj4jGbMdTx/Ygx2UfAraUOxIuB
3rLx3aPKf0mVxuOKWGrPkXnuoD2addvd
U6elg8EXovlH6/xutZor9reMC/KJQTEb
uHSOqvcUZTTHfGCGnAliTwZYlorQAScE
Vp17kydHpTeGLUfypGT2bFOm3RIZyIAP
BZxmGlE/4TCn5D5/TQ2upKayWHg7e0CR
AC0//Mv31oxWJMxpbNserHf9SCZdUC1b
6haXmrrZDTzCQ+f1Wz9qUo7SuHm2C5fa
wH8WN0Hz82W/ifT47PYwHwq+hinkngbJ
aNyMddMTki3ikmxuWTRcCj5gD6jvvZKk
ib9YxOWeroOsuGHahcqobX6oiA6JmyYv
1V/1lPE6SHsc1mU9NRQjl+ez+p1OWsDl
x0L8qqG7Ajen3HQpQ39j10uM7iSOwffX
OHXoEcZ9t/5bmspPA/i7cg/S1ZlzG8Qx
UzXwfyEuz9l8/nxNMZDbNOri0N01H+7A
S0KjeC/tIwLMiMY769KjjCHzs8SVvsDJ
LNsSrATF9tyAGiuchAqjxs9+VfTSpaF0
pVmO+zPaqA3kQpjRzo1pvhknZkyiPTUb
TNVzf6upG6OjDNvMBOj0TeRmh+nybz/3
0UhD2C2uC+jknZzwec7QdMro+HKCrr/o
UnOTPazPuVc1JAsjw+sclfxzacjoXq96
1eWng2n3WBvRWXsyqWHdgQDu/ft6RQOF
oC1al7NkJm9oe1Pft54JZAb0Xl3Ml1N8
z2I9zygSsb8Y70buVtVWdHv+7KkdssEi
6yoRoiMIgs8EIbLIhY9qfmKd9chHmzCQ
H+hsVHkrqohHR8NHP39cCeqKepVkh/fK
pHhqx6QhG49PwCp7UmR+Lsp5b/bfcFkN
rR/bFw3UGWK+1p86JV+H4bUouishfO+V
JYZlkBIRZwnwDHDI/OPPzT5+vTsDQKNP
TJVwrVpo1MG6kyHOMsKEgO2hxMjh/Ky0
5nw5pcwtv6gRQ2ahJT7eZ2FmT8MDToNh
0k2SycQlEqqhK5Pt4POdescKtdyelIni
i+DgWsB573sfCXrJbEsYR4rkbRtL2pnQ
ifZemnANEJbkzOfSSsT//JJGCEP7nkv7
HR0thIZ5yPGrQi33yYM+3rY18ihx4h0l
O5diSsuarqDNZBCrxy8AFhIz9k3tHIoA
30Ryr4b5671U10xHnt3KKVWseY+/8Azr
sV7fhSecA/lR/fl1emhvc58RjiWOT7c1
+5ypZgtSiQTueCt5QoQ9t9ghZuxI4Srz
5nUm3nNeqiVNXp0CP4eTalLIzK34m+u9
pMHnYFgorZ4vMUvzaO9BE/zQbL9L8cvT
VuIEOce3SRFKWtuMlOeGozN5RqVFKuPS
G2ADOzVt2+N7h8btvsxvHA+eF3zOE4zl
DiRN7ol8hQwAFjwPy8gVHOMElC2Wn0RF
jeOTgj/B01bi91Yeie1TfaJJbT6UZLtR
MqRP+j7DltznJeet6gn7CVjzSmcyy0V7
NyS9W11zVsuhNXE6qSIL6A1lsFzzUgLe
NAGhHs5xKvO/NDJFcO3fG+RXTBXpZjzH
ojHzGF+2Q20vkbPSfrsyykpfzdmW3R3C
6hpgEfKH3ZNLbUZOpSFWl1OZo479zzvh
RTG3HUyneFu2mTmPq2US9HygE3iKKoWG
l+a+ZxkAgeynJcqQhd2FmaXaY3CK32/S
+fBUANBEHeGEwM0fASPTOv9T/RR/iZ0i
yKozEv95ADcEq2Fi+DtTUhRfh1b/6cPm
z22+UC02qpOcBZ8lU8psUgqO0Z0Yl0rj
YFNJ23VzaeIMQ/6MReG9MpxXwjnCi5RU
pN3UqbpGy2cWaHmqhrBU73G+LhHspcs9
WPe/S53Ub7Kc5xSVsbrZnP6nWrTVz/3Z
t/+3szYUShzu6zoI/ONlbXFsJp7++bzn
Su0Y0570hkc84F8diU/jyJAUG5ceKfoq
ct044f4qPrILMp1NnK9Zvk7oWRJn2XqZ
1ftmG08OuH6bz6QlvOKVTk08F7UWiYzG
kXXFldHIXdwph7bce2lQ7/IVq1nu/lya
GJwtOELUlO/t8UfbfWg+r4wt/v6SnJ6N
tfVWCFFdicMFHEQpPzy041JNnp7k224b
9kcn2ek3PNvXT71jimYteEpPn0qqW6l3
K6PX+eJtQrV21FOhPD/yuXSL6moUdM/f
PX+B2/kfPEs47R4BRdP3i4L2HJXLuDLd
yMievPd+/JlYA9L6bcS9LffnCO74SHZ8
QoxXxBpchOgirypL9Z7IFm/10damR/2x
nNgrDCBxqBIrTxZ7LikTACahQGn07fdg
Rl1J+YOdXKSPVHfmBfxvaBlaCzhc8pBy
gp6VpL4lDEVoLTmLw2JhNqib25RClZBY
1VxE82Me0kCINDyT9/UpgPas3dovvE40
rnNq3hP/XrJwob3rQUEDAMA6Y93LlC2Q
dshvi1jm66c6UC9A2kafHzr6u0dCLJqx
tOeTSh5yd58+B6+mvaikdXLqphxhsfGf
hJH63kZ1MqkrWA4jttGcV+0Gs4fCmXUs
rj/3Ck9G5JQCJfTO+GS5xb93J2p+TRCe
ieyEkx2IxgrRQ0JxRQiCKscdKiu8TZ+4
Rk8dlipK42j5iZ5w4pqdY/lISkxgf7Pv
Gj1lLIdeqjoNtRzQPTNQSaOBtjlmXFsw
PXU5xiHOOwhbmr0tQxEuKsiIf/F2P7vt
DcOCo0M6HUy95VIt16kqvpVzlGyqdF4p
ndKlpaYtJupJf5DJLp5MD7VMJyjjCZC4
45EzC6OHDTuGcDoHM9y6qV8KRz9kOn2g
kSoVrprTs/7k+5nUrupCI/UBLsw0Q/ua
E3aKCiAKd6GalusxpWrELReoVnC175XB
MIZB+JS7W8hWdUv2KM77uZK3yj4E62Us
mmiUiNBzIRG0W/EE3Ns4sB1qMZSHCvN9
Wo2d3Rs1waDS3ulkUURSISwmFXrRKyXQ
s6YPUKSzpJIvcu95PnNo82WJd4+skX7C
t+WU0ea7EhZey/tKGJlZ2h6uwYaaslgB
OeCGFo5G+kriC+D4GY/5WVefg5uzB52u
HZf4sgkJwTRKo0dLyWg16ZybZ4uAcdGN
UCmZgji29HukUstXCHdfGbzMkpMbJ6nM
Dv7L0pLanERJm+C8Miw7pLqOqDKQuHYq
erbOC7CgIgXFEAX5ZobPDkLDC002dCx6
NfkPFL+e6TvrBPiwyJHGGbLaot2ewN3b
mDaiHvTp0yVpNfqEhKDhzG8YFCLpjzdp
wUMoVcBr9D4GMB4HB7gQuWh+ZMgpciMu
I/Lhr9VRJMJhMPm7EOLZ0hat+JE/2a78
VE930GgzykshXqLF4uVso0Mizb4PrO28
0EVI7HDtMajxTOo2nCHl9++dCU473p0z
E4QFzOzL1DIj2aDtjyxPW8X5hFBwoTbt
Hk0kRfLo4bO2aMowdhCdu5q3G5UkG0wf
wdQcrpuuhf7ZbMABN3FxbPHFZC8AKdbg
rxnqznaZRgMrifJ3GxsMEVMi7evCUlYM
1UQIxm2ZflTPfmBDbytowM3JiuicxwR2
QovMV/mO+MZJ1FSYsIbvkbDnHKlF72/X
hkJoI1O72nD1qsuJ3XTDpjsZLafjw912
zsFmueMFOxTwsyeD6hTTZ8m5NjihUPTa
OB7NXxMlAFqbo3XEl9GSJ/Y9Laj5idZQ
HSkf3frI8SI90h2q7d/nZpDGvMK31vtd
Xmt0DdyHwRrgepAzeKLJOiI23qe4wGLv
6Y937hdqSzlhL45F/x3pAu/teN5W/qgA
ZL+dWVVeo3COw/El6lZEHGLdOUJhPead
TtVpwSfW6XVEjSGOgeIC872/oBiEWy4Y
M9LgcbKg3xgPfI7fm3InD2lPvkm6e9Ph
Zn6rt2Dber6w+gvqFEVYMs6bVzLM+e1A
D55Xz5UlsSScTYYQy4r5PAXGW1EsJ+IJ
EDspb7xLaxjmaXBeBXVwyfpFgSoHY3kr
K9m3J+/n+Q4AuwMB3pPZYQjk+ViuJKMU
G6iYG2e/1phpahgW/fs6BYPjvj3L/p3k
tmuKAEyEr+SnKS7DUjWn7smk7ZMvbeXf
6GpqWuwR65jRGs5qIM+AYf1h5pVhSElP
OVSQG9OMxH+KUcg47JsWgiRYWHuUl0lo
OpdRGiQXX86Zfp/lukS5FSXmYs4hBlZg
pHlbEoQQp3n6IVJg9u+tCfV+00lRQMWV
utBtAB3OXx9+G+SM5H6dYbYteVF33qee
oPMWQlzJJzpQ3kIY55UhnEP7ipLS8Qa/
QZ+3Uk1Vs93m1nPlCIrVNAmNG5CT/2/O
LMTqpJ7T1Uy9dFNJnI6BJNdtWRCFGxoy
Pv9lrJzPsqAzW+/t/L5TJWLTYmBLq8G1
spYBW5Ygc7FI+FJMMbveBt1z17HF5AXW
AL0z9EHwyzOyLeJUegzdYLqOLYNcCcZs
uxd67QRJZF9S/yDhlnTvjZ1C+cv3tGTc
qzagFCniMvAVoRDzel0k+FTGcMxxsTgX
Jo5JnsuW5LnvXckY98ZNfooZ+9KrtdNL
g+fGqn/uyrVHXu5b7AmXLq9lVNSH74JP
B00r0nbePxT7+veM5kKLGhuqGrJEIWBr
oM/wz+596plJsru9RW7BbsyUVaeMvNFp
3pQ33fKG6FKPAdvgxmLxCoOeOuUsJgI4
nM7DPfHPtkTAX6k5vzcnrBVWhsgUkGfm
UX4bBq4zhKz4PE+FCu2qmmHStjRQn01a
KxT6KgpM0lod8Lhvl4+8g5tLg+wKyket
5n9wTP29Sg6yz2/MEnWUHKKekj259k8F
abrZ51fY9ai2SG9N7ejKroAQr9gu93mI
KqB5KkH5LgPWULyagbGe7fxu2yVH/tG3
buN9nx/l7Q2UzaLe7hC8aagpJpNW9F6J
la7HSwxXonRNXseWZipD8XulhmF3tq7R
bHvEOA5R91T6E/zpb66AondgOIao2gzN
88l2PdUb9HW4QOLFhCZfir2ajO6cwKbj
PlpVXd5RECfJa0t+hUPKTz8+7w3bxPP7
byjOkG/LwgrEacya7JArtpeaYVwhtxZq
MRsnGtJGyphR1xiW9RwD6J4n2KmCNMgk
DYMw57LOCLaNOAHjuqUZExJcCSnkGCW7
QiWttnvxtNeEqWB10iWeg9XA6U5jV+s8
+odycifqTXdTV0TvYr9/EcT/vFHZE4VQ
mz3rG8wIHLbvJMMlbCGshOgEkgXNCDkl
9RaqzdDQGKGPvbuWOFZq8oPnnscGkgbu
FsRlsFb8uGJU9EIndlEbTXA7V98PpWsM
l1M7Mc+27KM8Oqi4Yj+GUdGg/WClJ2qb
f1MNh5HLh+YpeJ0QT6/ko9bju3vveyIc
QxDxqBxLkjuTlvaH/0u3K3XfHjgNmF77
zOdDvObxpJz6JTpslmLDgnMJcdzYMI1C
E0aZHx/JVbVj0DIvLvSDeL1pEFMMJYCa
7vE2zSgLGDIvj7fD2lq3kUC9JcHHlH+R
J+TLa7yVDR0XhMGwtCe0MalE38cn5WYJ
0TRn1U5fP//sEFmM3sHTfBYtjqVF63xJ
upeRRxsfzYlfb0vO6hH+bltQVMyOB2Wu
SdvAjNtmOv3FJdhn7uJXGVzWK9rQDAVq
0qzxu+uSCMUlE9tUNXrJT3c25tI+jxCo
lqLczElsFbeblj+mTN8DSy7fRWdUzU81
AaDf9MfmU0gYjRbQ0Jee9+y5sicvNl2E
ssaURuD81Cmquec51L13nKUu2OLPnT1g
yy+hAfENYAsu9450cESvZPiy9Ad1zSUF
sAjmVzVlem5FgDrLG8PnpIZwciU+Du0b
OizFclmiAcxEqPWL5I4WLHiJSVSv3Bjo
n2enZRJBcnInsvNYktjMPt3Xyrrfx8yk
onyHniZYZ37VPYJlNl3xQ1fGK6y5Z0mC
yPK38DW2/ChQszuXJufM5pGvenr++jkg
fiXJVt2lVknTxW5aP1tmZvnRMO4l3Ij9
2WsSBBPo+Iwm3y0hJbHcXtCI164eVLIC
XCf6WyHvHrELdIzACphn6XiED+acySLS
coe8fTvbCwHGYmbNCFzGgIP7nQ8l8vEP
M1zblw9t8ageaSluviEt2xSHp61rmSOQ
4VGJZKpa9jztmhqD0QP39D7DCV4k/xN9
Xd1LgkDIMLKKbTFQxlMZ63NOwMZqOkoh
Nz6/bjn/aeTULQ+RYpAF0Rxl9suJxU0D
jjPT6HzbDCwiCmiU5mCrAhpfmlD8uGfG
jaY/CQmuoOKLTDecA6NClhkfqsjWymQA
mH2Bw7mLeE91ngeIZ1RRp0V2XIt1P2Rp
i7hcqPzNpnhMfH6DurCPdw4jQQV2zHPM
yovUQF1N0QTfdyz6RHFnKBOO/dtSUWjv
5ExgiHgaNdaoxhm6DUqO5coWmyec6p03
rEQogV5hNDLJO314kBxgUJV8qNis4D5o
s5bqItmBV0YXfMJ+URSN6P+OdQXySPm3
T+Pa30de0WOgMC2Zu360Kk6kGZgjGeWV
cB6zsYvn5+/utP+28uut60mRenNKtWUx
jWOIxhfDiFtbkvkR7mMkFMs7k1yIhI0+
RyGfyT13KQ61kxZ5nTnuHdvMPURzUYW0
qwyow3SsOj97CQI9CBYPkyXkz2sxVVmZ
FZTOgK2HCOztg/JU2C5wg0R8KY2v1IVn
yFx6IGIVkTPO762JcYifhaRGNs/8qD0G
VMSsQCsDaOb3Tayngm+5h3mhasj2w3hI
F6BXf+x9eenBrIw+vK6JHowpgAmwSo1Y
Yoa3kLOe38dANFx0OZrab3SrSx4q4ZbQ
13CF493ck6B4DcDffKd0GTTpUTUZ0VLx
aSPF2aOj970zAUK25Ofilp7ntXShT3BO
wmT1baDeGmsSWEABZpUHKmy8bcoQX6ku
nrEzZ/x8Cm6M9Ssrbb0GTZynzwF5XwJv
gb+IqfoQauY3dKKWH1AyMH6vdA5odxjc
tIHH/5J0HKlXtqL39XR6OTkiQLrawBCj
5UgwAJMr37sSextTjPlSS6T6PEcAk4NC
3L+ZCUFNUkQDWpwNSig6zRck1B8eLTF0
etpHW7SxrHXc/p0n//l4d9DUNd0vVeMC
K1C8kgomoWtMLRWguPpREM6/mH40sRd2
6IzwYp64rwEBXJeuPeeBiL4H/BDCSwcw
S+T31mTgrNXkcKuV/xHcysHzqp84bQ5j
2WBqiG3XuU50Y2hxwAWg0qDEDrr3ULTn
Gf65ty1sVG/v3sPB1q20hW5bog0XO0xS
Du7oI7dgdk1WNbQVSNvULKSV48bt/Blt
pIOGujGAkvNPykasLb8lbWQQ1LXUCAIJ
xv+5O/eIDBk9GlGy96oiMXqE0aO/CyZH
bCtnggCraUiufqYMPHvccD36drX9fH2L
5BYEV4cETW2A65yq7qzzz0s17ZGkrPrM
RxbUnFO9AF2GDk7zokYW9p7hhUyoPIsl
2AdLzTGNyTkDOqKZSIgMNfqNj4RSXvX7
vTU5/eUYGERdP5cl804ARrpN+lg5USav
PGFX12wFIhc/K0Ygz8+nZDxgWMoRsvnw
+ZF+ZSU96urIUTrCIWhBrz+3Z0ovMnCw
LULS5NdEnWi7J1Z24Kx4W0QIOK+08Dmj
BhMeccCzYs9p9h0Xpq4CxdWeOEE8F4Xg
mS/ne3cCAw564UjQV12+OhJ2uSOkNxkG
YuA2oMutfHQS2lBbstg1HvNW2eSz+5jH
LCHSz6P37IWCov6iN1nWMJP0ZpZn0ZmH
SbKxd6Zxp+dP4visgPu9tmruZDJvsqUz
KFArsjJjppR5RuVAuc44PcVc/giA9xHI
BA//t65JIo6a02q0S5apc9LhAVHR6+Zd
GZ87MR5ZbeS7zgvLkDBqbm0jQ8Cyb6QH
f7ct03NjD6McRpI2Ehn7lXwWVf8a2HBp
zrcaZt9YOaOZMiLQgZ1H1H0Qw8ziyDLC
ThLWAjPpSHzOJ6eaCDimMVpmyJPZ2fOU
yS3p9VvZJHfF9hPCYAJXFyM2ZYk9riQN
IxEC2nP74P0tGi6MSuv2FiphFlkjXmlK
Vo7pQTdbu4DubqEzWZp6og951+klprU8
HAPtziv8q2xA5m+R2z7XL4GV9in+YC2E
I5OWS0iJVlFJf262NzLkDBe45zOPNExI
OXBz/nmpksvhfKK80fpdOKK5ncZKRihZ
c7ocVNOR4EPmlSxt2CCXUuVOD8s6jg2Y
YfF0NegodDEwtEVKr1uDysi/9Z+2/lcK
KbKBb5G3Et0GDpckjY31dwFXbiUxg6gj
tecj2QaTi8TW/l645/3lQSvASVaxWC6c
pyNg/9yafQuPmCxRTOxVF0VRUmod4MLM
2Qe5mD3dS9UWeUumNQYxNrAaDcfltSM4
oMifmVDcF0zIfNXcj1lFrJFZyNvUwcES
Jmw9FvA9ZwDrH9kT6fMxZUD6ruouXaJr
JHhEtUn7B7w8D7mkgy0ZEM5m4XInLEZi
HGDL9l1wMg2iiFJhMSv3d+FsCZPAWtnc
HNYontQj1gQQyb+f2VKD2mZAtw4X+qLv
IESpHf7ujG37KUVilKGzDrsxjUcnkKu9
O3NMh5cTtxPSOej72jdmFs+j09+fbJh5
R+gjMtYHPttaxw720u3zbxErp/0c3nuc
BMTtJzYmGvuU0o7bkqGNc93AokEkLYda
NY3N0WHvGILI54UWsmzkNPdvGhj7n8Ty
MqI2OrcYdvuzrUyIKIEE9BtTeQlYXOPc
kTXK9eXhIrkM//hMFFDo/1hRaXi4FXMc
Cjkii4P5+UhnP5FkLXdm1UGIQjq8TYEA
Zr/Y4mRXvZnZ/vPMJMmLuYNQB9Vg6XKk
PHrKmdi+skswC2rzCuTdVhGMye+5jwSP
pKr0JDjTKMJ3LLq74ODMOU5k5mz1gTSZ
qvUlJg/yBsb7iixyBKMBAhfaD5yPCZ/F
/CAzzs6XtAU9MSnDThizm0j0Y+XQtig5
AjlrjSfJV/ZdhdPL9yUgXEp/aH0Z72eE
VbeUHqqkO2pxd+VZ0WfTrSchwQpwocgm
CDTJBGJyjiW5GoIDYeUGnnzWNlWp3odI
IQrgeVx4frBos/zr16jH/Uj62xbR7YSR
VZtizeJR7gGS1lBV4CXMcgJtcWi2uIfM
e1R9zpcyDdlM2gzaHLdmyJfiTu4DjjUV
oNrpew6z1kKlrk52cJji/+ZhyiMDzFVD
q++uvGIKrUn4mqJXNjXTMw0bOT5n1Jpe
dSISZ5qFkX4aj/Bn80vaRwOoJAr3p8xn
9kgb2X5ORD/uYhQukDlyqOeTSP6rDRWR
nnDvpJdy5Gw/dMe8M0lR4sUCoQmJaTJq
TQlZTqPdCVkfIZO63hMyI5zdPceUkpcv
WsSCzVYNJ5F5lyMARHg7PRIKqZSkJaF7
VvG6ajkIKRIOoPMQtyYujjOlcV+fptIW
THyPXVNsU1oFFQLvTKrS1F4YvmVWstuQ
MsM0SHseeumU1/cgFe+TEgNgVs0jt/T/
/oxKgZey+5iGDaebBLWeB6HVicxgZyDN
kcdGchHDEP+HBAUO/3PapJSN6vI8p/GG
QXOBjxvubtNOTuonbaH3UTnE2mNzliEk
sGs63gS78QRrKD1rcdgVJvCWFmPavpjj
zhRhXrYj5Ax7G3kSetPP8Tv3qQE/kbV2
R3ZbBvnn5yIL9ulWhhIdxJIndJyOQjfh
NSJu7CLU82AXLbcnKd/oTnWIZN4rE0ml
AW/uEo+4DS1I1dDB5p+yhXSCHW/iH6/E
ps3SPKBI99My5N72IGfb8HR69eOsaKYi
/f3p15GHKUCDHrfdGHLm94lNd7k9+2CG
H+WX2sN2s2Thcpfy6Gs0bCNAJL5kS77X
YYKZnGJY54647SJl1SlrCbvd93clIcDd
gyDrNuRByoIbSwTK88dMxg0N9m6igsxY
BjO5yhfASm+vI7mKyFNItXgo4nJRX+r8
h2U3f3Sc/3dELFuPdAc7MjToPTDy5cWK
+pqVE7E2Oe0Zrf3fYuFS0vSEP4wvWcdB
GPgVqd/kH9xJTiIxQn0ZLpv0WBNBfrU3
ADxh3yPKBhFqvAXAORTmUUJOY1gVM6aS
i48iH0qN6nA4PH7vQ4apukd7rKF9H+Mp
SzIepsxTkEyAjUOcFsEeomHstprKw3PU
JRV8Hx4fdrJo8VU/5fg2V57EmZndmfie
LSgJGnb2uG5LXME+deBZWoqWIHjpke9g
KNs1WTg1hlw9hLhh+whT1zVASb22xWwb
rU2LMz4NH9bmEe2D9v0TV52/jQhyqSZU
RA5TkUsRDIF2zfPCTs4nIFOI2DTx+ass
T8FAMpPd/freoDGm2TPXH4O16W1mmA78
KGqvVNz8+0wWHrZj+bZhYhwInL+frTp2
1yNqdF6NP73BuBSNlpiKNKxmd5BLLT3z
uRoXZn7oljHKlrNnUKicKF7vu5VllQxp
0unOCCBfObQyZkoLE326teVr0h5zvu25
PYw24WOR9ofVt9ydPBUbNAgrL+PK+1Dw
MJiS80s/n5A9zkm8MznVn8xoXJlcbyJo
r3scZ9zmbGByHs86X0VKz1jkWN6HmduB
/DlJnaqOe9qaW43CSSRDIh3CBjD+0Pmt
UZG9lxKTO3HhxY5V3ATtjGCHGGTucrol
5vDx8Y4ZnnkKLW6mrKMNv9wgMjFBFzGu
sNDNGiGMCy0+MWSD71uBililHO1mJ6AP
ihjF7fNrB5KGDd60gto51fmBbpO+qX/w
JHgQUcVidtKQf6ueMybw5NdAKOYzcQPl
HScwZjaIkseQytNB3mf+XI6Qffc/qlgK
DXPPI7yGiLezvUlDO9ZWV2TZXXaZaVsm
MMsQ78zAUP+WnMSOYE67J/VKCv1bgupv
VuoR7Zwh0ayZC15BRMyDalTpehUWwiM4
ouwBMcjcd5vHJr/vppQax+sWMr++3hF2
Q1/sX55xcm1bkgFFQNcljvI72YbzKCbQ
3mRTikf+Ihw0RUnE9ft35yol2wEJBbHq
nnHY3zNI3t+JOdmn7TF3mmY34vwePvP8
RsB0j6C1xR/CAVeZpukxyA59rwSbZ4dF
J+Vyv/kc7uTGCeKZJzw2cRJdvQiCJRrM
2Jh29/W8VpET2BKbUKggIZA5b/H1nIvD
8Bp0qhjLPbT3/3CjSiLIW/0hseet2X8I
iTNhi/RA7/K1JQCBbOCmmh7cOOTwzc6U
RJoJ2cgoYGfCFtGVTy2B72ZUCW77XtqH
jA2MYsAuStTbB0d2Zr3vlWCCHDqVIqkM
QkSL29ew4X6h9cbkBnSXWJfn9xr++pHk
ipp7ntPqrnOQPCYRGQNqQEDFj5ckof7P
nl4GZAPJwJEY/mieJ7hKOdfi0C15znY6
1SOdT3TOiUoK5epOJtF95/6go+M74sgu
TKcDmFcXeDMPHwTA5IsL/DwIVd9LaZgJ
ClT5R7aPAbfnOiE6avOu385wqqmEx6S0
3xKNsAc4MOIkf5deW1QS9HxPNZCyjwqz
10jvjv5dlcP4sGDfBluBM06n9JgV+Pzn
F3gW53zdrPoUksXwdp6jgKu0gq1Vz36b
mymaYFCg+4JoMCPvEWC0H3YqvLhqCeji
8Sb4gSSfZrISOR3jGSpJIPUiqhnfS8Pn
L7HBs16Ox81oiQHtKT/a8lc5xgGUSlpo
o9pFglCm9Z9gbblDV5S5qDz27/OTu6Fd
xCKaBXF0ianEEpukWlymwk4wMidsV0Mk
foYCEDHSqvhDCMHrTCBazIEUD1RKsAGU
Ve9SSpQINMHHccYM+XwztQ9c90JsIu17
Fqg91tmnros779kjEfOYEuZs3QOFjShd
aeB11RwZTycO4/iuQMlgqVHGXoIJ+5JR
v4/ERxzK58b5mc/rL29S7fzUkUv0n3F1
rEwZ8+UcxQpDC2bvmbM9WQDsSTmO3onM
8w5vJDPemiWvr4lbK3dqmafCGp5bypEW
+1FfbviR9Bj0VAh5nlvWxegEGI0XEbjz
kdTLeEF7PK8tziw5uvv9uTm7wzoeAAKO
49r6EFIy8iKDjAlCGVjPKApYW4nJ/i41
aCILN3I697E85hP3fPX7XNGuHpiibRwe
O+8re0FCu85YCN5CjzcDyf7KxKYP1I3y
+SLxva93fS6p/vgAcUeOQUFQFD+/qknw
sza8byEtsjayUkUJNJok+ADupOjCz6u1
J3RZSCNrQzhTc1JzXLlDOnN6lL6Vk5Of
IKsSMv5duQUDJfQqccmeCRlLPdizvgx/
mKgYRc7EoF5HoqIcMDQX234vCSD2tyMp
AQj9d2oU5DZNqLqYP0xK8pDc4XElwVon
HVp95w2aD6SCvSZlN2ryOJJL/G3q8/qT
0897k1/PX5xsS4EOyxMLOE3++LwnV/s1
M1SFjF+2qvdKWfEoRIFhjmhG3EIvIn3L
lDDtf0ztJuFwKNnyntxhLt6lLQMvIZVb
SdDnneZfp9euCcnTDHk/lEiqx4H8Gw1f
4XxoURrgLTJn7MTDEPJKqsj5v32QqkzA
9+ufejk2SzvIsCxvdY5+rGC6MMYHgXv9
fAFc6YO2t7ShgxKp9I6lR3FxRVQtrgyj
YJFWI9DcKTMJLMNJ4FYkL+f3nQNfv7Oq
i1RTHaybj6gfFJP54HJhAs1Ne8qQmLRB
XudiWIkFz9MMsK+FtkedH2AQaYyoge2f
WxNOSCi7VDmRhs31a8CjkHJJdi/yOOX9
lb9r2a1IuIgYe8JDg5iOPZ6yglZofqQG
J4pXRoTnDw7LUJ+hwjKT2INt5GWm2cpn
qkzif8nxf/lMVRRJoE6qXzMKoTNhTseC
fSSuw9QgjcKFUIDD6veEsLW9fO9MD5n/
eTGHtXBbIjHozFSbYdWOA1sNpB4ra6vL
wXIr8QTGqEUK0v/HbQsogaa6eIi3pMKK
VrEIZQNKKKwVzCxj7mpbgq/oKulRrkTl
HHuUCzHOLVH02nKkT96rOpxucqGQCMX/
LnYptuCNaBT8YqhINVy5rkw0y6qVrDqn
z122uj7binDA1TxPaHwlQYgkJ1PX9MiB
W8WNzivvdA1UOBrPAXbmPgt81zldHCdX
4hEMVRVPmUqHvuT7NC1cjr81dmUjgTNw
Tcpc1m6b3bEcQRG3DqBED2rYyIF63sN8
tYQKnhlXCqVzDNqQnvXVDFoj+u7fezNg
ikwzvNBHXyGzIfdB/xwqtpSdXrxeg7nc
xljz125jnqKuQlRODQ8yYd20sbyf+Gwl
XGbn6DOlPD1o/VMZS3qYZLuc70Eebl3x
sTE/Z4znslPPfRK6VaIowBEznP3X8Sfl
SKz4vgQF2ecGvtDjnLZXapjUiqzJ10vp
Gncnnj5BW0Z9mqRL4ueGYKOj6Pzk+yDy
Nenu4aTNOtYh8cqkz0k7zkNbRMDrf3D8
/HaX04BNm4AneVrGMS0uAbzz2bqAAMX1
7xH25j3lSjK3MF5eMocArlEuDIz7gIDu
zm80w7UvVxKs4KIXJ7I056tYDLFb5U+5
NW/MwAfanBjNdF8n/z2PnnvcUnGl/Gp7
Ytt4EfnK3ksTuqQXpks86OqNQCYOegk6
81OpStK/YKatx+jOi4U1dgLlnZ2TSggD
jxqIUA5Bu8fVy8/G8X4oooZJA3k9Et9v
RhboLS/UNg9hNhIasdh26z1akjlWySar
5z9PTktyATmgA6uNZWHROn8P89SZXeVK
c8kOZrg3N6ojQGeG1GEvzYTvDEHtXmSx
ezCQCbv2B/a4YC2lDg0qiPmjTUGryjbx
zKFu2WnbHd3o3DP2oKGlf3KmVu1N7MJD
GpUjx4RQPTWHc1v2etJcG2/qFk93LaV9
HxxZVW4sINEeLNTfqg5prbnfjmTYX7H9
NuzWeFDqdPjLsTUZABHfFJ2ar5ploqUC
W/l7+0ByOpH7qX5+CgTfyhZESHiM56tu
5MWiEkKzr8Fi3E4CRKBpY71L8WH+6eBF
aKSL6g9C5BExC/H5dhg5IES3CMFg4h6z
QMFGVZVwn1v9PDOhRdt7pfvExPMWfnuk
+kGtI9Cmi3yZjTeVx5QLIw2z+KZ20PqK
J9lf0O9MkxfCmZbRCH2ucd708BTwlIy0
2uJXu3IPe+ZYz66Wx6vEHV2JTGY7dyyB
8VcajXkWoPo6B6yRzBRAs2TItCLlyLyb
zqpFmtjz877bVGqRmmA7JhNhsTMXGM/I
u66FhdgM7H5Ym9zI+1XUOCoKY1LjKDJ0
cZP3RXAVf9ZyoRYH1x2n50AXMyVv+zZS
mf+u3BOJRUSp6bPdQdqnNaD/60z+XmlZ
3Sm+S0BRyUgB0EOgNT15h7nC6KwNWk6e
58TIVMwW8KHrB7+ftyaL+rN8HbSygrwW
6XrySp+C5w56Oq73PBs6zQreReSerf2U
wakiyX5Ga8x5cJOJzisvkCZiRz9Y64xi
Gj98qriF/VDGxKnVfxGXaUgnMWhp11et
1JI19Dyyk9nV1DnY6H3RQzEOo0KU6JcT
+kAA1MKC/AvHm7dlbMknpSglh/nIPBsp
HMGFe/JRrjB3NaaTPLOwB23lgulsHlfe
piOu6KRhI+i/V+r5B5LP1/+X5YDV/Gz7
hoGzknNeNWbcghEZFectPG94QBeTgH9S
f6bQBF73hOAabMqyX05wLawuTVytvlzp
+6RAcP7/54kZU/8sMR6P+jqvLfHBFBlr
c/A8h98DEXAnw2rT8sUkQWJz5nsqIyFX
SS+05RDB93ehR/gONUojkDqgCvPp4Wcw
o/wutN10LTFxJ1xLzWz5jkPTqP69LT2r
oMwgCTppIQrKUVwUHfa3tk9LUQitOXDJ
0REUEY8thp/yz13pA1Jzg4ac+MMzelx7
PP1jM6NQpEJ+4K0FiZorRwx8puGqDxEO
SfASepEq6ZrkJ3s13t4WH8YQ/dvAIBo0
DKZ4eqcXlEFRB6MhWLhTnsDJR7tklDtU
OTSbGp3pSDhLeL1IThd6kGZEHMAbFFhc
JRl8qSDJNO5/bk1EXXeyUhXDbW6wJREy
hAKbwX4SBRNyunmzDILfC0vcH1YK+t1Y
d3Ynk6RRtEXsJvbLDzPDGxnhJjuyvy7H
vnvpg5KO9hj3nj8+8E3SAQEfPW2cKbw8
APhqyCclHgtdYo4GhUs4JLN3Ef8arIIu
a9T3Aa0OPuLHclZHl5i8/xgCtemN5sDI
hIr4tx6DliFbqfIs0cG9gpfnT2OgUOgf
OZe0/5GPYaoz3Szx7fmffQwZWzqMSrkk
vMnxebcbi2fjhNY9HtHBZt95YsIWngHl
Vviw8p3QR5OvJIg7rbsZVBjm1wWO8RzX
sBEtM8QnEaIpqL53Zt90BUrCwvbktszZ
qxVeUro+D2PI+EMyE4CnvsqqxtPtBcTR
F8yvF/ZrwOC9LdF+8ReAG7WYKG3bhxyr
jcJZwt975Z19JMghXbMAf0pmkRVOcPHY
0e/vOQfXkUZss9kTimY7no5G278tXzcN
x639T25iPBzm9Nf1PUDlxAgKIPY5htx5
PtixTRB2VJ1Xdk/wE9EAgaNMZebu2fDs
22+vqNX9/9nDI1lZrrQPifV0zMqRMYjY
OHdxbufBNkAzIgsS5DbEVxweiR2+5zPG
f854S0avdZaaQal5yuxTmC/EDEdnqnc+
zjyMsnnglfcQkr/35kg1qJ8XmSlR098n
6Tc6wj+/vB3dXeyJsxbnoMpZHGd7ZpjS
M/Z98KxowZQp/PTzI48WBBGEWR3C+8Rr
xA+unTy9SFpc6LjP31KwJdQ/+h0NLarV
NZs9o0KUcIyK7MgmUSXHiwlaFKKRyBHH
b7MrN/tM5reT+/VtZO1nNjtfl7w1NoNZ
3T6/VxUz5tU/fxutDEMWI1X5uzQAzQAZ
ipI7aWY9qslvGmjXuU2Y+ji8bjRTMQdc
SeMyiFQ6/F0nDrlFWggtk35cDq+XyCkq
hncr24OrkQdGddAiFqw/BJTO8OsS2cNi
M9OSAptKBNkQ0WYHMWn/rDVpau5J/3ie
aEaIRb98j4bTmb861LWeTKQSDtVn5og8
ANYSr3gOTzg6Ip/TPHovfP5KlkiTifSc
UgmLeT/YrMq0SDYeWj0HTt9kJSuuSKAU
2wvmC3lXM0fqypb4ZccKDK0zK+D7gSFa
brLXImNP67UKyhyJHOWfl6kPC4tANqqJ
a/HoQpIZbCZqcE+bNG695NK47fOHkle2
wfO1dCeg2SaBL8oFPhUmRq6B5powjnR7
qmLDT2LGeVY0EonfGM5/lKxReD7P4qxI
n59weLTBFw5ZVLkvGMyJrtMMmsfULI++
LfO7kRZLUqvnbebxvTE1sqTnGbfdaLMc
bTmVyNRlbKVfa0lJ81sg8V2O7/PZ0ii/
vNXOfMnHPYOZESMIyz3Lmjs5FtjHvY3k
HkoBYzemgHuesPDkn53EyZgUtI/zXSrK
y2FuoTg7fEF2ytU6w817HgkNOLvEso/x
pZKXp91835E6yY0UyWGdur+3JjHhW+xJ
xvOtve2eKmyEbMFM83nNIkpK44msq04g
aNXwLBm7juQdoiQhIVanzuJ2ziuf+70H
ZR2mnM7L88+kzxlZ7W+PBt0EwpOx+Re+
7G8Q3Qs2U95Wqq4sZM6dDOCt50LdaJ7a
SwT7e6GbomSuyflIW9Oo0CHFDODua1/v
+C/65sGTyujfwrrMYKFtyPpuboszK2sP
Wa9tOTDOCQaHzqkSjmoo0/ZL2UNQdvzG
zr9Ry5ksR6l28rkcjBoZnWNMrNdzkz9z
3gvh93nMogqwKHr9kYSWQCrpBx5lqoNU
h6RmwuY3Pdk5LCaJ6fINsSNaGYb/8GvS
Ybo+LJNjjFosinwxRsxt6sQpXWODa4me
zyTPn3ZExTWfMI86CTM7m0rl+p+OhOlr
MLnzCXN+lB/C+1bHykpkABj8l+Ixni+K
tSRzJmNI4+ASf3aoaJ6fPB/ZHiS19VCP
ORfWMxpx8t/1OcwkQsdTxFw63Fl7tRrR
Lf55aLJ93Sa1jJB/dPUUH80BkCxKRElM
Xr4uVI9+lSX9m/te+pDfF+Kwxw92ZHOz
HB7LlepqJAVpVsf5Q6L7LpJNci6OHL75
qKkhnOroCzzvPU/9HX7XrFMKMtQVhMoV
1VDCHE/nFUa7pUiSKKLwUfQGbRfaNO1k
/DTfB6bnJdF2obu+NAD+7rK9Rv4ujOuW
NljyikSZHn90xXwb5UoZK41HgRo6uIEb
SaXd/D0+eQ1RuElxn6szsj+i4xYNXSdi
WsOZnpYd5NB50Uy3DCYn8NXX1ppsYu0R
GMEs/yXPNF92+EXvSrNH5yWKmGEy7xyN
WHRAEX398yZdY4ZsamjYRCvx1lsb/oVv
1+RDzTjixOmL2gwOiMw3AdmQdXX09IiP
lDd1rR2fctNNdsitOWSpkDRjTB21m94+
NH4GWX6P17qMNuGzAO/xwpZ5rHwOST29
EMfPSJQTbK74DKRslqNGmyfUSkpqkcvP
wqTRcDvDtu9duXMEOyLjFgA8b/Dzr8Ro
GMOH3pnvwvS8qQzKMTkG2CqE6G0QZPy1
7JqmvoaV17F8ZL+x3hjkmFIcFEviJmIo
WXJkeXl6Nrz4G70gd9w3fIAmC0sf04Ja
Yg96yr90jJNeEEyuQLd5JTcKX5xJXRnB
ePeWMj+Olu+9Cf4aPKQp770Bi5IBrJQf
w/nt+Q4Dlr7SLzp95bPl6ZioKb5lGwxF
4YwkTF2rO7FwlbREHA0dcH/o8jvgfAfT
e0G/YVSoG3a5eIkWEVHKunXlTXt/eADb
OX2ePj5KmRLVd0eJ/yTiokVJrNWCD0g/
1ihHETr879s0MB1sgSiI3KILCvQ6RpjB
GbdAIjucnCil2bnWVoShGgjSpXum14X7
CtKfbMDJUHD4y9lco2IgtqpY8fMM5eJe
xPScD6HkOKf6LZP+drYkWM4YCdua2WVC
fCN+lR9zqPqeE0PKgffHM4BCS0Dk9LEE
40nzmQiH+d6ZwcRvBhhBbZQVhVQHn+UI
vOwcvTALCMHRvUZcokh5nNjQjsHIOSIJ
gvZe5PmJuN0iJQ56NY1FhBpdfWnBC8zV
iDzPQkLP/Jrhx/GtaFfN31Lvy8T54p4Z
kH8gHOmDR3Sg8+9JehX+FvRXwF3PL8Le
z/77GTod/8XlTsj/rJ6MmGWxK1ubElfo
dT73K7pKJExENevgtKgmKX4npb5z2Hag
5k2xP25z+n+HkHIlCUCilp/NEmD2K8Jw
Zh7jHxxGms/y6ll0ZWSDLcHoc/m68bxq
IBLM5nacO7pPn/acexbTvGOgMsoJ+mjj
hEw07++p7QMvOf5LGWhIdWc+Dfj/Tq+8
DPvoaNwDwXshDsQD4xg7oTypXpS0z3+D
tnw+8tnVzfw3PbEZYkJZmFkKUsnw2Wqv
7An2uq1h74dGpmt3U6s+tziXqi8364dC
ZHYiNmnoGH/PljiqKXuis9tWj1kiXWF2
UKPL8Qzln2iVxjcTlvq9Nb63vApibQ7L
1hIVxXVkV5Bvf+yj6Snpmf9xe0/bHhlz
AYESTtglJ1mWwJKkp3MxSofmBbiAmNXG
dOXUjruG9H8OhpGENM2C1T7TSQqOmy2a
THj5YopwjT2hKXeKwtYHO8AY43ni5wwZ
HL4PBy0kdZJjIdaSfN7L/S32Ai9xI4kp
vG9TTCnXRSVrrXh+gXip4XopscJtmCLE
UIi2WF01AgIsYrviAq8ek0VmS0TeBG6J
UguT53kfgIjP4IjnShduROoYmZZX8iHu
cCV3bLQFJ1jSKu9gPMcfPEh6bU2DGiR6
HseMhmoCOn/xV2ychn6u/3cljmr2jsHP
QDWT1/ejYPLiQtQ5T6qNJDOfwT8wX3mD
Onmp25B1pvaXD4xTCbV8LbFSTpMnLEai
A6ObZ3fZlYFQkOvB8vkddfnYzI4xc0Mv
LPtObrnN6eY+/G+eoMo1kVwKMwiTq1SO
S0taUqgG8J6Uzryrzx2jcjTV2L6Lzh5E
QfJP49t/dtIFnvis6Hvi7aAQjwTbeF0M
uvj6FkC/vrCwYUr3mmSKPXE4Mmjazxf6
O/pKe7sSnNjGVgWtc8eQa4K4r3fySA6V
TMveQ45MVdGS8b4trX1l0B2N69EHDhLX
EqBxP1cgfB0uaJl5zzKPCgTRwsOJh3Ns
361q5/kyYL/UbGQ37xKRdqSQZjXnERnp
UxqyVd0jb+E9BNASdpLswyyPrEU5gNgn
w+Otn/do8FT35kBDl4qfqIHA7jM70i3H
Q3oZM+gImPTFYgpwDJ2faI5g1rfxsV+R
CDWNKf4xR8f3wpL+wZmjxq1ThNXvVBFT
e/9nvVHaY2T07DEcON+mML5Kpsw5A5Bh
pAECTDNbveR44rq8zPvvtHBEghiDwqKO
1nUkLfHgpwNvQiSjyHTynk+Beo3yhRHl
+Q3SBA9SJm2rBQZpjAUna5rjlfaRBEkM
E+KMyirNppCPMABtJIt20HgkIVbS753J
dptEGoPt6m2da81umdnAANgTfoNpoG5T
ipV3iIpJWETMWVIViPz2++Jirtibbewd
6Uw+20pIP0AEwf4a886fjvSfyNXnIbzv
SBBQUTKcbEvdTk4gVgc/lpgySv27hnaS
HMGpmk+sxh2Ih18tFLfuGCcfrf4av/Pm
pKcujqHh5ybicq5a0kJOLzvkbWYiV08a
lS7kcqh2oC2OXVwq2hABI1w5IXKUGmW8
l9JlZxqtmxl3hIAneWUtAcoL1i/5YiqX
e4SW6dl5B1D/FzuPr9QmiUxfSsa68SCZ
+POOLvp/lq4tqZi4OjXKC1YrjktlePne
nTujcymz6hEd+H3dcdNk0nNXKSTir2bC
cH0yh8zSkMYRNHj5Uj87vG8J9l4C/kj/
SGaq9PhtuLv0D0vGptdS7ho1O0CaajEu
2HvAGBJP5jA3H/GYjghcmY9zGyV6bYmr
v+xGc7totkgqZgnpHjK5yAdqDpPx/T1n
hmHC8KZwz2l38HZ/emo+gi15YVzdg4mg
KfJ8V+G/TdIBePTh5afb3+Na5c3CfRIG
c+zz0i0SjqTgYSu79FDvXPFTWj3+79VU
R8TOKKZr93clnwS71aKp5hnZorZluBs/
v2YaEDj/SnrIigVzaluwXBStBS0iUJWE
Nf/fbIaeOuiBZWAk0GLse1nsi4eYy5uf
Rxmmw0+myTCL1bJMbSPTsO/foSzKM4r0
GfPHIjW/wT2dQBuqXujzgWL7SqQ6i577
xwtSz3JjnhlCRCOjuXofk/Zl4FsSyFqN
W0uuBLHQhT5nnCMURGCiecb2fCQXwNmS
tb3Xvh4dziE2Zxhpmay1iVihAixBLtPK
tIyob2N7z4TW3ly82f+iVKFiuMZM3jLQ
o426J6+Sm5Ejis7QnkK2OzAToRvOjbKb
gFJk8f72qHLTfUlWhfH1goMMA0w5FPB+
knNqNFJ0oucCjuTVvp2pktcYHbIQstQ7
znnte2fOrF4XmOidYNs1JK4ALDxPGyng
8L2xoyob5PIsf0gNnmDLJOSMxH1P7rXV
gWZtIhs0DPuZTmmLuRPSrIsN6MeSN5Mm
2eEPo4TIyomZesTveesOvFeq5tx92sru
jIj68XxFaE8QDrPrsWcehj9Vs9R6R7bI
eerWPk3i07/D2SK3gq7n2QTn9hO2jtFx
OiOZmzLi0fru6Wy/F9qRe/LptBfH5FmF
Sq7a7yXAFRj8DIkmB+BkPVmObG9CKGdb
68JGIIWWtFC3Ie/tEcWKqZqDZbSJjANv
h/5ryDQqPHh6A9dUpF9iBbb0t5OlbC6U
NUuU4P2ZYJ7/ZeQXRSlIuydhDr1rmEMp
t1nSrRyGyw7ne1nsGJyUpriaxE/BZn5Z
AqkDPTuWPXmoYkK82RMW/fzo2Bi7eS+F
23slomog3QryUWI7R6aL2a8lPo89QMsI
Ke9Mcd8Jb2FT9MOmvCD47cuqaUfswxXB
sgu84xlcXannf9lAwyVMe+HtB479SRNR
H/4QIJzJjA3AzgvTeC8SPaBBi7WGxtGH
jfuOAfRIDNUs58IWDpNUilMmjRiomLNp
jc/KJoaImBZ/b2fNmBNqe58zHC5VOhVH
MLq6eGdNcOUShg85L9TSOxOAgXGUXX57
biQW4dm2f16m9FnA8o4EEG7aie9tjizi
V/WXawTQRot/OKotFkn4qVjnHRuvRKud
SRXQBxBxsARbqWhup/PMEPsw0keFl9CB
96l5ThPSifmPmDb6/2Soadgpwo65+Dfs
xRYqPEFINDHP82EsSjq9aEgsSMHWJrA7
K/XB0r4LIHve3et7Z/a8bDxVmfK0BT65
jUQWD0yJdJHtxQni5qzrS0giDjqrnbpf
5muurAFHnL7R6VfMyc/HciXn3Eg/2NIR
NqZcbIjIoXd4HBTLw+dnKIwTc5d9+eHl
6MNHD3xiCGxSBH5EWH8vVx4kT5qBJfbj
iw3++VdNXWSdnd9bEwVa0XmK56guoByW
gqxPCZH/u5LwqI4tZjZNDK+SE2UQFWch
MK9G0HEsVGFQjedbCk5bqZJAOWdhKrMj
SKr5MhOm5GebpUTCYvbUIhWqK/+bZvyO
BZF/zq/ZRqxel7C0KFgMSMrAOEEDjJle
e76aEqrT/q1nUtQ78ulMS+HapqR5k66S
Vq0TWZ5UP0/FoghZ0Hb2NWmjIWNl/Ccm
IP2bZEcvDWUqU2AavidAD3K1PXMULoBr
1jQA1cS8nlt13wiRhF8Tx/MsPvPCpNMk
aWRr6YuXoKbYMI59GmK1bINfRv/UHvba
Z00aBLmzfSuacmbvLEDmsV0szCZAElo4
JXzGtpfureNqtXQt9UeIvolBtiv0Osw+
uuSGEtpM70OtX9D0u0ycYwHWENLLDFtk
qmhH5IzwGmSXHnfCFtWvgqTNY2MBKREI
qBw+h2zyKcHh7+hnlqOoRhvh3xHYd0SO
drV4exmF/n2lBgoF6kzwQWPm+zuY1CR7
CYsSivAjVQbV2lIw1P2lQPDpqt/kIsa+
MzjJFEYlW8/g1edS7t07PRbH8nsYBnFE
082gUPm7MuBpHV2msIEbew5qpjTRGl91
fuYR8xDNFhH2MH+dQ7Jhz3/xchoQXklZ
AVibofrc6Gkqm60Mw8/cwsudAuN5BPcB
X5tciS2MgmaS7Mz0k3FBOKIKPufP5RAj
JiFya26BXGkgJwVMyXMtp5hbskFeJS54
tTqiH10tdOy2LvBXsoqTBB5fTaNE4sEX
0DTPZMRQxv2kZSZIWY1zfuUYhpOan/ms
ExqJdVgl4pAGlrKpeSu+a86+Db0ho0Q4
WQsa984E5/nqifh7G4b5Z81Qpj43bGoX
Sri8MgekJdZk4XpuMwxWC77XtehY9cWd
eDPLZ4fkfVCs9jmyCwUx4ILrHL0gHtfL
4VCP5l6MfnvS0Oj/qGKsTXHUMAeQo8+B
YYtNRWv18ptExe4waD+JmuB7a3zHpLft
DndQnsffD01sCQG8xefnp8xB0lHyanMH
AqvAjfPLCyHNMTl8GQ0MANB3NtT5aUkY
L3Ynuz1hG0HPyYT6fuKdBpccT8rmABHI
UAyqDJqnzy9pUpQq+FHxmR4Bh5zJo776
/GvOw7HHBEFmtE8MwqgJaXwKtn8emfhk
mIeOHkroGmoeCloiAOJf0zjMRHkfiOB5
YoQebFHfyig9ho6Li3mPR2GJihbP7Pc+
cbQyoCEQzpmi8KPOgwpEvz7Noe0atZKt
kqGbtuCeXs7CMR2Ltyqz+0zggUgGrFJ9
uZBD06rruJGUisTqavDbOe/9uxbvmfR2
5r68/XXdfTLCIvQJwSseMJUYq8aWocHc
KmrOfdLEy5gJH87+W3QS6xTONuYITEW+
necPR4+W7GvuS3xypNp0/znQ5dfMec34
nUt0hh3siSi9t8FHzSFdpyBoArSlOfBB
Nj1M70HMcxvpPOJIPT95vuPe9J8BJpw8
RNy/n6lDqAHBh8tOl3l6t9rWMUz8+zPE
rybwiocgSWLO3uxuYP5tovL52KFLIy4l
gs0UIjtLzsPtVQ/TuxXnHVOGa5wBQvGL
hn0h5duC8WdLKIFDcR7Hsg7Lsxlt76QQ
kCnA+uQaxEh38ZfXoZmp9/ccldXtCDtW
6+J5ThaQIv3Hs/Oq6uox7AwGW3zmJupz
RHIOXgW9Whu2n8qqn4N6mYaoSKyce9Rt
V854ceDooogNmkM6L6MWAzmFPmEkjEmX
ubQ6F7+zPduQz+k9MBdXhpxjJSt10XJZ
BcfKERdfnq3LerRHdnb+swSHKOYIohNv
oV4mpzpqhBNoM+q6HDFND0Lb7MfqpgFD
ibMLfXJcmSblUPzNC1nQrOBUSzmK0m/V
mLekRs5+HKKqbiIqdIYadcuMql9Rvi1w
kTh7L6IjFpx8LVwdgX1vsxd4xtiyOwsT
Y0UV4Rm+Yp11alvuDPy3Kte1N+lZlJPv
sQMPLVmmd0howwHgqCB7bVuyiGl+tKvF
zQH7aSZZI840OvtykHk2BDyKQiTXUrry
g5lmn0vShka7rSrm9fgArhSmpi0bqcis
fWpgXHfUx2PUWRKPt6c/tcRdItiVRLcY
1zlCCfMVVUD0/RnTtRENCTAqmvXSkp9C
qjOU04RiC1nJDbZZZXD0Ayz+WkQccDk4
aH9kiJgpGb1DXXVjQD+k4oSCQy+aQSNO
8PMobPOE3lpehdgwttjiimeCK0mReUxR
ZiIN9xKKmcc58k0d0m0gKOaXoqZM1rZi
LKrWVNQk3VecFOf31tjX7bdxKqPbXcvs
C7NWo0ALYgTNg/c+T9Gn3xU5GfGKu1ZG
PyIorWiyF2DHEVrqVZOamYrCLNjSnqyR
e2l3OarlTZDSEorucwzR3WQ+WMzdRlBX
9FoBCjhMH0O2ritxLwDh/SBwM+Wmkw7Y
SrV0R+vVv8ZCQyjr+JUYa1u/k+r8fs9Y
8s2vgd+UtaGniE0rSzJ0G2+CgkIzP+R5
Xm86sSMNtKWcOeN36DmruTLpNdYrwvRj
/eFHjc5VtzkORBPElk4IHOR0YWkVZONN
8G2P/Lr3JKncxuDrSqhHeEfN++uMQSAn
Y/7+zFfaYNYE/Toi1q/6wl3IOM/BDsnQ
bOBK7bfH6KGcf0M8lFT5IbcpXsDUGgRq
+h7zXP+78PkYYyeNNgDWkGUsRjlJl5xL
/z4y9jBzNvSdfOZmHAd/fSrL2/vDmze5
JpK9KCucaGuUzKIOfjqcXPm8guh1zTx9
H/R3IRJVMKoCfD1Stv8yPwsN/yBqE5fy
t/dLhRtR2j3jvNh42K6P6M3P9+lCuuFh
IN6xmDmkAi3wUnhSXxWmDKJTbLC5g8T6
5wiB80oJG7j31JATA2ki4rjfdwZtoDtb
cl/69Bw4iRxRdNK4BDGzx/psy1DWvhda
RKkn4ybPnQ6PVMTOZWrzvS0/JrukMqDy
Z2+YhHIJ03kGzN3DmCRAES9Db39NsG8i
FyytqnTyuLDouYJMq8/ghP4uJeun5YMU
3MbztRNzZ1cSGfDXNNizaOAuGtTvY+pL
Dn3Gb3rXP3htIEVErjBUzy320yWr+HwP
c3/Hw3gFe072CXZC3NKFimSUd6F+cqUa
vXlJM67GKuv0PwMKzmCLBatpMCT2IPl9
2iZlYOjeSxk9HfcYcgI1TuCbjInTz7on
lf1A6opXug2osLUL59Mofxmfxx7L4kN2
sv3iQiDEacPT1HuvpL9sCX1UlF6Dacx6
zBz1fHXzawRcHKOR53cb2QNa/PxsDluD
aDzvjjG3AN/EQIsOeFGmQz9OXmFAc+Te
cPVYCkm7j/L3hyjjeUPkOD1/qr/jDNU5
wYUyLN8Lo/QYB9MBE649QgGIaRzc94cL
xaZVMTq6cxMzojXRCeBy/nB3xdCzDTSv
X7MisvYtFubhOHLl8+Cfd4roPbhr+Qg7
Xf1zLKoj3Pt7b/aETKA6+A9/XF++EUJh
X6egijt3B0gx25b++jGB/nxY0S7Q8ZTk
CUihwNDQW1rQ/0SzYalEaZ6v+Qp+xXRP
afS+gpa1PV1u6rhzvIHD7Jem2/xFyzZc
TfSqW15V5/hM4HlmJo5WQ1jiNR1cC8rO
pESY0zCT1n9uTxz/pKthv36A+f6qmwYA
VKmPHvcJ21uSV7gWN06m0OOpTBXgib4a
TXL/+rwybu5ducWj5zOfJzOsL02XpW52
VFVWkc492xCZVFJZEzK65Dsrx+5jlFH6
oNFTSY+kpix/rJ5RMF10lyf1cr9HO+Si
YU+75tiP7zlhmGC8O7S3wWNcSzWgG50N
LPiPFELis2l4sBim15pH6DKrua5R6AZj
LDANR2Z2AsU4BstHB/QsZsobfkotAQqG
ebwMhitFk8o6jfMyMkYVYe1zhtry7NEe
Ja+tI+lp4T1/l69iToH9pXxp6CGRcJ1R
qxleK4fqt/QrfUy19uAXDjbb3yedDvS3
1eU2uw/qy1bD0qZJ9mYQJvVJ9SSoJZ+n
40BuBLl0v5UXYshRMl0hVC9DBPqs66yi
GyLW2+GwdSB73GgXpcVjXgW3uznbnAnJ
B0wGvb49b8alhwhawaN0vJ+o26+03JKJ
PHKT8jebXBAq7v+8TVGXmHefSTFb9Cqq
kEqFtMdF4yfq1DoxeZb2pXbFvD2H3/HO
sS1OQHbdjdprmcdDs0qbCFV9oPjqmGRs
x2zTpBFJ1r7luJuhyDlG306DZUmGbxH0
Ri3iWwlaypeDn3qZVC/PICYJsBSbRMx0
HgM6QuyPH7v4vTP7nqyGPRmHRskr2yMe
V1aBllJcR7g4DTJxMYLOh9QozOkhWIjk
oOQPpd/kPV6EtJ4YaiQLaoC2QNV61kT/
y7D2ILShsJdNu2U8HaW7SQuVz6QrHEng
8X4RaYRKcKYcuMB9ZmNRASvPSvNM83OQ
fGUZaZ+TsX/vTNzEnq9Db/Y4pxRAGzg+
GwPq55HNfDiPPN7ekm0ZwzmUkmH7MU6C
ECMh2q5MRhQbuqIjFMxgyzQiIw5OQ/+9
ME44l2oU7wOFYB6WseXzu072CdVZH/m9
XHp5j8XayW++FnnjnVQ1gogjNq8hZ73i
J3sK/P2jjG0E1bdsiQs0rCZZ9++T6PFB
U9KOfMr0SFZqRn9Oyfd8DhS51BTmN9pv
xhJ6r+FzmNotn1kyJk4C9IgW4ePxVNAY
HLOxKAshMH0feqSb1II8lNLG4TDbwYar
lEZOG3VAJnmOxLZqyJdF/++MJ3CzJIU4
B2+iY/k9ZUSMLLcmGUFA+Rq94Af9rY/C
rruzIVKQJVmMEk1JnuJiXvisqISzEl7B
GlJzIaQrPC5D1TIv7VLJKXl2HeSoSf0X
MXnESxOyn7NKyhVys2SBVN2z3kLNWmSn
inWaSvKVMiaRAkaiD6P0mMDfGyIOY101
meInvSOCuKz1q+yomSOUSA4J7pxYyhJT
+FxusX0KAeLvxGspv/aWMPp9ZQJ7Qg4W
GJ6SXHjwIzE9CVXu81DBFEWGoqzI7+e4
Zm5sSGxC91ZnJaOXbtIjX25MlIO73oJ2
nwJeImT4zcOh9PkqQqtF8pRfGfDCcgJx
1unpyJ2joxDD/BUZJ4rT97gQOqibQCh5
B1E0qck9g2wz7usXdnNFY7OHun4vCmML
5Jlv0Dwt9yczzkbWYtL4Xhl/u0nBliNN
zj3RUsbXc9b3GFCIiNloYk88x8hbXraU
xi2TwxlUIkCXiYZUJycGjMEiZcn84J4/
Xq/LmWGL/i2buJZu4GQX4M/+uT11S6eU
xl0WwVWWJFM2GQxomefOh1nsKAJSKG1l
aUEFnBa5ioZ5ZLQcj+QN3cBk1n1YRgnv
xgsYVacgkhhXuT9mEcEhm+w8tX9YLXt8
zXQjuEnzQr1Gr0xA11FWOHE1U+BjiQFT
KBNFSph5dg9/jpMD5E9UK8uy04eauiRv
NdMlHdMZbiFy1f/HadeSRkM7R4JIqlv2
trwFXfTXruMGY5Z0jZZwuj2Jeft8CxLW
dcbyrgxJcA7HCxwVdG5d3kKJgQcGCzzX
SMMks9Y1lYs1c0DAc5CvGN/K8YvsUBHF
uVyPmb1IROdNlgZtsDESnczrqTwoMdYC
sP+Xo9mRAMctjaLrXP4Y44lzmGTGZ0VV
ZI5bEmU9rywyAa2PV5x4OTirZiD3geKO
+ZlD72y2eSelqKaHU/FM63rPS2bn6MKM
Gb/l7EokbUlT8b0UzYcSw7QNGrwM3e4f
R3OfbQr1SoAKO9NO7iTlO8rHCfzyaf31
/zKcpB5FY6LgnH1YUPbdTOKkwMko0UTO
pM7pc+HMdkASi10254wxEykJyJeidc4D
4ImuEF74sbR2ZZtostkOt8UoDpADgMCm
5FXlnRnCNDOixZUv8p4akzryeV7zrtJe
6Q0/teZSqxohNQe/0MpquDfs/LS7chuv
/fty9VEge7M0A+q0kpNQH9zemFqgLAOE
dA3DFofeRLzh+zjkaws+JV7S2bItMTTV
mUinOU48cKYwfR5/9Q61imAJb9tr2LYK
OsilhUp1ldYo+oEWO+3WLFefNUbjVmdJ
xeYvoqYTw7vln72fmbiWEHoRFBTyYriA
h6+Yg7735soUxAQoYQzbNX87nl3nlR5L
z5Bqt6QhYDXsE9K3p1fJfUmyrzUaKBDw
LYVEeSX5ZI+HWOeE0h37kM9RK+IdE7W8
H3l5ZMy5KaNrRnEm6aqVQW6cChX4evan
AcqJXEkL/IpcbKH5SYJAzD84Nc+ke3jU
UzB6a2r73promWoio72yovXer+PI+pJz
2hm1rD3dai85tk/bpqO5/etZgxLk9ZSp
RLEqRO3kOpWHfKZVxZUYl5w48cV111Cg
litroNLWDC3JAGauRKtHoTnZe85vlnTL
9zbmXQgid5CG8Wm+V6q4L5OiI27/HDhT
4Khzf9Of98akY4w5q6sCGboy+61kUnyK
PO+RAwDmp6vtl5kcfrzELZUg3t0Ygjiv
O9+qm2Y1nbhV/nrn8SvCfObZms7wvezN
15Bh7/FLPfWAV7Qn+/I5FaWF/V7ptJqA
ksBCXXgm0sPADb3+vTC9V1FYnezF+mAl
S5iftuN9f2+Nm8yA0LPCtW0Riut2O/No
/u15mzDWDy5xdORF3UPYokXQjJLynOrL
bWErCXufF9Z8t2i6jEbxE2lo8BvSHM5+
I4OosQEj/ejKoJDk6PeXDfwbbBtXNRIP
HLL0eUBeYHUwXWaX4ghXbjPNi0YpP5xB
1nLWyvYBeff/4rg1xBFmguxSp29SBE08
dSLp+5DP7IGYMwNKhH+v3G09Ts2dxGds
eidNXc1ccpFdsQtj0bptNewnhAZcHwK5
iR0RwUsaTm+PMBsNs0bTFZ1/WyJAbKVB
CVnVgv5T6ySn9Fmh7hVVeWutAUjsP5yg
lUs/Dkl0P74rTTl/IyzuPYeWbeZl5Axj
xdwTFJL3xIsqob6n8zYHVIFpjCQPD3Eu
1ftHxN3su3MEo0RjwPQ1txzzHfz3Epk5
JN27LSOJ0NOFY7sHwaeQJNWLS/698Lkn
UvnUnucx/iJTijsJh8aY7w+/Akg2xae6
ylSHnaiekdXt65mz6xXf+fSrxAikYJoN
Eg3zyvgmEbNGUWv5pLaNefF9jxV95EBw
oSqvG8QhVpw96WWTuuOYE50aaVk6Pkhw
yNUDAf7BrRwpmYuWbMjkOq5Qt5Vy9L1Q
qOhzgT9JEkxe6J6RkJ17sVHpnDhVeRg2
v6TouyOGcFvUP49NTgKhVTOKnumW/26y
Sv9M8Av9ax3NgivteM//ebwzp53kmj6x
K8WP0S7Q4csvg1P1ThYb9fdtVNeSxTm+
uRaQW3T77+AQfTxU4DMomF8rYMOZoj0r
b5q0gaHphnIU+b6OaeA9urmXw9Zf6Y51
niX36plWjpFOTT8m8bj3/nlwIiKOx9du
jUE7nU8aidSIJePlY0QgCIc9QtxbTP5n
RGGnMvnZmiNO0JGC7WZbL4t/DBHdTNF/
tLz0boNDwx3Y/ba0rD3/gDEGgjmXwvOU
LcWCFt57pUMfrM9VkpgawaAEDosEO9Nc
nQzYeS6j1D0HuSwFptu6f/QlfVBKbNUZ
atRjTQVLU20LRYHmNeT2K0ADeoHznD1P
yJi+BUo7sjjAfxxdFEV1iSmxdom5r9HB
74HaH7pEz4Zmurz49SwB8ubi/mi5kvow
QJSOSrjcRgc8Z4F0R9uVqRMAOyfBts7b
nmJBexuFN4f2U5BdjyJHufJJYetDS6yN
i6OSOLTlQEMgAbQ9fG/5O1rIfxJ1F2lr
FzXMes/6/hxM3G8dRb3+jG6Xfj+phjw4
BWC86ZopQckRbMyNEjaM2x0oiAYkn9mi
gAK4mASlH+uInGUnSE3+oZbEIcjIwj2/
7RoGew+75Pm6w02TPHYHji6w+XtzrLIy
zc48PHw6S36k/kXUTNzMIyrHXMKxOxPS
d7OKBUnrQbfjGvOfk1aIX7y2H/rt1zhT
Imr2KCxDgd0T4KANWxbfhNBsKrIKkdZH
YFQX0C4OGNLs3f9MGJVCusjn6OQClNGg
Qgca7M1mWBTLNzn6UbJTUorYgJ1lPkFs
fbTW46/LQBU2ZeoaFUzJEzwSy3eL14UE
sm70Ze7qlTD2QO7X30m8t+4GWNmx+EsP
VFdscEl7z7+Qtn51zt2StTD3vxpWL9US
O211ZU0ugTFKWRNIaxb5I7C/5+aNPRWG
h3zE/H4lY7SMD7X97zLOpDLAQ8g77uNb
G+/pBh32BeCEG31/SWU1atuSaevBzcmh
hJlmM5KI/h4HaMu51aqh+N8h4xZwzBPe
xzT396ndPnXF77mHGtDSxcwuX+btsRSI
h/PAR5f93HFJfdH0cz7P5qP4KaeLO7ED
fTxkhIpMDHAIb+nmRM+tb+7Uep4MMIie
hhgG1Hc7T5Y6NbjfDiWwT/GgM8OzFQDp
G5me4wB2pctrzb+WlOk92fD1TLT98Vda
aqqYov9YnL/salMBqyDAZlK1rHZJTsxB
8P3xhA599GcvyhBnl5YhgVveptWixKZo
4BaYioYGveRzg59NayFEurfPvygTXl2X
ikiYpnta+3Z9Dw+7ExoBU7JIOo3Z+7Lc
dMn3NQrNXzN/D+kvG3l5JVQlptMaYhSe
e54xClnO9yuMnbdXH7akUZaqqsXirfo1
5MH0nvQtLSzWLAT6egwfG2PswRDErTrz
DUXbUUYd4YrkSnaVhkNVp9KrhMOuHRA5
4H79JhnEwzyRKPKfOmfAStQccRIor16p
oaHMdWUvZIA5BtWjHYET+zkLVeQmKrTA
NM3/CHacDeNLt93MI0my45rTlOChdCbN
lTw5ppP9fRbjND73WGr3Mq5kKabIL2EV
zytBl55nEMexZtLjf+3CTrP8T00aRmdi
dbewlRWMIfJ5eyE36vf8kNqXzlfcEN/K
2ZZjEIyLKU70iblQuzNdG0PABY8i188Z
R+R4dQzTi5VZeUaOMd+qDMgDZLF8Dp3Q
kf6awW/TOZjfji+hN3R1BzGX6qHKM2YJ
nBKy319tJJxuUj7UHFIFjHo2lU+H8IIi
womJqIXkguyOrWmUWvqok/8/UEsDBBQA
AAAIAAAALl1Hj2HVQAEAAJMFAAAgAAAA
YXJ0aWZhY3RzL3JwNF92NF9iNC9jb3Zl
cmFnZS5jc3aN0UFrgzAUwPH7PksOGo3W
Y2HbqS2Dyg67hExTG9DExThxn355bger
EYMgQn68+PjflRY/StJGyN7wDg1ClmpA
rOu4QbwWlfisOdVq6NCNiZqX9KtntTAj
rZjhSMhCNW3NDacNkyUzSo+01bwUhf3s
7Pm31SU1TFd2YFfcednDFJj4FAWo1aJh
ekTH49sJ4eSQHVCAQkJQinAaEPJgzh8X
a9I4siYKsX2vzPklP8IcAgYH4b+JH8z1
NYc5OIM5OHHNubw/w5w0ieE0C10mv56m
u2Iy3RXN/7lQ8iZ0w4xQ8m+5MMHg4IHP
NYLt9tC03i6C/fbQtOAemjZcopDsV5ub
rWozs1ltbraqzcxmtZnZrGbNfrUVclVb
Ime1FXJVWyJntSVyVvOI5tHMI5lHMY9g
Hr18cvnU8onl08onlU+pX1BLAwQUAAAA
CAAAAC5d93ATipwWAACtWAAAIgAAAGFy
dGlmYWN0cy9ycDRfdjRfYjQvcm9idXN0
bmVzcy5jc3alnF1vnMeRhe/3twwH3V39
eekgu1exESDGXuwNIcuMTEQSE5JOkPz6
fU4PZWve7nfUQ8KGJVgj1lR3fZyqOtW/
PDze/+fh8+2n+8+/Pt89Hf51//nnh38d
frh99/R093z7dPf0dP/w+Yn/wQc/3Pff
/fY/3z98fn589/R8uHt6vv/07vnu8Nd3
n+4//vvwj4/3f7u7fbz7+df3z3zy9u93
j+/vPj8fnn79iZ/6Xz4d/v7IX3j89yH6
dgi5ttp/9wd/+/DPu8fbP7iDOzoXYnEW
qk/NtRCqHT4+fLh9vP/5w93tL+8e/3Hw
x5Bb8jW1lLyLudTD6Wt/992f/3RBSniR
4rsUn0LzFlLO/AwEla0YdywtNcuhtJyj
i96viTlXxlcrOSTXfAjmk0eb+w+/PH/4
6dNtPy4+1IxPJbQuLViwq7W56XKKtVZc
y2YxpBrqVg6fqrW5YMbBxdicxYuCSrTp
5YQYi48lJ8dXjvyQ8XaSb762lKPnt+mL
lO//74cLUs4vx3FQLXLLyfnoyuRuAqoU
nzEQbMDFtiZlo0uKlrOrMVmOXPX2yPzR
qlWfY+H6XUkxXqvLTT5WX32omfuPnH26
u3FpcjUOK+FjKcqkQ8YqL4qqaa4QNmTc
bUFcRr2Z65SaaomlumweP3sR8/1///jd
BTG/axSPyTxe0bDliErlpND2erAz7qY6
yy5VV7ihNUFbfbjf7EvkXKq3EscbCok/
0Q0m7/n1an26teF23jI/JTtrPnL4g5Pq
hyd8ORUMoYXf7mcuqIQ208fVVqovgX/l
IXxgOLhE6HO1/5eQ077Y9fd/+Z8fL4jZ
6EMkKTFbxUEx8TL4qDtGa1xeLDJLK+F3
O7gsZxPbouGGOEgNjRvyYTw21OXYiBMh
8l9fr1WnxzaXK85XYq2VAIfhTRzI0NkX
LigSESpJ4bKkHKf3kxs/h2sibLlS88Sw
iQiYNldITiD6uS9yfvjfP87s4Iuc8wsK
eGEJBdONWLWfZjh5DuEvpeiSC2tSNtdT
Kt/PCsdWyQtpcj1ECgIt/xrBp1h6jTLE
q+YIny5yScFNpCQcDxPB6jmxFP1FKbjZ
9Gr4luQUpYNAwkyTlJAUK0ikmHSTT7+I
+fEvf7okZqNMTvq2PiiL2og9OLLksTTS
Hxk3+rIm5FwXIw74qsgofFPDkBD8sXhP
fsUASuXgF6V85TZRCdLL4wAF4I+U9xJP
QInsvQEKdKy17sjy1R08GcMdzH2t0Al9
8GWrRxufCD3mhtvhUxlJrpI8grlooKmP
YEhzX2DlvqxNcAOwRSIot0yEzCHMfLRl
xWkMOlc+FJdFbW3OBHFq1E0TI2emDRjC
6PgaxFIr9gqlTiGuegd+AGZwn7nWMrkq
kjY4pWA3nDB2ekFWBZeREkGB+PZgfqY0
FglvAr4pj9GHGIftELiJu9hoPrz/5fHh
8wOfu3//7uPtTx8f3v/t1l8QuAl3mJfP
geOp3mOKE8wISMqk1dRyEK68WuAGN3BO
zkB1ZEAyxnBvxFdlfKEY3wj07Q0Knuwf
FaspsGN5VusAIGT/IFoAilc0Bp0t6Ehi
xWQI4XwYLDWYJ35bQyHYhVYx0lma5+At
KU9xCYIDh4937/5590XY7cOvzxcFbl2P
EJ8yn0rFO6uD5eB6KXDRMSnvJOJAvlLg
5h4Vl4j85EEHlkwT/EdWAMADdzh+wIy9
TcFAmhJsxT9KUik24kDyFxlZRSNA0ctR
LwnsZqPSYGam/C8CfOzhMGRX/KQEPRUn
pO+o5FqnRhMuSNwoGCvIRWWviutWyiTB
hUKMVjinJGmVL3i1zG3FSDJLVAPApkxA
nThjbi1TiwCpsDCzNypJluGGyN/FQD2z
SxToQTcCakbVuqDhi+FUn+rMUIE2CpSu
o20S7gQ4EtxAfCRrohPY79wxwo6h/i5v
o2JMJRBTCSXc0uwWSeRJyA4Nc+sQ8ypx
Wz8kMIN4cfwEZBzrI/wQrTgE7AsEkKy8
Wr2XdJi4XLyay/FRoGSSDpMDOzUioFG1
9Tx2SSQ1wSFRFCEXBLoNpRg74B/fV88n
eDdBFfgnMcZHVTTKZn5qNXZB5sYXSfmk
AVxMWbhHkiF8q3EUJRcYU/M8TV2SuLFS
bN01AptShrDrcIsy5Aa+CBxG5Eu9QcXT
PSaHY2cnjErN7GaVWyDxUz1iPcUpccwT
o52bTyYPg2u5J0qpSfoH/8VeBmYA9Ag3
TA7E90pqZvkzw7HRVjfCNq4Y5BFeOYjf
EzInvmgq5aWeeVyjXilvq5wv8kYqrlRz
tEnxgNUYVkWJKYvOr1bvJgOlK96Hizri
l5Xd7hUWRdHUQJbgqRjmIt8/fP7r/eOn
d2oOH6h68KKkXwd8SpCsslYZRhiyoj82
DIpSv7WINxLmN43MXTljJQEm8hTyIDMj
FE0qCVXRmeMEyyQi0aqkzZ3xbQmdFHtY
OJcyuTMhCmVlCg11/16h0kshBmLwQJfY
QCjBtcltJepnp/qyKlCncL1OJ98OnFqm
LlKvt8Y65Dx9KhWKDK6KiiLoo5uu5tpN
gcpSqzmrhQB0HnvO6Zh6G8qrZUhyXxZz
dk2GM+GoVI1Uc9WGdnA+Kt1Eoic4pag5
cLUyOhK+J/EVDJpDBgnE0aHsmImcKigj
99QsbXvoi2anCjhoeMFlEw1nfdqongk4
QYWetCubtuaiK3n1T5t0isodflIZZA1a
aqLeInbl4lcFbTt01MDEdMERwumQwrrL
gpaJDxgcsl6nj0zOSPbEa/2UAbOqDihE
xFoiAUpTqNepQ/WGD7kmdyTGDhdEXDDl
foCjE9xJ2z7toj6YNrZNPvTyfCDFJKZS
nKq5WjlizC+sCtqanKo/fBRkGnPvzgwX
JODd1HwgGdrQ4V61OIBY1WihqTIqgxOp
3k4G5gilw8jfQs+1CmFqWRAUSNE0iphY
tpMR4NKkLb6Jbdqbq6HbqZ4pwDw5vp/3
0HxV/5NSn1hn/IVlWRuzA9w5RCnJYn15
UpM1eZDmSA48TFp/pVJB1blVXwrf208c
9gZRnvAO3q/8GseO+mo+Skro2JOmxjnU
8aZugroFeK1mCSTIGvKm17mqVaSkorqi
Ztfk1iapTwcYNVCiAKPGrNtm9DIUElA3
1aihttmEEuhimhAkajzN467XqJ+dhnW4
FNDD15qHcitQMgPNucHGZQriXZLDRw8c
kJu5E3ArUAkTVaPC3nh0GDm2ooyfY++C
qxYbGp378s4Vi6qO1e+gMOXWZliyKWtx
AH1QO+0VrypX1F0hxmoWknIabsuOKE9A
UhhOp8z/KsVuXtoqBILce1+Y4jgb5VOt
qdvOlyq6NJLhN+TFejAC2WEoOoz7Ig7W
PjOvs7lyLZo3qhmXyCFc6253c0/ilgGQ
1RAii+unTuOuWeizLepjFXPpQkd1Tcso
9Azwy0m9vzrklABcqhqnaJ6ldHChMf5t
LW/KUY6Q5OUBaBxsr7riflXocB5AdsLB
6uGqjZF7aV5G9N6c+d6s0fiCiDbJNmam
7qMH4FPdpZ1e7r7I8xvt2U09sEB9XvOQ
3jhdMo66hZy+5r07vdwVFSVPGc5JgaA2
QB5aDyaoSOyU9ioo29XyNh5JsCbNCUeJ
y+FH1Hgj+gXBgWwH+Cer+rogcmquJ4Ga
poCmcEmK2MEpb1RVqlFPdogq0UvYb3Qu
2OsJ6FPLCew74J3PY8v6pndXO24C6AHU
64V27ppbeqo+aiBSIbFQ4/AJ1gtOf1cw
Wk2mN2h54mzJJTWYJ6Ypqk8G5yAVArha
mJlrWNXxgrkGYo/K3AZSx+XypBtBBcfB
O82nOYtU5w3PVX/EEk1dazAKsGnkooRj
573h/7E1zUOulnemIP4gMxVaUhtsbOlG
Ap1yI+7Yu7Fup6G77I+RnCdQyE3h/zZz
R7y+iCxHEASc+rggkTRjnP9hnOY4ZLhm
eEYUl2vWTvLARhlVFfUjzOcOdknk2RWC
ZtW/rOifleOHGB6PVC5AfGAC9swdXi/w
/A6zq9aKQCThJI2ti3gk9ClBqs2pVtRb
NOyXWGpS65H47TW4Gu6QrBHF5QwmIEz4
XZTY8iGHiN3kr3VMx+Br9IUk1ALRte2x
20AkQIBeQ+W0mTrO+51fC9xWftT4iAxZ
vRGgzrSVRrJufVDgRd1MuVwpcxNtyBfU
JSKLxtN4aCgsxK5MhQ9QM7WrxW1nHcAz
J5/mtJqNBYYj+eOF1AXopqQxO9Jlyq3I
Hypamyg0M+iYCKLEI83HNPI4Y/au8oeb
fFzDtT6ano1vkkqOKEqvWHK+LEn5XRc7
cj+kWPWDCcF4wgwg9tAq9w8KBEEzRnuN
Oskp9JpIdr6KGDreEAFOaTG9kKYvSdlh
3OJcuheZX+gazQ6N02pyiT68OaPcrjBu
X+jDGSRJyPAq08dQLH/S7VMwA6eD1yRo
SdC2rw/iEyFHDVScc9InFB2qihulTotG
La9QRwNtLElVfm9Pz4oEU4tX/BIT+zqV
dEnQHt2WXCknFRTJahSMl3Ma0XEt9VQ0
fNX1XCDbni6Hb6emkjjenFoo08EB4SkV
NYaAC5SutiRpSMhN3RMxgNTCm5g0IRX/
LMqiOFor1yr00rcjgsbQO9NYgp+wh536
vJrkoq0VxdsLcvbYtvgcfqHBQxZNajL0
FyGFsIbjhKIYetYtXmHbeuxVHA1+TLNu
3LMEeGq3if8IkBHLn6C/JmrTDCK1mxIC
eVS9jEls81G9IF2QZtbZX6vQlyE0ATrJ
pgzVZp0S0F/sPWvNVYmmlwXtsIc1cSax
UJTV1MjlkxsC2ZrvrtxkNr8ZXG92rrKH
szgIJXQWVh4rdwUd9TfVXuNgcaIlIdsx
cxNd04XOGRop1+7YWT2UuVVjN/PlFZr4
JFql+B3UUH2/Ykw6SaReajF0oV6OFw9s
wrfNxCwLamU7gcTdfQUvyigJiQ8qPp31
a1e5w2rHkQSAadamSyuxh29yaKTgQk5a
krJpUimAVDWvCV1tAFLiNpDAtdtRNVY6
5w4vs6Br1e4Fnh1wiHGKjCpNyKeIv8zN
JT+XsseuFcGZHAP4JKpmdAk716L2rzKe
63w7PxJsF0nDmLHaEgQtheE28RknqlGf
vWfXRIBdlbRtrBHRuN+stmzM4hXMwo0i
HmfspBpyh87sMheao+eKRJol9cwCaBNn
syYRqpJ6qvuSLlJqARVefxhw2kb4mRCk
dHRk9CKSURYHbt6lXCQNazHFXBEzkpMq
M3KkugQJSFQJC8TTa+Vt/KrPCNTnBkER
vie0GsEikh7fqvWFwTfp1znz/CDgulac
bIa4nTm1iKkCKJrwhL1+8zKh1uMIrfAz
tTpJRTYpjUIVugNlEDN9mNFb1+m04saL
O6RVGpFqJ9KcFoPUX8LPU8dO18gbhs9c
Y02CfPzMOBk+B22kNUG9qrI3vVG/5vuC
Qes7KWNLC3lUGCIuV3lQzZfVu0wWThQR
TRwyERHFbBjTiwYvWG/UyJbYvdOUXOYK
y+h8VNex9HWXCZzR9qX6wAB5zQavlTjs
W4ipIjQtksz0PIG+uE0g1hFnxaR6k46i
x4uuCi4EJJRxwCo+ifgdgrngA07+mwK/
RRXWIiGoranYnpL2TRxi4CphNIHkVO4N
Xcl1prBYV+oqo51IcxNx+KAIvp1OkcN1
wragsWjAW9QlJF2MSFs0IPE1soiQOH6d
dVyXdVNSIlARI7XpNKu+NfgIqjJF46nR
X9buMkWYsopoDdilIA1uwqGRO9TeUNIw
nX926Pq2L3KjX+NSDOciORiBbZJzVRo5
YTECHje81/1c1REwUaJqTFIv0WqC+/hb
GjcF/lgrJzsuv66iOCZ9Q43qVg2sWe0M
KKNOJ+WqpNlZDrIzs7nIoPXlNBA0QXAt
3k+gjJdHxL6Hy5XP+KzLBGGcgRTfjMAF
LK9tlnO191BqXwJHpM0ou+sEYaJ/6wWv
qds2rpJRugFNo5jseEZ1V0rb5kAnFikO
UdVuHcesyrlRVOW+H0HRMmUHr1OXRMb1
ShTeenUzYVubapeKYg3Mm8/blIuEmCAO
m7ZaxQAsMz4j6KGqY9CIn1hoWxSz4VWo
GyCEiL1RFcZJH17tJSqWqnW8nN3r1BFE
AcYAsNBGVdfEscVHJedq2Z4s7K/W5zSS
Duq1EbHV25g1qfoeo4+nNqLwoW1blYuc
YM4kVk2jAgba/OBVRj2S+EPMgChltijk
fN5FANaeUCPR9GX8gRCijQpN9J0Wpex6
TfoGbt8coIDzCNEYaCgY4zFp9p2pKqtS
0Gt06UtDgUJS72pks2YT33E6T9wK18ra
nmrnfcpVbmYRDBfHOYrnlWdQFRyg5Vwt
mGm0nxcFbdyHlO417tHrDcnF2UZLjhol
mDo72u55nULaju+M7Iqk0UvVfhdNRtsI
WlKP8XXqICXoqQSxffVYwrhERvmiXFEz
taqFcN6lXCUDmwK2KLEeVKuqcpQD4jg9
7RCE/lflbCpq17d5tRer8D+JbWCpVPQv
YJ8yN71KnSDn1LYYqYWvPZstAm47X0Xt
HTnbq9RRpRA1T6xcD8Fgtm7LrQEAhXGJ
gXz8vFm5zARWj1+vgATZWx33HbQ1pd1+
p/1mrcu0ZVFb/rmIj6FzPrmoGZ+eDCcn
DpSUFNbBXqeSV0pRw1CvgLiRFq4JVt+c
FvOLk3PBXa/RC+vDVIk2qh2FoVkC0jjb
axCq9wiIeOe9y+VbIjFwJLqCwiFN+do6
My4xa9lcI/5FSdvXFaip1HbTeyCYw7gG
rdOjBqIK0nNVQT2qq3U6DQCDzJsIozdb
JgxITc7Rmwyhzo7P9ZKcHaZs327RApc2
kNXUv7uZNTNOpAHSXcYutGPnB7rsOg2Y
Cr6IsR21cF3HsbM/hlCJV41QUjoj8BpZ
5z6V5I1qQmnLP5VZh970xkERqUpDj0lL
dpkFLDp1yr3dKurtuO4lAMkfZvTO2t2N
A5t6lYZHRj8tAJ+e2pmEP80cgyw19h77
vPr1FwRug4ZW7aMakUVM1DbUvwKTMlmN
O5O2C7Shd63Qze0FvepCzEVLzHvSBqZi
IldpQ1H78RfazotkQ+szlmR9K7yTtAdw
Ie0oNIBm6mzuPo9xBf1X1IeGUwiGqTyb
zXhxup67e42oB6WmzctVumH1Os8i4mhq
ecye/dUR4dJMgk7JT5+OWKdTpqz5QdLq
hChVs3Z+Z2wAuFRb+Z3W7DLZUOQNLUNq
u6SlscASGTdq508rDXisWNzfFLhP/Q2q
BrVcqQcGyS8z+jZIMemhoiCiYN4x1LBu
qFXzWoAncINcPb5apcpO54ChpqbHaq4W
uClYtMpnWkpUvqll0loQlIjCf6YtL9DZ
G3Xsb09qcToAXU3LT5N+FCZA2NaMVLsQ
bb8jvGqrXs8JiNrXO1JhBvypYzUK0zOU
qtHzvCe8TMXP4vwC0EEvRLLJC0BJZF0T
CgvaqL1S2mYthkIi6N0fWU0aQQWeT/KN
TSUuAZ/Pv1q5kysG016M1pm0mBrGASxB
PPcd8s7VB1/MqMYrnNiXia/XKElvepnG
rVMkqOcWo6KRXqvRqwi7Xcxl6q/1GZbe
aHFjiR2O2KgWWdS+1K9Xy9uwt/tcrehN
pYKuA4APRz3GpY6ZAGTbfatilfgLbCa8
mTixtdkYv2/CUQ9YqvgTySXuvOE0CNyn
xIqcqOfl1OcvwkuTAOdSXzwNGjRyGH4z
v5j2Mi+wYr2eDNPbAtrxbXlG6dKQWG0u
whXKOteuEzjyFSkisPiImfrZg2ZaGTc9
JsuPODG836CfXi/CREPVo6R9YWHoDXsw
sNMbnkGfyDPa7/8DUEsDBBQAAAAIAAAA
Ll3NChvlngEAAFsEAAA2AAAAYXJ0aWZh
Y3RzL3JwNF9yb2J1c3RuZXNzX3B1Ymxp
Y192MS9yZWZlcmVuY2VfcWxpa2UuY3N2
zZI9jhQxEIXzPYW1sYXKdvmntBEJIiJA
5C1PTzHrxd0e2d1atDfiHFyM6kEEBBBP
1NWfn8t+r1zziat+br28tVV3/sqd15n1
p0nQpaxDKv4+1/3MZykHj1Ga0Lkt18ob
T2joD52WPL5pHltZ8sZ6LlNtr8fnuVye
Hx57O+1j4zd17O1t4a2cs+JV1azk/7rL
zqwVv/C8b1mWzjyu+88fQ4pDU3ne9n5o
pX8v+UmtTQ0WPMvhWS3tzLUNtRY15+VU
pHdVW9n2mvuTCFXna+tbViPXSz5u0X6X
j9p4PTgP6VKnK/dR5J5HCsYTeNQmxqTR
RP0h18Ea3nkfyQSKJroEgHgg48jGgD5a
bwCcoAAiQ8E+mkTu3hP4+P7zdH5dtAmQ
nNUgjkl/6fthGCFBgBDIg3ceMApzwUNI
jhI6S5Skg+hixGiQAoC1KZlw16Yd/Gvs
iTxpk6L7a+wIAZMzNoAFsY3hlgJ5ScIT
ehR9uqWAQRIxhMmDCejw3lP43+ido2Ad
kI8yThvczTQYm1DedrIheGtujCQCikjR
WZ+Sf/gFUEsDBBQAAAAIAAAALl1PHVU5
OQcAAPsxAAA6AAAAYXJ0aWZhY3RzL3Jw
NF9yb2J1c3RuZXNzX3B1YmxpY192MS9y
ZWZlcmVuY2VfY29udHJhc3RzLmNzdu1a
247cxhV8z1cQAvIUKujuc05fIOdBTmxY
QHSBJD/kieDOtHYZcYYrkiPZ/iN/h38s
dTi2pYclzQjQbBBwdrXi9PA2XdV1qrrZ
1le5LW+6vvmpO5Z9fpP7fNzl8lmFpuvm
OGAr/7BrT/u8x+aQh6Hp0LrrDrdtHnPF
Nv3WWh3q4W35pj407Y/lkMePp6vetc3b
XB66fW5/3e67Npd5GJtDPeZyGOuxwZtd
edV14zD29W111Xa7t1Wbj9fjzSfNfb7N
YzNOd/Gxdci4vz4fuvd5X+V6d1ONddOW
u6Zquw/6301zfVPi6PpDWbdj7o+44Ptc
Nkdsv6/b8nhqz3+qXd02V32tV5ju6zSc
29ENuEiNrzP86UHfXZ2GMf9UaEf03QG3
tK+LfCzausD72xO+Wl0W+d95dxprfLTP
w+3pl58HbOg+bd6Np173RQf0Tf2oOHbF
kNG8w3XrYuqqbiiOTbGrD1cNzt0W+Nqn
tu4fYccC3dD1Y10MdXtd6110580HpRV0
fj3gLG11m/sB/TpBaiUZ4dKGEEu2ofy2
bodctt111Tf761zd1P278mtTmr+KhGR9
CjZQNIYZTTY6HGwjJ/yz3pcvnr96Xb14
+eTp45f/qp49f/rk2eN/Vi+ff/39q9fP
vnn1CodQkOhdiD7aJBEXPeT6WEqZ8Cqd
cd4kE0q9HlG0Eb/WB2et0esx7la8dT4G
4x17NBljbHndZ/ClL8cP3cOhASkLfMVd
Po5Nm4skf9Z+HIvm+F6/eHcsujeFfgoa
7h8qisVtiY5Fv331t8KAMEOtPN4XvxOw
ODTH01B0VwN48ekHj4q/WPRz3wM5pcbf
nz998f3rb/5Rmv9jNtg5NhhJ1gKslCJ7
cavoEMhJFCNAW5bpkNiCbt6GxEz+zIcQ
fHJixXpLbmrbCHF5QrgZQgCpYAJBISAR
YRUdIrN3EAlOYgJG+Dwf2AiAJxFH1tHE
hui9CU45QRztRobLkAE1dLy+Ovxawudr
BQMUQJZcNCQ+rqKD7ggzEU1KEP0ldfDW
slOFMBJcmuiARsORHBkvdtOG+6HDbLFA
/Q6O2aCeu5DWWQeGjLjg8RO8LLFBdSSg
zXiO8cwGYU+WIpGBtLiNDffChrlKoVYP
L5cwjGWtdfAGUhIjuBAZDFp0kjyVIIE3
sRMbvIXpIFiWRPAdGxs+nw3fPX5Z7T8c
SusNtBYdDr0uX/enu1MEm2i88T6p5RPD
YUKfULgJhEg6cNfECKfKYVwKYeLLAvgw
BaCjUsWzSCJ1Ci4SBxhJgQbhypY2/C+B
v53F3xktBAagxSnXrSEAcAN2CAWMsb2E
f8BlKCVcDRlBZMIfwz9Ewx6RJcJabPhf
An83i79J1hBqAaF8AIY18AcL/fZAmQRA
LuEfhZEbo4TADsF0wn9yHowCIDiHbPB/
EfjvCAZ3wx8wRJ1ESgQLYNckA+eAnWGr
L4xiWjCDKPQ6MwG+ROBuETxweIDcaPVg
JZ2JWzi4CAPmC4AIBiIla01EOedVDHBi
AlSDElIBw9EvWYBkPPTFcUIemMy/zhKI
xfiHGY3WbQ7wIgSYrwASPHDH4Hc6/bNO
ApzTKcFAKAbOL0wOQGAccUyo+izIgWf8
JbLqAgRHEAc2/D8TfzJzeTAm+HL0Pf3B
wgJrSCcLwJxJOv+nkCGsO6tLC8EgLfIq
T6CO0GJ/gdNDrJsnhCMDKxCSE1yDnZ8y
QXI2BQRFTs7LFgnviRB2lhCUgEfiBCnn
dRXCq6dQn4A/qC1LfHDiPSghNCWDSSB0
lsgnPRSegTeHcD98cPN8MIkQ3x0TJ1mn
D0QxeYHtC6gGC5nBEemiEpP4YFKycuaD
RzhBkIxgVTBbaLgQIe5MEXdWDMCa4PTI
6JrgGkKACiZa8dAUyP8CH5xOHXqnq90x
kZ/oYMEk1AzYFoLf2fThfuiwUC8ijkS6
ZyPwm2sLRpIQExwqh7goEPoIQxCkUJCA
zVkgYC4RZSPhxxuzMeJ+GLFQMUQzQ/AU
rGGzZj0ajEgINWnCdHGaCSXDBn00ZVrZ
sOFsKTng8+TUkurq1caIz2fEf7fMAL+H
HGl0NEO3PZ0JEFAadObHT+uDK/C3ycH9
kQgQndahF0Im0oeeGkTwkBs/echpdRPF
IfmgTyTEjQCXIICdJQAHUTkIjHMAqFUM
4OjVDULpRRcplhhgk9VZSaML1EoWZQAk
wGL4T79meyblIgRwswQgvGcRE734aNcR
ANFQPEDVCavoFgngoj5uEvQhuEg+TARA
jFBL4TS8Qn82BnwRBtyREu5mQIKgI+cz
RwdFXlcENFJYhmDogyWOFouAziwZjQvT
ytTkDBhykAyTWg/eJpYuQoD5GgCVhiGE
GxP4M17zpIlNxC5icDvyREsLDdjHihOk
CKLI53kkStZMS1soP/78/MGG/xfHf74E
eHYQaBaAEGkt/qgCukwUdfUgLS01eNIn
TQxqBhR/eg4R9cDrtIGTac2B/rcF4D9Q
SwMEFAAAAAgAAAAuXR3u/hHRCwAAbVEA
ADgAAABhcnRpZmFjdHMvcnA0X3JvYnVz
dG5lc3NfcHVibGljX3YxL3NlY29uZGFy
eV9tZXRyaWNzLmNzdtVcy25jxxHd5ysI
rxmi69FdVfAqj1l4YSewjWwFjoaeYUKJ
Y0kDw/6jfEd+LKcu2VcGMpGuzQ1bwIwo
8oozLJw+j6rue9i+3R3WH44P+1+O9+uf
9vfvjj+tf9je7Q8/rx93T+u73dPD/nb9
zQ2ueL+/f8Sjx93j4/6Ihw/Hw269e3za
322fduvb/c0Bv4tvH/bvP/zhi4fj20+P
T7tfVrfHu48PR7zR/t12tbtfHbYr/Pzx
E35zu17t/rm7/fS0xUvvdo8fP/3n3494
kNccdrdPnx7yWrz/w3775er+uHrc4elb
/Ovb1d3x3e5wfFzd71e327u3e7z3YfW0
f/p02D58iQtXD7uPx4en7epxe3i/zf/F
8fTwizXV9eldf14fju9vHvbv3u9uPmwf
flz/uay//tObNbXiwmulWP/1zXd/+far
v3//1T/e3Hzzt5vv33z3/Vo3zYnMmogS
Sa26+2Npa9nUwu5O4V6KaEzPtk31sGhR
KhtxcD47bnm+/fq7V+sjG2lKuAJ/mkkl
fOS69k3VQGnUnLjpVId13TQrNYxV1RoX
y0tHrQ4tA49RVFNvtZi7xDN2hFmItISy
6Rk8LdTIXGqr5Kdnxy3PQvDUSk1raGHm
Eh081QoKxAps1eIdPbVKMeBG8fLY6OGF
1BOACErEFvh2Bo+GOaoi1sIayQwe0I2j
OE2aswwNHl4MHooC3gEBB2toRw9h1YF9
OJqjbjN6qDWrUQygKjIUeiCyT+/f3t38
eNj/a7dYuEwMHxRqBC0ikw4f42ZUueVL
7mf0ODWUrCogRGCmodDzP9VZhh41Y1SI
iqEY+Pxn9LigBOBsEsby0zN6vBX1AjEj
1ca1jQyfpdJlRU201dR2stbx0xoWUcDm
OKt13xNMWFxigtKFjwyfpcqlnioUUCRo
UfUZPsnViRBwT7VufKDmuCoI15H6xM7D
wmehdpkXmcxPQ0Waz+jJxRVpDrGQOngq
fI+CyY2a29DoWSpd6nDAHq206rXULl0m
BQsrHaM34+jocQF28FzAR8YQ3HN7vP9h
/4CwiPf6P8ErrJY1188WiDcWoGCwTkEN
a9BUCd4IspiCdwThKmZJY0J9iiKiFfHT
tYPXZwLRSwWyjZgbJz/D0SAwTKXQjQqq
Bb2vhUBDNj1LGwRUqoLCMaOsY6jXixWi
JQiqCmeIGmlp5eyIE0IV5OwFTBNwy9Zd
9cRK5pUE7+pDGKBXKrQIQ4AFygT2DYrW
OogYKQvoalIj19WMokxrDP0KxhXjo4iX
oAhMDS5CNIf3myFU2E1ggLDKmndnRFhw
CBjVmjJKODyEeBGE4HXESzXB9xodQQRL
rSiEOlg56jOCiuRiDEQQIh4PQZ9NYq8g
CCvJa6AWIGqRjiEEj1oM+b1JaG8MsWbn
0GtD7chHpKHPh7GXMaQVXlkiCERdK58x
VAEURrjnRMrZCyWEDMqvpowcojQ8ghYp
mcAmFqwokHK0DiCgCS6oKUwhRG4moaZp
EgUcZH6ipsHrs0jHEDwhSgJZoiIzCRFX
hfNB7kLqmAEET1mKN6QO1GmIRtCLFVqk
YkgZXGooF1CL+kxCtYBlCPm+wTp3CHm6
bFQyG4w8RCfxlQItglAICFog4K11xQKG
Cogp20GQMeS0ZxJySsWzgsCm9eoxJOWy
KZhvClfkrijTcKL2pnPJsUZDTMvG2Jm5
aUPgJqowQ2BvVPX6l9iL5VkS6OsGtBPI
7rDMyF526vCgFKqwP6UJoXg5DestaoF9
hBdiJ+V6/Sr2QoEWdROBnyCsF3AKIbzP
TfmA7oNmEMfquQsy4UcDKxDYcc1etY9d
nmX4wcswx4xQShFOHT7g5wK+zkYI7CKf
4aOBoMF4xeG+x6afRd1EwAfSDuppAAwL
zejJdmIkDRP9in0UwUwBLMSS0q4/ZLxY
nYXoMQT4nBPC/GlvJyb7FHCPpnghzM8j
juwuZjSzKKyD0fPvGYX5hgtPKb4CKNFn
OtEyTNQm2Rdr0uGDrAbRb1oJINLr73K8
WJ1l8BHEdSZIebaXT3yLSmClBaWKI5Qp
d+7xBnuIKz37HHb9k4wXyrNQuuBgwL4w
ymAgjZl8NKOpoAzIYN7Jh3MeBDDBDk0d
kbHLsxA9CvMDLcIiAiOfRutJPlhvHmmA
jJWkW58AkZPAVeNqeKCRC7RQu3L+1Thy
ttVYWscPZ8Gg+eBiOe9xAX6AG4MTAAmp
h4wl7b9rFgb8CKjEdFIvcp/hA2uswuEw
OzLTTyDGCsI+0FWg/CPU58JZWG4i82z1
wAG2vllMkn/gm4GeSlhffdCsgFpuVWSv
wE+9+uz+enleze6QbLBug/0xBk1TOdkc
22R/tSKDhjfgqE8xak35coQLwYK0699G
9lqJXu8g6qYBOgRjGMURN3uv0FtGDSw6
9nYyygkhJFjJwdA0tqfrn8a/Xp8lECL4
GiH4vkCg4pOGtQ1IBuJuWhAoYraItWLN
FWL8EjxQGcIiXjgI003NZGqK2ODIYaXO
GOJM61rgHaV1ccsoC9pWScst1z/GeL1A
izCE8OVT9yd39lLHEONnyBgLl5hbiBUO
CPGsNst9QTFESr1wFAYagiUmYbgbQWDt
/WafdvWiZi3MW28Nae79EHihCpYqdP1t
6NcLtEjKsJqICBVhajJLGaXSg58UembU
MdQULho6n6axtSH6ZBcOwxQOJ2sD3WK1
MtMQHGI2yMDeiPjMMw/lnCeyS49FGDE8
hhZqWZUUddAxVH8ygclDFrkhr+EvEE6b
eahJbocBp6ezjuEhtEjKGIpljvVVUsY7
DSGVThP6UhH1rSMIVcFXSO6IsTI+DS2U
MoX7ywlzbjCb7RAyPnCC+B6SHcRnCJlF
bk8sOY2+fkd94ZEw3uQMg61Uy6GF9N4P
Qj54ppbcDwxtj/OQ1aOlkTYsyJz6XD2A
Lj0RRrkJsVAeQ2Gg4tSAx6qjSgjtgRwr
OTKs5/LAORaY6wJWyi7twOBZ1E4EeNo0
64k87aVW64weQAkZhB2FKn2rWSihOIhk
bCjc9W9tvfREWILHamYMoKVG6IyeZjT1
YZVA0DaDB4EEMR7ZXxpffzPxwhNhAE92
ObJNBpicTlCesSPQ8JY7guh5q2uwssAL
gMNzcj8weJa1EifwIKqDUMgaEledqQeM
xJF9ouJEz+ChbKpRzjfKAKOMC0+EpW5x
nhF0z3On1LfUNZghqhDunCH6eZSR1APa
UfI81Qt/eP29xEuPhAE+LnnOIrdlhhY9
oweSjvUl2SAy9b648Ejc8zgLhcj15/cL
z4PxtL05z8lVU8QqncGTnQxRJiR17xvL
Ird5IOtbDp5Drn8b4qXnwRI7gAJz0Vwv
fD4PBvBklxBmMDv4c/sn0cMFpUSFlBSs
NTB8lipXnsbw3PyTD0qb8cM5ymEvEUG9
PFHz7gGIaTmFpwGOhF16Igz4sQDDIJmL
url0/BCebQ5KCoPEy4wfuGWZ5hpZTR1B
uy4cg9Em21/Vc9QjDZ9/3rAxbYTKo8vT
LTl6GkMOAwHhqyrlToUBIHTpJEw21XPL
ijVgyYg6GzNLoidFDBRkvTmGnNrgELMJ
HWfqHrlArzcPASEwD6vnzpYIsU5CAAks
EOVpeY751FwagdKKIL4r8Qj+59I5WAKI
8wBK7mgROyWqE4IKzDPlyS9Xm/GD1FVg
HIEr9hEaGxcOwWjDASxwbQVesBbyZ/xI
NXy13JjZnvHTsOqgdWCn8+7osQu0BEAo
AGxxaM2DKPyMH6w7Yyg9zetONxD5yBNP
ue/edLwV9ptHYBMD1Wk7nUOwYj6VQiUQ
4SHwmjeQMp4hhBSGQF/A50lQ41doAYSm
WyQUplxKdbaEDKc8nUoNLCrt9wQqzFkY
WMe8W9B4Iv+bB2DgIM/jcpT3KYHhiVnD
Sp5HqQCKSu757QiCeUQcqa01szLAgbDL
z4OlisE1pzXOY3CnUxcThCgo77xV8hYv
0e+uUPImQg36lkOeAe4ecPFxsISQ5646
KnlzDg99JiGJvL3WPKiYAGQk0167HP3Y
8EtsmYo5PnbNmbrmHVtmDsLDlrvGELoq
PQNIM72mfZw2wU4V+i9QSwMEFAAAAAgA
AAAuXSRdgD2kAgAAoAgAADcAAABhcnRp
ZmFjdHMvcnA0X3JvYnVzdG5lc3NfcHVi
bGljX3YxL3BpdF82MF9jb250cmFzdHMu
Y3N2zZTNbtQwEMfvfQqrEidc5O/EKhxo
6WEP/VC7RfQUeZPprsGJF9tpoW/Ec/Bi
TLor6AF6AaTNIRn/xxmP5+dxcAsItB1L
vL1tMrRx6DJdxeQf4kBvXe/DV4piSS4X
etagY+mHjFaGnH1EM8UAFHLxvStAc3HF
46ClixhLxv/WzSLE9lMTYFiW1RM5wRqK
L49BfqkZoKMJ+ngHXQOuXTXFeczQNyHe
T5+VX64o/u3uqQsF0oAL3gH1A9p3LtBh
DJtX07rgF8lNKzzmNeaNDl9aXMQNLeS9
/RQXYy7wQNrYr1PsMaXOERhIcATH6xG3
5iiBj4BFcujqIK/H798yGtOcAG0Z0zQX
C5C8OyRDJBlQbnFdR/rYQYiZDJ60rl94
jB0IbnsMLh3iRIJliKk4kl1YuimLuDH3
KReMck1DXDbJd0toVi59pke8wdqk5gid
htVSUMUtvTi/mjcXl7PTt5c3zdX59eXx
STOfnZ6gdv4BlZOzq9l89n42v6HsFWPc
cCM5r5iwQnChFO3BDVRTiw8VTBhmWUXZ
42QmtLC1EabiVgqFKU2qVFpqqWRltDKV
qCdRWrpMgOcg0XIfD7LvoCNrSC0MxQcg
Vr+Y6lOIH3AL0/kh8ZZMXkjQHUx0yJpi
wbAer98Qhgchu34dMMrPg0V6P4yZxEVG
3k8dh+Qlx/qlhEQm5MfnpxfX85N3VNZ2
pzmb/4dZK80lN0Zoy/Arn8XMrahZzapK
KiYFV2LDWdbS1lIbjjGYspOozC5yVnq3
Of+hn8UWNP8b0FzKSluDiKyQWtfPgpZS
M2lsNTU205qZTQxsbrwXdG20xJHY4he7
iFrynSb9+47+F6CFkariRvFaKMMR6TOc
8XpHxqq2yhhW2XoLVNeCWyU5gpd4/Vfb
5t9FzGzvB1BLAwQUAAAACAAAAC5d466t
96sCAACqCAAAOAAAAGFydGlmYWN0cy9y
cDRfcm9idXN0bmVzc19wdWJsaWNfdjEv
cGl0XzMwMF9jb250cmFzdHMuY3N2zZTN
bhMxEMfvfQqrEidc5G/vqnCgpYcc+qE2
RfS0cnanicG7Dra3hb4Rz8GLMZtU0Et7
AaREUTL7n1l7PL/xBLeAQNuxxNvbJkMb
hy7TVUz+IQ701vU+fKcoluRyoWcNOpZ+
yGhlyNlHNFMMQCEX37sCNBdXPD60dBFj
yfjeulmE2H5pAgzLsnoiJ1hD8WWzyB81
A3Q0QR/voGvAtaumOI8Z+ibE++lv5Zcr
im+7e+pCgTTghndA/YD2nQt0GMP2p2ld
8Ivkph02eY15q8O3FjdxQwt5bz/FxZgL
PJA29usUe0ypcwQGEhzB5/WIR3OUwGfA
Ijl0dZDX488fGY0pJkBbxjTFYgGSd4dk
iCQDyi3u60gfOwgxk8GT1vULj2sHgsce
g0uHGEiwDDEVR7ILSzdlEbfmPuWCUa5p
iMsm+W4Jzcqlr/SIN1ib1Byh07BKCqp4
TS/Or+bNxeXs9P3lTXN1fn15fNLMZ6cn
qJ1/QuXk7Go2n32czW8oe8MYN9xIzi0T
tRBcKEV7cAPVtMYPFUwYVjNL2SaYCS3q
yghjeS2FwpQmVSottVTSGq2MFdUkypou
E2AfJFru40H2HXRkDamFofgApNavpvoU
4gc8wtQ/JN6SyQsJuoOJDllTLBjW4+07
wrARsuvXAVf53Vik98OYSVxk5P3UcUhe
c6xfSkhkQn58fnpxPT/5QGVV7zRnyf4j
Z6tqKxlSUhMq+SJnabnRBr8ItEa2W86a
M8UrrpUyTNtqo0oldpK04jtN+pkbLR5J
878hzaW0uja6ZrWQWlcvk5aaSVPb6Woz
rZnZroHXGyeDrozGnuRCbBtgN1HvNuln
7vS/IM2mscuZwdltmWUaqT6D+mATrpSy
xuD9ryxOe6v1Y7/gYGCYA051yY2YVF6r
ahdZY1527xdQSwMEFAAAAAgAAAAuXaP6
aDgtBgAAlS4AAEgAAABhcnRpZmFjdHMv
cnA0X3JvYnVzdG5lc3NfcHVibGljX3Yx
L3BsYWNlYm9fbG9nX3JpZGdlX2hhcnFf
cnYxNV9kcmF3cy5jc3bVml1yYskRhd+9
CmKeb1RU/lVmxezBW+igJaYHG4k2oIeZ
Hc06vDGfhJsT4SXUgyS4IKRD/n0ni8vx
++my/X69nf+8fm6/HT/Olz+2f2/30+l9
ez9dHsftt/Pj2/30dv18v3+7f31s/8S9
+/18/bzjJn7vx/nz/o9fbtfvX/fH6c/D
2/Xj5+36cXqc34+H0+fhcjzg/s+v0x2v
dTj96/T29TjioffT/efXf/+640Y+53J6
e3zd8rnnj+PtfPz18Hk93E+4/IY/dTx8
XPHfXO+Hz/Ph7fjx/YzXvhwe58fX5Xj7
FU883E4/r7fH8XA/Xn4c87+4vm7+spFt
l+uPb7fz+4/Tt9+Pt/9sfePOo88eW2+9
Exmp2RxddI4YvlGEtskRU2POGZM3pbnR
6CG8mFgqsfMltit7nzSNZbiobTS7NB7D
usmc7r6uVt61Un9q7dNVTcOHqodMvBVM
0cI4yAlxFVlXq5RW2uPqHi5del4lxsM8
orEjqDrwXnRaV6uWVt4LdkAt4kccbL2P
DemLgp2GK/yM+7parbTKnsPaSTpydgwU
6kurNAonRbuawxfuTaO06ktrSAQTvg3D
j6zX4RM5HEMI9Wo8x7pivcTaS6xqCPEg
dGPuYyLuY46mGEbDUbASCwc2Sut4afXo
YyBlO6Nsp6VW7y06urPmW2HrSp0l1fc+
DFoAQ3i38N5fUq3hgtGUZ70u3Jyo0Ili
706YpdM8U7eLG+I+lJuAJtClQBMsC4eW
ip1o7mo7Ryd0KSMz4VRrow0CJo7kCWhc
WG3RExc99cAkNRUTsq4z1VrDRQyeVBtL
qy1+Yiq1KpgywdMUreqplhoLsjpZEV8r
Z3IRFO8ExeGMkE6g8pCRYqO3GSbZn2dC
5bpai6B4J6hByFuHFswcR+5CrGhjNbQn
TB9aedBSIRTvCOVgJJ2wPcz40XP8yGwu
AqRINB5jYYSiYijeGWp0m0LqZihZz5oe
ozeBKeA0BmS6sNiCKN4hCqI6W8DjDdDw
0FfRGumwSH/nujJZFEex7/0YLm/2HDMi
QS/fM1ugQQ0DWGEQL5zIXBzFtYKaA2TI
GD6CKFuqNW/mYI0OtRhNK6stjuLaQUET
AZDtpc5esxYXEG9Kg7DyrOXiKOk1f1DC
sHMCZ4vizbLVpnjGc3lBK89aLoqS2kLR
gBeAK4AbUHieFOstGaMz/EAuV9cVWxAl
BVEOBh6EYSvwdZJ2YEbDRIIrSFvrC/Mx
F0VJ7aFYwVHMEAexmruZCYqCow9LaIQ/
WFdsUZQURSkMD1rxzPh2f61muqlRlixy
emGtxVCyM5RPmT4oZgbWcg7n0m2k9twm
96XnbDGUjNoco0FJLk4FQzXwXni3BpPr
IEg8PlZuxoVQsiNUJEQF2vDEt2RhpDE3
tC/BBYxdkPO6aqUQSnaEghlIDzBUQ0cO
plwxGrzAyDQeY2WtBVAyq2bVhQhDdoD7
Zb4WUYFwO4p3ui/coKT4SXvtKjpoyfNk
i0GGvJs8g5dP1vClT/EKoHQHKMwWNgUm
wtcJCPFlBNCqNCyXbr6yf5ciKP17DSVI
2glEDmUY2AytNUweDCUM2pWXx1IApVK0
aNYN+YraZUdgnXqDMwiiPBdYOq7FT6pl
eToqkwGPGtNzk+FdWmJif55tYQovrLYI
Sm2fPHMgXZUMLs8NFe0C755HQkp5zrd0
GhdBaR3l5dbNkcme8U0bMLxFAJ3zUwbI
8pW7cRGU1mHeDGONoAjBDMq1TGDOwh+o
584NDLWuWi2C0jrMI/XsxahYnYOeJ9K9
KcI8nuZ95UW5FkJpIVRnwjiFcyemHuPl
Z9ViMBTDH9jCDKXFUPb3WR7ICbwPtlBG
7WZD5gbPA66A2uCV87gYyvajPBgAFvAx
SMrMx3PjFg2IBY7MAdwX7sdaCGW8j1oj
2Fc0YBRvZnNOn2i5SqbsyLC7K+dxQZTt
WygGLamSOEA44T/ldkUiq/dnIqsvTBZa
HGVai3IawjDyAOQYiVE0WxpdepqBWHlP
roVRtmMUYexAK5P03D49y5ZbcCJ0jh/g
1cJqi6Ns30SxwakDKpDFwMR0tc4GC49I
9+enwxbelGtxlBVHmWOucn7UwCj3VFm0
3FP3zHNp+j+x/wNQSwMEFAAAAAgAAAAu
XQtG+DkoBgAAlS4AAEgAAABhcnRpZmFj
dHMvcnA0X3JvYnVzdG5lc3NfcHVibGlj
X3YxL3BsYWNlYm9fbG9nX3JpZGdlX2hh
cnFfcnYzMF9kcmF3cy5jc3bVmk1y40YM
hfc5hWrWLBZ+G901d8gVXBpbM6NEthxJ
XmRulHPkYnkgu12VI/SGRYm0rCegHz6g
eTl+O12Wn9fb+df1bfl+fD1f/l7+XO6n
08vycro8jsv38+Ppfnq+vr3cn+4fr8vv
eHW/n69vd5zi736c3+6/fbldv33cH6df
h+fr6/vt+np6nF+Oh9Pb4XI84PX7x+mO
zzqc/jg9fzyOuPRyur9//PvPHSd5z+X0
/Pi45b3n1+PtfPx6eLse7ie8/Yx/dTy8
XvFtrvfD2/nwfHz9dsZnXw6P8+Pjcrx9
xY2H2+n9enscD/fj5ccxv8V1P/2yKC2X
64+n2/nlx+np5/H210KLkBRqVBdaiVij
FJZizFqrsSzq2lYXDmfGdbfFuC1cqKpM
ppWH1rZrdQ0odCMKNg5IJVulGs6ttVp4
XqnSpTLtUqU1UsQ2Cit7S60iawa0FSUS
lphXrA6xvIu1JqFIVw2l4iVzmOracCxe
W2sI77xibYiVXWyQGyKKbBavlRRiua2l
ihVFZAmi5xXrQ6xuYqlEmDb3aMaVa0aW
bXVibo7LovNKLUOq9SQm9y2kbpWqyO5O
6VTElfKGiZM4hljv9uTVpTnsiqjaFlZZ
S1Br0jLHdeIcrkNr6VpLq4Gqo+QB+92s
2NaIisjmgm1c5xXbhtjYFyzgwbVqoVJM
rPa6I0xFomRk28SR5QFPPODJJOBLgqNC
YKaxrgyQaALIAElN7E886Ik7PXFDxjbQ
YhELLptaXrnUAtRAlZ15zfLgJ9n5iULg
vKQiQg6Ukl2sABadC1ixtYnFDn4S7osW
suFDyNtISl4UNLyKI4GhsUHuxJWHBz+J
9MiG1VK2WmswKe52XLkAkzON1SdWOwBK
OkB5VK/WvBYcqXpSha+4oTqsuiGdZ47t
YCixjouVc20W8QhtG0NpWRkr2cPy+sQd
Hg+GEu9iDUaM1cmeyStbIvvqXJHHGdoy
cYfHA6JkhyiAYVTDErWoNcy3PI7VmqJ7
py2PJxY7IEqizymAxJBHLXnKCneuAFTR
1s/axJVWBkNJZ6hKYEJVHKCPte59TyPP
dhZioXFitQOipPVF66Lo3JOk0P9QqjVd
i+AnMIQWyT1zbAdFaZ9CgfqVwFBgZC/W
6j5chGOBjOHH0mautTIwSjtGhSKs6PFE
EyS8T2Yc3Z/tZEET11oZHKV9DuWOOhvo
4UlyXqF7bBntrmQ3EFMv24FRqn04U10Z
eAx4wgG/hYIf0Q5IcQIfY1VPLHZQlHaK
qmTFFO+gDwhkdOZxWVF8apHZB1EyKEr7
JAodfGOqWMSGikMpVn2VAqoqacjFJ57O
yMAo7bMoiDIUH8g1yNvUSlvN0OqS5ayK
Z07kwVEaY6SKpamgY7EgjW0YRTmMAlbV
bQo5r1gdHKWdo0TMava2AUjESZIFrxSw
ZsnfIaM9r9rBUdo5KghFlSSXL5ZtS7Vq
SORQkZozi4k9SgdGWccoU7iTMGIoDItK
saIr+oBM4dzfmrjU6qAo65t5BR6lEsFo
2tEVZPGJFb081Wi55TMzROmAKOsQBaSo
FcBYCluwyu7HKLNo+CLTvE1cfXRQlHWK
CkbTU9HehTiQqux9rRjwKScWwMmJRxY6
MMrsc14Olypm+cCBWdljK+hsNZmx5RRj
XrWDo8w/nyNRxHCbwUVt+76PpnVt0Djz
vFwHRdnnMEqBFCkUHLXNoiRWQy+AmLcs
PhNrHQxlMeyYnSIqqUfspaeuntPkAo5C
AzSvVhsIZR2hLB82qGoKK86HS2BPtOYe
UKs5dpvZnWwAlHWAqhUJ2zxHUaXENj0m
dALpTCSJFhMnsQ2A8k+AShPSmgEuUIXG
llffmBjv28zbAjb4yTs/MbAf1kQosbiw
7/jICkpGGu/btBPTog2A8r6bh2QtDv4X
45AoZd/fIqvq28ZlzLwxYAOgfDwOBaRw
y4kieKlY23elyWBNW5WtPDEc2wAoH09E
cQNJKBp3JuP9KSH4MaKNhZx+rDOrHQDl
/jmaAStZbgpY2Z5ORVjXgj4XnV8C1swm
NQjK+xyq5E4I8pa3bdnAskVEV8EPEOgP
YGITax0E5YOgiJHSJTcBIvdqobUijYMq
tYqqxP+rtP8BUEsDBBQAAAAIAAAALl1J
py9GGQkAACQtAAAzAAAAYXJ0aWZhY3Rz
L3JwNF92NF9iNC92b2xhdGlsaXR5X3Jl
Z2ltZV9zZWNvbmRhcnkuY3N23ZpNcyM7
FYb3/BbNlKSjr7PkAjsKqoANK5djdyau
69jB9sww99fzHKk7TnETWFGUnZk4HbnT
br0+5/2Q/HQ87X45HlbPu8PXy3R233eH
7fG7mw7bl+PucHG7w+N0mg6baXU67ie3
3l+m02F92X2b3Gn6snue3GF1ns7n3fFw
do/r593+h9scD5fT+nxx5wtnni+7jZt4
fF5fJrfZrfZcnx9Puy9P7mV1Wn/n8em4
f3aHr/v9avrnZpq2a17x7Gx0NS66Ou9+
mdzD+jztd4eJa5zP7mU6babDZXWatl83
l53NYlof3MPxeDlzAy+rh/1x8/NqPx2+
XJ7eDJ+ml+myu/Rbvo6eeVl3fpk2q/PT
Oubym5Ddy4m7Pv1w/9jvfp5W34575rPf
XX6sxtz5G+a6tTP++off/flPv//tX/7u
Lt+Pn867LRfj/D7ZkLz7I3e9Prmfwur4
bTqtfvKu36v/7L1PuYTafKzVhypJ3ac+
nFWKr6VkbdKC1H5ySFkjD6mJhpICg9Jy
cuGzdxyIS4zYE6klTozJJ05njKurz9VX
34K2lF12ypeLPhavvropbcrDVB8fig8P
YZ22cbt+zGmS9abJw3aSB78t2032YUox
SYiPj5sWYmuP6xAk6/8er+3uFTGQ4rU1
txqKz764T+lzC0l9Y9oltyg6ffJ5QBYy
N+yBK/iUtBVDiAvwIwcvjmN/v7BduDbQ
9dZY5ZfNZSCYowSrtpBqBYS41JwdaxAf
IijrUnNUIVUq6gvPlY7LwLFprvrmyzGe
7w3MOIMZ3vasr9GXnENoMScrowEgfdxy
ENGiJeUaZwBDqbmlBowlVWEwFY29aTkI
M2DCGG+I1yLJixpgqdXCxXzTEqn0eoOI
Xbu2hFRqLamE0iiDPEMmmelSKLWm2Aty
YCa+CaUTSy0R+ms2qqV1wgNXLfWOYfug
a6vnXlIK9A63Ab/p5xDRCciQYmtgcGU9
EYmVWkzSPF1qRYfEGGK+uVj0btD722nC
K7yrrKrWiIDQSk41loXlwCxKKi0VCTzZ
T445Z9paKnIiMJ7BGP1gOUmJ2b8yG39F
NbdIVaLSpiNqDMl/iciN3h5g1yalUrI0
VZSy5rA0aYjSjP6lRdo3FxmQhYZnkeyl
NQMxaccSv1Icx/VO4fqgOYsAHf4j51Zq
i21RhBgRCetGbEnQMrpTmX/2dB7aQOUZ
UOCY35NUxtNdIfmenkpsgBLwIppK0fwK
nvJEg8cMOlmorZjNQBuQBlrZRrNQc6an
HCxtmgX2smIMisD0MVxHUKn4bW3Ize3h
1Rs1ftYKTEbvJRaqorP+p5nyEsDEgnYC
T0uzBVZzYtg7oTKldFLzoSzNyvH9gvZB
uwolVUo1UGjGhtcdAJYoUkPGAhdrzblf
oyrfPnNeyHRzt3BJ3utXxuOtg/m827qA
G/hPsVUxbLhZc8KQHqlgtnOWsgK8htcH
4RlA+tQ3nlMmT5ztPkR6P2NjAq8lM8dV
UTFvjV4Df9cT4rFpOO4EDaq13SBmr/qK
FCCjga6VhAOZhdRHEi0ymim7iO3SgRlm
pPKLZgxzyL4b49C71oO8C0vT3iNm73dt
4A6s4iokp+p1gQ8wNUGHNBXNWdOMX5Oc
WsPj4v8WC+y73SvRHF29N/zek9ZK5kz4
D+G+M3c79ynVgyQoucGbn2ij5vB2KGoo
OJbUYumx3RYKXhvVVGOAFjXjaXLDfjTp
STd5iEB5jwgd0MOtkNv7aTXD9zEpbkOD
8bX7VIhG3JTkCnkXsLqGrRTQVcup0cql
l2WI1SANsJ5/80XqR7fvF8KPzLFZNmFq
CAW8v7iVIFFJoCR9XAmFOKstXEfGhyk1
UJ59RQBI7Ac6YsQX76YIP06vMcFHSjaA
12zRbABJyE814xwUJ0KL9tGEk4G+gi2K
Dlfih92gc0FqXhU24iSbNKvmnGz5KdHO
toQacNmRV7k9rK7CmmOs0Bk3yjdFNVcY
8UGV4iBa2Dq6pgXbZj0L5dOglKeNtsF2
v2rYFu4WwPe7NZpdQ1NrK4pspOIWZ1wy
goGSaiyzyPbIK9Vsi5in7UXXF9c9lYpe
+PvC7j2FpRNLrZi3lOH26waO/U9GbwA3
WxUKT8gVwhQjZVn7ZoR2pRj6qq/iEAqJ
rRo2UGfnwFyIDcU4kwzB87eH2FVeKQQL
RpIpKYi7URChiKCkhkMDz3aVV9QAMaAm
Gz4t9MHSa+zXzVpeC+7+8PtoVZj4TrKi
yGz/Jsrrujo2uXlgxunltkCJHfTY4dKF
tWOYxlpnV1bLFTcOX99Y/i97rpJsbmq7
X7ZpuGTXhHVoJXgojgv4PCgOK0JXp1pi
jTI6VjrRAUOyd2rEiGjmLyafo63CwHSU
NF0O2REpxC4QbxKya89WfJXNMRAokiwL
nAF6S9rDBJZLwpJdtSAStq7ew5btK3QO
7DuwHMc7Bu2D8Bozzta6B94LlN3riknj
14rpTVZnuugqsTVroUSljRWnkUwpuap6
d+i9I6zRFpYq0xIz+THPwQyfSx3a3o3l
s7F7g+tvHqbCxEGBfnjfMGxIssQwVCF6
TZRvhSnBrdZqa8a201X5x2PGo9wiYFcb
XOHx6mHsMoh+YEZV0bBYE9BEZWcx4EyT
joQ5bkjuvFgcxzpT3yW8Y9A+6NFmCx+F
YrIPO1SdSy7hZU00Ndu2/+x8MXI5qOhY
tJMB9VhHwda41u4GvY9DKobNEjx2IkP9
2NtlGyIzt2gsB0row8Am23K4bSMG2w4b
lNaz11BS4tiAjChXMBjkCqQ3MZeeFpp9
cEASJuQG8Xrz4SWakLe/u9xIOJh3WD1k
HiX33IBGLAtL4NWMzpHeFvpycdR61VFb
4LhPyD4wuxozSUA0gY1J41JvNYQqtuVA
eQHlXJwUoJLviROmmX2XdXxozoyt49jf
FXr/pqGjsDTanmBJxE2Fi+Z6QyM8fIQo
5B5Sl4W3Fulki/IivWtlEFtvUI7nBg2Y
O6oS2bBgrz3xktHIGzBiaNmObxCzt/EU
TrPP2YBRSvPiOOpq+dQ+WFOIAnEWAiAw
EU0oAwU5Pqg5MhXZILslUt0naB+0acRj
2AcVikqju3RejjPwkAPbSBXkYN7+4iwV
k4fI06H0/pRuXfqPKu3/Wnb/AlBLAwQU
AAAACAAAAC5dLhloKfIgAQAroxkAJQAA
AGFydGlmYWN0cy9ycDRfdjRfYjJfcnYx
NS9zdW1tYXJ5Lmpzb27svVuTZcdxpfmu
XwGD2TyNSMX9Mj3zQJHsHo6NSI1ItY2s
rS0trkC2ClWlrAIpqa3/+3xrV6HOzrOr
AIgXkWNNSqKAzJNx9o5wX75WhLvHf/+L
zz77/OcP7c2b9fbhzXrz5vHVyzef/2+f
uWjDX7773aunxy8ejx/aZIp37398+nCw
9fjh3/30lz/90d/9+P98+MXP/+9/4Odv
n75exy/648v5+PILfvTf+Vf9oL1ZD6/b
y/Xi4c2XzcXErz5vye8dV7Kmuun6MKba
POJcYZmV/a6uGBPbGCHHHZLPxoey2mip
Tf3r53/5bvD16/bi6/aWh3sYr+a6fcO7
L+cT7ent427j7Zu/enodjg/91fs/Wj98
/S96FmNSWD731bsxzezuYjSpuGVGjNXM
FLZNs5Tu5p61hWB8r6sVV0Pa24b3z3L5
rl+751/HD95/o1tjR7uX9yvl6FupvPVO
hW+rzo48U84pV1OKn54nmLWEVdy0K+Y2
P7z9p7/xn9f4+sP7+eaadfyZt9GWshh6
DO9T5ZVCrSnkYVtwNu8SzEh9axpccJv3
da7V7/q2/bTWv37zZSO50OxupsTRyoqh
9G1i7I53XDv0VfNcvpuednat8JKNBzN7
zGX23v67vuwr/uvFm/dfVlrowYZqOm+X
eo591I3J2JVGmrblNVfsbQ7nrJmjxbqi
7dWz3rxbSJ/8Mv/uy9oXXzytL46l8++/
su6wmT63VzBOL8ia5dGcxXJ9CbP11m3Z
xZWYkxl95Jqx4VF9nq26TxvL+69cX71+
+y8Pv8GJXv3mm9cMbsQ+R0572eBZkWk2
Zm5zGzbmGYZxTsPbtsoy3Xs77ajJOONj
Ci1+53d+MNBv3jIGa3tLMfCdPZQ9TS+M
ZWvIdfbt6si+B2NWra2ZtljhEJj46Udc
xn3nNz4zmTbyKian2FmWnrtfDp/wsaeM
ve/VXe2RNZsJlMitth35RdmORU5+te/6
sseXez2tl+Ob78PwN1+RVkn4XXJ5jeXM
anWMFUcepcS2MrOJWwABwQcfw5xuhWL2
qvO7vu+ZiQIue4/uedJpRp1gSe7OG7e7
N2P3XIAzm4udHlPqrrfSTHfBp9b7KOuT
XxYuJhq+WTw/Fv7gml+W+ZzDR4zFm25t
mcntbQJzi4MCs2O3WXBFkK1sAMDabb7r
K2/mEj4YKFZf4gIMcTMGxiptyq744pKV
+ZfQNp7AsrlcnKY1uuTiqiWH+N3f+AzP
nOPrzA7DldZYkLSCqyVhN/xTcME6cC0U
rLYlmxpgwHT6HBz+EEz9NMSEj+GZtYH1
2sXIIqufHYscQE5K21s/6uIFemuAq8t4
5M4G25reMZ+Oh/quL/uKiXx6bC8e/3U9
vG1PX6y3H8BtpF2JgHZOP2uNfTU7Wo4h
pnG82ww94fRzmJ59mLsP54mi3boUxzqB
25un8VdfzTcpmr/6ar19ehzffEVzKVgf
iC2hmdnBylkCvj1SJEz4YYczLW6/Rs74
B+jqmjPFLMI2Xxo/+hVPr93F5aLLITFj
c/SdfNhl8NydMDFcwqVzXIS6nTLfMHLB
UsHtEXydNtfIO33yi/7pxeM/rodX/b+t
8fbx1x8inks+GTtwLU8kNc774Tfm7rcx
tmCjvGuIrla+txLsayYulm5mYaJL/Pz4
tv/xDcc4rA+KcaYVr57meuIH/+X9D/iR
jX/1+unxq/b0Lx8e991Px6uX+/Hpq4Om
nH/10c/fffz9L/7rbQLecTExnacpRlY+
/Ortl0+rzTcPr9fTwwFC4mvPXuWdaZ94
WPGGINxnHRAw3j5nXwfO7OB/M9TWCTcL
qIhmxhkWMGiESmUP70Iq32Dv51/CHP+V
p/rq8SWOehDI+P5XT+vFEgW8fWnuEd4R
c20pNUYBkExNaTpXHGhgF4aHCbu61+4F
BgHjSvCSabfz0X740jev13jcj+Md/7uN
bxouaRoQZy3eOLDtsX3Bl4agx0WidYeJ
8IV+QXbc8mkNCFkujVD3TYz+/J0/Pvzj
Oozq6dcPNt796qv28nGvN2/PzHaFYRde
aMKI2421HExhLubPREt0DAvoAxNrByv2
4UuFqF4Mbu1tuvuGe+IMFsEBLavWQo2a
x1VDj2I/4K51/Ks4YjaZL22mitWO2lNf
mwgz+93wT+vNqxdf389fhJvD1QqQ5nyZ
kCrCP6wqErIsZBLGDgnAp2Ct1hOD5zZQ
ntpLzs7tb77jHYvRiN/Y+V+8N8XPR3v9
+La9ePjiFb/e7cWbd+JhvPrqdXtq/fHF
IyyI/yd0bC8++NkH9/sWGfP+9x+TMu9/
dZUzxy8kVF48vlwPB6jwS/NDWzA8w6tn
+BuR+vbZV6/evnn71F4/9Bevxj8+vFgv
v3j7JX8TP/KRp/V6vX18+/4rK//58KHx
+PDl4xdfHl9mCE+ETugGXCgofJ4/9uKY
SX0KfqV451K2FS+0t68EOfjCN2815X9t
H179GiD461N0nV89fNnGw4v2xZtnj/r+
568f3v7m1cObR4z13VfFWIrzWHJOEBJf
7v/gzVtc783bx6FZ/mG1tkY+zoSZYLP/
8Gkc5FFx7v3zW6KytxYDrfByF26rtv4Z
c+fbn62AMyguUT/DO5+WYLevHl8cnvni
1RcPT4/zi8VDPf3T7XW/+M3xlPOxffHy
lR7zYW4Zyrd84PW7Z3Q1AxK11JT5XpPi
t/zJeRLy8ac8MBEFSl79+e/0wa818Z//
6Jf/8Dd/+6tf/OpnP374yc9+9J9+/otf
8o+353798OWrF1+9e5Jqy+nnT+29EYR4
s6F3MfBpza/H4cjA/1gv3x4fLJLRQKw3
ALsJt7FuD/PjX/zN3/79r376k2dB79/f
0T61zH8YT5PnYIexJDDNWOc+4WreR2IP
S4rPGW0LfNzX3Htfs7+9ryFEAgELos5T
w+PCtzmb+yGMrwADQBNhILtP+pqFAcWa
jp0FFEb5dl/D3HBeE1Ak2dh/H1dzNtYS
+WYczdicvrerJVtqSK5URnDxd/Y0Fz7q
aazL9/K05Fyu4jGsXy7V/ul6Ws4Rcy6+
etD9BC9/GE8L1riUCuxfWJ/SJxxNO4GQ
b9azEtlOdvr7jmnYSo32cGY4TQj52/0s
OFhVyC6VTFRL4VOORrRIyVYkL3ObzkH5
o44WYySgEjHh1gT7j3kaU/j2i/7V+z/7
fXgaJgMKlKAtK6L69w9qVq/F3zmCdUi/
s6cx6eb8n/AxvwPXvoff2R/aVCuLY0XU
CNp/um73qSX/AwU4WBtTgz6w5ggAV7/7
wWG3rAZ+mXPhrWP9BJf83eNbxj0qc+QM
a0VY/Tav48EcisblnEMsNoX4Ma/7wTsu
jOQhZMHQIczJfqff5cTcg3ouwc/Kv4ff
VUK094TWBPqZ04J9u9eJfjNpQfENUgQI
/c6OV8xHQ9yzH3/a1ZhvCxISNlLWnijy
43vQyb94v3dxSLsX6y3rcdrBOJ/VfO4I
wT+w5geu/PC/vTl2Wz4v07aRfIth9N5n
BmWbKaXslnWIEKerYwQ4mvbC00wVB+1r
wiBXLX5+s3y3oeuHoe1Ou5Xecg+Ewtrq
2M6u0roZyaQGRE4d/2hPtvhg0bx2JeQA
FC2G1tv90N58GHqitLNJBIy0B0bMtE3t
soaJwt+7pDWNHRMOWvpwafnd+D8bG7TL
ze4vQ9sPQ2c7eDk38qhheD82yj5i1hYp
bkND8XtTweqe53R8Y3TM3kZymdain/H5
0PYH5jb0JBqu0csaPk47mPhhDo/dZrmZ
Cq5mR3M7+WQ7c5Pa2mF5vrqvjptehg4f
hk65557aqH6n2NqIu45CuLd5rr2Y4xR2
bSv7WG2323cbmS9ju9mj1uUvQ8cPQ/vY
t+GZi93DYpclMdV8W9u7jx1as1iLWyb5
sXi9wdvEkbbtpTHruV6GTh+GDq12r40Z
N2AnbvrlKuGzBYupdXgN5jCTGcuxFHzp
6osVmbGwzDOm69D5w9CjtlRdh+fn3XQ0
oG1bn5oxkzXdS9s4mOeexTAeppxH06Ej
FIRffdgPvQ19c5kRx4JLF1vGQIkC/nNi
2zXkDdjbgghoeIbtYfI2dfIPGMZsjnnH
yS4WYm8WgoIoOyZoUIt5RBau+VUlMFqf
kANCtsO+h7aNVkUxQbV4rwmFmo2FvQzt
bsu4Uyi9p41n5DjaDsb0sHb1C96oA4oF
BfMjrNmyAbkxw4yxL29nyN1dhvYfhtZ5
8ujLIOknhJcAvgsvumPbLKtZtS85ubbC
mbLF/yXsnImOw5fZ+mXom10fp48tdG22
GWt6XClgXtvIGpp1lvkKC3gddiJxpsVO
gc80csRvx3VCbnZdY9Dft+J8c27XFR3G
gJ3hhKyASXi/0SlLiZ3/khd1y3p3Rq82
tcvQJ1Ado+ukMKxQZ59mzr1G9rNvjxjO
BaductNmtHVuG0wCjuXDmKvOafdl6Hqy
ED4Hz5m7WURGd9Fo25MpnRHzxVcKsNR6
Ye0EK4ZX8GGGkvrsOa77od0NVI2PTKUR
i3LdLrdQ5Us5A/wDFhD3BqHLBJtsWKxu
YSlGz9K/HT8fl6Fvds1yVJMwilq20ak3
UQdWyrtPA1y7yDflwmOnZd2AWsXAtLE8
M0V+fnF0d7PrFfJ0WNHQuaXp1sFWDG60
dj5MAwwMzizcCGEy2hrV7JxmntLEENvL
0DcLmQtbXYzZfNM01txa2tsZUJ8osdZy
e+r4uQa3GuuJsYI2jfkpxpTLMrob8kEX
CRna1x3REVm3A63ABzDPZNay4Ne55TzQ
zHWyMg1LtL1G02G4MVyGviGfNZMBiaL8
LWsZpnd121o7jMzsOFPbZW7vdey67NCm
P0+fDcEIUMsXeDpF9I2fVd+IKnkSvg1/
BsYTS9L22YGq3gKBBMbUAjhtYFeAebE1
9IY33LmM+4G5LSNmgMtFfI9QGxwLdyQB
pDgZZ3rMMYYKSJtOYMvL6dzExI2r5uah
KJehb/C0eE6kL5pxBdYwd74JhoA5YyPA
Yg/8WHt4JjSd9u+9aymm67w2hrwuQ9/g
yZQ5Z1zEXcK235gusLaTYbzOovmFGfMS
OArMofOVvjeeA/4wjiOyy9A34wPCJiwo
AmyEXzxsZGw7FqzRWL89syI5T3wYpRdo
RO8sSQfH4G2+78vQ6bSMFlo2CWAlEBfD
Fi/rCZTlHfY2Zmtikhs+uMY/1zyJ5dCg
BpNivMvQJ843F/wG9QGrKPYdtoEqeEgb
Ov6c2iidjNWBkd5dg0W2MN1WTsAol2W0
N3jaRpsXGnnC+AzxbyUiWfRVHNU0nLwQ
JgGMVuCbMEwivYVtWjAKInUZ2p7wejIg
jEUB22DfwVhQM3aef1RYJvwSjIG7LrAw
sSCmWaBlY+i87LwMfbNr6+DODopTzewW
z3UKggH+IVMv2MkIhHqdoxN8QrVF+yAj
4PtJZ0GXoW92HQyMPMyF3boJSCjHaJg1
dsm5EjOhIwMLBvEUnJdWD0riFH4n+HN9
6pOFEFtzS9O1CMOrBD18ZnUhhe+ZZfTE
MIs1lD1GxOJLskB6W4hiB7m6DH2DJ6aQ
ULqjZZRF0IKRM25LrUmJOIgs46YFiSD0
EHrbchnHksZDjiRzGfoUdpOtCV2LixVG
5Y/xvhSgqBPuvsUucyjECwt+S9iGFgf4
UnLaxKerhZyQD/HbqvJZphI4xKKhYQO1
EYDSHmCaFkKijA/nAqEYYlPBVL4X6rMu
3ngKu1CxXoZfOWS0iROIJvkDWrbGLDAC
AZhgLJvnJhpCsiDbUO1IRPcXC3H+ZNc5
QQ0Ip0ASCDKAI4LWKNvLTcaC9FkwYMRS
EDmVkGkSBgJ6W3DLX4Y+ETNWuvexIhF7
dwNOxF2OJB9oziAGY/G9zrSLployO0y4
r/TIEve8DH0yvk7UD93vw8Bdw+cTzA+A
DWi93YHP2SqY3bNDkW7lYOEuhHSrQ9Pr
XN+ML4Gi3YOruaCuOwSHf8+17waOF5iZ
V5ZCbitgLoXAbuziFxuP72vFeD/0SZJ2
8HmDHqw9ZCkcocqWXEE1oi1xuxUMcayF
N7ptvQ4LJrwPtekDH74MfYOnWJCW4k8A
PYx/QvoWfiGAQrDjfiOjYFolGJkcAqhl
iMYo+RBQOeO8jBE9eg67fg5jfAO0UbYs
PiQEloAsxS0XXoTHKYSnIS44qgB7gIKZ
UFGxpnQZ+mZ8s0lKpInAm2Ce3NtjYYR0
SKolPmRr8Cj4XkVueL6uOxk1vJNIZ+Zl
6JuFuOa92JPFSpUoaomMtjitktduA79H
+CGYgLom4gdEwkGwywAHfOYy74bOpyjD
RMwe85rATmgsUlAuwIR2O/wDFCh5ts03
4v1oYcIi6iiKE2PX8TL0DZ4mS8K3Lxgi
DK1hdosYNZFaMGnJ8JSYm7nB5gTPVkzr
HupgtBPBa90PfYqNTts4MDokC0Q0R/gz
awqZKUZpqLB/x2Pu2o12LHJxWdoHFWtT
hAZdLOQs7jBbuKrsDQ61rfY6YlCGEgiR
o8yM+FWZa0KN2Uhjp+QmOEQGVMtlrk/i
ztY2I8jeJA0kn1MdIH+H83logihOhIeM
Hhp4qIUjXBKJoH4JGCuXoW/syTDLuyjv
DsoL6SIi7gxfCqj76vmdB+W68qMgpi0K
81rNDqcUXrt6GfpmfA07Yw6Ywh2JdzMT
+iKDoK74ifYtmshMHRM3JNLj6nZj8Bki
tfy4TsjJ+GBlxNGobRBBXQY14JdH3lke
ezj4DRQZE4wokw6COWX19sbkN5b5fuiT
AmNpCjoCd6/GAZ5MiyaW8O4bGsb1Egj6
ShLmUwOHgfHEheNHDD87dxn6hiEJyeqH
eB2g1oLZNoCzs27lx0ogEGKSeAOYknpu
doZI3G+sEfQ2Xoe+GZ9eC2a9Z2FdYHK2
j1FdMErBQ4K7QMSfnvnSj1EeHoBy0GUi
I3E+j8vQpx2zEfC0OkS+NvE7+oHfQ5rE
T4eyGQGYYfDMgt5v3kQEq2Ss5I97tvHJ
0O6sGz0MqHRUEFKlo40a0wKjNoKLSijm
7ecKSgS0oBa+CSBmnWumRKRe4TL0zfjM
bLwthNIF7bSNXgh4NXrchlBWiGFVGdGr
CUOISDwuyBT9dO7YCL0MfTM+OK20q7TD
mlhF1Rk1foJDDGI9QR4Ln95A5e3YXWKH
lS7apOa/rbkMfSJmxO4E+/S9Ku929aqE
fAQqEg/+mAE5bRRF5D5iKVsIVVvRGWMH
vunvhvbnKAOuZywfXOpEJSKZGfB18LAH
KWvbtLeHDCbMm+j0flb7mAOrAdNGuAx9
gicIbYTgIGgFbIUpJDbOYxcHYMYzEEgw
eKihArE8HJ9qHqrCavh+GfpmIW6KJrXk
iGNr+Nbshr+7ClPIh16QtEUtAgdg9lZC
WIP21ChrJ0pchr5ZyII1taVdj7wylgtQ
pGC1AcQcQ2SKx0Na81B7CGtX/vdgfJ3C
RqTadeibhTgUNLaLIY9k9qzwdmC+NTRM
xpMbdoJiwJAxD+0Z4ZwWG4UQiSjPcT/0
KYB17ConkCTOTKANzmNlMjTmFlrNlKYN
M/XLSj/j6C6OMPHOopW+TshJ3IHqe7fC
WC4meDnMgVir/HecR0i1YWOresh7ZsLX
cHNU7ytsB8Lq92XoE/IxX7Uh81dxJunM
0bkJdE7MAzIfWjlSLqLD/jwMFf8kBCH2
Om/yfMfs3dCnsBv5e0ILBlvCCMgZgiQM
TyyjuA4/hcaniCZnNU1sS8m/hFsPGLtw
Txb8Oex2wvKCI1QtTNHubkSNB+g6z1Q3
mG0YNzIfCUSYEqKL1SVqNAJBv3jjKYBZ
HT9AkvbMrFTLETbV/Zr81UIvQoiPwp4t
/K46TJ4RZtvhzFgTNOAy9Ik9IU+qKl+m
Z2LERrcruRO0t44HrM5UDCEHhLKwm5oI
OqAWES5VU/d1rm/ibqiapnl0mLRgNw2W
YEGSVpFNZWrfPHftseiohyiUtoQ8lCqD
Btlenvok7gB6VBBBdduIngA2mR4o9ODJ
Fs4+ffB+23SYdcSvILQwH6UfEhHGZa5P
ER35idGCNAQQhgigCTgN3dDOGcuAsfHu
kOoSQBqwYKFzkuRk2TnnCzydxZ1vOSC8
tu0RDo+V1Lp4Mkhwg3LYjT2iAvB+1LNI
SANsWJLp0aks6mXoG/KJ7oERTOdQUgHk
OvJcDShZvItdHaTaHcgFGjeyzsKREZGo
414QxPYy9Ck2Gthb6UNHW145ZXWWEVpH
qPDvHd+BjRHaZhuYHdpI27UGeSDvtz5f
hr7ZdXXJLsI+cw5FgnmNHNaRZA1uEsHR
UM6OCNmrEeLDRBG7Omuu06RbzdVt6HNs
dBESsJAQxRKzIm6AgssmbBSuMXhpyWFq
v5/fKDQIgbEiIihwfhn6pBuJ2SjiBXTC
9vYmYqUYkZvDQ42nkLs2VR+YdmRxz8Ja
a3PZw+FanXdzHc5Hmf3YftP+JrRSVV9o
X+DNIwgmkGJiHHB6GBban89sBeBBMLYj
eADKX4a+gSrvZKKHd1hCI0CqobU160oZ
QKuxDd66J+y127UWwNLNxBV9QdWDLpeh
b6BKmEUezx3ExJiXIemrY+JMPCuqeTJr
LMm/HmYm4hurTFOkDx+E0F2GzqcoMyf0
HiKN4AKicPBBODV4p9G78BVEw0HAqaXr
nBr5X3k77QbAevpl6JuFYB6p78xD5GCW
KvA6swxOExKAcjyywYKhNJgGz4uwBla0
ubVChsPVy9A35DMDgYhlI+o6k2zxeVxw
EVJGJ6hg6Tw+nJ6lZIq7AXndWMqJF93M
l2U8RXQkxrGPgm3hPSASq8l3odchCqIn
UFOoY4loMYeAL8T3rROLpmzReDG+U0S3
Q1tfyFffE8+qrQ43eWEkgjYOYZhJcSxp
MdxUwMHhJ+wHNpme7/O9G/qGfKg21BDU
APzXXm2zXSeNypsFnFUfG6OKJdyh0nAU
t47qp6z8k9UvxneSpBvzzcoSJoKgDraH
lPcQprIaIxwVua593w7NmVsJjsHX5BrE
wfK5dn3qG/KBFkK4EVAXBNiy8PgMtHlt
dwhEYdnWJIfa6Rj0NtjfRLoKXUD36zLe
7BrQhQAg6+Ezvi5NqvUEP53KToc3qpp3
A+G99FK25mrkpK2uY3P/4jKnAAYt0kRX
5BG8bkZYPGaOM6PG/c4WrgZZVWmUKK0z
gqaKg/qKNivr4jInSardpaz8w2SIW0eK
yGyH+CQQGmKBgUNZEEr1nsT8NtNxXkxU
yDuFy1yfJCmGUGUaUHQDMd1HltO20Fao
u0pARNAk2CElCQPx28FS5o7IKjGsy9An
42vDqZ4QxZgE+MgtQKRm8X4lPeo03S1T
Og9qKw7DCwYVpTQiRPPjMvRJcChwoZB4
eaKvhRFU/tZMQnDB+cFplo1X6mBKhfP5
Jn0TRVAay3p96hs8KUzN5dEliJ7mxjS8
pWkIoUPuJWsJOIBnxlISfAhlgNLDvXRe
Vs3F0c/njQldlepiveJKHYXkbYT8LR2F
ItIJX5Zg2N1RnwPny5o86KRyvCGV90Of
tmvLUkFcFnv3zfBvABI0AwXTVKyk6oSq
rWu0ETCm88ZlzNBhmBhPvAPVeI6NWdRR
ZZNmJFVsouXKcMyF1TYriiARFBYLQniY
KGdsPSvHoUDwq6v9MrQ7wRO0Udt2KojG
c/uAYRxH3cQWFg/KC5T4ECDwsxNZZtgq
LwTEYgeDL0PfLEQCjYct2LQ1ShiK2syK
FRhKRzKRhAHBC0QJRxWZqqBL80j4qVOi
y9A3eCLgy/7jEErDdgIqueH7RrlmsEdH
LISKEMx5HYxOkiFiOfOYxHJ96hs8dYEw
j5aHchwM39EP/FtRUCjeSiQemIYO/hF3
RaIa1Q69ZWKyvQx9s+u2cbq+ujJ9iH6o
opRgkKpSknVoE9HiQk6MlbUg8myv8ku5
mNvBXYY+JXGAMygwlAaqgyfFsHREvxoW
AVvQaZ3le4eWFjq9lC0WwZgBWTDpnpjF
syTFhLGhmIR1jj/Bb1hHBLXZyq7GKYyZ
C+KAXpgdbQOZylPWMxP43i5D35CPFZ+s
BmzGLF53bUJUU3oLX9SJ4EFsQbuhkBDl
qGifxFToLbjIMufL0Cfk40UnNEFJbyxm
Rc4pJyJIdjLlxuqQBy1KzOL3SyVcUTvk
gC2yOK7L0Ge7xqLcIPjBRLIwvunocvox
K7Ra0AUiZgSUZT3EIGyMoBh0MOx9j3zx
HHa9NoCczia7kjYgwEH7q535hPY3l2pS
MS/en/PGInSAjQyD+A24bLpYyEmSroh9
pCXtGS3C0QjSeH2IMKyaEIV3Qx0j3Bic
VZ0tIA5dntDY4l26H/okSQHJjkweSfXG
4WCs6E/lq8WxIL8QHR15eRwEWFoOZta1
Z2SEUlC1y9A35Auaw2j7YOIwQMiMuD/x
G72fYWqspNG2iEpqtG+Zsmqr7eLThSBy
wZBTRAfuxREYhRAFE8HqIu4ggbqmJGkp
xg4dKPcOvGKeSFarwwmLlKmXuT5FdIQM
qnk6YrdzOicyBHWbI7gXNzQhTzMQOsAe
MlElnkzzgo7roDC5ey0Tz7oxI+a7cr0y
jEbScxsMPXsd1LC+QLLCZFXKLTzNO+2u
Kg+38UojmEsoOIVdpm4TTJEcwaNqHBOo
bJhNXFO26hbv5q1EIgwvFmV1oNk4UrHq
vDj6KewaAJ95bnD9gEHkDXIPpYHlArtT
N5HqayE+FjDDEAycRz+x4mBNs/mC16ew
C/2NR3MPIpUrRl1oWNQpjgFOE73dFLsc
yrFquI9yXYK2SmOBPoW7sJvOsXER+KW0
zObJalYcJwoYAhR4TxCHay7QhYnOOieO
R86C89NPWMBo/TL06RxdSblZmUimodhG
0XYTgXBmZRIl7wV3vAL+pAzprTkmxJum
cwN7r2XSeZN5imegICIMl6+pOo3oYcg1
dMgKZWRSPF8SQF+o5jFpwze7PAhyv8mc
nm0yOzcbfG5gcYgY5qYB9MwHYb5FLRho
NyQuleWs8rtWUDF8Q5wTYL0MfUI+TxyF
81bFUykfw9PwBcg9OAJBZw+EU9KBfdA2
XJ6YHv8wQO112WRO59iIsNSWkxoKBCh5
kST1xOIojCAIMEtwBG3G92axFHWE6Opu
4rfdiL37oU+StI4sxhgg7h2oIJIEhk7e
LYVh5WkhoVFxMWbsJii65b14A9zfE4Av
Q5+0DAtWvctbeagD2x4EF14AbeRXRu97
402ZyqlPeU+XHbatdMu+oSP3m8zpWeIu
70q8YJDihxK6E+Q3WXQ48g2FzcrCO4pO
wrNkNswsBN+U4dw0LZeh/QmeQspHdxms
rirxMikFRAllYCx2TRhQvORDe0wMpmLc
SRIsJeUMXoa+WUjcR1r/VIOZoQlUuwkC
IeRInTVgxgmWjUwIcFl44VTWoG0IQPCP
QHcZ+nRKunhbPmhTtUATnFI0J2u/BRXg
4eyslXRoUHSHwXd+yMQ7nYgDxZehT5yv
6Mg8IPCA4y0v9rYhU/rCcCARSl7dW1V2
bUKglg5SJzoXzpzq5Ww3ncNuxkNKbyDZ
xs3a0OEq0QtLgJLljjlrLxFPdO9Sy/D3
rAZVQ6eRq12Hvi2jba0BfQQNi28o34K4
pIMDBctZNV/gX8K0o5DMSv8RehDwEvE+
XYY+ETPtqYyo/gZHlgYccBnt0kZftXlm
DeMroz7Ayo6zpALZNzBtwgRfchn6Bk9Q
gxQdqI2krXCQlqTiCuFKKkiNG3D8hFFq
6Sywh97rUNutFERk62XoU36I9mOIu0Cm
gjc4CjT5MYx6azQfUcxQsSViOpSnNS2K
aQ0DfgzovrsMfYroJSqplYFaT9oI1xZW
RW1FiMgWVMExxUCAj60mX4Y1gcYh7SXX
Lt54io1YLorKz13L3JK9DZq6tIO6m8tQ
Gk2xXS4U1swoV53pA7CqFGy6Z0/5LEnV
18ojvPKW3kzAWlD0tgmCVrVOWWkoBOUA
kgS1AAF3F0/ekSBYymXoGzzBbAnnTO9U
cckmRMUOpG3i2UjK99YBRG4smNpowZKX
RDZiPszi8lqXoU/ErKitWtvxSKEwGFqK
Cn1QPRS5MtuYXOBkzsY7wRUcIYm4ab06
hIR+Gfq0rVUE//CnGUE2JZWpZGMpY5dY
hjIfFYU+9HKTZ4j+yHCrhKExtQl/GfqG
ITbosaCTKLephh4M1uJcjuVSAQ3zqoxg
wuQg1AWoWoSyYaQWVWbuo0w+x0a4hY69
skGyGUA6A1LBl4rlbi9H35B6plZbfTpU
QL4ixYzIiMeVyv3Q5+3aqfq3gkwOY+es
jbAkhqOGSHhoJIgr1mARQMxUqikrKL5K
kGB17WXom/Epp9OgV5KqFUpruYCDDK3d
YISiLXZh88hynVMT1jK/bBFRjCwpF7Wb
z7qR2MKbxzot8hmMd8koi3ZVlQHyRUGl
Ca0qQ0YVOj7XjUHiPPgDtPbiMuft2hYI
jEfO1krKex+H/mQchJEO4rFCSQWr3F2A
hhjQZD128xD2fncyP4uNzsDtK4+Dq4cM
nYR/ZBX7IO+V8rBArpJVaCeVgN+WwYuM
5dMY6/5sN59jo7bfwiCQVp4KtaiWYNng
Pphuxes6ZKd76X4i1o7CGghc7SBuyus+
YSY/O4DVKe7MyShjBkmhegdYTkAPYb1E
CVy1KL1oK23cAl6ZALo3ggr6d59Slc+6
kXDB53WySqQbzvLmYpJjI9PNUHrIsWux
1Yorqm2kEkP1EOAf0BUvQ5/yr6HQkWii
zl6AZz0qe4yqwKzIJFRDyTlFfSZVUtgb
BkVgR6AlbT/Py9CnsMv6IQoAkqyeQEqm
FaIcW0TIfatq/E6cVDKrUbqgDmnt1nGp
l6tehr7Z9c4waN69qTVKXkRtXgBHHkQe
JoCILhKsdB2iGbKXWAHSwFQtcjO0C16f
i1pCg99blTyhsmCjVQeN0uD4IV9Xlg5q
VFxUAkoD3yYITQQ3SgnXvC7jzUIqH4Zw
iUwvXB1B61GpxL0dmFlYo3Posyz9T9gi
xkPgMdKCkkZMzOuEnPZDkGzaU3aAEpHW
qUcS6BMwd9YQURDsUG8xABAKt7XnBddE
VwaW2l6X8RR2Ib74uivaIBdrLhY3t8zG
LgVpZlaB/zG2K2AXusngK+I32UCdy/15
Yz4fwGIabRB54dBNUVbbFsSugX2rJ5rq
zaJd6kCnoxNWmNExvwkrwUvvd8zKOaIH
HboaE9V0sANSiF9AGvKgUovmK4wJ4Qvv
w4cQIcJsQKR60M/Pdp9YUJ4VtSzEIeu/
+CAKDKViCcCtWtOCUh0LktTAAE0CnfDG
pbapCVal3dvh2mXoc94TUUN1dPBr5Exs
ai1otG2qFOaVlro3aucsyA9tloLm0SMo
iMxO4TL0DVTByz1DTcQvFDIktYGebcG5
0aYANjR7YWMLXjlVbVwEH0oqHRWbvT8U
LGeyoModpy1PUb6mHEQxmVxQGjI39R1A
clU1CFTeglFxKjQZnohQ2fE61ydQLTon
BYmqjhiiPYqqClYQiYAlMkd4P3ZNyJBT
wsZVlQKzl8CZ+TLX57wndZ6FITYdLaj7
DwTNGQ8mEdmO9FfsGhrILFgiedsgHpiq
XCbA/T4zrjyTpDCCmfBh7X9jFZjEICIu
PC0qkRYEB6EqM1D70o6FqI5JavipoJwu
Q99AtR/g0V1rw1sidWDQqMNSmGusREf5
e3M6UGqMh1Kveeh00ugV9r4MfbNreMUc
k7dfjFEIM1GlIXgd8RypKHos6LDKF4ZL
1N1H0Dk7EDV13nEZ+mbXXbsdOtfd2mlO
OLuyaFXvdXRs0/GcEvR9L14tOpvyZfFX
VaCpkdzF0U9h1/cBVfZKgQsOPxlNjLSU
fFTsM+lbncO0kYtEOM5i9ZbWHEk26X6D
qJw3mUUMddpPHAdNQlPFAJxrdyXEGGF4
V28Lp6VsSwmgE9ZSiPIbIXy/1VLOavdI
MlSxMnFW+VQJCqmDKaLZcay41dZzqATd
eSgwvsrUqOenmpr2+9OkciYLtiJKtEUF
wfauduAKATeHP/YeJ5ii5NScxloS50dD
yoTkSRA4YPw69OnwJCrzCxGGWFE5lvoP
Nxij8q4768m/FmWcqQEp1B3o88o33YG4
h1y9eOMp7HZV8zBSz2r9F7yyUPD1rFOO
nmLTyY46AVZArzqdEiAeIJOpOqjZfd5T
Oatd6B7j8aeCqE1MBNFErHH7CSkh/gWz
gHCrCk10CRQO8DYpgYigw/WpT3QSMEIN
RGxJe0JMANZGtLWSkF51cfCbUvEOn7Qa
ypoEbHdSJ8h+v4VYzmQhqL/BHKrCIj6D
22Ni5SbooFW9XglvCGCvBgCrq3/jVDKE
6sWcVS3uZeh6AtUMsjMPxnR0NFiVs9KM
ib2zHVvwhi+zQEEzolAWQ9Rhulc3MiL2
86HrWe3KA72G7Ju/hhRpHw8VpAO76LXl
p2SJOlAZ/LCJL6P0YN+QQISIvQx9znWf
wgQwjoWHj4ajDzpe2bTt1FIJ3SdonjLS
QWi1bS/Wq3Ycjpv7degb8jXDR/CT4XkI
SJ8yBxMeg9oIymGAKTl0KbxjdJ2kQonV
qzPpEH+A35ehT7nuA5jUZlafboYMIReL
2gVEJEC2EdRfoJqpcmYdIuNaIG/RGRUR
bvfL0DcLWXlphxvTbhB2lPJirpWribWI
OAAVA1SBHwQnlwKnJfvRHFA52/xl6JuF
BG0yddW2K8QgJhbGEXrtqKIIgHYWtGad
KengRwUjDVQCFRwqLNzjdX2WyUwYn9pH
HyoOBnnUtBsmpvpR50IC7DDrxZdv1aPY
oe5xW0XOKCde7TL0DfmMy8pYqcoodom/
SAL7DjtQ2vSufXiJSKUPqhKK4FCnumyj
i51OqC9Dn5BPBdFpantluFBz3/toyYL4
hVCmUXNQsrB6XajPTkxqXML3qVyQx74Y
3yk2Es03S9NmW8pRhfzto0fIKtmq+3JY
mcedTaXHKD0ijg6wEiSUmB/uE2bqs1Ic
9Ik6XValP2FkGb7rGLYoFXMl42FriCdY
aiayzBJn6ERR5S6g5a8ucxLSU+qIANmC
SvjEx0M9TipDYdJVxVJVRqgO406l7/HI
AZKKGQi2e35dzxF9RAeN8Qu2P9ED3gbY
E9Df4Xb22ERsSsfGnTBJ8QqMeqsaODDR
1c7L0Ce71uPx+URIRGA1Xf9gM/4m7Yne
UPVyRp4SZPz2R05YUpbtwLJgxhdvPMfG
FrANwDl2NR4ZKjyG10FYm1fXHVVGhNwc
7D4uyKWyFeEVKFYdNt2nntSzkIbfbOAf
ORGQt3BU5l3NfIe2NZekY+vWl6PNDype
VS4worwIHIUgXC5Dnwv0lZpva7UWoJsA
ZnBGbR9USZHUtaaJasNVACQ0pbI/K08M
tCB17hVYPUd0LE0NSF0Hqp1OR4xaGyeY
6BxQeBbUWJ13K+dOHZLbUAKMCrxZh3Bf
hFjPEV1J+eACzDRmEX5tph9QWjthUDml
CFwV49doifLw1Fy3lbVrEypdjO8cG1tj
mfA0/kMQhKGlDBXzPTMHfAUqzAXxoN0t
7AQUaaMq1QVn0Dnl/dAnIZ2XR03hAIVZ
B/R4NglUTBazRLh3ojAyIMehfuS+sSAD
cpnVeDra9DzsWnNWuwq0gAauIMqCXyhD
DQ69rBgZD6YWVkjy6ar3wj9iPJpg8jmX
xl3Jgoa+2fWuTV1O4HuEQO3yYhGgZseD
ICEsmXorBlfSdqrKU3UOzum3S8BJbuky
9Kk2qXuV8ahjPjrRiK9LahnVTg3lpqZk
dZg+PaKUh4fTxw1jLrvwX2Vehj4XIY6q
7gpKyNR4ENyB2AgBYZ0IrxY7Dkq2Ylpg
I4B/VekG8izGnO8SZjT0Kdc9WiThHqqD
wDRgjDrK9BWiy+PJMdD724tZ7apm7km4
BTUpW0rbXYY+cT4HS5qKpEfNmJo5mFIg
0Y7/h1y3R/oZLp71P1PXYURFYMyjEN/C
ZehTSlVtTn2AeIxeN8yfv0dZ7cnjIyJD
QSeKSC450URkJsBgYRsIBl/8ZRlPEV3q
p3ilZirlOEQlEOXttIU4UsUziYK+BR1H
QKoR75jJlqiXI7W7fD4NfbMQgsyRdEpg
AjWOrQgMAsRQgbjpDnDDRVQoFwIy2HQM
JioNGWa7souXoU+cD/zAfA1A7WEdy/Wo
HCqYY0vqQNKRBdCShCBV1QRROaK3dUyw
Zkj5OvQN+dRGDZkI1OuUo6owOOkf6tLF
JS12xK4SNF0gRBQnKuwIibXNAt3eF288
RfTAUo0CPgQLEC0kpzYSuoqQIcHqNgRl
iJCPpUqZrZxpSL2Qp3Ydr16GPtX+a4N3
A/IOrg5vmnPg8/J21o+p1WmdJf46FaCp
xtMRQ5eDS+CzxLH7oU9qd9aQ1aOKIDuh
wsrccuqip/omtfREPsUKC1SiPjpR0phH
38qoz66Wi12fOydtfzTglZgrspIKC8a4
IwFQrQuIf7bBe3Aj5xXrlIxZkFNHm4F5
mZBTRA9lFJX2ojew7nX0jYQ/gFVKrrAq
UfSQEmirqVn6EbvvFrI11JUjXzDkXLcL
b1Qbkqa7VtQDhgGxQqKt6onVavHYqVzw
nwopzIn5sUP5DeBevgu7Gvpm10N525iz
Ku/NhHSlYZU7NozFLgqcxoN9BDDlQCLT
jZouQISMCYm1vD71qTUJk0A4cDEpNyuo
Qwazm9MEjzEKxG0zeyYYd1ZftsBUA1gu
q7ba2Xod+lTDocTAxdhHmzTItVntOO6C
vsK1lS/jVJ4ZVIZoY16IZ6PDcUSjNh8u
Q9+Qzx0hWy3pwNElYUOw0T5ALwPmz5NZ
IyqZj1133q9WtO9Ug5gI1l+GPp9IE7lU
BcncFR5zFrWay5Ak7W96t0NdyjjbyunQ
BitMcmsDAwI8Vbp1GfpUj75RaEp942l4
cl3bgOBwKBWmendC2cSaJfC87XMe2pqF
F2ZtnP7OZZ41nti2FrViJMJuVKJZ6lzT
ed9jE82pQ4REFACNz+jYrRyJUMm4uXUs
fRn6lK21ENFxYcvD4o1oOMvs4yTAHQ+4
1gAUVcSno5Oo9kSsRk9q3Md0130Z+gaq
Foms7Bsd4eyIEMhFnYZggnPy2JGnjTIb
ZQ2j2nEZCIAuC1EbuXqX5vO8zSIhXMks
OvJbqAKk1ZHHcly3o3RhCJnqDbZSJEAQ
vR0xwEHVKqKn9MvQN5cJwykpX1diJKBh
WMZD20+VNukKNqWIESvVm2NZp9QFuKrx
IM5EusVxP/Qp7Jqk2oGNE4CBBHIJ8mqX
IVDCfWtygxUIOqtXA5Tad2AxhralssFA
zWXok5CGgOEPLEnJhFTCrvYlB3rCaCfU
93D0HdB5vxcvPnoyDqIHr5Gav1jISUgj
PIXJBmph/bsEm9x20OE54XUi6+AmGJ6N
2jeCHnohI+rbBw1tL0Of0iG0nydrUlqg
qvmwc5VoOqYyq4Glcqh4Ax4AbsAsqesi
6DHQeHOZ69AnUEXJLrUtZ5mcGg1ZwhKL
tCEOZholHAAyXmWTaQsNUGGpKSNUB1p3
J3f2WU8L5AlmpT40EACHY6qVyAbeII3J
S8LgeIkIq3uetHV7dKtC/U6164vXuT6l
wBKVctGQOjK3S/m5Km912cSWqye8EAGA
rKntZm2JQ08zVNhsHe5d7PospDM4CiMa
O8M0UF1bXZTUIcK3ikyaTCjocbQvhFg1
VrJiUDqqGtrmuB/63GYxYtAQrY7sVoay
2uIUHaonJqUZbUxuWGFUIoH6TMIYnN2N
B4LaJnNBvhNZSMMEp+1ouDIKNzbEi8fq
+DnEVGf/Q6lwcWcV8hOM+cmRY5WDbqS5
PnU4k4UmxglUWrU6QRqmgCNndd0MKlPC
dTIjpabLrpAY/miCDqu1CXy6DH3K0IcP
YHdEQwj81DZyYy5BPxxE1EZdBHtU/xmo
ZLejVTcUfJS8y+Svy9CnTAtV0BHLoY+i
MEa39RANrQvKCVZXSyVOpzqbpgTBgfkw
J5gPOnCa64Scyp6cxDKujX5tyZpCFE+4
y1TbsqQeYfANDIRoNbVV0lnNob6aBTo0
73bd1UTvdMLBp23TuQKDa6cIE1GDI5lG
7LrOzSiJy4IBMDJV9fK0PLaaSsKL/GXo
U5vF7gmyUnBZTSV0AYtK2AkLoU3n3M5Z
RS9LzeGRomorhAOpWWb2qOx9GfrUZnEA
cBstwUsqf7vyrq2rP00Eylk2zDdgcxiI
oFrKVMnoC3wP4X538nmbRZ9QXVFZQ0zH
mKIljf+3N2yn96Gc1QCy5kVYTOrHCdM0
2icwHpT9yFOftMxWTQ/zgVjyR+9ebR16
NOHwPmjrXZjnVIuDbNq9qKRwKDcHVe3v
ssY19KmyuygFybmoxmtEmqpCrxXCkCDd
asKrGgzVZkIYrHYomxqtExxbO10qeRv6
Bk9decU8XEhDODLFb313YSrTWE1nhaae
2KttgFW1e4jRsAJjKE/iMvQp7ML2ky6L
6LUx3WiU6lRu1nWGgbDIw1eViSc1bj5y
81B8DVyyR5HtXZ+F520W5bkZ011KZ4Yr
qdUGRHoQKgPBxwMVeLZwZsIqjLpBqZRh
HXf+GXex6/OJdDIbT9B+uFpP7q2NwahW
FEXNYYiDanKqFvbxICQNWdOPLehclrcX
bzyf7cI10ihR24NB9zAK6AdrFSyqz0x1
NhtKctTJo1HyGqZYVJ2ji1fXxUJOajfx
+kPpWinOqot4eX91q8AAIWabaNNmhrZC
2rG8phYDA/KjM7cCAbSXoU8JjjiEbrAq
mK8nnjT1iNj5qGeLKrZlMYmEC5U09W14
UtJZBCHOILIvLnMKu2rW29U3VNdV6Jxn
Bgilek/AO7w23tVhPCTojctqneG0NcQ7
bVxhroujP2uziNrMuvx0Eg8zQVDXuik5
S43yorLJlw5H+X6jHpyEe50wK/8pq5vw
/dDn/evAM7qivA1IXFK2CJOfshISXcnh
HVJ0zURXtYKYEO4LunbVsJvL0KfyEAeI
zaOViRq2b20qQAei2m0cDTMJFUeHLN0U
964KT+1hyy5WW6IXC3mWCNa3WmjlruYv
8ag8JAxm5Ut7a5fOyztgtbTRED1PMiER
/FEJRv3MLkOfqjJL1KaNWrYp1RLI0SBN
Hd4l2FtXcw9drYuyjR2WV3TY69XaIcP6
Ly5zkqRZ6bRFjQkXggnVASdQtvuEGLBg
iANtcxEhp4iEmtqurThTtDEy78nC8zaL
pkfAVDkUOh3ReUjQ3o826ZIA1bbkVcCc
iY4s6PQoGS/BpOYO5WIhJ0kalDmX1e+5
HJsIWfenBh2vMclCIKYZZaNTSV4gL+2d
oRrxKtTuKucJSXdtFrW9ga9U9Z32vKG6
kiAzesF2Vc2iFNLdqtcVWCotU4NwNWFc
ioxE98vQp+O1I/96GTxG98fokiXdLK2+
r6rSV6aCLepPEsvRCQdJoowURO84+oFc
hj61FRjKrMHPuoHwA/NEQvUeVlGTP+4L
VsfLpe7zikaguocAVF1MC2r7dRk6n8iC
97oBQdWzuEbSfha+o57cpSEhhYZFucjW
o1dT0hGQq2oP6QHZZ80h012bRfXkQcR2
vTHLpiRjNfzAxqDzUGuYDAyPQK9sOQ9j
ZQhUZEVGttrsda5vdi3ipHOz3DrxFPEJ
EUGfq3FoIYaDFsFNNR2oKp0Ud4Pp5alc
0F5DqfdDnwKYTqjU1qBtpWuotWpHD2Cy
Bt2obqrHtRDG6SZRtQnaccj41EjquHrs
MvSpysfL0dtRkmWCbkGABOiWorIhJy7p
EhtUWuepcVmj0tiNjhjQe113WS5Dn1qk
AXo6PpkIz6o9VeIskhkuU7tKunNTQukY
jQFb5tO449aqm6qGfRcLseeOBapkUWtZ
QgnMFgtxOstdaEi1O+KLpD33VmlS9aov
aupig4xQYsp16BPyHc2WwU0nbHfqCezU
n4EXUcqXuubWLmaphiH9KPxEuqI6kGNQ
LHs/9DnvaUoXTBV96sBuaRNZpYJqraNM
6KLSsiPBux29h7d6Y1f+oir14lnBZ7rr
4OjVqUG6EEKA2vTCTR1BI2onElH1omCM
7ndwzBFBDraQ1F5wW9XDj8vQJ/Y0ejBp
TTiq9qTrkFGnMtT1SJmkzsEpAf4pnsdq
RqPtbL9Rftm2Z5kW6a6DY1WfIIBJlyPi
z2oorXYwuem6ZZgUxtsQOlEtIXEivCvJ
8DUdqxh7setzn2CkN/peuVNJ7SdAv+NW
Wl2hBdjLD/HSlXVnJryqTY+9NDQYIl7t
Mi9Dn+BJ/eC6WlnEqhz3uZWBGqUL1N9e
KfMYBmER7Vi1SyRkmKpSVyOvdrWQcxdY
EEkRABacelbnGqn+tleDeB1dX1F6iOqc
WUAvzEX1LA8UB1HQy9CnTYtY3pXpKgNl
lLrQ5h39v0tHM8FzC77ZdZlBqxXyC79h
sYkScEq1Rrwf+nzhTtctBEhE0TFh52Kq
sWZlbsGmR3YRhYub4i5YOCRKxkgMXhIl
+27oZ5IU9zBqWogGQDeqC7EZA0YVcEk7
01RWooMXlKPUrqPQVYXDtzkdQ+16Gfp0
DNGDkviLmsiajbpT1bV2brwSjZFOTDD2
a9T8uejGidFKlYzYfLg/a+HwbuhT+1Cl
OdiqTEwotoxOWddRTGmpc4VXH2PQRTKX
tfNOmdMSVh1q0p9dO/Fu6FOG/tDp8kyV
OZ0z6dLD2Y8sWLUuR8RFFbsqKRoBz7K2
MBqBCCtEwxP8L0OfEgvU/1u5CujFGr34
ZNP9N0q+1k0vFVKIULJKND5qWQ4xqYM3
5Y6E6zLejC/ixUzpjGlW7eYnpRJ5nFN1
6OJPqBf1K3AgS1YeDZTSJKUL8wq9+/uh
zwew6pmqRq3qVwPNOdIy1epE5ec63oFI
8/4AqVN5RIWW6RYoSLlRXeZlrs+d/0E4
plpJGuoku63uofB2tgSNJ1IWE2EMfK+6
QkavS5SsOpTg6VAVZy9DnzhfU5NHKAF6
GqmujsUrJ1BFR129SIZF9QtV2lcyCGDb
jvvAIROWiHqx61NEd031Ok5tBJQ8H4La
QFoVL6v+WK1VtjpHZ5U9Ai1VeWcaGd9X
Msa8DH06/V9d+/k5H02Fk5iq1yVRuLQa
o+H+qyYPWuj0y+t64aliQdArw4jHdegb
8jEwrqXOJxOGBz2afAWhSuFKdSZVl4EV
HeHZrStaqhnqSTxx+2QQvJehT1ktFgzN
punwEOfSntuA3hyXUAwLYKPWkU0s5cTE
AZOE/6vLHTgWet33Q5+3awsrNjbhsMND
29FGF9azrHreHmcfTXVtTV3+1dFc5frS
YRAMRFC/Dn1aRnUEAx4jeoMl429UNhN0
aVnHfnW6o/SyAkZj6tpghoQc2YNWF/xc
HP2kG+E0+MNWE9GiW9Cq/DcSfYMaS6oR
WZsErqZjOEO47zpFCGBTcTjYfdh91t04
s3xqzie8VoaI2AHYodaIZc513E+gOkKV
DanvnZd0tfmQKNr3vgx9OqhS6zMoRlzA
vCiGVdNqi+RYYv22DuX+89TlaB0GBRJj
5j/QYljyBZ5OEV0Apv420AEVmmkbLIHH
8pSjwRZCH+LU1ILWoeJzV0GYxcgRJCqs
eT60f5YTDEx6NdJTCirPrtN+J7RSl7dm
4TQ5OBS1et7paNCA4/gjjuTVqD5chj7t
qYaeRpcCCmNAT8U7pFyV2azzgr7VwZwZ
aurNh9ED2ovwYjNRo8dyGfpUOhm7ayYo
qz1rW89ph9JFpc6rHp0glgZrZpQgEqpO
XaV1EQhL3abXvgx9Kp30cOemsgRYjYrE
Vf63kbwNSn3kIW3ZWVT35DRSqjr4h685
uErEHy5Dn+5NEn4Sw8TATEcw97GwtqJe
YrqHAomDrlBTekeAA6KcVVf+PQHc8fwo
893Qp8YT2AIktyrteLi2cTAh2nG/UW9B
NhIQY22pqDarbF8dGY+bDAu04rKMpwCm
ig21hMTmVPepxAvoSNpofkir1ZZ1hlv5
ov7/GJ46Yc+AQFcTqxquQ58aT8Bt0C9A
pyOc48Y8lxJJay9BvQ+cwbT5d52SJqVM
JrBL2UnHzYO5XoY+2XUyqtEN4IKqFEbC
rCy0rBwtZ1s2rAP4wjrjAckfTRiKWgsE
4DvfI9+z7sboHimqjmkIM3goIgIWN3Vi
3MTRoU/Teulgo/sPnT+SgSZPgUddJ+Qk
7o7rGCyBX1dZqoY3zqCuAhYNhvTQVipk
ZzPdqu9WWbPao5QQTEHTzMvQp8YTHsq0
fLJ1KVNADVlQ0JYQQDRAt4AtLiuJu/NS
xPM1GTkpiZWIj6C4DH0qGgchLbJH94oV
pTkgDZPUIcR8w0ug2kzNUo8zTF5rG8qs
PWBMxIcSL0Pf7FrVlqXqYryWjx4vVttv
RsVl8A985LhIYFcBrrGtbOu6uhAqa7rx
yvdDn9UuoDpLl4CpuvTBWVUZoKJ55Ig3
F52qHNG4HJ3Xg9KpiW7OqqXyfZTx59i4
lXY/CbLKrUHZ676krL6CStvVIYyHLDG5
xR8LAeHMaNg2gT23ecnL0KfYuJz6iyVw
fyn9QT3QlCxS1SQhq90G+BEAgwrZ1EUw
QUd5RM4UgN58wZBTbMTLwAIkAK+H+N7W
quAu6+pDAE5tZop6LDisCGRUxS3uPts8
jnBKu9j1uacFehNZDujvpmoWtRuqOo+C
XJqt/RQR9JYarDDi5kn1LojNaY9rVS4B
7HwrTtBVL5BbpVpb0bmem9qTqtpAPQGa
bqhRVnZSCdCMQztVak6XLCzjMiEn3ejA
U62cpPr2Q4upTgjwgql7JgMkTz3THGw7
N1y+8gHoU+oywJEuy3jarnUyhgC6I3sX
QRf5gTQ9mKJyAVED3RRVDKuhMnSotQqq
YtQq0UcAPx/6WXdjWF3Dg8XalV6TDg0U
AQ4Vowzv0x7aftYxDZPt1T24JrUZRKLt
55eHprvuxsyY6D40AAqQwSCpOVZNB+BQ
JiXqyl11IUzI6lwW1AdMHdgWKGPjZehT
UyadPVirRPoFWy/T+46XaScHJpx80O6f
LgdQLxFduaKCIDVcQdPgpdenPt+bVCG3
TteANfU8w92Keiwu6Aj4NMJxn+GAGpqg
FjDV6VYH3dXWj2apl6FP1RB9KGsdHF3q
wdR3HFbnuKxfjGpa2NWPMsemW/9YmIom
Q4Opk16TFV6GPtU3LvUJPCoFdTUXLEys
ebQ+Uf66GTPogEN3vgR1+FTVSQQYtTuu
iz/W/dD22X6IOqSoBztYr1zcrp7GS+km
W+dJFUQHGCdiVAQFgaeLtg3uBN0flwk5
d/7HSVp+dzgQvC6aA6WIWbrtw2Q1aRiD
r4eeJlFuJVktSeFa1fHW2svQN+RTxS68
K1v1tLbTqqYK+ji2tmBGUmvWqOtkMGav
jQwiP/iyUTZQCDvNZegb8jF72QQsVG1e
wM6ZpmoRZkZz6G5joZDEJ4FWR5BmA626
lKRpb27OcRn6hHwLjyCW++R1aarWrvNd
RW1JlPu2VJy6mKYkErHVfwXr7Doqi25s
fxn6VN+oUpU4jkPEok7wgQDC5GakQI5q
3RNV9KIi25qODXroag8oX6g9IfV+6FNs
JJSoLagaipsWtcM8h06RmdiO61h1WFkb
YWP1rHtI36mbkppk6VKJy9Cn9DVdZuyD
DkFVFg9LV2pj0JXSVtfiGqWKq4YglaOT
WmzQ6yXNltXlpl+GPm3GqW9XIOKqz4ju
ZYi6pFtxLCkLumxQ36iTRlYXl6zAb7WD
3X0euuHgMvSpXQZzG7J2WLquVmt4gXHu
uLMN6bZyxGfg2sc/8kUKXCDYGJjn0YX7
MvQpfY2VyuoYlyExTnpQtQnWN6U5DVU2
K/FpshRqUm8h2kqEUFfLqJh+HfqcNW50
K1oj7iv5C5nkfRy64A5xq8vBGtC93ZEj
E6c7Eswm7EQpDCDvvAxdTnatlj22pdW2
WsKAaEcfRXg85N8UnQqoN3oIPDbfkZi6
uFw4+tz2+62WZ42TO9JZTWiCbjpBMOuO
D0vgqtBe5ter1qUBdh01spa65qJEVQ1A
SNLW/P3Q53YZRVcx81yYVGARl5WEgZ8a
XU0LxO5hvBqAVfSW+meoPx3gFJtOwe4x
5FnjZF1QpRsylc8H3hTl/qqNIhCRmYeu
bAqItBH+6bh6w6udVTl2QF6bchn6tF07
R0HdVnU5AsuUJ1fcZnF1Mw2Eug8vFG9V
18tW9YbUtkZWVwQLf9mXoc/Ip56xoVrt
1gY4/9alPbo4VR3RY3KI0lEUz447sHCb
2WCA+Dtu0Lu9DH1DvqM4BfedPWm61ZQ0
dHw8HFeJq+RajRfQvVGF33sCHWqIkvgv
nYVd5/qEfLpS8KhrcurTuJJdUErQUP5G
pKwdb8V1jl6C2vhHo/KeuptYZ5DtMvSp
DYwTLQCckUq6jDlbcUh1tmxFiaTKs91t
BGWigadDrqUys6MTpp3rfuhz538YMzhR
fABZdcbIGmkvD66b1LCH+RHD7lG9N21R
qwRRRP6m22LuT6SfNU5GoOgUxlqlAOi4
VuA01TelK8FTnRWCk+Ql0uhyRN0QpEN3
p8KwSyh41jiZB0XnbwtbKCrGGVXtWR3g
AY1EAPSWJWXg3brt4qg/AKUxaBSCmpZe
hj7fYlarwqruUJkZvTbV9tvKv9U8YI7F
ZOhyHd0Ro4IGq53KGeFO6s85L0OfLiCO
urRLRdY689rHDblRF8PpfNuqDtToIhtd
2WqZtiAPA8lM5k3X8+uv3g19OqLfSGX1
rPVHE/aVjvu5Ub24x1AllCurEdlg7gRw
XZFo1HMV4dHDbv0616c6sK0btcUWoJXg
CNQ6uSOdQ/1llDiipF1iAKu5leGHWiUC
WN3bqjbz90Of7/LRRi/sveq4f6bjGp0E
XjridtbxKP/bvM5HlYoYW89hF11SspVR
cr9B9KxxMvwi6Z65IVkM0Ym67i9sZkB3
yZmm+KOLOAhZYoF6A4Sq08kMM9gvjn6K
6DBQcEFdrhPxZmTlLU71AguxGX1lzUG9
JnYG6rQ1BE81ukfXDIj+voCqe9bX3UPF
HPrycEElyW+rAxndlqI9SoG00W2u6qbW
DfLXV/QrMW6k6C5D3+AJZza6ASPF2ZV8
qrt08boiAuhU32mnthCVM8IiMseq2fG6
EGHosDtfhj4JDl0/5VC6aSJZYvHFE3zL
cQ3kka6Bfat3ZC2NMJ7R1Tw6z6NOYbo+
5zL0yfiAjjgkOUbEVj2z04lnXqdAc6rk
9mhzhoJeulVb9z3pDiQ1x9PRyp1dp3Ns
3Agw5k7HAriw+i8hJEtFYanGkH910k5G
9x5bdb1QX8uti0oKtvq8gCjdNU42RjlN
gxHg1FuNo5X1GXU7cOmqEfYNmT+Pb21T
N9V5E7Bqbc/Bn9pl6NMdsBIDVa1jgg+s
otqKKcdYrReUyix7czhqkJZ3W3tgukED
9KpD90tfhr4h30KlqYYDvx7Sj0ntotRh
Qvu3STx+IXVKdKqfaRPW4iMA7FSYB72O
l6FPEV35RwC0dsdBUHiUcagCXeikWtQI
hqjvxtIZE9amO47hY0pmThMRuS9Dn/b5
PMvjjotXAI2le30yExOUMKh4MJKR8EL7
qa8tTFikCZ+VR8pZL0OfzsDCPq7uBEKJ
IfAmZbRnr5Yp/TiWB0kSsGd0lo+y8U77
XQEm53X9ebof+qR2rXpbSUwsVCnU2ukM
BgfV9eroWqM4o7NFAqTXzRNq80DkV3Gv
6da7y9CnkgWr8iacVnUz6hDJjGMRLe9B
1A3Ks8tqA1OA2oDiyyq4MzoeSsNfslqe
NU5WG66atFLqytRy18GwCwCSTUN9F0q0
SugIvBeELOqSLyUXhcF3+nidkLOF+Kns
m2m9uj6r4ZgqEBvsXU3FjowomIev4lcE
NkRNUUPHxPut7vNl6FNld4JAGpWrBd0X
YIdb4DCqRfUzOo8PaEpv5XwDuqaubGpC
qeY2Y8R7VfCscXJLSrlTAUfWIQOMcsHD
DtA3qk9zRbkA4PPgvWxW42rEpIrBtF31
kQk55bqrtcZR3hiO/lwE8oRGr0cj8wTm
wVGs0iDUnSRWldcQiODExiYZzP3QpwCm
JleRBc/K/Ew6xIDebqVnGSAjHTefttXw
0JaUgs5sAzpWlQy11VYvQ58Sd6fyQ3Yk
Ttmiw2L1qsE9AHAeELKQ0BmqyuFDG7aD
lA/KPZkF4hrud92fNU5uEm3VVNVjgcFM
tM6eo7qEq6mPLrZUb6mu/pgwtfHu7o+O
oK9JPc4uQ9+MT1YB+9cd2lZ51Vs3xOm+
MbWFOHoS6sCnGWVmeuWw6RJ5FagCUwru
l6FP9TLEbvGYot7I0FuNjzv2nHU/LQDr
jttmRwar1yyGECkFX5yqXez9rns6x8YG
mC09XdrhXfd9r6o3rarawusoGcAlhled
iB392ZuKQ3kFpb9clvEkScF1da9TTbuS
WIwavVTdvoQTZ7GwoRrCVZ32ubBPuJiN
7kAz7QzcGd+zxsmI26U+EMSDrI1aKNk7
tluw6KQ+vLtl3cCALkq61HgrwU85eki9
UNdl6JPgUE9h1X+z/CVEhe1xZBibpkZp
qv7SbWMGClEP4odUKDqeNjDF+pGnPlXR
6xakos723erOjnns9TrlAY6tvkCzaNcw
w6Z0oudU2H3seBJvAu90GTqfowzqVRmN
3aajv+Qk0gBGuDmKhVgVl+PvdK2gX7q7
Fh4rnlwm9DNehz41Xw8wRaNrQppYYtGp
NybtdBwRoCRq/bSgNYmwrDR0v8QB4e5A
uW/3aT7PGicHnpEHKBXlIWfMTKpTMZ1u
kTawpuNkKmr3RX1XcVziOQub1P/zcij4
rHEyEKw95DiRzEWbmaa6gahJQcebeTrJ
BrXjL8rCUY+zpbsVoUF1j3yPfPnZhTtR
e0y6CUMtqrSRHnRNMv905Kw23QoH3ch+
KtsbYbmX+v0ql1+2dX3q007w1OG5tboq
vgTbjtSnEiZGBxpbFdI4uJ8ySJWzpJSn
xFTUqAau42ohp9hoJdiMKlRRtLxjU0MZ
a9VwA5SezDHMbqiOnh+GpISFiEQHu3kY
f09xnjVOVlNvpgxnrlHJTR6TDgH+HHqM
qavj5igbB0mqXbbgItxQl1E4hd9x8cZT
bMTloAhVraiZDJ316aYD+fAAmBlETc29
Gk0QitS5OqoVxWIWAzwnXIzvpBtXqDiV
SpjVQFWJ9Js15Hsi6hOhl0E9f5ShTasu
5vLYpkOhYfGplC9Dny/cUfKGaQ2GAV+d
uptADZN0W1pRj2mn9q9EoKi2x7HFFNVz
W3doTPVQugx9apzMaC4dN8Ad956ox+fC
E6fSCXgl9KKDUacIeMENgwpa1Qb/2NB3
7frUp7NdQFU1uSEvXQWOMWd1XSeOoAZg
SzpkbUO3s/P4uierE3oIB30fXevqZeiT
KlAnXVxa97DAe/ka9Z5ELyIT1Kql6vDH
gatKE2F+topxRp3EYJ07hcvQ5x2zNZS3
bGEiEHWEh/LmPY7vdN8z3BUSWMRtnK5L
1F3ymAjicvF3JcTL0KeKKqv6SlhSbpPF
V9ks8Vo3WPIzZqvz97oviZg5dXmOklpT
76uOoRTHfRn6VIoDOwXGIByxSgj6ftzi
UHxUx2q8m0iYxVHQZAAkxCoSl5Xepw6l
9znBzxon+5Z1lUBSabXuD+NhM4CEwcGq
wFZ+gv8r95HoqOuEysDKDzzUlTaXZTyd
7QJmOmpZzra6sTvEl45jgQyvfnnQeSXG
TeUALdlGr4gzNbyElvEH6XNG/h8a/vPx
6uXbp/bm7RuG/S/H9/3347/51c8f2ps3
6+3Dm/XmzeOrl/qEizb85e33r54ev3g8
fkGsZyVPvzr9UbD1wy/ai9dfNn5mfmji
6Ydv19PL9vbx10sv98XTavzg8w+/7+3N
evH4cj28ePXmzfHHSifRIYKFLqph7vWj
//Ti8R/Xd3z21au3b3j51w/9xavxjw8v
1ssv3n7J38SPfORpvV5vH9++fyW15fjI
h96sNTVLLBjqIX/4xHh8+PLxiy/fvTf0
NuqCNkIg8I6oOH/sxavfvP+UkURUhSJO
4h2E7Pax90umufpr+/Dq1+vp4a/Nbbrm
Vw9ftvHwon3x5tnLvP/564e3v3n18OZx
Hs+qhYilODG/rJQ0X+7/4M1blubN28eh
df5htaoXUNG6djmy//Dp9XK+fvX48nis
Y/Yfvlrt5e2xFkN8xcK+fz9UT0KmQZmV
JOvCza7WP79uL3m683Lr+Ed1txV+7E5r
+OGz5/X+1Id3++rxxb/o+V68+uLh6XF+
sXjBp3+6PeMXvzneeD62L16+0is/zK0F
/ZYPvH73Pk7HqSoXR9apKX38lj85T2g+
/tS8z3bARM9/pw9+rSn4/Ee//Ie/+dtf
/eJXP/vxw09+9qP/9PNf/JJ/vD33l//y
+tXbL9ebxzenv/m7n/5fP/3xr376k9vH
Hl/u9bRejvXw9OrF4Wt/+3c/+5sf/d0/
nD+C6/26vdAvsZMfHHby2ev1NNbLt48v
1mc1/i//4bOXr95+9vgSw5OPf/Zqf6bf
MvT8wcuvX7z47PVtvJevvnp82V5gdd98
5d//8qc/efiPv/i7h//4s/+Xf/rlT/+f
v//pz3/809OfMMRBpdeLt+2z//3/+Mw8
/93DaC8e+1OTM+pzT+tN++r1Cx7zw8R+
xpd+/eazV/0NL3P+xX/47H+1n41XT09r
HH/+fOD1zwMPbkyQJhAI/vDr1w/71dPD
XOPxzbtvZcn9+ddP7TfXn76bNbBjfn18
3TuP0MdKMUcDBDU3L/WEAe885/YX78f4
9j96Wv+N9zn8+e3T1+v0869Ah/mw2vjy
4W171LSaD789m+Hnz331ZkQ//sXf/O3f
y4iOX/2Pv/z/UYj4FAx8NEZ88sN/pCAh
0Aci1XvJH7VCn4gSaHij63gVLnTEcnvy
Z2HCvQ8T9rcPE4Z5QQnqsqF3Z2rfFifc
D5EGyD1CLtox5tvj/1vDhFVNUj1qOOHT
8Wb0HwsT6o7LdAVvczb226PEJz77Bw0S
zqqCl28mRBh42fcOEgmpGI5Gd/DRZ7Hl
zzHijxojbobzsRBhToB3ihHPfvzpIIEw
yzWg2r2ua/9+MeJTf/PnEPGxEJFzBDiL
V1tFZ8v1s2fA+OSH/0ghQkk9KZWEtNaN
i+n8sVOESNZAS4UeFTVxAtDft44AmWq0
RxQqumsp3//F8wCh2xxtyA6NrZ4fJ/7/
b4sQsO6ULHTIG6WrnYTSxyJEjBGNg4gx
ymb9iOg4r/inPnyKESzH2y/6V+//7PcR
I3Tcru4DR3swa76/kLCaAvW4UIH3s9Dy
5xjxx4wR1t8s52NBwvp4+v2HIPHsx58K
EvaHVpfOlmy1yYEE/R5B4tN/8+cg8bEg
8SkU+GiQ+OSH/1g6QvefpBqqbioSj75G
iR8cKIr+IYpAGZx2Wz+x2/S7y4gMWKuy
7uimjHr5thjBg7mYk8s5qzgghRO0f/8Y
8YN3u2nWqXlcVoszl+x3RYmcWDzlCSZl
J31HlPjEh/+AUaKiqnSvuI5hdJXn94wR
cgUWQD0diTGqXfw9homf/+JXD38OFb9b
qFAzi2+LFem06XMLFc9++qlIgRtYJZBW
n3LQ7nO8QcanY8W3/dUpWuz24s0fMlzw
3/9Vn/p8Pr4ZL9rjV2D2O5P77Gc//88/
/eWv/uanP//VZz/6yX/+2ftV1QffPj32
r4/3ebNAsdme5Iz/Loccz4LMB5v+7U40
/liBI6iLKpoHMencaYPk/pQCtYkcCuiM
Yt0ntp8+oi6uCP6JJfsU79cJNpHEat+r
uOw/st3zPTYDv3u758tXL756ePcx4ti/
6uvD819+/bL9Gitv/cV6eHswiolVPbx6
uS6ucYXDX/70x7/4+U/+sID4DN3+eODm
T9iliXsveGL6CKahW8N3g9q37on/G0CI
ePknw1p/G+z4k9u9DjaLDhVp0no+1XqO
HkH/G6Mu0FDDIP9x9PgI6/w9oIfjBeGU
uahrQfh29Pit94r/DB6/J/BwPn0UPbK1
H0MPiPj3QY9P75b+TwQef2r7mur3UqVc
Y1G6a81X8Hin6bSxGZVE5EM0ttTvv7X5
u8MHMhnksNkUa4Gvb1eTv/U24p/h4/cE
H9nFj8JHyR9Dj3zisb/F1tv/VOjxJ7fh
5YKrMELd4FwgRR+hHuWHakTkYwX81e4t
rh+Y77/h9btjh3opp4zVMHGx+t9uK+rP
4PHvBh7h2Qb+d4FH+D779t++r/I/EXz8
qe16RK/CNwPLz7pG8ZPbHjn7nJQ6bP8/
9t61OY4jS9P83r+CRrM127VtasOP+/Hj
PrXzgV2l6tWsLjUlddtMj7XBIBKSMM1b
86LqKtvd377u6SQRCWXgwvBEOh+kum6N
TAD5IiLO5Yn3nFDz00LjspfKozYtNbhZ
+c1i8erS44g9Dh89bGfwcCHsih5TuAHL
vSX2kIXoUU675yWCbH7qib568vZzDiTD
IZD6pNu61aqIDn5nF9MQSG27Qsi1eCr/
KbsjyT7qEFcKEQnJct3BEtMnOeuOceTO
4kjeGUam3VHE3SSI3IZ+3IsYMhoJqTt6
S0jQ0lnlOoYUFmKI1AbM3FRan1g3A95h
OVI3mOb6MMipPjLGzcrYIwkZMozM7/lv
3YbJOyOJCzcAqbdDIfcilAyHRaZUWoZQ
b+dqyfZxGarGIOUo1n23OfvZJ98/Fwlp
qo9XF1+fK5HnXdURjIwZS0R3Njfl67t8
Kttf7oVGeoaTv/vgMzl7/urtX0/+cv7i
6cu/bNlHWqh5WD76m3cXM7MXIWgTSdp3
X4SaiyZ98/KLch5deofEJNtv2nrt4tbG
FZFq89K7F//24uVfXsy+fZq9vomOVffj
x3/6+uHshV/K+//2ftj6+cP3X/9///5G
4tL14qLporhykHuL++Zfvu0lzl8vzvK0
fOS0t7hvvvzh8R2KixdzBb89cpdO2Q7i
vv/jD53Emd1AXLJlcRf+k07ivv3nP/Q6
chfzdFecluaXT8ugncX98P3X3U5LvV5c
8leIu8to6ac9qIuWls/LuwyXt1anN8h0
lsPysbvLeLkXddGWs8GdBsxbqzN3A3V5
+bq704i5l+vO0hWZ/C5D5lzd5r//9e/n
defJj3/98PtvWoJO16mPSxXapRe6FJiz
F97rOHnaGsG67aY+SeKRpNsd389PYIYL
9BNdoEMLdI8mvMBAF6h0gZEu0OgC2Yne
lX/RBQpdoKcLpOdBR8+DDh9F2e2SeyTs
dqkIpOdBoedBoUdRoXcT0rubWL5rvYf7
0jcRyE4T8mhiB5kikF1sCx06CR06CR06
FYH0KOrYxbbQoZPQoZPQoVMRSI+ijo3u
hQ6dhA6dhA6disDeUXSshrcIpHcTcCYj
/ZnMaALhRiBBG4H00eTQ0KkJBBfbTSA4
ijaB4CjaBIKL7Y1AMpNpAulBhmwjaQLB
ZLsJpKcJMpPZCCS7LJpAei3aHVkMJ5Cd
JgTtk2kC2WmCzWSaQHY34ekdvUfbSJpA
dhT1dCbj6UzG05GFR9tImkB2N+Hp0MnT
oZOnIwuPtpE0gWAbyUYg2UbSBNLzIJzJ
eDqT8XQm4+lMxvOZDNwnE9ALc5pAdrsU
6Ngw0JlMoPtkAno4ayMQTtUCnaoFOpMJ
dCNQoBuBAp2qBTqyCHQjUKAbgQIdOgU6
dAp0ZBHQ+2Q2AsnDWUWg0pmM0pmM0o1A
SjcCKR06KR06KR06Kd0IpHQjkN4aOl37
uJu49JyiQwmk50E4dFK6EUjpRiClUzWl
UzWlUzWl+2SUDp2UDp2UDp0inclEuk8m
0sfrIp2qRTpVi3QmE+lGoEg3AkU6VYt0
qhbpTCbSjUCRPl4X6dAp0pFFpBuBIt0I
FOnTZ5FO1SKdyRjdCGT9odNAu+6bQHYe
NLpPxug+GaMzGaMzGaMzGaMPZxndJ2N0
JmN0JmN0JmN0n4z198noUrGthym24dDJ
6NDJ6NDJ6EYgoxuBjA+d4CuPEp2qJboR
KNGNQIluBEp0bJjo2DDRoVOiG4ES3QiU
6Ngw9ceG6pc6en8Ygb2jqE5LAqfDCGS3
S+nWTie5VuDSKXoogfQ8CB+vS3TolPo7
ncYCv4nudEp0bJjo2DDTx+sy3emU6VQt
06lapjOZTLdyZbqVK9OpWqZTtUy3cmW6
lSvTrVyZbuXKdKqW6Uwm061cmW7lynSq
lunzg5nPZNhWLjfBnU5VIDoPVoHoPFgF
oqNoFYjuJqpAdDdRBaLzYBHIpmpVYPco
KksC5TAC0d1EFYjuJqpAeh68LVW79DF3
KFyych1IIHunUxV4y27i83paQRVI7ybY
VK0KpOdBttOpCqR3E2yqVgTiqRp7QNI5
OnRycCtXFchulxwdGzo6NnR06OTgVq4q
kN0uOfiAZBVIz4NsK1cVyG6XHNzKVQTC
saGDD0hWgfQoyrZyVYG9u4mh1lBXgew0
IXSnk9CdTkKHTkKHTkKHTkJ3Ognd6SR0
6CR06CR06CR0p5P0dzoNtY2kCmRDJ6FD
J6FDJ6E7naS/02msjl7o0Eng43VVYO8g
M1oehBuBBG0Eio8mh4ZOTWDvYjssXYPh
zq/BJhAcRZtAcLHdBIKL7SYQXGxvBJKR
RRMIRvdNILjYbgLB0KkJpOfB7jYSTUvd
RDqMwN7F9kDtUhNITxPk6bMmEEy2m0B6
sd3dJzPcEaQX292hk1963oS/80eiFIFs
I1ATyE4TbCNQE8imakKnamwj0EZgdyOQ
2lKit8MIZN+b2IMRaKxSTehUje2TaQLZ
6J7tk9kIJA9nNYH0IEP2yTSBbHTPNgI1
gew86OnIwtORhacjC4+eXWoC2VHU05GF
R88uNYFsZOHpRiBPRxaebiPxdCbj6UzG
05mMpzMZT2cyns5kPJ3JeDqT8XwmQx7O
agLZ3URArzxqAtntUqAzmUAfzgr04axA
p2qBTtUCnckE+nBWoA9nBTpVC3SqFuhM
JqCXGDeB9GIbTtUCnaoFOpMJ6C3NTSC9
m4BTNaVDJ6U7nZTudFI6VVM6VVM6VVO6
00npTielUzWlUzWlUzWlO52U7nRSOlVT
OlVTOlVTuhFI6dBJ6dBJ6dAp0plMpBuB
In1+MNKpWqRTtUhnMpHudIp0p1OkU7VI
p2qRzmQi3ekU6U6nSKdqkY4sIt0IFOlG
oEgfr4t0qhb5TAZuBDI6dDI6dDK6T8bo
PhmjMxmjMxmjMxmj+2SM7pMxOpMxOpMx
OpMxuk/G6D4Zo0Mno0Mno0MnoxuBjG4E
Mjp0Mj50Gn6n0zf/8u2nCqyrSMa/BlcL
HP0aXCtw+GtwtUD2NejGJ9urBY5eqq0W
ODqyWC1wdGSxWuDoxfZqgexE78Y3460W
ODqyWC1wdGSxWiA9Dw6P7lcLxEdRdrvk
xkf3qwXS8+Dw6H61QHoUHd5tuFpg725C
lh4VLZ/2qOjVAtlpQsY3460WyC62hQ6d
hA6dhA6dZHy34VqBw7sNVwtkF9tCh05C
h04yvttwtUA2uhc6dBI6dBI6dJL+bsOx
Gl4Z3224WiA9TQxvp1wrEG4EErQRqC66
R0OnJhBcbDeB4CjaBIKjaBMILrY3AslM
pgmkBxmyjaQJBJPtJpCeJshMZiOQ7LJo
Aum16PADkqsFstOEoH0yTSA7TbCZTBPI
7iY8vaP3aBtJE8iOop7OZDydyXg6svD9
bSTqFwReeuHOBPbuJoYT2DsPhqVTNBzo
FKXnQTiy8GgbSRMItpFsBJJtJE0gm6p5
OpPxdCbj6UzG05mM5zMZuE8moBfmNIHs
my+Bjg0DnckEuk8moIezNgLhVC2gh7Oa
QHY3EehGoEA3AgU6VQt0ZBHoRqBANwIF
OnQKdOgU6MgioPfJbASSh7OKQKUzGaUz
GaUbgZRuBFI6dFI6dNL+0CnYgsBLL9yV
QLgRSOnTZ3pr6OSuPYJp6Qje+aqHJpCe
B+HQSelGIKUbgZRO1ZRO1ZRO1ZTuk1E6
dFI6dFI6dIp0JhPpPplIH6+LdKoW+1M1
XVr9d5B+MNKNQJFuBIp0I1BEb2luAul5
EM5kIt0IFOnjdZEOnSIdWUS6ESjSjUCR
Pn0W6VQt0pmM0Y1A1h86DbTrvglk50Gj
+2SM7pOxPfhkwoLASy/clUA4kzE6kzH6
cJbRfTJGZzJGZzJGZzJG98kY3SdjdOhk
dOhkdOhkdCOQ0Y1AxodO8JVHiU7VEt0I
lOhGoEQfr0t0bJjo2DDRoVOiG4ES3QiU
9oANZUmgHEYgPQ92h05jGWJT//G60QTe
1ul0/Yjr0jV4kCHlRMeGqT82HO4UpUdR
uNMp0Z1OiY4NU39sOFaQyXsYr1ta1hEP
kQcz3emU6VQt06lapjOZTB+vy3QrV6ZT
tbyHpVWDJfruTGastWO5v5VruCPI7iYy
3cqV6UurMt3KlelWrky3cmU6Vct0qpbp
Vq4Mt3K5Ce50qgLRebAKROfBKhAdRatA
dDdRBaK7iSoQnQeLQDZVqwLpUbS702ko
JlMForuJKpCeB29L1S59zB0Kp7EEdt/p
NNo1eFun07VWrri0KOBQR5DeTbCpWhWI
pmpVIL2bYDudqkB6N4GnauwBSefo0MnB
rVxVILtdcnRs6PpjQ12ykegBbCRFIBw6
ObiVqwpkt0uOjg0dfK9aFchulxzcylUF
stslB18FXwXS8yAcOjm4lasK7N1NDLWG
ugpkpwmhO52E7nQSOnQSOnSS/tBprBug
Qnc6SX+n02hHsDt0CmlB4KUX7kxg7zw4
3BGk50G400noTiehQyehQyehQyehO52k
v9NprI5e6NBJ4ON1VWD3ILNoiJ0OIhBu
BBK0ESg+mhwaOjWB4GK7CQRH0Sawu09m
nIa3CQQX200guNjeCCQjiyYQjO6bQHCx
3QTS8yAZOm0Ekm0kTSC92CYzmSawd5oY
rVQjI4smEEy2m0B6sU2ePtsIJEOnIpBt
BGoC2d0E2wjUBLK7CaFTtX0YgdyCwEsv
3JXA7kagsWpRthGoCey+SHzpgRp3/2Cp
JpCeB8k+mSaQ3U2wfTIbgXCqtgefzHAC
6cU2eTirCaQX23Cq5unIwtORhe+PLMbq
JvweZpeG6iY8HVl4uo3E05GFRy/MaQLp
eRCOLDzdRuLpTMbTmYynMxlPZzKezmQ8
ncl4OpPxdCbj+UwG7pPx9OGsgF551ASy
26VAZzIB/eyzJpDdTQQ6VQvoNdRNILub
CPThrEAfzgp0qhboVC3QmUygT58F+vRZ
oFO1QKdqgc5kAn36LNCnzwKdqikdOil9
OEvpw1lKp2pKp2pKp2pKdzop3emkdKqm
dKqmdKqmdKeT0p1OSqdqSqdqSqdqSjcC
KR06KR06KR06RTqTiXQjUKTPD0Y6VYt0
qhbpTCbSnU6R7nSKdKoW6VQt0plMpDud
It3pFOlULdKRRaQbgSLdCBTp43WRTtUi
n8nAjUBGh05Gh05G98kY3SdjdCZjdCZj
dCZjdJ+M0X0yRmcyRmcyRmcyRvfJGN0n
Y3ToZHToZHToZHQjkNGNQEaHTsaHTqPv
dPrmyx8ef6rAuopk+GtwvcDBr8HVAke/
BtcLZF+DbniyvV7g4KXaeoGDI4v1AgdH
FusFDl5srxfITvRueDPeeoGDI4v1AgdH
FusF0vPg6Oh+vUB8FGW3S254dL9eID0P
jo7u1wukR9HR3YbrBfbuJmTpyVnySU/O
Wi+QnSZkeDPeeoHsYlvo0Eno0Eno0EmG
dxuuFji623C9QHaxLXToJHToJMO7DdcL
ZKN7oUMnoUMnoUMn6e82HKvhleHdhusF
0tPE6HbK1QLhRiBBG4Hqons0dGoCwcV2
EwiOok0gOIo2geBieyOQzGSaQHqQIdtI
mkAw2W4C6WmCzGQ2AskuiyaQXouOPiC5
XiA7TQjaJ9MEstMEm8k0gexuwtM7eo+2
kTSB7Cjq6UzG05mMpyMLj7aRNIHsbsLT
oZOnQydPRxYebSNpAsE2ko1Aso2kCaTn
QTiT8XQm4+lMxtOZjOczGbhPJqAX5jSB
7HYp0LFhoDOZQPfJBPRw1kYgnKoFOlUL
dCYT6EagQDcCBTpVC3RkEehGoEA3AgU6
dAp06BToyCKg98lsBJKHs4pApTMZpTMZ
pRuBlG4EUjp0Ujp0Ujp0UroRSOlGIL01
dHLXCgyDCaTnwe7QSZeWdeidL+toAntH
0dEEwo1ASqdqSqdq2p+qDXcN0ottOHRS
OnRSOnSK/ZnMWEEm9vfJ6NIR1AMdwd73
JoY7gt27Cbd0DbrDCGRTtUhnMrG/EWi0
axBuBIp0qhbpVC3SjUCRbgSK9PG62B86
aVxKE/EwAulBBm4EinQjUKRPn8U9UDVZ
EigHEQhnMtbfCKRL9wcvvXBnAnsX2zIt
CLz0wp0JZOdBo/tkjO6TMTqTMfpwltGZ
jNGHs4zukzE6kzE6kzE6kzG6T8boPhmj
QyejQyejQyejG4GMbgQyPnSCrzxK9PG6
RN+znejjdYluBEp0bJjo2DDRoVOiG4ES
3QiU6Ngw7QEbLs4PHuL+YKJDp0Tfs51u
7XSSawX6wQTS8yAcGyY6dEp0p1Oizw8m
OjZMdGyY6SuPMt3plOlULdOpWqYzmUy3
cmW6lSvTqVqmU7VMt3JlupUr061cmU7V
Mp2qZTqTyXQrV6ZbuTKdqmU6Vct8JsO2
crkJ7nSqAtF5sApE58EqEB1Fq0B0N1EF
9u4mhlp5VAWi82ARyKZqVSA9irKdTlUg
upuoAul58LZU7VKw36FwWhB4oDTBXiRe
Bd6ym/i8nlZQBdK7CTZVqwLpeZDtdKoC
6d0Em6oVgXiqxh6QdI4OnRzcylUFstsl
R8eGjo4NHR06ObiVqwpkt0uuPzYcqxZ1
/bHhcALxURRt5aoC0c8fLALh2ND1H5Ac
7gjSoyjbylUF9u4mZOkIyoGOIDtNCN3p
JHSnk9Chk9Chk/SHTrq06uHSC3cmEB9F
2fcmhA6dhA6dpD90GqvYFrrTSehOJ6FD
J6FDJ4HPDxaB3Z1Oo0XR7k6nsTp6oTud
BD5eVwV2DzJLhthLL9yVQLgRSNBGoPho
cmjo1ASCi+0mEBxFm8DexfZAlUwTCC62
m0DwHd6NQDKyaALB6L4JBBfbTSA9D5Kh
00ZgdxvJaEGmv41knLtLTSC92CZPnzWB
+CADJttNIL3YJk+fbQSSoVMRyDYCNYHs
NLEHI9BYpdoejEDDHUF2omcbgTYCyUag
JpB9b2IPRqDhBNLzINkn0wSyuwm2T2Yj
kDyc1QTSgwzZJ9ME0ottOFUTOlXzdGTh
6cjCo2eXmkB2FPV0ZOHpyMLTkYWnIwtP
Rxaejiw83Ubi6UzG05mMpzMZT2cyns5k
PJ3JeDqT8XQm4/lMBu6T8fThrIBeedQE
stulQGcyAb3EuAlkdxOBTtUCnaoFOpMJ
9OGsQB/OCnSqFuhULdCZTOg/fTacQHqx
DadqgU7VAp3JBPr0WaBPnwU6VVM6dFK6
00npTielUzWlUzWlUzWlO52U7nRSOlVT
OlVTOlVTutNJ6U4npVM1pVM1pVM1pRuB
lA6dlA6dlA6dIp3JRLoRKNLnByOdqkU6
VYt0JhPpTqdIdzpFOlWLdKoW6Uwm0p1O
ke50inSqFunIItKNQJFuBIr08bpIp2qR
z2TgRiCjQyejQyej+2SM7pMxOpMxOpMx
OpMxuk/G6D4ZozMZozMZozMZo/tkjO6T
MTp0Mjp0Mjp0MroRyOhGIKNDJ+NDp+F3
On3/xx8+VWBdRTL+Nbha4OjX4FqBw1+D
qwWyr0E3PtleLXD0Um21wNGRxWqBoyOL
1QJHL7ZXC2Qneje+GW+1wNGRxWqBoyOL
1QLpeXB4dL9aID6KstslNz66Xy2QngeH
R/erBdKj6PBuw9UCe3cTkhcEXnrhzgSy
04SMb8ZbLZBdbAsdOgkdOgkdOsn4bsO1
Aod3G64WyC62hQ6dhA6dZHy34WqBbHQv
dOgkdOgkdOgk/d2GYzW8Mr7bcLVAepoY
3k65ViDcCCRoI1BddI+GTk0guNhuAsFR
tAkER9EmEFxsbwSSmUwTSA8yZBtJEwgm
200gPU2QmcxGINll0QTSa9HhByRXC2Sn
CUH7ZJpAdppgM5kmkN1NeHpH79E2kiaQ
HUU9ncl4OpPxdGTh0TaSJpDdTXg6dPJ0
6OTpyMKjbSRNINhGshFItpE0gfQ8CGcy
ns5kPJ3JeDqT8XwmA/fJBPTCnCaQ3S4F
OjYMdCYT6D6ZgB7O2giEU7VAp2qBzmQC
3QgU6EagQKdqgY4sAt0IFOhGoECHToEO
nUJ/ZBHcgsBLL9yZwN7FdghLAsNBBJKH
s4pApTMZpTMZpRuBlG4EUjp0Ujp0Ujp0
UroRSOlGIL01dHLXClwq1Q4lkJ4H4dBJ
6UYgpRuBlE7VlE7VlE7VlO6TUbpPRtFL
jDcC4dAp0plMpPtkIn28LtKpWqRTtdif
yQRbEHjphbsS2N0IpEvLG/XOlzc2gb27
ieEE0vNgd6qmfukI+sMI7B1Fx7oBGulG
oEgfr4v9oZMumRD0ECaE2B9ZjHYNwo1A
kW4EivTps0inapHOZIxuBLL+0GmgXfdN
IBs6Gd0nY3SfjO2BycSlUi0eRCB8OMvo
TMb6D2cNJ5BdbBvdJ2N0JmN0JmN0n4zR
fTJG98kYHTpZf+g0WqLvbgQaTiDbCGR8
6ARfeZT6U7WxrsHU3wikaUlgOoxAdruU
+huBxrpHn+jYMNGxYaJvBEp0I1DagxFo
sDwI3+mU6Ngw0aFToo/XpVs7neRagUuV
zKEE0vMgHBsmutMp0Z1Oie50SvT5wUTH
hpk+XpfpTqdMH6/LdKqW6Uwm01ceZbqV
K9OpWqZTtUxnMplu5cp0K1emU7VMp2qZ
zmQy3cqV6VauTKdqmU7VMp/JsK1cboLP
D1aB6DxYBaLzYBWIjqJVILqbqALR3UQV
iM6DRSCbqlWB9CjKdjpVgehuogqk58Hb
UrVLnsEdCqcFgYdwGxaB/Xc6jWSnrAJv
2U1c/7SCpaVVUQ4jsHc3MdwRpOdBNlWr
AundBNvpVAXSuwk8VWMPSDpHh04ObuWq
AtntkqNjQ0fHho4OnRzcylUFsm++ODo2
dHvYqzbSmHkV2DuKDieQ3S45uJWrCGQ/
f7AKRK9bqQLp3QTbylUF9u4mhlpDXQWi
14452YPTafFJygd43kQVyC62hQ6dhA6d
hA6dhO50ErrTSejQSejQSfYAnWRJ4CFM
CEJ3Ognd6STw+cEqkJ4H4dBJ4PODVWDv
bmKsjl76Q6fROnr2eF0VSA8ycCOQoI1A
8dHk+kOngW6fNYHgYrsJBEfRJhBcbDeB
4GK7CeydBweqZDYCyT6ZJhCM7ptAMLpv
Aul5sDt0Gi3IkG0kTSC92CYzmSaQnibI
yKIJpBfb5OmzJhAMnTYCydCpCGQbgZpA
dppgG4GaQHY3IXSqxjYCbQSSjUBNILtd
2ocRaDSB9DxI9sk0gexugu2T2QiEU7U9
+GSGE0gvtsnDWU0gvdiGUzW/BxvJUDdf
PB1ZeDqy8HRk4fsji+GuQbaNxNORhacj
C09HFp6OLDx6dqkJpNeicCbj6UzG05mM
pzMZT2cyns5kPJ3JeD6TgftkPH04K6Af
7tYEstulQLeRBPpwVqAPZwW6ESjQqVqg
M5lAH84K9OGsQKdqgU7VAp3JBPr0WaBP
nwU6VQt0qhboTCbQp88Cffos0Kma0qGT
0p1OSnc6KZ2qKZ2qKZ2qKd3ppHSnk9Kp
mtKpmtKpmtKdTkp3Oimdqimdqimdqind
CKR06KR06KR06BTpTCbSjUCRPj8Y6VQt
0qlapDOZSHc6RbrTKdKpWqRTtUhnMpHu
dIp0p1OkU7VIRxaRbgSKdCNQpI/XRTpV
i3wmAzcCGR06GR06Gd0nY3SfjNGZjNGZ
jNGZjNF9Mkb3yRidyRidyRidyRjdJ2N0
n4zRoZPRoZPRoZPRjUBGNwIZHToZHzqN
vtPp23/+w+NPFVhXkQx/Da4XOPg1uFrg
6NfgeoHsa9ANT7bXCxy8VFsvcHBksV7g
4MhivcDBi+31AtmJ3g1vxlsvcHBksV7g
4MhivUB6Hhwd3a8XiI+i7HbJDY/u1wuk
58HR0f16gfQoOrrbcL3A3t2ELD0aTD7p
0WDrBbLThAxvxlsvkF1sCx06CR06CR06
yfBuw9UCR3cbrhfILraFDp2EDp1keLfh
eoFsdC906CR06CR06CT93YZjNbwyvNtw
vUB6mhjdTrlaINwIJGgjUF10j4ZOTSC4
2G4CwVG0CQRH0SYQXGxvBJKZTBNIDzJk
G0kTCCbbTSA9TZCZzEYg2WXRBNJr0dEH
JNcLZKcJQftkmkB2mmAzmSaQ3U14ekfv
0TaSJpAdRT2dyXg6k/F0ZOHRNpImkN1N
eDp08nTo5OnIwqNtJE0g2EayEUi2kTSB
9DwIZzKezmQ8ncl4OpPxfCYD98kE9MKc
JpDdLgU6Ngx0JhPoPpmAHs7aCIRTtUCn
aoHOZALdCBToRqBAp2qBjiwC3QgU6Eag
QIdOgQ6dAh1ZBPQ+mY1A8nBWEah0JqN0
JqN0I5DSjUBKh05Kh05Kh05KNwIp3Qik
dOikdOikdOikdCOQ0o1ASqdqSqdqSqdq
SvfJKB06KR06KR06xf5MRpfWjumdrx1r
AtlRNNLH6yKdqkU6VYt0JhPpRqBINwJF
OlWLdKoW6Uwm0o1AkT5eF+nQKfZHFqN1
E92NQGHpCIYDHUF6sQ2fPot7oGqyJFAO
IhDOZKy/EWisKGr9oZNbOkXdIU5R6w+d
hjuCvYOMLl2DeqBrkF2LGp3JGJ3JGJ3J
GN0nY3SfjNGZjNGZjNGZjNF9Mkb3yRjd
J2P06TOjQyejG4GMbgQyPnSCrzxKdKqW
6EagRDcCJboRKNHH6xIdGyY6dEp0I1Dq
bwQaLQ/CsWHqjw3Vlo6gHUZg7ygawoLA
Sy/cmcDe7dJo1+BtnU7u2mtw6Qge6BqE
Y8NEx4aJDp0S3emU6PODiY4NEx0bZvrK
o0wfr8t0qpbpVC3TmUymW7ky3cqV6VQt
06laplu5Mt3KlW9t5XKf7mk+lEJ6IoRv
rcp0KJPpXq5M93JlOlbLdKyW+VCG7eVy
E3yTeBWI3lpVBaJv0leBvaOo+iWB/jAC
0TcnqkB0N1EFovNgEcjGalUgupuoAtHd
RBWI7iaqQHQ3UQXeMg9e+pg7FE5jCWQv
daoCb9lNfF5erioQTdWqQHoeZFO1KpDe
TbCtTlUgvZvAUzX2hKRzcC9XFchOEw7u
5aoC2e2S28NitaHAr6NDJwf3clWB7HbJ
0bGho2NDB1/LVQX2bpc0LaWJdBiB6AnJ
IhCODR18QrIKpEdRtpWrCuzdTchSkJED
BRk2VRO600ngA4RVILvYFjp0Ejp0ErrT
SehOJ6FDJ6FDJ6FDJ6E7nYTudBI6dJI9
QKehPNsCnx8sArs7ncaiatLf6TRWRy90
6CTw8boqkB5k4EYg6W8EGiiKxkeTQ0On
JhBcbDeB4CjaBIKL7SYQDJ2aQHAe3Agk
I4smEIzum0Bwsd0E0vMgGTptBPa3kbgl
ge4wAsFkuwmkpwny9FkTiA8y9GKb7JNp
AunFdnfo5JaGlN2dPzKkCNyDEWigmy9N
IDtNsI1ATSC7mxA6VduDEWgsss02AjWB
7HaJbQRqAruv/hss0ZN9Mk1g98fXLT2f
L9z58/maQHY3IejhrCaQXmyTfTJNIL3Y
hlM1oVM1T7eReDqy8HRk4enIwtORhafb
SDwdWXg6svB0ZOHpRiBPt5F4OpPx6Gef
NYH0PAhnMp7OZDydyXg6k/F0JuP5TIY8
nNUEsruJgF551ASy26VAZzKBPpwV6MNZ
gU7VAp2qBTqTCfThrEAfzgp0qhboVC3Q
mUygT58F+vRZoFO1QKdqgc5kAn36LNCn
zwKdqikdOind6aR0p5PSqZrSqZrSqZrS
nU5Kdzopnaopnaopnaop3emkdKeT0qma
0qma0qma0o1ASodOSodOSodOkc5kIt0I
FOnzg5FO1SKdqkU6k4l0p1OkO50inapF
OlWLdCYT6U6nSHc6RTpVi3RkEelGoEg3
AkX6eF2kU7XIZzJwI5DRoZPRoZPRfTJG
98kYnckYnckYnckY3SdjdJ+M0ZmM0ZmM
0ZmM0X0yRvfJGB06GR06GR06Gd0IZHQj
kNGhk/Gh0+g7nX74/utPFlhXkQx/Da4X
OPg1uFrg6NfgeoHsa9ANT7bXCxy8VFsv
cHBksV7g4MhivcDBi+31AtmJ3g1vxlsv
cHBksV7g4MhivUB6Hhwd3a8XiI+i7HbJ
DY/u1wuk58HR0f16gfQoOrrbcL3A3t2E
LD3NXD7paebrBbLThAxvxlsvkF1sCx06
CR06CR06yfBuw9UCR3cbrhfILraFDp2E
Dp1keLfheoFsdC906CR06CR06CT93YZj
NbwyvNtwvUB6mhjdTrlaINwIJGgjUF10
j4ZOTSC42G4CwVG0CQRH0SYQXGxvBJKZ
TBNIDzJkG0kTCCbbTSA9TZCZzEYg2WXR
BNJr0dEHJNcLZKcJQftkmkB2mmAzmSaQ
3U14ekfv0TaSJpAdRT2dyXg6k/F0ZOHR
NpImkN1NeDp08nTo5OnIwqNtJE0g2Eay
EUi2kTSB9DwIZzKezmQ8ncl4OpPxfCYD
98kE9MKcJpDdLgU6Ngx0JhPoPpmAHs7a
CIRTtUCnaoHOZALdCBToRqBAp2qBjiwC
3QgU6EagQIdOgQ6dAh1ZBPQ+mY1A8nBW
Eah0JqN0JqN0I5DSjUBKh05Kh05Kh05K
NwIp3Qikt4ZO7lqBYTCB9DwIh05KNwIp
3QikdKqmdKqmdKqmdJ+M0qGT0qGT0qFT
pDOZSPfJRPp4XaRTtUinapHOZCLdCBTp
RqBIp2qRTtUinclEuhEo0sfrIh06RTqy
iHQjUKQbgSJ9+izSqVqkMxmjG4GsP3Qa
aNd9E8jOg0b3yRjdJ2N0JmN0JmN0JmP0
4Syj+2SMzmSMzmSMzmSM7pMxuk/G6NDJ
6NDJ6NDJ6EYgoxuBjA+d4CuPEp2qJboR
KNGNQIluBEp0bJjo2DDRoVOiG4ES3QiU
6Ngw0bFhokOnRB+vS7d2Osm1Av1gAul5
EI4NEx06JbrTKdGdTomODRMdG2b6eF2m
O50ynaplOlXLdCaT6VauTLdyZTpVy3Sq
lulMJtOtXJlu5cp0qpbpVC3TmUymW7ky
3cqV6VQt06la5jMZtpXLTXCnUxWIzoNV
IDoPVoHoKFoForuJKhDdTVSB6DxYBLKp
WhVIj6Jsp1MViO4mqkB6HrwtVbv0MXco
nMYS2H2nky7tstAD7LKoAm/ZTVz7tAJd
OkU1HkYgvZtgU7UqsHce1CW34aUX7kxg
725CZUmgHEZg725iuChK7ya6U7XRjmD3
AUldemrPpRfuRqDrD53C0hEMhziCrr+V
a6xT1MGtXFUgu11ye8CGi3MTh6hkHB06
ObiVqwpkt0uuPzYcq5twdGzo4FauKrB7
uzTaKdq7XRpNYP9V8IMl+u4DkmHpGgyH
uQbh0MnBrVxVYO9uYqg11FUg+x690J1O
Qnc6Cd3pJPD5wSqQnSaE7nQSutNJ6NBJ
6NBJ6NBJ+judxro3IXSnk9Chk+wBOrml
U9QdRmB36BSXBB7C6SR0p5P0dzqN1dEL
fH6wCqSnCbgRSOjjdYLelB4fTQ4NnZpA
MLJoAsFRtAkEF9tNIBg6NYHgPLgRSEYW
TSAY3TeB4GK7CaTnQTJ02gjsbyMZTSC9
2CYzmSaQnibIyKIJpBfbZJ9ME0gvtsnQ
qQhkG4GaQHaaYBuBmkB2NyF0qsY2Am0E
ko1ATSC7XWIbgZpA8PRZE8juJvbgkxlO
ID1NwKnaHnwywwmkF9vk4awmkF5sw6ma
pyMLT0cWno4sPB1Z+P7IYqCxgiaQXYt6
OrLw/ZHFaKcoHFl4uhHI020kns5kPJ3J
eDqT8XQm4+lMxtOZjKczGU9nMp7PZOA+
GU8fzgrolUdNILtdCnQbSaAPZwX6cFag
U7VAp2qBzmQCfTgr0IezAp2qBTpVC3Qm
E+jTZ4E+fRboVC3QqVqgM5lAnz4L9Omz
QKdqSodOSnc6Kd3ppHSqpnSqpnSqpvTh
LKU7nZRO1ZRO1ZRO1ZTudFK600npVE3p
VE3pVE3pRiClQyelQyelQ6dIZzKRbgSK
9PnBSKdqkU7VIp3JRLrTKdKdTpFO1SKd
qkU6k4l0p1OkO50inapFOrKIdCNQpBuB
In28LtKpWuQzGbgRyOjQyejQyeg+GaP7
ZIzOZIzOZIzOZIzukzG6T8boTMboTMbo
TMboPhmj+2Ts1tDJXytQBhNIz4Nw6GR0
I5DRjUBGh07Gh06j73R6/PhPXy8I9NN1
CusukuEvwg4KB78K1ysc/TLsoBB+Hbrh
8XYHhYMXbB0UDk4uOigcHF10UDh41d1B
ITzju+F9eR0UDn6vsIPCwflFB4X4fDg6
ye+gkB9L4d2TGx7md1CIz4eje0g7KMTH
0tE9iB0U9u4tZOl5U/JJz5vqoBCeLWR4
l14HhfDKW/AkSvAkSvAkSob3Iq5XOLoZ
sYNCeOUteBIleBIlwxsSOyiEU33BkyjB
kyjBkyjp70ocrAOW4W2JHRTis8Xozsv1
CumOIWE7hupafDaJagrJlXdTSI6lTSE5
ljaF5Mp7oxDNaZpCfKRBu02aQjLzbgrx
2QLNaTYK0V6MphBfl44+W9lBITxbCNtP
0xTCswWc0zSF8N7C43t8z3abNIXwWOrx
nMbjOY3HUwzPdps0hfDewuNJlMeTKI+n
GJ7tNmkKyW6TjUK026QpxOdDOqfxeE7j
8ZzG4zmNvwechu6nCewNPE0hvHsKeJoY
8Jwm4P00gT3ZtVFIZ20Bz9oCntMEvGMo
4B1DAc/aAp5iBLxjKOAdQwFPogKeRAU8
xQjs/TQbhejJrqJQ8ZxG8ZxG8Y4hxTuG
FE+iFE+iFE+iFO8YUrxjSPEkSvEkSvEk
SvGOIcU7hhTP2hTP2hTP2hTvp1E8iVI8
iVI8iYp4ThPxfpqIn86LeNYW8awt4jlN
xDuGIt4xFPGsLeJZW8Rzmoh3DEX8dF7E
k6iIpxgR7xiKeMdQxM+uRTxri3hOY3jH
kPUnUSPt1W8K4fnQ8H4aw/tpDM9pDM9p
DM9pDO+nMbyfxvCcxvCcxvCcxvB+Guvv
p9GlylsPVHnTSZThSZThSZThHUOGdwzZ
PSBR9C1KCc/aEt4xlPCOoYR3DCU8TUx4
mpjwJCrhHUMJ7xhKeJqY+tNE9Us9vj+Q
wt6xVKclhdOBFMK7p4T3RCU8TUz46byE
J1GpvydqMCKc8J6ohKeJCU8TM346L+M9
URnP2jKetWU8p8l411fGu74ynrVlPGvL
eNdXxru+Mt71lfGur4xnbRnPaTLe9ZXx
rq+MZ20ZP3+Y7wGngbu+3ET3RFWF7HxY
FbLzYVXIjqVVIbu3qArZvUVVyM6HRSGc
tVWF3WOpLCmUAylk9xZVIbu3qArx+fC2
rO3S59whccn1dSiFcE9UVYjvLeCsrSrE
9xZw1lYV4nsLuCeqKsT3FnzWBp+wdA5P
ohzd9VUVwrsnh6eJDk8THZ5EObrrqyqE
d0+OPmFZFeLzIdz1VRXCuydHd30VhXSa
6PA00eFJlKO7vqrC3r3FWNuuq0J4thC8
J0rwnijBkyjBkyjBkyjBe6IE74kSPIkS
PIkSPIkSvCdK+nuixtptUhXCSZTgSZTg
SZTgPVHS3xM1WI8veBIl9Om8qrB3pBku
H9IdQ8J2DMVHk2OTqKawd+Udlq7DcPfX
YVNIjqVNIbnybgrJlXdTSK68NwrRFKMp
JFP9ppBceTeFZBLVFOLzYXe3iaal3iId
SGHvynuk7qkpxGcL9OxaU0hm3k0hvvLu
7qcZ7xjiK+/uJMovPYXF3/1TWIpCuGOo
KYRnC7hjqCmEszbBsza4Y2ijsLtjSG0p
49uBFMLvW+zBMTRY1SZ41gb30zSFcKoP
99NsFKInu5pCfKRB+2maQjjVhzuGmkJ4
PvR4iuHxFMPjKYZnzz01hfBY6vEUw7Pn
nppCOMXweMeQx1MMj3ebeDyn8XhO4/Gc
xuM5jcdzGo/nNB7PaTye0/h7wGnQk11N
Iby3COwtSk0hvHsKeE4T8JNdAT/ZFfCs
LeBZW8BzmoCf7Ar4ya6AZ20Bz9oCntME
9qbkphBfedNZW8CztoDnNIG9C7opxPcW
dNameBKleE+U4j1RimdtimdtimdtivdE
Kd4TpXjWpnjWpnjWpnhPlOI9UYpnbYpn
bYpnbYp3DCmeRCmeRCmeREU8p4l4x1DE
zx9GPGuLeNYW8Zwm4j1REe+JinjWFvGs
LeI5TcR7oiLeExXxrC3iKUbEO4Yi3jEU
8dN5Ec/a4j3gNHTHkOFJlOFJlOH9NIb3
0xie0xie0xie0xjeT2N4P43hOY3hOY3h
OY3h/TSG99MYnkQZnkQZnkQZ3jFkeMeQ
4UmU3QMSNfyeqG/+5dtPVlgXm4x/Ha5X
OPp1uFrh8NfheoXw69CNz7zXKxy9aluv
cHSKsV7h6BRjvcLRK+/1CuEZ343v3Fuv
cHSKsV7h6BRjvUJ8Phye6q9XyI+l8O7J
jU/11yvE58Phqf56hfhYOrw3cb3C3r2F
LD21Wj7tqdXrFcKzhYzv3FuvEF55C55E
CZ5ECZ5EyfjexNUKh/cmrlcIr7wFT6IE
T6JkfG/ieoVwqi94EiV4EiV4EiX9vYmD
dcAyvjdxvUJ8thjefblaId0xJGzHUF2q
zyZRTSG58m4KybG0KSTH0qaQXHlvFKI5
TVOIjzRot0lTSGbeTSE+W6A5zUYh2ovR
FOLr0uEnLNcrhGcLYftpmkJ4toBzmqYQ
3lt4fI/v2W6TphAeSz2e03g8p/F4iuH7
u03ULyi89MLdKezdW4ynsHc+DEtnaTjU
WYrPh3SK4dluk6aQ7DbZKES7TZpCOGvz
eE7j8ZzG4zmNx3Mafw84Dd1PE9gbeJpC
+J2ZgKeJAc9pAt5PE9iTXRuFdNYW2JNd
TSG8twh4x1DAO4YCnrUFPMUIeMdQwDuG
Ap5EBTyJCniKEdj7aTYK0ZNdRaHiOY3i
OY3iHUOKdwwpnkQpnkRpfxIVbEHhpRfu
TCHdMaT42TXtT6J0aWuE3v3WiKYQnw/p
JErxjiHFO4YUz9oUz9oUz9oU76dRPIlS
PIlSPImKeE4T8X6aiJ/Oi3jWFvuztsH6
w4h3DEW8YyjiHUORvQu6KcTnQzqniXjH
UMRP50U8iYp4ihHxjqGIdwxF/OxaxLO2
iOc0hncMWX8SNdJe/aYQng8N76cxvJ/G
9uCnCQsKL71wZwrpnMbwnMbwk12G99MY
ntMYntMYntMY3k9jeD+N4UmU4UmU4UmU
4R1DhncM2T0gUfQtSgnP2hLeMZTwjqGE
n85LeJqY8DQx4UlUwjuGEt4xlPZAE2VJ
oRxIIT4fdidRgzloU//pvOEUdvdEDTbp
nPA0MfWnieOdpfhYSvdEJbwnKuFpYupP
EweLNHkP03lL+TAeJB9mvCcq41lbxrO2
jOc0GT+dl/Gur4xnbXkPm7BGy/jdOc1g
28xyf9fXeMcQ3ltkvOsr4zdhZbzrK+Nd
Xxnv+sp41pbxrC3jXV+Z7vpyE90TVRWy
82FVyM6HVSE7llaF7N6iKmT3FlUhOx8W
hXDWVhXiYyl8Y3lVyO4tqkJ8Prwta7v0
OXdInAZT2N8TNdp1CPdEVYX43gLO2qpC
NmurCvG9BdwTVRXiews+a4NPWDqHJ1GO
7vqqCuHdk8PTRNefJuqS20QP4TYpCukk
ytFdX1UhvHtyeJro6PvaqkJ49+Torq+q
EN49OfrW+aoQnw/pJMrRXV9VYe/eYqxt
11UhPFsI3hMleE+U4EmU4EmU9CdRg90h
FbwnSvp7ooY7ht1JVEgLCi+9cHcKe+fD
8Y4hPh/SPVGC90QJnkQJnkQJnkQJ3hMl
/T1Rg/X4gidRQp/Oqwq7R5pFB+10GIV0
x5CwHUPx0eTYJKopJFfeTSE5ljaF3f00
A3XATSG58m4KyZX3RiGaYjSFZKrfFJIr
76YQnw/RJGqjEO02aQrxlTea0zSFvbPF
cFUbmmI0hWTm3RTiK2/07NpGIZpEFYVw
x1BTCO8t4I6hphDeWwiete3DMeSWqjZ3
GIXdHUOD1aVwx1BT2H1j+UBPtGoK8fkQ
7adpCuG9BdxPs1FIZ2178NOMpxBfeaMn
u5pCfOVNZ20eTzE8nmL4/hRjsN7C72Hu
aazewuMphse7TTyeYnj2Bp6mEJ8P6RTD
490mHs9pPJ7TeDyn8XhO4/GcxuM5jcdz
Go/nNP4ecBq6n8bjJ7sCe4tSUwjvngKe
0wT2c9eaQnhvEfCsLbC3XTeF8N4i4Ce7
An6yK+BZW8CztoDnNAE/uxbws2sBz9oC
nrUFPKcJ+Nm1gJ9dC3jWpngSpfjJLsVP
dimetSmetSmetSneE6V4T5TiWZviWZvi
WZviPVGK90QpnrUpnrUpnrUp3jGkeBKl
eBKleBIV8Zwm4h1DET9/GPGsLeJZW8Rz
moj3REW8JyriWVvEs7aI5zQR74mKeE9U
xLO2iKcYEe8YinjHUMRP50U8a4v3gNPQ
HUOGJ1GGJ1GG99MY3k9jeE5jeE5jeE5j
eD+N4f00huc0huc0huc0hvfTGN5PY3gS
ZXgSZXgSZXjHkOEdQ4YnUXYPSNToe6K+
+fKHx5+ssC42Gf467KBw8OtwvcLRr8MO
CuHXoRueeXdQOHjV1kHh4BSjg8LBKUYH
hYNX3h0UwjO+G96510Hh4BSjg8LBKUYH
hfh8ODrV76CQH0vh3ZMbnup3UIjPh6NT
/Q4K8bF0dG9iB4W9ewtZemaXfNIzuzoo
hGcLGd6510EhvPIWPIkSPIkSPImS4b2J
6xWO7k3soBBeeQueRAmeRMnw3sQOCuFU
X/AkSvAkSvAkSvp7EwfrgGV4b2IHhfhs
Mbr7cr1CumNI2I6hulSfTaKaQnLl3RSS
Y2lTSI6lTSG58t4oRHOaphAfadBuk6aQ
zLybQny2QHOajUK0F6MpxNelo09YdlAI
zxbC9tM0hfBsAec0TSG8t/D4Ht+z3SZN
ITyWejyn8XhO4/EUw7PdJk0hvLfweBLl
8STK4ymGZ7tNmkKy22SjEO02aQrx+ZDO
aTye03g8p/F4TuPvAaeh+2kCewNPUwjv
ngKeJgY8pwl4P01gT3ZtFNJZW8CztoDn
NAHvGAp4x1DAs7aApxgB7xgKeMdQwJOo
gCdRAU8xAns/zUYherKrKFQ8p1E8p1G8
Y0jxjiHFkyjFkyjFkyjFO4YU7xhSPIlS
PInS/iRKlzZ/6N1v/mgKe8fS4RTSHUOK
Z22KZ23an7WNdx3iK286iVI8iVI8iYr9
Oc1gkSb299Po0jHUQx3D3r3FeMewe2/h
lq5DdyCFcNYW8Zwm9ncMDXcd0h1DEc/a
Ip61RbxjKOIdQxE/nRf7kyiNS9kiHkgh
PtLQHUMR7xiK+Nm1uAfWJksK5TAK6ZzG
+juGNCxli3Aghb0rb5kWFF564e4UwvOh
4f00hvfTGJ7TGH6yy/CcxvB+GsP7aQzP
aQzPaQzPaQzvpzG8n8bwJMrwJMrwJMrw
jiHDO4bsHpAo+halhJ/OS/h93gk/nZfw
jqGEp4kJTxMTnkQlvGMo4R1DCU8T0x5o
4tL9w3iQ+4cJT6ISfp93wk/nJTxNTHia
mPAkKuE9UQk/f5jwNDHhaWLGb1HKeE9U
xrO2jGdtGc9pMt71lfGur4xnbRnP2jLe
9ZXxrq+Md31lPGvLeNaW8Zwm411fGe/6
ynjWlvGsLd8DTgN3fbmJ7omqCtn5sCpk
58OqkB1Lq0J2b1EV9u4txtqiVBWy82FR
CGdtVSE+lsI9UVUhu7eoCvH58Las7VLM
3yFxWlB4qGwB90RVhfjeAs7aqkJ8PoSz
tqoQ31t090QN11vAWVtRyGdt8AlL5/Ak
ytFdX1UhvHtyeJro8DTR4UmUo7u+qkJ4
9+T608TB6lLXnyaOp5AfS9mur6qQ/fzD
opBOE11/mjjeMcTHUrjrqyrs3VvI0jGU
Qx1DeLYQvCdK8J4owZMowZMo6U+i1C/l
Q38ghfxYCr9vIXgSJXgSJf1J1GCVt+A9
UYL3RAmeRAmeRAl9/rAo7O6JGi6WdvdE
DdbjC94TJfTpvKqwe6RZctBeeuHOFNId
Q8J2DMVHk2OTqKaQXHk3heRY2hT2rrxH
qmmaQnLl3RSS7wFvFKIpRlNIpvpNIbny
bgrx+RBNojYKu7tNhos0/d0mA917agrx
lTd6dq0p5EcaMvNuCvGVN3p2baMQTaKK
QrhjqCmEZ4s9OIYGq9r24Bga7xjCMz7c
MbRRiHYMNYXw+xZ7cAyNpxCfD9F+mqYQ
3lvA/TQbhejJrqYQH2nQfpqmEF9501mb
4Fmbx1MMj6cYnj331BTCY6nHUwyPpxge
TzE8nmJ4PMXweIrh8W4Tj+c0Hs9pPJ7T
eDyn8XhO4/GcxuM5jcdzGn8POA3dT+Px
k12BvUWpKYR3TwHPaQJ7U3JTCO8tAp61
BTxrC3hOE/CTXQE/2RXwrC3gWVvAc5rA
fu5aU4ivvOmsLeBZW8BzmoCfXQv42bWA
Z22KJ1GK90Qp3hOleNameNameNameE+U
4j1RimdtimdtimdtivdEKd4TpXjWpnjW
pnjWpnjHkOJJlOJJlOJJVMRzmoh3DEX8
/GHEs7aIZ20Rz2ki3hMV8Z6oiGdtEc/a
Ip7TRLwnKuI9URHP2iKeYkS8YyjiHUMR
P50X8awt3gNOQ3cMGZ5EGZ5EGd5PY3g/
jeE5jeE5jeE5jeH9NIb30xie0xie0xie
0xjeT2N4P43hSZThSZThSZThHUOGdwwZ
nkTZPSBRw++J+v6PP3yywrrYZPzrcL3C
0a/D1QqHvw7XK4Rfh2585r1e4ehV23qF
o1OM9QpHpxjrFY5eea9XCM/4bnzn3nqF
o1OM9QpHpxjrFeLz4fBUf71CfiyFd09u
fKq/XiE+Hw5P9dcrxMfS4b2J6xX27i0k
Lyi89MLdKYRnCxnfubdeIbzyFjyJEjyJ
EjyJkvG9iasVDu9NXK8QXnkLnkQJnkTJ
+N7E9QrhVF/wJErwJErwJEr6exMH64Bl
fG/ieoX4bDG8+3K1QrpjSNiOobpUn02i
mkJy5d0UkmNpU0iOpU0hufLeKERzmqYQ
H2nQbpOmkMy8m0J8tkBzmo1CtBejKcTX
pcNPWK5XCM8WwvbTNIXwbAHnNE0hvLfw
+B7fs90mTSE8lno8p/F4TuPxFMOz3SZN
Iby38HgS5fEkyuMphme7TZpCsttkoxDt
NmkK8fmQzmk8ntN4PKfxeE7j7wGnoftp
AnsDT1MI754CniYGPKcJeD9NYE92bRTS
WVvAs7aA5zQB7xgKeMdQwLO2gKcYAe8Y
CnjHUMCTqIAnUaE/xQhuQeGlF+5OYe/K
O4QlheEwCtGTXUWh4jmN4jmN4h1DincM
KZ5EKZ5EKZ5EKd4xpHjHkOJJlOJJlOJJ
lOIdQ4p3DCmetSmetSmetSneT6N4P42y
NyVvFNJJVMRzmoj300T8dF7Es7aIZ22x
P6cJtqDw0gt3prC7Y0iXNkPq3W+GbAp7
9xbjKcTnw+6sTf3SMfQHUtg7lg52hzTi
HUMRP50X+5MoXXIq6EGcCrE/xRjuOqQ7
hiLeMRTxs2sRz9ointMY3jFk/UnUSHv1
m0I4iTK8n8bwfhrbA6eJS1VbPIxC+mSX
4TmN9ffTjKcQXnkb3k9jeE5jeE5jeD+N
4f00hvfTGJ5EWX8SNVzG7+4YGk8h3DFk
94BE0bcopf6sbbDrMPV3DGlaUpgOpBDe
PaX+jqHB7uMnPE1MeJqY8DuGEt4xlPbg
GBotH9Kn8xKeJiY8iUr46byEn85LeJqY
8DQx4T1RCe+JSnhPVMLPHyY8Tcz46byM
90Rl/HRexrO2jOc0Gb9FKeNdXxnP2jKe
tWU8p8l411fGu74ynrVlPGvLeE6T8a6v
jHd9ZTxry3jWlu8Bp4G7vtxEnz+sCtn5
sCpk58OqkB1Lq0J2b1EVsnuLqpCdD4tC
OGurCvGxFO6JqgrZvUVViM+Ht2VtlxyG
OyROCwoP4k0sCvvviRrKfVkVdu8tZOks
lQMp7N1bjHcM8fkQztqqQnxvAfdEVYX4
3oLP2uATls7hSZSju76qQnj35PA00eFp
osOTKEd3fVWF8DszDk8T3R72tQ01rV4V
9o6l4ymEd0+O7voqCuETllUhe3tLVYjv
LeCur6qwd28x1rbrqpC9zczJHjxRS8+3
iId4vkVVCK+8BU+iBE+iBE+iBO+JErwn
SvAkSvAkSvZAosZyKgjeEyV4T5TQ5w+r
Qnw+pJMooc8fVoW9e4vBenzpT6KG6/Hh
03lVIT7S0B1DwnYMxUeT60+iRrq71hSS
K++mkBxLm0Jy5d0UkivvprB3Phypptko
RPtpmkIy1W8KyVS/KcTnw+4karhIg3ab
NIX4yhvNaZpCfLZAU4ymEF95o2fXmkIy
idooRJOoohDuGGoK4dkC7hhqCuG9heBZ
G9wxtFGIdgw1hfDuaR+OoeEU4vMh2k/T
FMJ7C7ifZqOQztr24KcZTyG+8kZPdjWF
+Mqbztr8HtwmY92Z8XiK4fEUw+Mphu9P
Mca7DuFuE4+nGB5PMTyeYng8xfDsuaem
EF+X0jmNx3Maj+c0Hs9pPJ7TeDyn8XhO
4+8Bp6H7aTx+siuwnyzXFMK7p4B3mwT8
ZFfAT3YFvGMo4FlbwHOagJ/sCvjJroBn
bQHP2gKe0wT87FrAz64FPGsLeNYW8Jwm
4GfXAn52LeBZm+JJlOI9UYr3RCmetSme
tSmetSneE6V4T5TiWZviWZviWZviPVGK
90QpnrUpnrUpnrUp3jGkeBKleBKleBIV
8Zwm4h1DET9/GPGsLeJZW8Rzmoj3REW8
JyriWVvEs7aI5zQR74mKeE9UxLO2iKcY
Ee8YinjHUMRP50U8a4v3gNPQHUOGJ1GG
J1GG99MY3k9jeE5jeE5jeE5jeD+N4f00
huc0huc0huc0hvfTGN5PY3gSZXgSZXgS
ZXjHkOEdQ4YnUXYPSNToe6K+/ec/PP5k
hXWxyfDXYQeFg1+H6xWOfh12UAi/Dt3w
zLuDwsGrtg4KB6cYHRQOTjE6KBy88u6g
EJ7x3fDOvQ4KB6cYHRQOTjE6KMTnw9Gp
fgeF/FgK757c8FS/g0J8Phyd6ndQiI+l
o3sTOyjs3VvI0lPJ5JOeStZBITxbyPDO
vQ4K4ZW34EmU4EmU4EmUDO9NXK9wdG9i
B4XwylvwJErwJEqG9yZ2UAin+oInUYIn
UYInUdLfmzhYByzDexM7KMRni9Hdl+sV
0h1DwnYM1aX6bBLVFJIr76aQHEubQnIs
bQrJlfdGIZrTNIX4SIN2mzSFZObdFOKz
BZrTbBSivRhNIb4uHX3CsoNCeLYQtp+m
KYRnCzinaQrhvYXH9/ie7TZpCuGx1OM5
jcdzGo+nGJ7tNmkK4b2Fx5MojydRHk8x
PNtt0hSS3SYbhWi3SVOIz4d0TuPxnMbj
OY3Hcxp/DzgN3U8T2Bt4mkJ49xTwNDHg
OU3A+2kCe7Jro5DO2gKetQU8pwl4x1DA
O4YCnrUFPMUIeMdQwDuGAp5EBTyJCniK
Edj7aTYK0ZNdRaHiOY3iOY3iHUOKdwwp
nkQpnkQpnkQp3jGkeMeQ4kmU4kmU4kmU
4h1DincMKZ61KZ61KZ61Kd5Po3gSpXgS
pXgSFftzGl3aZqZ3v82sKYTH0oifzot4
1hbxrC3iOU3EO4Yi3jEU8awt4llbxHOa
iHcMRfx0XsSTqNifYgzXW3R3DIWlYxgO
dQzxlTd9di3ugbXJkkI5jEI6p7H+jqHB
Yqn1J1Fu6Sx1BzlLrT+JGu8Y9o40unQd
6qGuQ3hdanhOY3hOY3hOY3g/jeH9NIbn
NIbnNIbnNIb30xjeT2N4P43hZ9cMT6IM
7xgyvGPI7gGJom9RSnjWlvCOoYR3DCW8
Yyjhp/MSniYmPIlKeMdQ6u8YGi4f0mli
6k8T1ZaOoR1IYe9YGsKCwksv3J3C3t3T
cNch3ROV8DQx4WliwpOohPdEJfz8YcLT
xISniRm/RSnjp/MynrVlPGvLeE6T8a6v
jHd9ZTxry3jWlvGur4x3feVbu77idQoX
Oc3BFOLzIX0TVsZzmox3fWW86yvjWVvG
s7Z8DzgN3PXlJvrG8qqQvQmrKmTfx68K
e8dS9UsK/YEUsu9bVIXs3qIqZOfDohDO
2qpCdm9RFbJ7i6qQ3VtUhezeoiq8LWvT
6yROgymEe6KqQnxvAWdtVSE+H8JZW1WI
7y3gnqiqEN9b8FkbfMLSObrrqyqEZwtH
d31VhfDuye1hX9tYRNjhSZSju76qQnj3
5PA00eFpoqPv+qoKe3dPmpayRTqQQvaE
ZVFIp4kOTxMdnkQ5uuurKuzdW8hSpJFD
RRo4axO8J0ro84dVIbzyFjyJEjyJErwn
SvCeKMGTKMGTKMGTKMF7ogTviRI8iZI9
kKixfN5Cnz8sCrt7ogZjbdLfEzVYjy94
EiX06byqEB9p6I4h6e8YGimWxkeTY5Oo
ppBceTeF5FjaFJIr76aQTKKaQnI+3ChE
U4ymkEz1m0Jy5d0U4vMhmkRtFPZ3m7gl
he5ACsnMuynEZwv07FpTyI80+Mob7adp
CvGVd3cS5ZaeUeLu/hklReEeHEMj3Zlp
CuHZAu4YagrhvYXgWdseHEODMW+4Y6gp
hHdPcMdQU9h9q+BoGR/tp2kKuz87b+np
gOHunw7YFMJ7C2FPdjWF+Mob7adpCvGV
N521CZ61ebzbxOMphsdTDI+nGB5PMTze
beLxFMPjKYbHUwyPdwx5vNvE4zmNZz93
rSnE50M6p/F4TuPxnMbjOY3Hcxp/DzgN
erKrKYT3FoG9RakphHdPAc9pAn6yK+An
uwKetQU8awt4ThPwk10BP9kV8Kwt4Flb
wHOagJ9dC/jZtYBnbQHP2gKe0wT87FrA
z64FPGtTPIlSvCdK8Z4oxbM2xbM2xbM2
xXuiFO+JUjxrUzxrUzxrU7wnSvGeKMWz
NsWzNsWzNsU7hhRPohRPohRPoiKe00S8
Yyji5w8jnrVFPGuLeE4T8Z6oiPdERTxr
i3jWFvGcJuI9URHviYp41hbxFCPiHUMR
7xiK+Om8iGdt8R5wGrpjyPAkyvAkyvB+
GsP7aQzPaQzPaQzPaQzvpzG8n8bwnMbw
nMbwnMbwfhrD+2kMT6IMT6IMT6IM7xgy
vGPI8CTK7gGJGn1P1A/ff/3pCutik+Gv
ww4KB78O1ysc/TrsoBB+HbrhmXcHhYNX
bR0UDk4xOigcnGJ0UDh45d1BITzju+Gd
ex0UDk4xOigcnGJ0UIjPh6NT/Q4K+bEU
3j254al+B4X4fDg61e+gEB9LR/cmdlDY
u7eQpWeryyc9W72DQni2kOGdex0Uwitv
wZMowZMowZMoGd6buF7h6N7EDgrhlbfg
SZTgSZQM703soBBO9QVPogRPogRPoqS/
N3GwDliG9yZ2UIjPFqO7L9crpDuGhO0Y
qkv12SSqKSRX3k0hOZY2heRY2hSSK++N
QjSnaQrxkQbtNmkKycy7KcRnCzSn2ShE
ezGaQnxdOvqEZQeF8GwhbD9NUwjPFnBO
0xTCewuP7/E9223SFMJjqcdzGo/nNB5P
MTzbbdIUwnsLjydRHk+iPJ5ieLbbpCkk
u002CtFuk6YQnw/pnMbjOY3HcxqP5zT+
HnAaup8msDfwNIXw7ingaWLAc5qA99ME
9mTXRiGdtQU8awt4ThPwjqGAdwwFPGsL
eIoR8I6hgHcMBTyJCngSFfAUI7D302wU
oie7ikLFcxrFcxrFO4YU7xhSPIlSPIlS
PIlSvGNI8Y4hxZMoxZMoxZMoxTuGFO8Y
UjxrUzxrUzxrU7yfRvEkSvEkSvEkKuI5
TcT7aSJ+Oi/iWVvEs7aI5zQR7xiKeMdQ
xLO2iGdtEc9pIt4xFPHTeRFPoiKeYkS8
YyjiHUMRP7sW8awt4jmN4R1D1p9EjbRX
vymE50PD+2kM76cxPKcxPKcxPKcxvJ/G
8H4aw3Maw3Maw3Maw/tpDO+nMTyJMjyJ
MjyJMrxjyPCOIbsHJIq+RSnhWVvCO4YS
3jGU8I6hhKeJCU8TE55EJbxjKOEdQwlP
ExOeJiY8iUr46byE90QlPE1MeJqY8CQq
4T1RCe+JSniamPA0MeOn8zLeE5XxrC3j
WVvGc5qMd31lvOsr41lbxrO2jOc0Ge/6
ynjXV8aztoxnbRnPaTLe9ZXxrq+MZ20Z
z9ryPeA0cNeXm+ieqKqQnQ+rQnY+rArZ
sbQqZPcWVSG7t6gK2fmwKISztqoQH0vh
nqiqkN1bVIX4fHhb1nbpc+6QOA2msLsn
Spf2Yugh9mJUhb17C41LCuOBFOJ7Czhr
qwp750P1S2epP5DC3r2FypJCOZDC3r3F
eLEU31t0Z23DHcPuE5YalhSGQyh0/UlU
WDqG4SDH0PV3fQ12ljq666sqhHdPbg80
cammiQepaRyeRDm666sqhHdPrj9NHKy3
cHia6Oiur6qwe/c03Fnau3saTmH/CcvR
Mn53mhiWrsNwoOuQTqIc3fVVFfbuLcba
dl0Vwu/jC94TJXhPlOA9UUKfP6wK4dlC
8J4owXuiBE+iBE+iBE+ipL8narD7FoL3
RAmeRMkeSJRbOkvdgRR2J1FjeaIE74mS
/p6owXp8oc8fVoX4bEF3DAl+Ok/YO9nj
o8mxSVRTSKYYTSE5ljaF5Mq7KSSTqKaQ
nA83CtEUoykkU/2mkFx5N4X4fIgmURuF
6H3eTSG+8kZzmqYQny3QFKMpxFfeaD9N
U4ivvNEkqiiEO4aaQni2gDuGmkJ4byF4
1gZ3DG0Uoh1DTSG8e4I7hppC8uxaUwjv
LfbgpxlPIT5b0FnbHvw04ynEV97oya6m
EF9501mbx1MMj6cYHk8xPJ5i+P4UY6Rp
hKYQXpd6PMXw/SnGcGcpnWJ4vGPI490m
Hs9pPJ7TeDyn8XhO4/GcxuM5jcdzGo/n
NP4ecBq6n8bjJ7sCe4tSUwjvngLebRLw
k10BP9kV8Kwt4FlbwHOagJ/sCvjJroBn
bQHP2gKe0wT87FrAz64FPGsLeNYW8Jwm
4GfXAn52LeBZm+JJlOI9UYr3RCmetSme
tSmetSl+skvxnijFszbFszbFszbFe6IU
74lSPGtTPGtTPGtTvGNI8SRK8SRK8SQq
4jlNxDuGIn7+MOJZW8SztojnNBHviYp4
T1TEs7aIZ20Rz2ki3hMV8Z6oiGdtEU8x
It4xFPGOoYifzot41hbvAaehO4YMT6IM
T6IM76cxvJ/G8JzG8JzG8JzG8H4aw/tp
DM9pDM9pDM9pDO+nMbyfxvAkyvAkyvAk
yvCOIcM7hgxPouwekKjZnqjNf/9r+56H
T16+ePv69M3b+qv/xy7pm89w8v7H1nfJ
toLTX0/Pn53++OxspsHFKXlZ+AsEt/3K
7EfHnX+b52fPfzx7/eaX81eLR+DXs9fn
P52fPd393h0f5+M3lGPXvmf7/UG2jsOz
t2evX5y+Pf918zd9+5eXj96cPz17Ov/D
/3j65uzZ+Yuzk2cv32w+4BduEpu8upTE
OZ3c/M0vX759U/7ur05+fPbyyb+dPDt7
8fPbX8p36c43vT57dfb2/O37v1Iu/+x8
25uz8pHKASqHfMqTzd7z5Pzkl/Of6y94
8e7Zs+0Xnr38y46vvz8tqtx/kJOX5e91
8g/u4dY7ytdOfz47efP29O27+rke/v67
b/709Zc/fHny++/++cs/P/7HL+fvP3vx
9NXL8xebn/jvz87/7ayo+vn8efn+s/LL
np6+/uvWu9+8PX/eTuFH0xdTzkEkTaH8
M8Voc2ln//Hq9EU5Fhd/96LfSfTiJaq4
mOZ/059On58/+2v9DM9e/nzy+vxpEfDL
6et/3764nj0/aW88eXP+t/oZwuWX3724
OO/fvj4rn/RpuVJOSiAo7377+t3Z7BvO
X/x09vrsxZOi+OWzzQn0/Ze//+7bPzz+
83+f/9pXJ/Un//ZQlMN/uuMIvTp7/eTs
xdvyV3z67kk9Ncq5f1ojwqMcv0iTxqCa
NejWH6t80jctbJQz/d1PP50/OS8/48FP
5y/O3549+Hglbn3H83Kgn56cnT755eRt
kXzp2qtH/7wcrCf1h24+waUX26nx7XcP
yinx1R+/+v3jf/h667x48+7H90HuL+cv
nr78y/tYq8+vjsjHsPSpYWmK5spVkaz8
23s7xqUVccnFcn15F6YQffA5+isDUznc
sQSyyZkPMU057g5M5Q/y9ucfn59sPg0p
MEn4IubsYrZy8pnY1hV3DE3H0HSsmLpF
phpecikCJpkmyVmuDEzHiulwFdPT82Ng
Gj4wHWumfpGpHMA0lT+ndy77XP6/Y800
bM10DE6fQXA6Vk1HzvTgXlVNb1+fP3++
uQpPX5zoqydvjzHq4q1DxqhjAXWETu0b
7lMBNUycsnVhyqewFKZknDAV0uowJVO5
SKxeU5MXndQfw9QnhakSdoJzIWrIbvJT
9irhyijlYimjphRMpsmp7Y5R1EpKyokn
Ei0El7KPIYa7C1EduXh1IhzD0j7CUrmA
fDKpvKRU2ybHqPTp9ClORVEuHcskpV1O
8cqwVN5Yk0G9LINZ8nl3YMIWT/GLUBLi
pNmVuGzTlnnpGJiOgelYLx3rpftWL3Vk
4sfAdKyYho5Lx4rps6qYjqHpMwhNx5rp
WDPdr5ppXxj8GKOO5dPQIepYPn1G5dM4
UapUBWldoPrNXfutWFVSwBjRatv+8Inu
pySlDggp1vvbEscIVfW6L3VeypMEn1Is
l7RPu2JXfeMU3eRSqE6aFLLmMWNZ/aRO
ckpuCjGWa1O2PumOcitNFkrFUQ5KUtUt
C/qdFFw/nT578wmxrPxxzl7/evps6xx8
8D5knT87e5D1f/ndgxcv3z44f1H+9vWi
evDypwf11fLjnz6qse7Bq/lPrF+pP+3p
2bO3pw/+84Pp8osnT06fnf/4+rSee/WN
r8/enD5/9az82o8R6sHz8xfv3jx4WULM
61/nL/zuwf/uHjx5+fr12SaS/uZHn/3H
k3LCnhbZl11FH4P25mxNW6+0sL05O+dn
+GLgnr4wV66/HFIOuZwZ07yr6cT963n9
Tz98+YcrY+7HWexP8HIdo+4toq7GKQbz
ZhJLnLOdofIQYde5KbkgzpvmWC3mu8Lu
o/cBuhTLWZKPYmmr1Bgo7m4+avms5cIq
hXgpi1wJpnZNp5s0OQ2uXIulZZkWjKn7
KyKPkfe3kbe6incH3/KKbr3yIfhefmGx
ai5H3Ek5KbRUGiGblvP+GH7Z4XfQoteV
rjWmmMp/lZp2u5bdqnlDCdOlvS0CrJyz
8ytjoNjb6p9SzphO2eWcJMT5lXUsej+L
0Cs2Pw/nZW/aDskfq97tavguit7lWzfH
uDtS3B227C3530dz4itIELf9xlngda68
HlK9X+JD3pYwWujVcmlJLB/XB/H+2tB7
LHo3P3GgyBvy/IqeR96SS3eH3pDnh/mO
at5j9P08ou+oVW/M5ic/JfG5RNWtU3s7
+mqJutPktVTIYiJbcGKs4Fvqcwt1VN/p
5F2I8/PoWPd+FtF3qeqdFqre6UaR99ZV
r6y95XaMwSPF4GEr4Kl8tORKxAq+Dr3u
isENptYlCVOOpbAUDd5tvXOsIDyVZJe9
lI+ZYqXVxwr4c4vBpSFbCMNOdofhS1/v
WADfj0Actn/GiIF420f26cWwm7z50hjX
hn8MG2m7ryYSbFLnXDU3bMPd7Wq4OjfL
m6KUmtmnLWgxViB2ksrF5sOUS9met1qP
3cWwZOdyebOPx1p4hDi8bW+Y1cJ+2xIx
Mz5sxe0rquGpmiSlWro1e7/tqz3AnbdD
+c3uTdxVjSVEln+XOGgWd8bKgzkfSkVQ
Imn5H5PtpBAfnA9SflytHLTEs2nrneME
3g/WhxydpVLTZ4vmwrXWB7PyDTWh6KRh
weZ/LILv1vvgtx0OM+/DQvy9/MLVZXDW
lCeXpS5wm+bfd4zAxAg8auV7c/dD/Zdq
OV9r9ZAGxsC3tz8cK9+xgq+kpXtwKfit
V2b2h5uS4G61b687cMfYe1+r36MB4lj6
vv+JA0XffTsgelW+x/j7mcTfYWvfWF6d
ytkYSgyaQto5d9GCmliJwKWg1JyczwMX
v26yOh6Skjkr6vL1A2/H4nes8OuWou/l
aYyZCWI/te8d33w7xuH7WgeXti7WSrgk
iVIMX3rjtg3CVRkhi8ScSsE4biAu+USz
uhTNtGSNUuMeK+HPLhTL1nHYdkLsnj6+
/ELHUviA4Xjz3x+ev3z9mfHwQ6A8f/Px
5304xf7ufXT/eHWdvHny8lX9Me1Zzg1p
/937X/fwrJxb7zaH+uTf35XjXj7Yj389
+fCU6fYtH5LExcOnHz/+09cfVT4sgfDn
83rSv375l03aiCl/PEjl6iqn6dOPP/3n
95fwx9fPXzx5WU+vt2cnz8tFe/r25eu/
nrwqB/T8Sfmfm9NEdfbu8oHPn5Zj9vrn
zWf5eAI9fPPkl3IW1N/14XOU4lM3r77P
dzuEfPMv314lxC54WAch/sJt91sh0yoh
33z5w+Mrj4j2FCIXvtBbCwnXCPn+jz9c
eUQk9zwiEvd1RL795z9ceUTsYt9WjyOS
P/2IXCNk60nzu06ti5HkLqeWX3uxXwS3
/3jy7F0tAz7Gwqfnpz+/eFnj+ptZRCyf
qhQzfzt9XYLq+du/fYiyz//2m0rkYVG2
ybu/feV/vnv+avbV2Yc4e/KulgLvy815
fG1/7BZOt+JmC0xbEahd4lvXcrtYtq6K
dtptnV/tAL4/Uh/Tw0/nr998LPbrYS5l
ZHjkpkeSNooe/vzs5Y+nz07+5+bv9vrs
f5aMPi9jLr3+PlM9/vrrkz8+/uarr7/6
8vuTP3z1+B+//e77H776/cm33/1w8s/h
5Pdff/f9P/25FYEPa4VazpDnz09Pfn1T
fsObHRlnR1PiZsV6eX3zU377ppjnb6o/
fEd742ZparsxmZ2E17Ql3i5+xrWNww0a
gRu1Adc3AZdbgFQ3jsWYY/mf9Ylm8/fN
OoDgq1fY1XUZ4mbUcbv6d++r/2kWFma1
/OzAburE9ucvF1mpukqxNvumy8YO01jH
YZz55NTPA/01sOTK+vy21fmNavPelfkV
dfneqvKd64CCXpzQHyty98U0++KHYry8
9+IYvTo9f33+4uf62conO9u+3E7q+XO6
iX4v3/5y0q6nn1+/fPfqzcnbX85enJzV
lPExHH38oVci64dvzn9+v/zt/anVokE7
6S6+uAktF9+06O/YVcrvTI3HqNQnKolz
NWtHJzGrT3EpKkn02TSZpOppn/+4q5lE
p6hUd8nWZ2lmdSGUIOWOcenu41L2FxT2
uriU543WMS4d49JtqyUpRUicqrNUzYvG
hbgUQ9K6ylXKv92sXLmTYkkkpPLn9LG6
eyXsikm7aeYxJvWKSaKyo1aavqi5akdY
Km8/lkvHsPTJYWkKMeeoddFBFF//WYpL
zkzMa/BTMDf7S99JvSQaXN0THtNmp4E/
Rqa7j0yloo67IpNOMe2ITOXtdoxMx8j0
yZEpa9SUU7KUJotiC4GprnAPWsqW0kll
n1R2R6a98aX6bJD6xOcSFbPNFgIcG7m7
C02pJKebNnLlvelzj0tbNspjZLrzyBTE
SslUXQAuXmzcvxyYVDVMFlJwUV2eIdC7
QUy+Tl2W2FSfQhTdhSviGJjuLjDZbL7w
usBU3vvZF0zHwHTQwFRez1OMKZdLPwR1
C5FJYvZa2riQqospzqwwd1IylVjkvJ9S
XVPn/Gwu/NjN3WHJpDsqpukLH7zsiE1T
0mPRdIxNa/h3TFZOrpAs51IZ2WLVlFwM
qd6+86VqmhZC0x6KpkbD6kqzWt1NyUoc
3XlX7hiZ9huZyt9ffxuadtdM5b3xGJeO
cWlFXPJB69xVadZ8KZ2cLsWlEotinYor
TZ+pD3F3YNpDzZS/kBIUS6GUJ+9LQ5fO
Hk26KzAdm7k9l0zpYprrWso0d7DeWWCS
npFpeULhGKTuOkiFnEKeimILvj63dSFI
TdlciU++FC+5/KFlgYXvq3oSlRIm63NV
ffnIR6/lIYJUKVpv7Gkq7z2Ap+kYpKBB
SoJFF60+QbD0btOSw0ms/E20YnHnzE13
bHEqNV75kCWeakz1qbk7sfixxdtvkCqN
/Y0rqS0IcAxSxyC1tt2rdqc6t+J9XaK3
1O6ZaizhQV2JaT7dvT+8FHKbZaR+Eu/z
7lrqGKb23PBJuvHcSnnvAYyYBw9Tf/dh
JPGXEgn+Vmfly+972/xim0t5dgKdPj19
VY53/Sm/uvKnPn3y9uTHchaWrxW9m996
+uLpya/+5NW712cn//Xrr/7vL09+OXtW
zqM6k37yzb+UcLNRfvL8ZTk13jx8/wt+
OXt9Xk/hNkd6vhkTfffiyS+nL37exIS3
r9+1+cJnp7+ZT4yPJnvk2/X78NXLN+XD
nJff8n7k/9Lo4P/1+Pcn33/5od0pLb43
8VmmujDiYhXR5m3PTn9+sxXONl98+aK0
P+9enPx6+vq8nmUfO6ecQ5ik3q8XmTuv
L+02uTjFyin17vmrDzFx41V78Puvf/jd
gzflSLx9UEL0y9d//d2Dn85flD/Ngw+/
73fl1H/xphzxcio8KJ/owcfAc+Mq7Ppe
7uYh49Xrs7cfr+L6lq9++Oq7bx9/ffL4
+//+zZ9++K6Oef7j43/6/vuvHn978vhP
f/rzd//tq28e1/dc/KAa+56d//Ly5dOP
f4f/58EmcPz9rw/+vwffvnz9/PTZ//r+
C//b7x78+p+Lbv0/vn3w6tm7nx+dzy6w
jzt36t0KV9psZ+XAiJTC9uKAlCB2Mks6
5aI7+el1uWKL3r/OTrUWBuqpVD/VT+Uk
f3D+/NXrl+VcfrD58oMSAkvkffaoLkso
AWrzAR/OvvXlj6fvT+VyVp6/T6XTF9lk
0kqirK6rnpGAK3ZGPPxw+OtVcfa6Bp/y
Y+sR+Pl1jf0PL/Y8/Lby2D7ngwuquWTG
UkUn8zl+2klvX8g0ZXFepaRXcVtI88Dn
/I6cfj/Oeec3T7PXXI6M19kN34Of8zkm
y5pLv1Z9TXd8yudJrW6aS9UyMdknnvHN
khW8TVFES3ApF5Af5ZS/OswvlJmAU740
PuUw1OV7UzkmYXbz5/CnfLkEy/lewn3Q
5O76nE+b8dhs5rxLKeYV57zUPVrZJfVa
/pfGYWqbq+M886R//6wLV+8lTlZ6Wyv5
d6DTvm6pL0G2rvYsGchZ/9P+Y8NSN8Sc
PXl2Wv7r5Mmzl29Ky/Hw4/aU01evnp1v
upgLyQ9P3548O6sdROmTP/TXbYnJpTe+
X2Ny+uzVL6cn08mkm5avnDDlF52eP7/4
SO/f//pdO5U2/eCHc/TN+59dlJzs+s3l
a05Lt3VS/uDPZztj3tSOrm4l3P5MF3/A
78tJ9v0fv/r4F3z45t2T0u+++ends/az
m/D/8fEk3C55Nl/+17/7sPPl/W8vl/Lm
LJn/DV/89cNn3cguf49nL//yG/HttQ9/
gs2l86b15o/evSj/+Z8e1D/Lg5evn1ZU
8OCXv74q/+9ZiRwPzl+0l+oJel5f+/jp
P/zoD3/KmZzrtlNu7fz79ur9j7vD14Z/
lSPfAqBufXkLi/28IS+vH87esWsFZCm1
J18tCt5SiLve3GLUde8+6L5IH+oSllD9
C6EEnbT9xtna3jrQmqJEc9nLLCNfX7KU
dzx9Xs7RJ79JUBevvDp5+5eXJ41FtsNT
einxrlR2tafx6bffMkck7ou6eFfLN7Tn
ZNp8q/BvV09eftLL5b2/S03mg93bJmUK
YYq1IC1/oK1j+/Hd8zNh+e032RP88182
6i+WW508/ake6Cvf8qopk2xamXKOJZRP
U9Qrv2n+B7bNN5cPHkvh421ra3P5zosg
NkuiFyuh5p//Q5w4fzP7rj9/+V++/P2l
bYq/zeh/+nNJxHtfmfmyxO6SJF59/LX/
9P2Xfzj543d/PvnjV/+t/K/vv/yv//Tl
t7/f2m66zUX/z4Mu2vRpHhFenfz08vVJ
Sajn79laOQ0uPZL0o5d4++vLq49TmqYS
LUpxbuWXbcWMD6tdP3zP+59y3be1jHop
Nb5/pe8TnN6/eOOlyOOlneXwsTPvXPH2
wy4qLmmkBNrSe6iv96h3Pq6uNedeJ1/i
Tk1BJS9Mt3tg8+1TT30e6ZScOl80+rx1
BuzIPfKF+ZBKdizZPWTdeu7e7VPPEutZ
SD1TfYRqXeXm6jOnrss8i+/ee+IRpzlt
HiO1uYcWr/ym7cQT6xK7KCViOJVLGeuY
dwbKO1uL83eknckvPPHk0gvLiSeKWK73
iMuVaXbTvLP8Xce0c/O0Y6b1kaE++1KY
b+3y35l2rnj7QdNOcOUdMcUYa2Uf4/Yb
Z1knuqkUyjXy5NL1bAXiffQ7JbZldZv8
lryFrV3pO5NOEPUumNTxnSnErR7ltlln
EbcuZJ3qyS9XU33irEbd2R7Nz4Xlt99k
G/+qvFOulZJIUyj51LybbtPwuPrnKN8p
pQEMlxLWMe+Mk3fc1rNXdiUed+nxsx8T
z6UXlhKP+8LFza0uVyGO6fzUW048V33X
MfHcPPEsR4+dieeKtx/68dyhmv5yqcE3
Ff6uzPP+8dwhVM+upXI4NC+Stj7tjtVN
0aE+N6OcqZP3v/2O+Sn3qG62q0t/zOrt
rxi2UsVt8s41dzyWMo/FclBLQSFRvO3s
jrZOhsW37znz5NILel8arlhqjGnr1Lk6
79RLphySUPudcj6VFN859dTt7Mf00y/9
lGvhGt4Wt1DXbMpj++tXPt0mWKmk6/MB
gjedB5nl/HP1980y0OWnFe0rBW3++8OD
b66MFQ9nt7s+3gi/dJ3+p3kq29yguyzp
cuq7zDau/Ak1LW998Pc/5rpHJdQ/wftn
8Hy4gE7+8evv/uHx1yf/5buvvv2h/jXa
Dbr5czNOnrx89u55vWV49qYdvYtnL/zm
OVk3eXCC6Gx+4ybPctvO9TLfUbJoB7jp
Q9xu+Qi36x7g5qrn5+Lj3cDT/tuqItSR
4pIbStLTFOIuc/vd+t+DBkt1aVWU0p/a
0oYYV3KDSJZcLuzSq808V9f5hm71wLbb
PK7tN9OG9eGepZJQ8V59mj01YEdCD5sn
Nxft1VSTd64PPQ4m7nsVlsw3y1yzCmu2
4vUKUBdiqVuyVNN0ST6WLrx516xtuNWa
z4vXPj54rT3p5peX717feFznGCqvC5W7
L9HDTQqV0D1JcDYVeW7xQRKTlk6qPqN1
spgtzcatrnOb3V2oDEVpmoqK6lqYNV+7
QqXPmmKwIJuGZjYAc4yVdxYr63NVbxor
a9l9k1jp6/04zW6qz0VRSReXzzFWfm6x
sj7YXKPW51mV+DPbQHqoYKkxO5FUK7Fo
JdwslZXiJ+/qI4PLR/eTu/kSnTuLlS6W
UlLUpRL+6ojLlWWlhjRlk1hkq+XjHsPZ
S3cWKn24eaj04QahsnyvmeQURHTzeOtj
oPx8A+XCBXq4HRlTLG11Re4lxARbCJSl
CDM3lQ+cXYmSOd98j8+dxMn2WMc6Ij9p
CkXLVCLlxV9uV6BMOZVGXWRyqVxax+d3
zF66s6UbU77xLury3tl62EVkLF+Ulqfe
/qiQyM3HlI6h8nMLlUOhykfhC622zfrc
2CmYZd2aEL68MtZKXyOu/AlKhyt59r4R
qsr3a4csViNqnX8oITNcGS2PtPLgwbI0
KDs2FNX92OUKmf0zrz0/PvnIzTaq759e
Xrs/9hg/7yO/nMyH6rWIKfokW89F2uaX
MUpMFSloXcy/8Hjug5aadVIq5PqQuhTq
c3Nnz9Q4Aswhw2fpB26+hbK0BDcJl50A
5jFcDhEuh0OYTifvU/lMUk60NI/gl+Kl
c3W/rpiXckKG2W6EIarNFvqzC7V6rjfI
ZU6wjgxzxGipaR4XP9aa4mVeg360E+s8
tu6ZYh6j5RjRcjSOOWlN27UuK/8jzJ5B
ffnmeB1rqXvBVUuPkxdu+Bz05nipkH3O
klJdaBbS7Eb/EWSOGS6j3XgtZ3nvbC3n
vkHmMVoOES3HQpnui7QZiS8VWZLJSw5X
sEyZonOuvK2E+TR/RtsQxWXDrZPLse6f
zLkE9BSujJdHlHn4cDnt8F2W6nKaP/xh
Vl1Oe7BeykLIvN3K4WP0vJcgM8d6I9xC
svIfMi3WmqVzDzVmhlzquLwUOw9ba5Yq
05dq3iYNztyRYw4eO7d2gl7HMcPseHbj
mMfI+flEzgGZZn3m6pRDqdJS9Hmp6pxS
NjWtM7Il5MSlp64elGl6m1I0K9eOD0df
5vCRU6PfUXYuNen+BjXn7YjmMW5+RnFz
NLrpS5yZrCTpnEoXHhYft6jl47qKQFNM
dYvLgBVnOTi53qXycbNsaHZH4Ug3hwyc
QfIO59HC46/nZrdedHOkyOlcnD+Z7JaR
03Ty87g3+8E+3/jBZpZuFTsnvXHsTDqf
if2k0JmmapAsjcRU9+XGw5ecXuuNZ3Uh
1qUoYUez3hphn3zpgeo9pFx3Ka8aMP/T
4z//8NXjr3sPAkVfd+OEVDdWu+lKzGmp
yDZX2voU89xnfezV767ilJll/YJzurod
cP7PjjhartnZHaXl1j3XZ/Gpd5LLqe1m
j1/cs9198yS3n+rlcwycnQLn4hV7qMAp
UymCS187BQneT7qrV09flKKs9ME+Jas/
Lm7fSbp90bmXyFlq+XpvqFT1m4rDXyje
FTlN60NuSl0yOWfHO0SHiJyWdoxQ3jBy
WrrR9LkGkxQ2z3Vw2c2eLXaMnJ9b5Ez1
qdmx9MfBlWIvHL5d1ymH+vjMGEvvOrmt
t81KzrqcI0262YBkMp9RH6TilE1p4bSO
07url3QkF3xQl6UchjxNx4eYz166s0Z9
hwmpzu5uTVleLB8PN/AhuS/iVKOuOLH6
JJ87Gz0/hsn+YXLpEj0c1XQV7znz5V/x
ioGgXB0+UypxXqRUZquGz/dTYEoplrX8
Va0+JyhFvbLCTHU0tD5ts/wj88Vix1h5
h8vf3E6Le/n6fE7oYv+bu4HFvRxbcaWs
tFKEBDVNs1P1GC8/u3g5Gsl0Uh/eUPuV
qFP0camu9JvNR6mOZVTb0cJNoAPWlZNZ
UVo+WimSS8VuV94COpLMAaLlvNO+qCx3
TwNNd0guP8XffgyVfHbpoq+LjUoEz2pe
ZqsTLoVKqY+C9pJLIy5h9licYUrLUlFE
Xx8S4EI9UPlKh+aRXR4+VuqtYuWNDJp9
WOUxVg4RK4ejla4cDJVUhybLJ9I4f9v8
OcgpBi3t7VSiTH2S7oA3yCfRnGUq4dKV
ujJfWVYegeXhQ6XLu4ml3/r6xVOr8t0R
y2OwHCNYDscsp1TtiElSnTUUW3Rilta7
VKBeTdSXwjKtGv7ZTxceUrlSYprqXziG
qw3sR2Y5QLj0ujNelq/vuhO+/eW9M8tj
xBwjYg5HLa3GGKm7iWrYjDtsRK2+nPym
wZ2y1qmaWb80Sn055fqAukoTcopR9Ygt
Rw+Ysru+FN1dX8qN4uWtyOVsHqavd/0Y
Ou8DxVStedmXgLM11HQpctbnHKqVgtTF
+XMzx6k0c6w3r8r/lUo4XL1q+IgwDx83
d20zqmHT23zT8NYWjwuieZP1RrdEmscg
+vkE0eHwZomIIuatdLtaHwyxEERjKFHH
nIZQznM3XvFZKW1UC1qqaC9hOroxBw+i
umPfcA2i087ac36RdmKbx6j5GUXN8Tin
lPqr/IQSayzM5/0v7zgqvXr5+OZKbZfm
G7yGKT5dzOU0Eckh+XDdrvYj5zx85IxZ
dzwXaPoizldozdYcbX25F+ccKHqKnwe5
2y7r0CnY7JXZz539lQ+2q0Nttm7qk0Jn
TlIrt4oPxTQefj1ctFiys1ldc1HiztJ2
OLf5yNGVetOVki4vPOfikCuOJMRc7wz5
FDSndPUT1XK5pJKbrHqpnNoReM5euqvI
KZp3DE7uXtUhc4PEFSVnXfZTEqcrCdSy
y1cvh+voaD/7tXygY4zsEyOXrs1DxcgQ
SgYu4bE05lNIuhgjN8+2aE/ZyWZ+wEeZ
Oy3XXNJkUzLvp6tny3OQmKqVs9oHJFyE
/GOMvLsYGe3GT50s771RaWk5lkNfTtFy
YZc+8xOfm3aMkQeNkZO5KUr0plOa334+
WIxMJVpLrCszy7v8Dgv7JkZGL1rvnEd1
Ms1vmAxRRn4Y4MyuXCN1Q3e8jl2WilNy
uYYmH3XrnDt24Hd3AyjYjunI3TGyvPcG
o5HlPAjJZasPEyjH12nMVy99O0bJMaPk
0tV5uPVuWt1OzlsUKZXijvs77TkW5cwL
dWu5t5wnWWXH3NNmzPpQolJCZguxfFC5
upJ01YlaCnpfB+PjRZg7Rsk7jJJux6zP
UpS80SMlyxEtobGu7XalQwp6ZwvdjjES
TCS9L6dVOVurk0jzoo/IWXlfqSZLz+3M
+aUYeUgi6VIpc+sTgrV8Pu/lyns5RyJ5
+Bjp3c5bOU7iztvg3t2g3+7FJD/FsX6M
k2AqKT66Ojauk6T6x1964m75zPX+oVlI
XoPTVTe991RMRnOpRHxNPpaYf/VCoiOW
PHygLCfmzm3BdeHzjkhZ3n6jNRt9yOQx
Uh7Z5Hak1JKqS9yoDwNzk198NnnbU5Sy
1P2WMn84wBAl5XsL0+Qnq120eT9fEH+E
k0NGyjpDsCtSpq173x+fSDF/3Ood4Mlj
qDwCyu1QaaHUYfWZ5Bp8id1L93GcK5HU
Yo4pWWnF84D3uuv0e8WOpapMda9xvuZG
zhFRHjpWprlJ/NpYWd59l5TyGCqPnHLb
ORnqsypd3Q9pPs31XQqVpZSU2rWWn1fq
z9lvHKKqbFF/yqWeLM1XCZXumqecHUHl
4SOls2nn4raYdrXf7ib3vG/JKS+u/b6G
82PIBCPL4KdaiZX6sjbaiz7KEoB8zCmH
aCJpIV4e1Ebpfa0pSpkoqVSNs8UNR145
ZLz03u+sLMvJvOtxE3Wc6ial5a145TFi
fgYRczR06aXulyjZ3pXOPKWlBW46xRJc
y+mYcp0hX2rHD0ouS4NdEpErny7FpHpN
kXlElwcPmpp2bVJfeAZ5usENnltzy2PI
/AxC5mgI05dzMSUrwko765Ytlt5NYcql
kCtlcolHI3osrZTJGktb5qZyoK5+8sQR
YB4+YkZ382md8t4bTevcDl8OEjFDmseg
W4ZMyRrnr8x+cJjtujlUzPSlFLv44Z8S
M6WOFE7lmNYmV8wffu1l0TT5lOozzpzX
aXFPcPm4pTl3FSdMwaXBbpC/H8KsRXws
+SgFr9e05qUhd5q0LvP8/9t7tx5ZjitL
871+hVDPcxK2b7bNHtnTVYAASVOoEvqh
exoERyKrOLpQoMTS9ADz32ftyDwnLc4x
9/SI9Ai3TBpJUVRkMBXhbv7Z2sv2RWfz
tuZHdwvMWZoD78bIVO81bMPbt82gQPQG
amqWs/40N05L/yMeAHyFn36coNwHlEsP
6GGn4+CeKDRZNQdh6lJ7NuLKEGxVwdRc
6HWDem7ESVWOIp7ikjIk/Hq3DMefhr8Z
4QEsXW05SXljUuZOswzcm1S7nS2lTTta
cTBZEAMh/uFMVe9X5ThRubumjE5gFrUm
GqfSx2tKg7StlbASIx5fnNST8cFLiu5D
VdNwzuWjOPZIiRLX6LdsZT0ML1Rx+Y2k
6kJD4BmF3/hwvHTPxpP3jnoSbegATA/m
WnONgi6v5cqRZpOSh1Ny8fE8rHNlTSxB
yeiwho+2WMSjRbkgWMrR87eOaFbi04ea
sFKqVS/NgWtfUVboDk4x+Fdknu80P7pb
uiX1W6fj9d6EnvOXlyVlqsUtEvWK4FFz
u1sPtknLd+9TQuM6sZkVcwHHe4c7/lAD
QSBL9gTMl28/pAGNyhSNN1OwP2ofa15X
ldOoPLp9Rur6lEzULQ33jbNyd/Apr0lM
n6h8504lkSaIRZKY+2iVFk90mEp0mVbF
W7O+riP6rYrDOZr6uiMGKzm/AMrpVB6O
SnJpD7w/sRILrGdV4u0b0tP3sionLQeg
5XhmpcTxt0uE1qUmcPNLXj5OoYUSlpRM
T6k2PmAbdMpMnMxNYzQc8XrV43Qrj6Yl
gqweLBG+9Ep5YtDJy7Dcx6+cqBwAleM5
lsxZsbwUYat7rmfva5Vl5KvlnBRbtbdp
a+MISxZLlSOtzpQQsq23HZqO5eGsLELd
KLycN2775FievXxzx3LycgBeDuhZqlr1
bFEjLumsme9nw8lyJsM6tEBrHtKzlJI9
i0e+v4dSXgXm9CyP7j3k0s0ZqtrtZ6nt
yJJ9TEtegOWr89EnN9+5gXnajBGNkyWP
f+oE5P6AbdszeRyi5ETL5zyHJlomjVo3
LTWX0yi/F0TmtC+PTrSs/SbAZyWOTaJl
3VTGc5F7ObH5VrA5oJOJzT2lglUmKcfB
ckdu8sMp4dKsJlYE6CvoPNbKjLpM7ATY
lhD5rc/CnVbm4eQ8m9HYkLOtMW7IaU2L
oX2szMnNt8LNAW3NkotXFdYUeetLUTox
mypnVjFourIwwewoxakPFGN8TbArBf5z
PkP79DWHBCeehH57IqPeifn5y3v5mgPB
U+sr2ClASm1+0vzeMyQfg04GYp6ZcRU6
M0KRmLXIp6G0x3d145hEAeaYOLbq1EVn
fogJ48ClMhRnpjyo4qQk0XpOoOTP3LMO
OHPSktWy4glrG7HPUP2e/TZ62OR+8XjK
tGkKRS5cmESLazLWu+Wv//3bb/+AFYGl
9v3TpZu83IWXCw/qceMosKiUPAM2rqXN
uTw/EELIm6JtJJY5RKfzYFrzUQ5rHAZA
ORc9eQlrxLTKUhIeqpjEoXl2Dm5+dDdk
9sdRLCJzyzyK9OAe3Q6KJSZLKd+tQHIi
80bIlGwp3MEUXnUzMuy4zhta8FlKoRqt
3XTxEL0kxyLULFRMWst+HI2Jb1prKtVy
LpXXXc0MmS+GAJ0kJll0kTmj8xtnaHpv
cm4M6+kPO/NNwXlmBOQRoVcIk0RXzjqb
yBwGmUtP6nGjfLhUXHUE3Ez4XEul5cnJ
rCqd5orVJAvIPPYM3VNSYoTnuWTwfRWZ
+DpiVsBZPGH1+etMZN4Pmdw3NJeQydsM
zdMR0CmOq6EGnv8vJjPfJjNHczKjXqZE
CmlO2ahXNflYBRRGoWMxSom+vCNWAZlL
yUzJsgq+83QyxwZmv2NRkm5QvqUEaC8b
85qk9onLn4mRqQm7sGTILaAw9YZFPuJS
M7SbRxpkdObQAWuBiC0jJpcAOrVN46eN
OSQvKXdPfuL0rkdMytvmj+9iY05kDoTM
0YzMmBdFxHHUTNV7DYyebExE7QlRuwlA
QyPamDFUXZN7xl8lr+YYTRfzeGIqd2uB
Uj8iP3/51i7mJOZAxBzOxxQGY7IWqEig
sDM18lFkxvTajIuFYCcV0hFLgZQp+q07
+B7Z96ulQNPGPJ6ZfY25FJWnbdlFu3iY
k5gDEXM0F1PVsRcbotR8ilMXiCke5mXi
mqtHKvuArdcp+tIlR5CdKxddF5nTxTyc
ly0Yn4HJZ+fnz8D0RpHu5GM+37d9c9gn
O38mlqbFfDN8npSrRerlAjvNXTx6doi5
tdOixxGb5gjO8Ow9Jpquj9udlubx8KRu
YmZKXUMz0f55mZOdb4ydo3mbnMFxKEmj
qEBnXmw6rA52AvfVAB1TXZiPdmwlkATa
tSQ3zs3FnfbmmPTMqTvMAq/3xOf5yzv5
mxOgbwygw1mdwKIVimLsetY1oel5FD+n
GoRl0NMHbXpkhkubTRC0J8m82ihuOp3H
45O12/Qoae2eDrFuanp0kdc5Hj3bdj4X
0zP6T/bhSe1vPQaeFsOxXwdPiyA5BjIg
aBzC9XSH7syuCOBzkaXUTYpev9UJmg5b
eh7yZF2JuET+fkXUVmk1292UuXLKEk0/
tHHHZuR+v45x1m3hTsnaI/fnjnG2KX+z
puyGyCNOFnCHn5+MGye7/+0/vv/x919/
hxXzzf+aqNwFlUvP6GHNjkpNpRSykJrS
Xo3PZvPG5NuasQJLlN+MOMYcKlMlkeAb
e+RWrbKSa8U2kWPweXJ+/taTlfeL07sy
8+zsvD1S31QWVCK/UyxmsRUmf74xk5Rv
jZQ5hehRdTyrPEJErhJV5OIlWhupLxVR
MjbpchKhiHeqDVgRlMLsANATmxcqq0fp
Zp6zBysZG4B0QTkj8hvPuuDaLQoqPU1Z
Np2kg5LYzinKNcJvuVuXjsnJ/YPvhQf0
KE5aRRBkkd1Y8I+9lKOnGReuWHyiVQlI
9QErgVhrJYledrnUpsqkh0nh7Bnfp9qZ
IzYpecdD89xVlFAPPeOSzkZQLvccjrFV
KcYHWY51fbcWcJOU792mlDhOFLBbLUbp
lKWayehD7EySNFsklo4nKBnxdBFB/J05
hgyvknK6lIejkvvJmam2rz+H3rxJUu5j
Ul6TzT5R+e5tSlYAJke7jcgBTmXpTOfU
eNhz1cg50hGTMVP17FCVceiUrDU2pk05
JCy1dmt/orl+bf7oupZaNxUC7eNaTnQO
gc7RfEtoXqoWhp9KqtYbAvSY5Qjo08lE
YDlLoRtGZiaAPTvHcHYz4hci8mlcHo7O
0vbNao1L6pYBnb98a/Ny4nIIXI5mX7Kr
hpWkbCrKvU5GH3lJJyCV4om8DuhfxtA0
VpPTGE+ox+lgjo1LwoPQw2W/k9GmRkZ7
+ZcTlmPAciwHcyZaTgtzJlrunaU+ufnu
7cyZdTntzJl1ObE5ODZHszJnCuZ0MmcK
5oTm0NAczdCc+ZjTzTxEaI6bj3kQNnNz
SHA5N53zAji1menyAjkhnS5hp5r23t5j
Z5VWxlyFTkdUzJ5FicIrpA7y7h2mQ2vG
kDEsObBk8SwoaSmBm8In9r/K3/yXr/71
t7/86lc7y01oDafsAXao4wbuHXZ6Yaak
OQuLZe7Cc0bpN87QNGsh+QmefjaF93lq
mjUgXNGcUiiZSvT4M9LUbIw3zmaPh+rr
f//mT3/6ZuJyP1wuPKfHNX0LTlotDJmZ
JHek5uN5UC4I3xH3WqoZH/9VUvMmvCTE
5lmLh61Zq9uqqwk1CmFdvVKNacPP32by
8n7HQf2RFsS12zNTto208KzYLXFrEccZ
bvKk5dulZUmEmAHMdI6GccfD0owRm4Mt
eBMVXczLxA894TdVjmHh9VXlP7cRl64F
0TaRJ4luyKtn514lI3ojOvVSnA3emh/d
LaE99bVlORvU+5GVmrZJS6XIlytRzBbj
BO42xHyy8gbKcuEhPSwON82GyMUjI9NK
Xc5hLxWBLiL1EJiyVCp5oLBMTDH7UrEV
ccU/59XyH0hKbBJUw7rNdXYTPoKWbtSV
ln1a4t2blCVh7y+W3JkRBuV6t/nlE5c/
B9/SVaWknEtlidPyL3lpD46NvUjxyHnL
lPJy++BDjUsNvS7V8ElN8jQuB8clkXaL
y8W7kTjevumwfCfj8po09knMn4F1SacT
HDIPlWmpl8qesYZFNGbcGjZt5tf2W78N
MK24x7FVLbiy1Cyx6VwOCcyq3RLJBV7G
fJctvNzHuZy4HAOXo3mXVJyT1Ojjz7HA
lvIw4zLhzSlTzayVqPmFowhMNayfHOet
MduF1qeXT/fycF4CZ920IqPcuprPJ+Nl
Uwr7PvblBOYYwBzOwMxAS5gD0blCki4C
E9+8ZoIW5ci7hz4eT2CqkyRhroLNKDlN
B3NwYoqV7nnPAjHx9k0HPjtZmBOZgyBz
OBNTQe4U3d6EIdKoE5J/4Ida3RG8U0qI
eI3GNDEd0VzSikssMX91PSafJubhxDTv
HvnwQjaR+aYzn8s8TL5V7vqE58/BzyyR
ulgI4Y+6paav1jM8ywNHTjh7McPH9lcP
kLwNPPFNPIqqijx+nVV4TkPzcHiW3K2U
5Nw7L0/N2eReduYk5xsi52jWJkt06C1Z
IT1Bxx43H/N4GGozIwgGPbE4m6yPUVQn
JXKpoGEE6a31On3NEbG5dHDe5+bWc/OL
bM0JzjcEzuEsTgRHBpyXbMWr1U5Low9P
es6iLyUUp4qUpsfWMIoTilii9WeUmasW
fQGd0+E8mp1Wc5edeL3Xn+P85b0czpHo
2YxguZidSu3rzS/NzcvHtOcAY5urfA02
IycytB1CxGqcD27PEUv5S0yevzpCx6IP
pw6KkSyKLxIZ7NI0qu0wEQSNjovRDIws
PT82N4/C//bjTxcS8RMtzq76R1acv7hE
ig8mYErxnEhKzm1SPz7gXx/ZhyX803ff
ff+77/ErfvHd93/+/m/f/uLTw7UKlguz
v3/zf/wCt/2X//zL//2r//Krf+oB5e/f
//n3P/z962//9Je//a+v7U+TKq+lyuKC
fwtQOaqjT0ClVsXemhR/gMy+JrQ4Dk7C
X+VsHJ3N3z1Van4oybKaVVNzn1D5WUEl
5Rj7lyVny6Yp9ZAxLlWOlCpJQBQ2SoLo
LCMsXTtwTVG5HEMc8B8R76uV2wRwB3FF
o0qNtZagqDSHJJMrPw+uLC34t8CVI9VK
HFS4xNF9FpXWROhgpUiGuAkPXjPizTv6
QgdhhfUh10q5OpZWNL/kCZafGVimt3Il
WNjSyV6xGj76enu+aa7cBStLeakTLNNe
eSuCJXRHjfQETilGb0x75XB7ZWJlGKxM
g+Vqg0WZElGNrHuS1Zk/016ZXPnZcWUa
LFdyBTFQJCsK2CI1lWmwjGCwTLQMhJZp
seySvlJWyDIdlntwZVNa3ETMNFveiniZ
uSyDmS2TMGMSZvouuyW2rEFmOi8TMj9n
yEwTZma5vBcT5mDK+PWQkaJ9yPAIkGl6
r14FGWzBBctfc/T9P7jtxxsSMpSCyoTH
iAslX/V5OUXP5eBQkkiTkR5i3lWsRJQe
opMMx2AlL8R0D8TslEgnaWLltVhZXvJv
gSsHTghXIkVIWREgpSrGq3l0maNDkjrH
fKE7tlk7hioc7cs4uyqVKlnzW0r7n1DZ
Q6tEqwJQRTmrexcG40LlUNeFwrTEFUOE
45Vp3XWpIsU5zq/VzLnHlXcVENGDSzm1
wNCM2FHTW6pTnGDZAywLK/4tcOXQbJec
8C0q2JKYicpaI+0UjbyyxN4NeBep754r
+UGhgpNVgkDz1MxlmFj5eWBleivTW3kv
3spOOXQTLNNdme7KeFw5yl2ZWBkGK9Nf
mf7Ke/JXJloGQst0WKbD8l4clgmWgcAy
PZbpsbwPj+UWKXKTMdNumXbLeIg5xm6Z
hBmTMNN5mc7L+3FeJmVGpcw0YaYJ8z5M
mKMZgzigXI+Zz6oDz6e7UTmaNW1Z5XUj
McVjwI4krhA0x6qZU9FxFNVyFS6FXbnp
VvCJPKexaYJIP4tAkLtTZX/VIPabhFTY
xQtLylw1A+a02qSBCiMG05KjtpFzV+/M
WcK3HewmzWDDZq5bcm9f/zRMWEozenh5
rlsly04qWbArtbOxd8gUXBvk9sRLQOOK
Ou9JzI/46hBz4Uk9bIhwrMTEiuWbIWek
tO9riJkpUdFoi1K0NlOth9BuT0M9SzkN
toc24eYT9nCZsDlkxw0oZu38z8nLu/Hy
WVu3tEzcpWVq1u/a7HXCU1VjhnDFCmim
505avlFaxnhbIsPm55abyeVH4VJj0Kpr
PTUpl2be9We0jEgJqjgmHufUQnUUeRmm
dFb8HZcXEnjVsS+W8Y1cPFohldr17Ofc
4BvPXNfn69YCU8VS+0f7rk/4JH1+LBbx
SXGjoTTVktZsuaxPYJ/4fAP4XHpwD5u8
TqmQMolbzdE5+Ut+fnhSpUmxiRfJ7CUv
hOeHWoX4jNAZ5owdAZi39Uk4xQqZgrNV
iqZuE8VJ0NsSNPpd9hCK1xs6foLm+cuL
pijuLTFuviF80hoiZX30+sTmG8DmYK4m
9n5LUik6QpM0KvhcdCLk0VSYFMF6IX7e
vocRnakkybmQqyseL5qe5tjIbBrqtZqT
rQ3en4P0Jrfl5pbmUmruBOY0NQFMd8Qv
Bf+VIL1q+7YGmAo1WqTiYzs27warI4jM
R9cLTHdLQH8trI3dME3NIYHJ3pzqNMQs
3bC89TpvbWpOXo7Ey9FsTQIri1uE5cWg
MM/e18TliSJFmg3RbJGUvTxfiIFEJigI
3S6RpkLi6ydB09o8HJqe+jKzNquroaan
DTJzLzNzYnMkbA5nZ7K5IJhhiZNxXjoN
AumzxigsMdHafvBxlKZBYcSJkIqyyAtK
c5qZRzNT6/NTuoGZWp9v6B28zEnNkag5
mptpCLoL2MEMqQYeLlATAjkxQZaCNRHX
Dqg0a/BcxfF3iol9q9Ccdubh0OzrTJIu
M8+Oi3ayM/k1We6TntPaTDHORZKAnBL0
aQPwM3oatGbkwyOwZ2duDtpHkZyUimtM
USYD6zWvzjmd5ubx+Oxbm/2Mo7RBb15o
bU52vjl2jmZzRq4Os5qExYkQaqk8iAvW
o1TnZDnOq8eTnkSPzXAMT04y9bVKxWly
DgBP4n7AztoVn20nmr1MzgnQNwfQ4QzP
hI9UCKJNJQYqfsnPp2MiREKp5sgwxwKl
hdj9UMMTmj4sCMN/R7LpNDxH56d09Sdx
T3+evbqb3/k+Aartvz8WQNsuHVd6n7kg
UnYwDeH7sf28HuvTkwMlplG34YjkFwRo
nF2Lx84uUjjrkPXp7iWHuVAYcrTZ6vrm
J8UXEoktge7YWHAS9FOBem3T3JsC9dpV
oNJMSF61PwVSQCFRKErjnpFx5+T3+/b0
eNfMXHpWjytRR8zuyQCZU2nlUv67Rf9B
vCWzgvlLUfuxNeoFgkM0VWCz5nVkJryF
qOKtkqfheQQx+w09kvSz39Omhh6eoikX
R5tRqyJtN8jJy7fJS/fo0eMl2mekpgvM
cTXqZKewmyvCb14yObOSxam1FYMqlYX0
94Nr1DUDlikyGGJPWo/RLeOq4T+4ON4Y
ojNIf2dF6poJkVEcBzrbYT0+JkB3A+jS
k3tolToboFNSNM3vnLF/rFJn/KqwjwyC
Li0kwx9dpl4zecHOVD0SVV4oU8dmlin0
s8Wx0oTo84/uV6cu3UC99IXn+cvrXme1
UhNVLImq6bD2HpOc79Xe/NkWqk9z831W
qu/hbe6RDD+R+W7dza216vGXGbbucJDK
gPmclxarT3fz4Fr10s9IKmdp8k2x+raU
zl38zcnMoZg5msP5My5Xnybn0dy8Vbn6
Hrbm5OZQ3BzO2Py51qtPV/Noat6uYH0P
U3NycyhujmZrRsV6lpg8neNcfYmb0Uod
gW+KVCUrVkY0NqtF7ShTyoKv8kLT4mls
Hs/N29SsX2Bs3i3rfQL0/ZqcGT/DQqsK
IZa0LOW9GzukZ6WKwJ6kDuhyUuRLFykx
kg3fqa6H69PlPJqf1Jed573gm7L1/U3O
yc+3x8/RDM/HIRNsgk9WouB7gZ+MUChH
2WV1LYh0B0zqfKpcD30PkV9fqBuafufx
BL1R5foFfudE6NtD6HDeZ8KKC/cTuphY
lo+MELzjw2tlzrWcoXYUDQp6WjUqeHIM
QrntrDLdz0EZ2lz38+r13rzL85d3sz+H
xKg2LfMupijnUhch2v7eQyBKMWH8lRxF
nBGar4StmEaQopZqscKnlHJuL/JnkXzy
aPxUqxeLMp7xhGjUlAoXfMoE1jeWRIeh
lYlco1EIrgyAOiP55x/drX5d+w1AuoG8
bKolwg5vkXNnFoOj/coEz8uPjh4h+NVX
//KricmdMLn0gB7W6QNqi6LZx0mmZepU
YQo+djTxrPjoGderfPshLTT6ONTxNETp
wpYzHpKoXF8FZYqzMnYXjXmLE5TNj+6W
C9/3PKmItEWY3eg9bzRAw/fMjLgia9Qb
T3C+WXAihFRXzbkUqj0o3rl4HeFMLkki
SQlrbDFM96rRBrkCtK7VFk6KDjU6i3jm
yIePIetG6wLTnJziv2IycjMwfgbp9wvS
S9u1+BmckrsF63h7g9CVw3aJug5EG14Z
qmCKzLfLyqWH9Lh8TtICyRVC89QnY4GW
FpMJREFMsmRLsDy0Tp1cavUUB27KVprR
iD1auhremwxfWkrtJiZNWt46HO/XDy3Q
Em/fltFZsKBZBFG5Klb03RokTVy+e+sy
HAIRylILovKcOnmc+pCVYpxLOIOKMGc5
Jj82jVNYLbox49u6yWoW0jQvj6cl125n
JKLe8U/iekf78prc9wnLd29gkmJNIXIB
aQjR65eopIccBd+puCBST1wErByx03u2
jI/omWLYe6rrynL6l4ezsiyw8my2+idW
li2s3MuxnKwcgpWDeZbQitiyK5QulGX7
iz6xsj5Ijbx2AVgyKfuYsjLqPkFLLkp6
mi63isppWR6OSuvnZuI+dnt42JbczL0c
y8nKIVg5mmcZ1YfJsMIQwMaHWjzhKQFT
ivFxzDnziEXo0MgJX0MppgjlZieanuWQ
uPSzZhzPvMTrbcvNj7w8f/nmnuUE5hDA
HM21ZIJmjAhbJIjZFMx8VvsjbCYxpywp
1rkNmHIZVUxR3CkaCdF1dWjldC2P5yWd
Zac/l/5Y7bbVpDZDcyfb8mY565Oc79/C
rJlMgMVihticOlJTHiSB+M5YkHGAbiuR
+aFSExcXkVg2KE3juj7vd5qYx6Mzn+VX
NkmYqeti5rap3E4u5kTn20HncI6mlIAI
F+XIUF8c9Jtrjkm/qTpLZVsI0o/VnG6Q
zoi4PXoyrcfo09I8HJy4+v28otrNWMfb
9/c0JznfDjmH8zdjpoVHW3dmixOfBXQS
AqDocMdOJUd/oLFE5yPejTy6SOGRccWN
Xe2yOQ3O4+Epxt1+cQvwxNu3zUm/yOB8
R/T0pv3eePTUVJ4/xVX09NgW8XSTIpxo
p6IcVv4DFiYz0iTVMnd052OnzRwNLDM+
M34fKQ2oO5kReWMXyJbwlGnzzHTYGScN
zrj7jv26NCXNM2K/Y4firtlJXatT6zar
06hSwUpWwj8+Pzv3yWX/9X//zQTlTqBc
ekAP7KjJpzZAZkwA4lLpT7SD00jP5OJO
WUc8RjeSqhlfJQ7UfbW+3KMBM0cKlUBv
pm5DuAnKG4vM2u/E0c3PlG3pmYwV6rGm
gUmogOffNEn5xkhZpERRYvRwj+f5+MZv
xhwTgNggE6Np65KkjKJzVcvxu7i5ZuMo
SqAPn66oSSB9tZ68EEcFvSMeJNyJrqCc
wfitOdlvWNQXlFvkpMTDlU8zSZOb3rs2
clJyP0ouPZ/HzbjwqHmMvd0N8cqSnqQK
1Zk8V3A1OmkO1h7zQ34ocXFLrsQxGei8
gLNLSsGXJoj60B38/N5JyruRsop3z3zw
etuR4yMsz19esS1TVH1xMY1UI87Y2ycx
3yoxh7MqEasa18wlGpp7sxk/A5MfKhk+
rmTKuQjlQet+EuAXzd2ioSf+aG7V9CqH
BKZoN4+dkljb4q0HzyS6rbh8D+vyFSnt
E53v2LyE0IR2jLG8sTt3yenqtUTpueOD
57VGHMcWl1ermcjxNUTlBXJO8/JwcraX
/VJyarOgb+5lTnSOgM7h3EzO2RNjdWUA
nb2TXfQ0SC0Vz6VG017xpeLJQ/Myc66p
qsT89lx9vS3HtDOPJ2fughMLstvCKOUN
rNzL0ZysHIKVw3manJTie1nySt15549n
5Ix3ZKsJH9uzJB1RaHp0F7HqVErGA/MC
LqeneTQusfj6FUCcer05zl++g6c5mTkC
M8dzNRFuR8ZicucYGLaYVxRVhtViMIRq
cVua1XuorQlFEeMys8TfXkgsmq7m8chs
necGmVzOpqd9QmaDgJ2MzFunrk9yvmNT
00rUF1YDCxOWbM/WfJowmWJwdOEKJVe4
aXU+kNqMHQCa2BWSnm11SO+0NY9HZ+13
6lggZ6pbOnVc6GROdr4Zdg7nasZgiphl
UaJKNy+7mqkQZXFxBPSWBszRBAyzn0wE
z/gfvlr1M13N48GZuyN/Un8ub0p5w8if
C13Nyc23w83hHE4wU6L5hkeLi9wZMvlU
BCR4hyr4iX9OtADOQxUn1LBCDIOfFu6l
rUbr0+A8npyIYrqaM8z2DjvPX97P4HxH
+Cw2Mj4R3L0Sn9GCJ0tFUBKnMZU72Lt3
tTn+rEIaU3qlrfn57DCdgjMU6s6rDOh0
Rl8bUa2eK24r2/qsSexiNakXZ8ai63Y5
mvH6jecBWTeBE1FDrxEx7ui2nE0vVhx3
Fhs+eePd3yXb/df/9NuvJip3QuXSM3pc
tjtYWYwyyVkG8WfVQeqVPJp3nFoUDjZo
Uh9MsAmBkYb95/MpHH1Siji0SJKI47ot
jSYpb+5sdg6F6KEtGPpUGLTN1kyMNcAl
SU5Wkqd050LKCcodQVnUnPA1S8Wvkw7f
7iwpIXCx94o7PhBT5/j8UVJmfGQuBMYL
duxmZNA4kjLmbkTJvBrC7XUnM5LgtaZc
VLK3h6wzHr/j+PJu67fUthZsm7U39tXq
6XlUaaiJRWGYTkn5dkm59Iwelsoedp+6
Ydcm47ZRzLmoZEPs7dWctUhe6vV2rHkp
fDKELUP6kub19uzmwojUa4X4aLzYycr7
jZi02p6Lr4pKvLfxOZdVZeDRirg5YoXK
9641n6h8x0YlJyG1x77sXNWXmhhh/VUn
LD5I4mQyYtkPF3CyOr5QzqplOpWDo7Lv
U3bbCccC3KQpd7Epr09gn6h8z0Yl9KQJ
tCUWVk21O7rCHrxq8hCW0eyIBh1c7vXU
WqRGXhSV9WG806k8nJRnzdTbYbzePdMx
3gDL/czKicshcDmYXcmpSnR8I1UpVBbn
/MTsMQ+a1hjMm5tFOI6yTPXUMyQqPjPb
es76NCyP52XOXcNyiZc539GwnLQcgpaj
WZYkptC5pFQqUN6bxfvISy2IbpRV3DWl
xZyhQ+Wl1GpYKClGHVdOZZqWY/NSEA70
gInX+53Xt9RH7uZbTmIOQczxnMvCNVLv
iwA0ba3j+fByUNKzMwkp3r5QEnmovCTJ
ZJpLeLB5/TR82paHw5L6HYVTW7XRnoZv
6Sh8oXN548z0ic137WKKSYbQzECjtP2u
vwzMHeItymHMqi3NrTi2R6bhYSGOPaCY
80y5HJ2dVrq93nAXe/DE2zf1xbzQyZz4
fDv4HMzVFPyZI6xB4J21+Xafic7MkbEe
I00jum0YO4zoDCMzRY95fN2isn4GND3N
w9HJ/RB9SXby1hZGF3iak5tvh5vj+ZtA
jdaaCGKtau6krz/KTnzixA7OVshPCM8B
VSdn3FBBxIY/WJpbNd3NEcmZ2/6sDTrx
ei/X6PzlvdzNd8RO5/ZfH42dSo15cl3/
t6KE/SEhmFBrMyIPE51YXEwxmDcm7bZN
KT6vkURYm1y8pOpnycXDyM5U8MnIGX9F
60RaZaefesVhOyxQqVxmxN786G5Jmpo7
dmc/nx3v3eB1RsMO4egFi78nMOPOI3l/
/W///NuJyp1QufSEHodK06oQXdkEumtx
1iQjMEd0Hima0VVtyEaZXl2jwbAG1ter
JPEVNNXsKZmTT2/zEFLWtLmcXOsmUKpU
Vo7OB5HvwXdvuzFJuRspi0j0AeJiEGep
sbUPG14eJyYaU3lLdNXoHJ8/lkgqFzeL
/sMZii0vZLMfamWqAOfYgApXKHZaTc8s
VCVmd2QI0bpwCjQD8huT0tqmQy+QElv0
FlQiqkA0rlgKir83Xukk5Vsj5dITetyw
HwO2we4EwgRtllBZlbIVKVXUy0Ie+2Hj
yx/Py0sMrtBSsAllsmcR0iNl9OioGaF6
yWke+TQ/umPZT2fQT5+TeO+2KT/i0Uir
IKBgBjCbYWqTlG+MlMMZleTsOblAUGax
vJiTqQBkjHSIrtcBzAE1ZVLo3crCuMZt
QD1tyiFRKd2Cnxi712FlSrKh4Gcvn/IV
6euTle/XqUQsDQ6aceY4r+mgEht9HIBX
Q2zD0YB30EHlNWU9zdiIQJxXo+/pUx6P
SuOOTRm5lr1Sn2QbVOVuRuVE5QioHM6q
5MIlI2g5zWSL0bsLAXiiLCwqScIGLD7i
FB9KBc8JQVJYzN1dzVifXuXxtMSW1S+N
NOnN8cHbN5VG7mNXTl4OwcvhDMskIl4l
hktkqmWxxCfHYJx86tJBURszlrikB5JI
sSStnHCNhV/qqj4ty6NxWal2jnbSA17v
ReLnL9/etZzAHAGYA/qW+CygYdAj2pJ3
pkQ+CUxFpAtxXNQlE/uAzYpCXxaB/gUx
TSit9t6Y1uXxyCypm50e1mMvIC9pU3b6
Rd7lrZPTJzvfsY9pMSdS3bLllJUXG72F
ghMzrVWyNrbSEFrzMeMyQV9ETXyJbUBW
50ROJ/N4chKnrtjE1tyNzTltSiW6zMyc
7Hwz7BzO2Iw8IU/RXcOJSq+a/BGd5k74
D4IfTfElxlOdpPh8SmzmoaNXg/Tpah5P
TsudzKJlclredAx0kas5wfl2wDmewwn1
VVLMW0Q47rokOrGDnwrOiYkLSxpsws8H
ixkwGpX6FZpDyzQ4RydnXejDUc8abjwb
nJv6cFxscL4ndjYdDwdkZ2qO/65kZ2JE
EibRbS3L8Q3gOEUdT3SlsKS1HRf9mer0
OHBXsRINO+rSdLRDvc5c8dykVIwsAYnr
7EwiFglX0e22KSqZAfsdszTbTuvP8LSU
e7ITb980yEKic2q0eI25T3hk75zS/pv/
9l/v13vjvdNy4SE9LkKnaKjB+LvjQy0k
tBNVRPFQBpwsmgMt9Hs70txkjhakUcmp
JE7rCe0lpgiz54JvTbPL8CEReo+TcbrX
4WRKW86E6AGhEqIGM/xHkzX/zsTkW8Nk
PJwSkyIgLmUAI7MgJI04W6AuS3802uMh
kBNIlKApERjVRCOmaHrRUxOl0L1Z1qf9
lFPzTBC/kiZtMlNnSH43VC5kaJ7ZPQ0s
N2ZoVs8EESJViIs3iboTlm8NlkvP6HH5
7MCLlTg9jrE/i7DMOZEx5DAkpaSl1htH
qsrwNpJaNLg35bRe+1MrNLLj2yNMq2Wi
svnR3ZoIW79QciEAx9u3FUpGg26OCg2E
36bP2nWy8q2xcjS3EtimFPk4Vqtm63V0
exSWjF8kKcY7RJciHVBYJjePeZiFBaJd
1mk53crDYem5m13EXLtBuOdN2UU7mZXX
Z7JPXL5fuxLUZpGYIZkgx7RBzDktTWqx
CswXQEmWYHmoslTOSm4WTM/u68mY07A8
nJb9BhzYv3usbCf03NywnKgcApWDWZZg
JKKWWnElktJZdP0pCs8PErnA0TFNPbrA
LXfgOLbkJwo+Bd81Rgd6WW2qPh3L41kZ
QdYF0rLN37u5YzlpOQQtR/MswZZQYtFc
I4E0aakJZsLq8+JqtVTV4ep8PvhDwrPB
oSkVN4rzSymX07Y8npcFK67HS7zenpN/
5OX5y7e2LScwhwDmcMZlYY06o0jr5ro8
nxz7ddGUcIPIWEsZMc2yFvOKuDo2I3Ze
jcWncXk4LrmvLlO27oE4b1KXl/mWN05Q
n+R8vx6mnMw/gdhSKbn0mnE85lyyZiKT
DIRq5cFKex4z7dnwwBjEcE5G08IcnJtd
jXnuVbYW5gaNeaGFOan5dqg5mJ2JjwP5
aEwxMAzRzJLc5AxqAq24QWzZBjwmx6eL
cvI4nioVBJ35l4Njc3Ey+YLc3FIReaGb
OcH5dsA5mrMZpkFFhB4jc6WktcnkcWak
7AmXoeQBBWdixgKKXCkxD/N1JmQOzs7c
lmK1s8mdeqrz/OWdnM13BM+i1vxkNHhK
bdy967q/6SkiToKHXJqp9sdlskdXtwqk
l1RorZYcWi4wK5lZmzmpw6jOBKmBj5jC
4aySbD1Ylxg06SaQOIlnrN786G7pRkm7
2ZlYrW1no0+l5Ek3ZWcaNlAqCKMYUUjr
jN4lk/23//aricq9ULnwjB6GSsSy2ZMy
IaalVBdsTSxrEmdxTXLe0mIYlUnZQl8Q
i0YG6XopeaS0RBivzD7bZB6CSm/D8Ofw
XKgHSmzVTTS/Ep4b4geshIQHTenKJPbJ
yeM5WZKppPDa8E11AFACGTmJqGcoXCs9
SUkx7tQzq4YMxnttzMRMSSV6LSVcXOZS
eD3RyDVoWvGXSTu4cIbjd+zu1m2LyWc1
48+srFuOzunBiZKxmXmBBJisfLOsXHxE
DxOVhoClRuvdIpTajqvn8bdAT3pJ2QVx
LkTxYKrygz5wHPqEweoxE9heTst0rZHC
yYWwaTXrbtLyfrR0b03K516Y7m1s/qkX
5tnLK70wES2QZyKxqmHFNNkQE5lvDZmD
OZYUYtddstWY+dOsyFZeRsajUqRDRZRT
dEx5Ge0vM4JrfNsi0JrPj8n0LEfkZebu
WQ9rdzYa9sFtZz27WJbXZ7FPWr5f05Ki
t26ymNSWmKUHy8dzZ8WPq1OcOSY6y9kc
QV+ePiPuDoWtig0glxeSMadreTgsRc5K
x5/FZbVezQ/e3ticN/YtJyxHgOVwziVJ
haisUcpDnlYOw/FrTCIUr6SLwfihylLi
g2WHAC5VE60ry2lcHk5LtbPxEy/RUpus
2lsblxOWI8ByOOsSWgySDN9LKeZBdjoV
PU7wiUZGMf+M2DlyNcdSlh/kgU0pKo5x
T1mcpnE5Pi0r2yXaEm/foC13NC4nModA
5mjWJZ3KCMnxH0tpMRh3iOJIs6wUhKUR
ky0516qK8DqV8A3WU4imcXk0L630z8Xx
eutofuTl+cs7OZc3zlKf4HzHLibC7KCN
p6zFaqH2fW0rDvOohM+Wcy24CiO2xQTR
c3wLhlZJsj5pd9qYx6OTqZ992a6us+LI
3bMvJzjfDDjHczSrqNeUKXuu3hb0fqY5
VQSalIsT/tYMqRpHcjoo6CLQ9NHBRqaj
OTo4hbvk1Gy903K8fQM5L7M0JznfDDmH
szc5VYHqyvhbKmX55JxIKUc9fEr1VK87
ouYkjZpyQsyealt3N+3NIdHp3O1kxJR7
0Tp2xg2djC53N0eiJ5TV9fSMw9z2X29+
r7SvHwNPrdS8+yp45pKBHs1QcNGJ8nB2
Ap2IbLNLiRMfXToakqgTsmhaJ1lzO9lq
CNX5dHxlUSPPTBYuclpNO4LEVjV8E6da
knTZOeP1G6cdqXasTnpoifop50h1g8+J
dZBBTjzUWAeCSInuNh/tj3gGJH3i1aTl
LrRcekqPC9I9qmagzkioIKxZ0poh4YKs
6lxjBtB4UjPlDNkYQ4KxEWluTh26uIxn
SiOVH3Ct3f5FE5e3ztLs5bQv4FK2JbRL
qXFw4C6WC5bB8780cfn2cOlJCU8yHukC
mVmlg7k79xe2VKq4A4OqejaC/DzzCG+Q
6JLg2Qj6bTB5+SSUGRqkaJwCcFpv95ar
Fibc/ULGbfu6GZnfL00T134rLvHebS2L
1ClTiYwPx7q+W8uiScubiMuFh/RAcZlE
ojG54GKUJR8TcFf8klqg3pgGU5ZP7Tsp
uUoMxcwlt3M2ei5m4oo/owMJtPXsjNn8
6G45R2dqcR2VeO+2ynL1ItjWczJs7bmp
vJisfIOsHM22LBbny0VcI1t9KQwXPvVe
N/BSvNSlWslDdSUJpAQ+n7EwQ/4+P4nT
thySls5nk9CeJ6RRt3MR3r6hAGg35/Ka
fPaJzJ+DdxlBS9GEYDxptrQIzQTQhyjL
BXKUuaH9EArzMcPIoRbxwNQqAPtLzJze
5dHMZO9ntS8wE2/fOChtF/tyMnMQZg5n
YFLUGnqNJrwU83wWkVlNo2gyRSjFpS4w
89BZP0Agrq1YMS3ayIvpXw6JTPVu3STV
1JtbntQ31E3u5WBOYA4CzPE8zBxhjEJp
Gsf/Wjrx4YKonLOYpnD/6kIa+6E2JiVo
xpjWie9bRWm9AGjamEcTU9i609EWRCbe
vmE62m5O5mTmKMwczMskcMaK4FNxEcSn
yzmYpViqiHmBTRlNYj7q4Mp4qIqL4lGR
PDMwByemimw/IxfZcEZ+qY/5/NN9E9cn
OX8OlmZMuKylqia15LU36OcpHdPApZxL
5PCMWGuejEq1ohIj29Reis6noXk4OZO0
kFwnZ5INVT+XupmTnG+InMMZm5FhRJnw
D+qUPC8dBhE2cMhkMolJvNr2cB9BdT4O
FS7iCXI+5v5I5RdU53Q2j2Yn19qJ0/vs
5HaR7+VrTnS+IXSOZ3G6OBOwGJ8Ka20x
T9MNrAc/cxRMVh2s3PwR79HqE9uSRT1T
7E9r7JwW5wDo7FmcS+i8hb85GDuLaAOj
C9lpYk0nkzN2khzfqyO6Sz5/u+u6HHG0
E8o1Bz51AHgWVStcRaLQZ+1AXaIa3Q0f
3WLewIAVQSJgfzQuDR+52vrpEJRpShUP
mSO866rOGbHfFp1Ybt0p5pK9d6Aeq/Nl
etIDHujoeGileIweff53bpzm/rv/+PGH
P/+AVfM9rt/TQ0wTm/tgc+lxPSxad08l
2vhWyZxKS8Ozrpr4HdCbRR1rUS0NVht0
EsV0EpqI0AqRWV7N3MTlh/wXhsqumanb
G25i87bYLN1ZF0m6Y9TSBsFJUCWUrYYh
z8n4biXnk5g3JGbM3jZoTcEum6sffzTk
FqnrxEZWoNJq+7Yzf9NP7mYxqQlB7YAV
QozgLIlJLgyd2WRUdYhZqqUgbDWNdIFu
G+IZo984DSmVrtDM1O1DjLdvEpoMoYmw
qRLFDNLn9Typ+Xapufi4HlYjFFGSRUO4
GNq72EozQ5NJyRL/UGWpB/GxziaUZoh4
5nAQynr3dlwQwvuounObgDWpeb+BvZQ6
B+qL1MTbt3XSzFK1aDTrKhUP+FSb74Kb
o9maZBQAj0kcYabbSl84BOeqZqZarB2P
MYrcTFJqsmgxL/ictazqzelrHg5ObZ+m
Nus9d6dRStrQvmMvW/OanPcJzp+RsUmR
866R9ii16GJ/uHifIZh30lrp3AAdQXE+
mq81MhQ0Zh2w5vXO7dPYPJybXHrdh8FN
425JetnSgHgnc3NyczBuDmdvUqQ9ZrMI
aarWvBSpEz5uMi3RCklTSYPlb36oD55w
WavHyRDlnF+a6TsdzuPRWcqC5GzX1zM6
z1++scM50TkYOgf0OHHREdoA6SVn6+Ug
Pc5ZSxBziH80Z8okOqLmLIbNqOArJUh7
LqvknCbn4eC0/mk690/TN6Zv7uNwTm6O
xs3RPE78kKN/EKJbYEc62HyM1XFnENVz
Po2aO3vfCIrzsUzdvIafIM4WtaGrsfq0
OI/GJhXtdtyUXFod+il18yyy38njfP7p
vnnvk6E/I7tTUjT4QBzucb7eOSR6BCgC
drKCTw392W7ow+hOisnuUjQafsTB7XqD
92l2Hk7QW4xGv8zrnPx8g/wczvaE/Eye
ayJBCGvd2P3DU41mLSnjfaKRAbTQ8uPY
xu+c8BmJcjFD4GarqfDT9jweorVtydFQ
9GxO8zNFz1/ex/acFH2DFB3QAUW4U6rG
rHFwktq3nUXyjHeROSWFDG1r74dRoql6
zpGKKhRtk9briaYDejhDifoMFZUeQ/H2
bdOGLjJBx6QohMBr+oCcg+qMo1wO7wRy
TvnrMBoTJKWSVyPsmcdH84hqoSxjmoYT
PpItdp9jUajnpJJzLt6UIg2jRT1TYdea
uJaivi5FBQ8bpDeWW+HmaZ7h/B1r2a1P
Uc9dirJtoGh6iCYwApGiJ4nAd5wG/O03
//ntR2B+/cNPf5vQ3AuaCw/rYczUml3I
k0QPEGr7vn/GzOonWwmxvocFMaD0zOpm
OeJ3KGRZrzAqXCvHYEzJbCLPX3tS837U
bJ+mDdQ8SwRdadtpcSwaDWjFTdq5WpOa
b5Wa0YbVThFwMaMO7e4+Rx0ha4boVTq7
9ucHR25VlQH7mK0x5LTLaP9hqZLh4kYH
6ReEplWpNU7LoEmb3hIzXr9fMXu3lD2d
R/HP/T+2tIinB5KSilbXIlgLfL8x6hOY
N5OZC4/qYToTeiu8JhDcUimLMtPd494W
oxLN3EZsDc8n7VhAd3fjRjp2kSkULfGT
4/JU6wbnE5k3TvKsvWnq6SFOyjvQPH95
WWZGj1ZNiDdOrcSxXCc1P/7gzVJzPEcz
I5CJ4T0M4Zs6B0NPRILAjM6XjBi9LkHz
0J7wKXu0iEgGCBYpqwlK09A8nJn97KQk
3eyklDZkJ+3mZl41t20i8+fiZ4IyztCO
NYqKckdnPp6kQxuzWXZEPDk6YA6oMz3O
qFIVi0qiPN3MwZEp/YP01J11mWTLOfpu
XuZk5lDMHMzNZD5lauBNgoi2jWTOoUkx
3QwajhUL0ZfqLw+VmWIQwEnDcYWC19Xx
GdPOPB6a1D8BSm2E0FCTtpwA7eVnTmgO
Bc3xHE2PMxOG8PKcpCc1Hy3N0xBJfH+A
06HN0ohaU8iBSzx1qpqF1jt+TE/zcG5C
OXQr16XbK0m2HZzv5GhObg7FzfE8TS3B
GGVxRYi+aGpmy4x31PA3pR2DNYzcTBXP
iqhnokyc6VloTFdzRGp6t2o9UekPz/AN
VeuX2pr0/OOd57VNgv5cLE5FoE7uKWuN
sRpLw4dYazFyzZ6K6UJn42O7JWm0wiNz
9iLtrI9pcQ7Jz3wZP/Mmfl5mcU5+vj1+
jmZ3OlYsFpxDsuWzPPHPhreVOLaOGkZh
Y1sYqnGo3Zlxe+LxSx6Voy9E7dPtPJqf
/YGXWGX95M1tAy8vMjsnPt8ePkczPhH1
IjKnqoh5OGlzkPn5bA182mphKsVlkBH7
dbIUZbJsqXrhNuKbxueQCMXj1M1Nwuu9
qcHnL+9lfQ5I0de1/ODsCwwdouUHQsRX
QhTsRCyMeDhDznEzYPowEWqRp8QS9edE
SyfuEhq0Srj9btlH7N2JBWLmYY4I9HTy
1SP36KAXJfpsLpKarz2D+Dt2/OhH8ao9
gCaiTVE8tIlJ1nrKV+YmIfSAsW48qbkT
NZce18OsT/dcovl7SRa91pfmE8WXT4LA
uGbHZ7cFbh4qPTUaSpOwWYWodF89PPKc
EONnrhKd9LqdkiY3b6w8E3fBidd7WfHn
Ly+Dkz0akXEh8Qx9YncrI5rkvCU5C8L2
AmVUqpgdH7OrRvFlPrXVdlk8cxeEwxSF
4MSu0MkDpngyO8UQJUhnz9JU9nWpaVld
TlNFvHYbHc94/caW58KZOx6x2v7Rk568
5QCeHjjafjlipaxxt+9WvT4BelPp2X90
j8uRz4kQ4Xi0go/vt6A848hI46PnOPSK
WWrjSU96PHgPVZ/F1gvYPWbV5UgiK4JV
MHt0Nj+6W9qSdBlKmrvnRi6bhCdijwrF
qV4yBOj9ytcnNn9OPmecnoM1HrPYsVEv
GZ2sCH+yRQmSRPHHgIftKZvlnKIGKjSy
rBZkTqPzeGxGmNDjJmuW1PzRi97x7zbH
TLe2Pfea9TYxupv6HM34xBX3aKLBmYmX
2yexiDEj2KdUKdrTjac+k3PoDTw2lIle
UJ/T9jyaovWs58dFEE1VNoxo380CnRQd
jaKDmaBCMSQNIu5U41h8sT+8ey0I+AlC
ulTnAcUoNgGmCNwy/juJr6Z+Thv0cIoW
fn5QL6Voac+dbm2DToiOBtHhjNDoBU+R
3ZnxmRYbLCWPrFVH2JwzuQ4oRCE3OOaE
xMVNyrzO0OmDHg5R6fcKOQ/zm97Hm4re
97FBJzZHw+ZwRmhJKbRnJk2ZSl06gqdE
aqdK0ywK5gyY8pkqC/QzgjYKcZ9eOIOf
Tujh4Mxd9UmUuimfbX7IXt7nPUe8TYju
pj1Hs0EtyhzDB1UWYGgp/zM0QSQ9OZl4
8lQXQvhjfdBozGIaPfGtaH2BotMIPdoI
1W7hEeXS9z51U+HRZd7npOhbpOhwNmiM
LcpVc/Rd6AnRD4zN/dTXLhegCfHxtx/S
gP2X8NB4lEYR5+hc1/SbmC7oiAgt2teh
Cwgtur/xOQn6Fgk6mgeapFhF5FtVovnX
UiiPnT3HMRKDT9WWBg0fmwpaQc5gPBnH
ZKNVgk4P9HCEsndzmpZCefZNaUwXmaBj
IvS1bUTOKDVcG5EzxF/XRiRbtB/mKDfP
VTrsu3dFUg524hPl1O8C+nHucE3BUGMq
JsoD+qHkHIVV+BrRxC7l9T5MYGjMwXPB
VqZdGToj+Rsnhmr/JInbvOMmF1S3nSRV
xhNm2B8Ti/MxY4f5rq2X3j0zF57Vw3Sn
AjHVRROuRKKlxnVGSgBlZnWOGcQDyk41
Z2jIUqp4Xi99LyKqjKsCdRJ1n5OYzz+6
2wHSQuNP7orOtK0AKU5opRLCOggSp0Om
Dk9e7snLyogRERhGGMGVjgemluLMpzaf
UJncASY/gJIRnIvkgguQwupcmNBxbMKn
VTwnYchajcyrNWTW00kZbkLcjuZgacbp
96vZbJ6lVmIKa1v23uWnN+m8a40/Oexv
08Lm1eSQ+cOTn7vyc+G5Pey4PbojG0J0
/CmpbVF0HqQLop4gbORRRsPPwTp9PhoJ
EI/RJVmtOEU96jpAsW+4WIkcrFTmYVHz
o3sRVL12D9zxeo+a5y8vURNLQS2VwrXg
/sYYWJtx+qcfvFVujuZtUjYlbO6RrIM1
tng+JMxRqYnfxqK8MKHj0EzPEgXvFp/Q
JZqGrAbq09k8mplk/UA990+HyDYF6vsY
m68dCzeR+Z6tTXJoXi9SCy6Esi41CpEY
P1yjI36KVnZDdvZ0UYR5YpDECmquKs3p
bh5PTYRd3VjdU1O+3mR2po3p8XvYm5Oa
I1FzOIOTGEJTc4n8IuO2+/rnQ4iLnSYF
1VwIfy6cCR2qNfXU/wmfUEhxW1e15nQ4
D6cmF+2OM1LpURPv3jbNaBdTc1JzKGoO
ZmtG8zbLFOVrrKotDM+hKYL3FkkxEsa4
NMUZ44hNIdFSshfHQqrNAzNtzTGxSbl7
MrSATcobzoL2czUnOEcC53C+pkeOuAnQ
iSBIu+M38kNF4MNWsOOLZx6zcijhWxR8
PItpwoXXQ/Tpax4NzXwGx+esd6zCF0/T
c1Nbs5fJeZcBcJOl79nw5KQIc9yxPIk4
Lw1xN9yZ7CzJkpSzgvdRFOhjxBaBe+Gw
Ml8aoTntzqMVaFd/tqZmO4N4g/y80Ouc
8Hxz8BzO92QERtCXWHOl4Gpo+7YnIWoP
kMzJmPGu6Fo3ag17isOiyAvMQURbxee0
PQ+np1O/hn2LFHXaVNB+kQc6afr2aDqa
H1othXuYT7TkbkeQxwkYWJUWeeileh1Q
iibyTHhsohuEkZm/kCU/3dCjaZqtdA+R
8Hqvqef5y7vZoQMytD0uu7wjSLSV7hOU
mp4qh3UEyW2UeCVCsTmSZcdDXsrxCD3N
JHYPs9bweZZOlMAkjeMnxZtrygOO14ye
SinjUjCkB2T/ejiP71BKGBSZ8W80mU4z
nL9fU6WcOkmf9NDK1I/8LG3XlpV4HnLg
dOxZSI0yN+WfBwyHkwnNvaC58LQe1xPZ
i2CPdiwzKaVTnvl0Ds9WFTFRhk6FWh1Q
eBbcn6S1AO/VqKyewpeKnQKPFrYLg0aZ
B0rNj+7XAoT6xUVsrTv63AKENnXzxL+e
qgY8o9t3fl6pE5tvGJtAJSFGrKm6d2l3
5/6dhYmcQRmWCuosHR0RPnfk1BM+NkL7
AcuL6NRDz41xbbl9IrtS01hMogzphM8Z
rT//6H595BcGEWvX7pS8QWxCqGIx4+46
I+7QalNrvgtoLj2ux9WyGxmII2SlpqYB
2LnWZNJalSvkKKsvzSE+tpLdUkaMHp04
LVqbPK+XHjYpSUxXhsCOFv7zyKj50b2w
adxrBtKP0CMJ6WVoYh1wtmpq7uxJERc9
/z9MbL5lbI7layKeEWzvIjWO/7MvtZwj
EUeQDrjWmI3OA3ZQIqiXSornJSZC1fWc
z2lsHo9Npn4xe+mrTaZ7Wpt7DXyb5NyN
nIOZm5zwiQQhetS052qdHKWnBsdQnNnw
X+ExcFMdMoTkfCqYCpWhnMQC9ev9Oqe9
eTg6iak9Pm/mDUvX3mTa1CZ+H3tzonM0
dA5mcGJThuhEeItIPOLcL8FJD6b4zNCm
FfxMpdiwyZ2PloIL4vSyys1pcB7Pzbav
aovN1M+Oz3kDNvcyOCc2R8PmaBZnkpgm
qJwJ/+h1sRkIUVVjNkTrkHQL2DzW4owv
IUQqWbER1PU8zmlxHg7OGBHbIyde7zbr
PHv59i7nZOdw7BzM5wyZRl5jLlHUsS2Q
MymJIVTPFD3YSXwhWD92Ont0nktSyY3D
4VoXndPnPJqdxNrvFd+kEbexum4qybzM
53x+Lm4/021CdDeIDmZ5EkJyl/AIsWVX
XWziCTIZ4nZACurOZCkP/lDLs0aifnLo
Glex9W500/I8nqIQiD2IpkLdJp7sm5p4
XuR4Toi+RYgOZn4ihsVHsqjUsOhMtHTi
Dukcv8WLakzcGPHAvUbnqtPpF0e3pel+
Do5QXZjylvtT3nT/7M5J0LdI0PF80GiI
WWNQGifvM/TDU/0RGy6Ac3TgqEtz3461
QnOIUHwTCSlqup4kP63QwynK2frt5LP1
DpHOX97NCh2TpJRz29nj4s4gZ7A67wzi
7S8+pjPIGemva1KXQDY8vtCizE0V4VEk
Nay0TEyEoF4sd8ran5rUWS5G1YBRP+vG
NIoWZcnZYvabZs/YEtYgGnWmteBhIyhw
aqY3zWj+jlK02xaEuJ/7qbShLQg9SPhS
2FBNtW3FdMe5b3JhI6WJyzVcLj6px2Uu
ZVUxE/IoM1pM+dRasGQ5xzwOUGmwI/jH
rp6nE66i1fHP3qC/R8wsmU9HYlysuTgT
mPc7RPJuh2QE3b0+SgjKNzVFdgFaawhO
IUR4z8yYyHyjyCzQccliI7SKp/X4k3dL
2SDNclZAprbZSOdHRvXxbCl6C3JMax9R
Y5JHczzFhlSg4tePjCQJV6hqOmUVdJk5
I/UbZ3s27bQ3iMw2lWJFZEbJXDEtztFF
65DpwhOZuyJz6VE9cAwHYwNnicIhTUtp
nti2S0ZoztCkNUp3xlKZH/KDlOKlcEmp
MuTFWQ5/n5k1F2C24l9pXLOJzPuNyjRv
2fiJmXi9l6x0/vKKuZniABZhXBVxLPzm
aGli861iczAvk8jiEhjFVDcpslzKHkET
O6tJjjyg8ZRmdJqDSMlVtNTVOvbpZR7P
zPaxa2Vmrq3H+UlmtokSN/YyXzvtbQLz
XbuZFHaeKMLzqg7NtdRoLobC5cwVm3gW
ER6xgt0h3s0imS8cBFrVmdPOPJyZ1h+S
ucRM2zIlcy87c1JzJGoOZ2hShTAj7Mtu
bNkXW8FHwxrGZw8LlKsOWL2eXDi+SsXj
Fa7mahXmNDSPp6ZQP4UzioI72Dx/+caO
5qTmUNQcztOsyikbLkMhJqOlo/NcwUrn
UvD7YvrbeFIzqutjcIYRFEbkHb1AzWlp
HgzNHjEj+OkJzZQ2nJvv52dOaA4FzdEc
TU9ZKQorSkzh5cVG8IVIGEGPnoTp0kHQ
wUXrXg1BemRnSlpl5vQ0j4fmUrqR9I/O
N6UbXeRpNlWdtxvZNvH5vv3NCiRGXQ3F
qILUafrxWE2U8Z4UswQp5rEOOH6ICIjH
1cXTU1Jen6Mxzc3D4endVkkpBlW3f7Tv
ehag7fn7Tk7nZOmbY+lwridHuhH+JEUE
b1Y6rT+eZrnVGEYkqlKyGw+Yx5lcIJij
Yr1GAVSeeZyD85S1n8gZRb0dhp6/vI/t
ORH69hA6nAXquOQmOXpfOhdfCudjQrsn
AzyxC9Qy4Gk7xcxtLzWXjJ2h+YTTAR2R
n9bv+9FGRK0CtQ2HRhdboHcHKP7+P+Pt
cVV/+uPfvsYq+fZ0M777CQ/JL37/7S/+
9NO38fj+4u/f/PEPH7774ce/f/Pj7/+3
X/zlmx/x0b7/P39K6Tv58y+++/7//ub3
35we7Q+pfkj+j4+/9If/66e//u3P355W
+f94kdXQ6l0Kc27z+36z9K+8JJ0+L1RR
B0QKWU2V27Sel2PAjyD4eLufFsDjfplj
9C3ie8TTmps94vlmPH7xr776l19t3sb2
vDQdJn5OLgNchS3naN2OAGqXa4N4rFpF
3O01Z8VmQ+Ndm5eXDVCNHcpSjbwFMuqu
mwVWr12bKvjVFtP2apjYb27dPKZzeGx5
2MpEorlxM13qNRcHv7qUGs4AnivVyLC7
9dXxZp7sbsDhMHQ15nfjPrczSF5JHKNK
pVqcgliT0/fFpfn1f//NIZfmZeDEMXFV
kDOaZibfizdco8NhBt4B49Q0kh3m0mxY
NdHWDlFB5OEr8LnLE0UPEikGWR0cTm66
8kCNuWo+5IdChTiyzCKUlfPZJ6/ETQLj
8btNYxfnbE36wo2uD4KYWygcF/DScY2i
keJuEseLFddw2QUxmCxem1//02+/OuTa
rK8dfUC4CyFSI8MmmvUsLZ0rkBPNLyKG
k5ysJG8zJYa5OltWDpiJ+FvxBJSzBu+v
o04086RAmcVhjb65lfOY6C9MEmOHc7SX
19xtdHyFAowrYlCX5iBy5RXm7HR1nOvu
KyeV6oWco98lREnzi173XBlitVROf48z
neWt/Nf/9s+/PeTabFg50PauWQrUH3b1
/lN1xbXR6JUL2MdOKM5rQD7q4myIq1Ri
hhnlwtHgrxlj+rqnyqP1f4HcZsXfaTmu
GnThPLW/LNH0UkspCK7oIoNwVegIliQ5
oKPQyNEW7uaXpylp2Y05ueJ7AD0x295L
3msvlxhcihtKOfpZaVq+OL/5b/91FyBf
fHFehg5DrTk7Nl7FRt63K66yuULhIF4z
0yjdGO/SbECOF9xUcTxVpWaxvZADwY1w
Fn8JxQHGckQ+8LKhmCiJeDOpx3i0vS5N
dMgscXJzas+ryw7gTpemNBVFu+EmRiPV
k1MR0ztsL7fCQnIXY+ziNVTm4rX57b/9
6phrs2HZZDs1QeXw/2QnUz06aBD2qWxh
MLaHlMNcmZdXjUAZU4moME4bCu/jVdCD
EyVDXGVeQIIBL80L8kbDpaNQZtVjRILl
XY0cxnLJRGJV41Ety/rvogtEJXUvEBF0
SP8CNSf1FyydpypJ40JYN5F66XJRvuLq
5cm4PKnEqbNEjlwPx3/E55T06XsccIE2
BFY5kyLkBDkRHbYNzV4rAGuOaBx7eC4x
FG/I67Nlx5I4cCga9ER8uNturk7YsnDB
EXyK93as4y/PlvAquiqCD9jQc8mlXJK/
tvp8KUKraOkI8kjK1lOCV16gItq9QAYZ
u3CwB0F69eYlYUspQqs4D7ZLRnut7V2I
r0B+hOeIbkubXLbeaZ8OuUobQi3sMzFk
8DFzsm3e8LrLRAXSp3LkuMbJ4dBXaYO3
jCciiUkuoDW+0k466OTKhqdPNXnp+YPj
XKQtJ+lYTEUifMcWFJNc9kJSlqoFe33E
vJS7xxOvu1DRWabP7hT5+P1LxeX6zQ36
rrAjZuJasMXt5qriYRYL3wcPduH+5t+k
VdGFeal7XqUtEgmBvGX830Z9ofTX0zUS
yRjEi0ks2CGgTHsMH+UqbWBTRAqI781i
7CXu/k5wIikpmshr5HlSN7Qf5SJtOQAr
MWm1JI9RTG2q0ivPv0xJk0XOGSme6V4s
e/1VWuR35CbsvsnhX0VErqeoinPyrmF/
TQbdYx5IzeGetamp6/TmQy7ThqWkhRFB
CML+mmp138swYpcYxBNn17kWs95aGudC
bckd8+jcWeM8Ire95V4pmnKt2ZPkrNgf
pMelca7SFtO6ACAanfEzFOZeYAo3H6sI
wVzGoupy6XVXaYXghWxBXb5imyPniLfS
6RDa024HZohR2C3jJiAI4he1AO+1zV1x
kTYsJjXnmLRQKiCyG5m8RL0KBAG0WG3P
48a7Rlv0Uox6yBmq0qKYep8oBXoJ6wdr
FLtD1JrLS0pg3IX0ZC8ZaFdj8Frk/5bd
sqIhuwu0GKIfiVoj2/eBOx/4evEs10vD
OGzVKUeNYVQJMPWPHK/JyIu55XiaIw8k
3Jmeyb3nZNzXXagNmqlQytH72sKKo75Z
eU28uzTkfcTLtGGPw06dKuKTMAYuK0le
I9PiIOcRr9IWNlnK0WwmDvY457RbZtHy
oNYbjVG9vGr3cuOyPz3zde7uQrufGxQ/
v/IKbZBMi+PyXqmZlvp4jHeRtiyjpQFZ
r82t7hfqj3eNXqzsWJuI8+rKjqV63AuK
Zf/6u//Ab//jt7/vJED843NheK/mdUv/
gW//n9/98aeoD//uxx/+9PXj52rfH81F
qH87WI36d+M8pWNzu4LOG5tfeqvGBp99
xZ2bonzhjFeiJBZFRFFhuJeewiOe8SBG
WlcUyzRP5KaS7e++//Gvf/v6P3746cfF
x3Quptcuptf3h/jc0GR8toRgI0U6gezl
GpyMZKvAbT0dONpcTOMtpr3JFId12AAJ
u3pkhnqXTNfEMBDltUTGv4u38fBcSqMs
pT25dIrzEAFHsn4JSy1FmvE+goofwsQs
kd4UC/Wz/iWjLKZiJfUbPDlE81I+WZWt
a8nL9tXUXR5Lq6ntMvPCaiqWmrzHK7j0
L1/9629/+dWvtmApi3uMXjNEwvsZdTVa
P5pAm+es1PRL2vIZn5dU9JL6+rvoFDVX
1K1W1DY8bV5RKUrUqyJ4PYFEdquoiBkJ
mqG/iWpr7c8VNdqK2plRfAIJWYp8kr4K
v0I45YRFJEwcab25J5zmchpjOe0NKI7i
85NhBL1Tcr/m65qcJz41OYw2FOZWmq8z
4JoKo7pYd1HhSVPvLyrLW9fUCJL87Bve
NLrjmEMRWTdFrZbSV+TXVOYiOipR58HV
a9tgfpsi//Y/4xfNVXSjVbS34URmVaId
TyouknaTTl4zfptU/F6JRj9zFY20ivZk
0YenmuZK0Zcx+tmwpn0UU6SolJiyQESZ
ayQwXmoQzIX0pnAE9UYGDkWfGyql3zn0
Cp3Ejt+bLY5qoiGB9cyBrcvoHz4d4308
qfv6d9/8Gf/Hf/3dT99+/Zcf8X1Oh3ef
WiP/41//8P1f/nJ+2vc/nn5F///9H6Of
79erJ4FXVpW/TPXHeWlf/+Wb73Epv25+
32OxDB50JTJOFmlj6+OF+HlJftGB+Hx0
2yNCsiLgyjEgJg7/Fx7aK2vGX/3F41g8
9FYRk7bzxKu/OCWrrMWrZU65lFd87/Ub
vvDcrH/vSKEs0aavEHGqO37tkB6eak1J
zHDPb3a/r/re0Sk2O8L0SAWvTZbF6xe6
xgGfZhbJJVXIpM+48rdvfvz3b8PV/vH7
/xcc+9hK/V/+9Ze//thI/eN7/vDt6Sv+
+J9fkz394Ie/nPrCn8Hm/Jo2y+Sp1qHE
uDhjjRrI533h+X1f//X7fz+Nc3j+2fMl
f5QAkcmLAFc8S1RQVu289eOv+XB5Ocvp
s9JDyTXqq+qpe79V+vxddHqX4VGSFM2u
Th7z52/iRzMGC7DiqYu5frjPzzvNj9/8
+Q9n3zVewB39/s/f/fDjn0595L9+2hba
B+3pen/9+8fd7B85sX1I+iHVvvz47C6k
nGO0F5cq+Dwk+fLbEHcBu2fVGO5aNJ1N
dv3yLlx5E6KZsjqiz+xGNbdtrZqbgHdB
D0K9RXCBxd6sh+Yu4F0JvwHRR6RQnFW4
P90F3usu+Ia7cGr4USINKmZagHnRu2vl
PnxYeR7EikUP2Oi2FuN2b/A44Op5TMiN
9HBsT6JLdwKMxZMZDVI0NxlLZzcC/3rV
HKE/1UD95zdCXn8j8oeEv2zb4xDNFFJ0
0KUcJSF8JZawtuqjm+GFWJrvtet9MOyP
zKcpodjKvyDO030wSZlET12q22fz/ImI
09UY+hG3NCl9fiN0lxshHyhvfCJO+0JR
SaHRms+z/XkIuhk5VmmK+p6IFW/BJc+n
oklOkRCr+QviPN0FJjwPHBWQHvWi/bsQ
aw63ChQopO0dfboL9vq7oB+IPpBuexyi
2y3IRFxZKgLtq3YHPE8AUvwN66vN/N/x
JsRgxOjdCo5SaptZnt2EHE0jtRSEyCVq
W/o3IYc6ythkmMAw+eJRyLs8CvSBF7oT
fH4TNOp7PE4CY5ylX7lHM+QktFJxjg4W
vIqk629D9FzR+KRaEV18scofP4tBeLud
yjchmZaIZLHo8Dy4K+QhffEs+Otvg8XY
F9q6Rws+jceEEbA96ll15T6sMAnPd5St
IqgjkuiPeosbETlSFI0lLKyHBSblFKc4
XrEo6KxB3fl9qBJNOKNhAvbGL3eGssvj
kLffh1MdS+yuEIOKZW1X3gcQOcWxKEG1
V2u8pl3vQ7ThKFjCOSL5pQfC4+EM0zpB
2vIX8cXTjcgCZknIVjxl/MUDUfd4ICjh
r42bg4RnIgXgZblOKz2NTsOmCG6f5ORt
uBS9aBC+Ya1Ex69FLj1NEyZJeCB86XlA
BK34tNH4WbKWz28Dpb0eiG37wwykrw+k
aadAeoZwyyHcplh6/UbMEG6PEG5TLP3i
jZgh3OtCuE2B9PpduCiEmwZfz+DbFEhv
2R22iNYZSL8ikH4RSBsD6RnBLUZwmyLp
DTv0jOBeGcFtCqVfotIFEdx0lrrO0qZA
+sXNYaOzNAPp1wXSW7g0A+lbB9I8A+kx
Aun1GzED6bsF0i/eiI2B9AzhFkK4TYH0
Fi5tVa3Tzrj2RHr9LswT6bsE0i8CaQbS
dwmkN+zQM5C+RyD90t4wA+m7BNIvbtEz
kL5LIL2FSxsDaSXEpyqSU1EInavuguNX
YG+OXu+B2rUw+vLun6cPyg/MuNVRWRHj
v9tqo7MoGjKj1Gg8DrnQ9vA5i6IrIJpO
40TqqUb2qih6v7xuOs3jxN6aEPjiv68A
UsxZVsVeGLPLSwzzafrkbA3dXr4L9BAo
wtYrQDkW8BfB8eNNwO7NMcEDDzZUqy14
GYFfNa0AgMSI6KtC6L3yuk8oKWGuUFSW
xcPejL64KHYLHwNExlrVaPF+g4chol4O
+8tLjWHdX5oU9DE2Tjmkd4kllrr3Ae/C
TdCK4C5GCXPSz+/DXdO6QyjFnGQEv1D9
2REGX6tXoytLxNARfjjf5j54TSCfWMJV
xj1ZiqFNmfFhBLcjpyWdVKTG/DWSXOUs
vNgvq/ui6I2ThwuG5wBPfLIrdoaT6sXy
syjlQlAEIOQ1K+NaLOE+4In1CBOFo/vq
QgAX78gIIyGEzlf6eRQtcatqzCyJxXOL
vO6LHCV8nNjNojVxFL/KdfeBY+gkx75o
AJzc5jbkaIVjEHYF+MtfXuCn+A06VBjh
neNzmC04rDEIElF0jZZy1b68DTtkdl8U
NnA85QGVIItcFTY8GeaAAPQ3RwyLjfI2
NyI0ZowcxfVNviBYc7jhp+kaCB5aD/v8
PuAeMUdTIodcLV8I1h1Suy/KsE+GrxZT
wLH7YYu9xlU6cQmLtFIq0cgJcPZbyKUw
IWpJYCiuMbaiL3fgp8gBm1zycMg4jNiF
/SFaQ0Ww7QjJIb6+2Kd3y+3eeCPwrbBs
OJUc0iGtbdNrR3BCoX0R3FZEpn75wc+m
2wBtjRgZIQSeuZIXtmkFkqDe8JGigfqX
x0OPt0EzFG3Yyw6Fq/ULq3uXzO5LSh2i
R9Vp8qLFP8iaal07gcN+H7IDTxTEuN7q
gUBwgjgxQbNWPMdCS09EjjcV8miQwV8C
7CmW9vjCuGMAkyjdMLt7WwgxY+krY+m9
UrtnLP2aWHrHFMoZS78ilt4rv37G0q+L
pXdM7J6x9Cti6b3KHGYs/apYesdc1hlL
vyKW3ivJfsbSr4yl98pmnUHcWhC3Q4L3
ZXka2FjtpDrNsMVdnS/DWIAVbI4Wp3go
VnOLX3MnYvRkJLPG3I2llFZwP7tFSpjE
2lh4KNSrJqsEXRVDhm6S4w3VxB+ozHD6
luH0XgneM5x+TTi9VzrrDKdfF07vlWU/
w+nXhdM7JhbPcPoV4fSOxQ4znL4+nN4x
o3WG068Ip/dKtZ/h9CvD6b1yWmc4/epw
es+yhxlOvyqcflE1NeH0P3xsD/737//8
+x/+Hu/5OGvgH/6/f/j/AVBLAwQUAAAA
CAAAAC5dwDRtcVnrAAAWkAcAKAAAAGFy
dGlmYWN0cy9ycDRfdjRfYjQvcmVnaW1l
X3NlY29uZGFyeS5jc3bsvV1zHEeSJfp+
f4Ve7tMFYRHuER4RtrYPmpmea2PW3TM2
oztmuy80tgS1YE2RWoLqnt5ff8/xyMRH
VZaUykQVEGukulkACBQyw9P9+OfxHz5+
uv3fHz+8/fH2w8+fb+6u/nb74buPf7v6
49t3d3c3n9/e3dzd3X78cMcv/PXd7ft3
f3p/8xY/8udb/9rDR4++8ecPf/nw8W94
y5sf/3Tz6e6H258efd9fbz7dfn97892v
/OuHjx/6N9z/47v3n28+fXj3+favN1d/
end38/72w83b9x/v7q7+9PHj57vPn979
9PZP7z9++5e3728+/PnzD4++/Onmp5vP
t5/98h6+endz893Vt7dvf7j98w98fY/b
/vbjB/zb3Wd8gEt59+ebt3ef333++e7q
5sN3P328/fD56ubu8+2P7z7fXN3810/v
PnyHS/Vr+P7dj7fv/371w8f3P77tH7+9
u/3fN/0LP394OLvPn27w09/heN9+/HBz
dfvh+5tPNx++vXn76eN7foqb/Ou791cf
fn7f/3r77bv3t3/69I5X379w81/f4srf
4Wfurn56y1+Al0/v/nb1082nb28+fMbt
fvfzt/x+nPG7D1efbn7EzXz39ubdtz+8
/YzruOI93eI+vr2a7u7u5z9B2PjOd3cf
P/xfMV/99Ak3+envV5JDu4oWqspVlJTj
VYrtKtx/KVWt8erz3z6+ubvFWVyF65iC
WTTLoZZck+WrfNXw50qCWGih4HtCSDmV
Gk2riZiVePWGX42l4fMmLRUN+NV69Q/x
LQXx9h/C1T/+6x/+7fe/++Z3b//xX//z
d//+9f/7u6v/9f72Lzi3mz/f/gg53UB2
3/Ga+U7BUosxaBbVrDW3fmUhSsZvxpdT
06v3H//89tPtdxDyD+8+/a+rdPXP797f
3Vz9x+/+8V//+E9f//v/eLivr6ajvX1/
81XL//d/++rDx89f3X7AlfGZ/+rj91/x
XyHI795QRF/9dPXdzfvP7776718FHOrd
ux9/eo83uT/2r6hsd199xKF/+uvjf/hv
X/0/8atvP376dOPyuypBcMjXAVePDxUv
yWKyJpZxbgVnaBCGS5nH8/9987t/uvr+
9tPdZzwWP3+62i3Jp+e1JMiokHaQFEvA
W0STLsmQgwRtKYdirdSWrv5BJknG3yjJ
hN9WA947aEhBeQhRW66WSpKcJNZaBxCl
WtFJlPgw8SUkjbnFYC3jhGs+oygz9CFb
jlkbZKN1UZbZWhSp1BgrOPdJKUWDRi0a
8dMaom1WymhQRMmxQmoWa3GlzKmGVsTw
S3NpcvUe1vjzn//041t/o1cpSU33ktSU
+GEp0moSyYXndE45HpzWokpKMBjRUqG7
sabS5QidKTHgZ1qEEBt+dotG+hMBZc85
wLjjNwQIMvuV1VZhbUVCrDgMHUCQOH6b
BIkPy9UbuYaxwv04+sQa9azG9Vdg8k26
zhItthRh/kpp+eZNyJNO4oegqxHvCrso
LW/SyW6prZji9xTDs4X/8phICePkNhUn
GR79cbSIOZ9Ezu9uXw47Q9EE8cMswhGC
/Z+x00ys0mTnmEqM2xUVFjfC7ieTCuSE
3bdBwRNGJk6aig/lJHieS5qr4DPCYdWK
7wNgQpEg9EkIUNNiUlRwzXgEtzu1pcVE
i0DXVhxoRsTPXClHQAUVFjASTyDo2YS5
BkPhDOFBo/7ggyRtdmuFEQrscs4wIoCP
zW4ttF6BwhUWHGYAcdCoIGqlziBqcOlO
gOj57OyvwWi8rjjdZNCcipgEweUDjsJT
ihH+Eh+EikveDqMRPpWlhGin4bHAkzwm
jCJso27isl03w3LIKeHqM0T2o2dx3n14
m3/69vOLgWgz+rQlAUiBaWHWVFjhRJGm
BvVqFO1mTYWOAmtSCRmebxwWQxOEMWFo
8rs4xtBLCXYlnkJyGhqwVOAhtUlnQ20l
FxjioBCBhW2+r7+TllCtFJyCpnHj0Wwq
9xZYdQFNLybWNciqOPBQ8Nw1gB0ijhlZ
8RORaFutVg11u77ishr9LDWagehuxojI
Cq8jTXJNDPEWkHWTYCOAcJZigYkPVwrb
WqqLFh7OVc1MOz6RbA0MQmEmQhEttqyw
mumcImixXAwaPplPrQojQ2+qGX7bk6Tu
v33979/8y9e/X5k+Mjw7VVJFRFOiI1Us
Fb8Ulh9+pDXG86/fEGcp/YnU9viPA65U
vDRRWDuN8DVhqWs6ykKwYvH2e9Yr9sr1
6PiW5CoBag3zGJIkRRiJR7FeQ4dgPbXW
wu8196keqexvESxsBt0mWA9/tNWPpxSA
Q4EJg2cVyxDOU6l6SrCFUryGD4FIHNE9
ovqG7zujYGti0AlDmyLUMi3bYlj41BJd
aJjGEGeHqsBWwtdhiaYIfnSjvoo/w4iK
GUv1vH2NiIBzbIILA/KXEawwNZLOfa9b
ARKvLVDGEgUCR/RxTikentcyokaCWwTS
F7WHNFJjTBIAsUA9KJJsVk+AJjQTvy6x
lgPj3i9NvPaj+COsTb1+URZEaV5Ji/Hh
BbcBpSxQkQSvsuKYzijONSgaYU1xTbAQ
iGpNbdJK9dJMZb6E8UzdqpXwDfHb8A5Q
fFgGr0SNiKLurYeeQwonUfNp2uHCuBnh
mCMOhdARf6rkNElSYmpRBQ4wrj+ZbVVM
PLrwiKzg/XiJbqJGxM38WJR2CifPJcpV
SBnxjnDOmAnEd2WbDCMCUPiiDdBZEM3Y
dtcWDn1rgoDUYPGhmoOCZWyOltom0DyF
lmeT5Sq8DJWBYJXKVB0ua87Ta4K6InKV
rFDLWjYDJsLgiEA28Fos9bzCgHiJJ716
squ3z0wvi3h5NoGuQszCwxaWVyhVy5N2
BnWzGFpmkgf2ZStkttIs0YC3Clc5jwqZ
0aNLmJqundKWUDO2X0opXBxBc+ajBo+n
erqpC9ZKYoIJKGoa02b8xG+lt4X/oO6p
V74HhM/SpYpw/PEfT+AHtz1HcHopGa+C
VkgQ8YkWGM3MfpKruX+vSIkZ0angOzYD
Ky4XAUvKsAwKVBg1Cs3+6Is7un7ax7h6
MaGuw1i4vcD/hkMvicnjudkEz0JFdAOV
q6wybVXdaA2XKtJS1TR3NgyIsdbcmbTm
UfX0coyxW4QrrAVMyfocEgRlj9P3GRb4
qWRbFSodcVMAqcv1NkPAUbUU5tghgDZ3
7vKn4LwGPJBVW9xclBH403SatKYMo9+b
yxDvwvmGD4CYKeYyAtjCoZ+7BCV7Ip81
FDysiOPhT+CWjhINN3/FlewU4eFBLTZf
JzxUkB6sLlzV3O4r2JH92GwVaqXo9pbd
mHHvNdcSalENPZ/bksClRqhKt1tSHEGE
eNZnEcJwMWeEGC/AYWLmG1byuPn6OUQY
SgwAa4VBrvRaF0VYIWAxlkjxNcjKRWjw
UenzWo4S6Lxs7wrEA8pufJb+bcZN6KU0
3HVQuGW8lddvXksqYW6gT33MINXYCts+
cDOI0tpxk+dzCPHwqJbrZZmhTURgI3BC
q01NKbi4xKYCLa0F2dG5wD4lKGAryfB+
vezE3jF8KIDPUKulIYQY270QvT8FdiQb
mwgiTFnKxxWy5xDhGjRUxaXg+hiiAK6m
ACXCnRXoIgwqXB2VbX1F3X+FLrMjNONt
4NTYoGiovW0GEOD5oVhO4eHTRMKlEFHU
vGse3m7lu+oMie59wWetmlPM23rmJ8cX
0RCuD746HpQ8LiT6ibEK76joAckiKj6/
INfgIofGEHd4Z1YMOvfg9kJKhccjCDZ3
9ssjTCo0nfBtepPDiMDIjAcuvTbv8GPj
5ElofH5JrgFHQdgY2XooGfEmwsVJJSPk
WqxZhXsJD2XHYBlwkFAInawsfjcdFB6r
Z9UnUeKT0wj5/JJcFTEmdvpFVh4R1fE9
prky/KENxDdDWdv2vngpoUEbYYQgyTg1
fA0IkhH3wSRA7QnmuISR8kt5gIsFkDD3
0BhoJ83qHD/i4NUaoMyKANG2h4/wc1g+
MeAidNGT1iNipapPaOD0vNkWd7WElWcX
6BrYhG9JPUbUB7PLduYZNSFqXHFtTNO2
tKM9nlNqDe5wq1ZznlR0PNjM7BKeem37
MR1j5tklui62hBhrwQ/DKsb70FJjSKFB
v6DnkMOO2LJA9X1ELQZcYkmDgqfF+xwP
PiyL0PlbBZqqzPITjoNdwZF5JFKFkoUn
IhXm5AJ+J82nlBP98MpBFZ9FwP+hzPft
8ALLG2muQ+Iw7uZYM7ixMDxYNWme7C6s
bcwIirTCDxvB6ooyZWyce4bVFa8M0PRx
6MDUKxaH2YL37xYb4H+zHA9Pa9GvhZwU
QMkgEEc99RWyGiuZRAmQsNVo2yfLYkrC
1E8tGgDQ0hPoBf9l/JUjTgQOxgCCNHfI
Q/NKpnq4UkRhvWC5hHPRC6m755KkOLFF
Zt4l0Xc9QZgAZQWq4eJobOeuIGPLe2B5
pKWwFTW7whdGQFrY7s7xtu5ys8U+5KiI
ctsIJa5Yewu0H49n8jiVZY25LY7dnVEd
D85qsVLZgIoUImtY+PY59eNTR7AlxhJ0
2zM+Br3DYwu8bbmV2qd2oY0NDzI763NV
HcH1qdEbDfDSHl44F5AZAFbFSZd8XNR6
Po1cg5HQ2sKRwcxRAogefk+5bjx6nLQV
UtLUPqqweQbb4LbzgWFCr1kaEiRLICpI
jJ6OLeEERj7NF1wSJWNMcGGNrc5AMJiK
ydnhsCWMYkr4F0txT0JWWGMuBbaomk1y
HA8lI1D+ynkv+jiUm5dFmDyPMNcBpdJx
JdsFZ0vglWiXQYIqK2A0edBQtjcNwM2B
c5yLD+nWKDomUsLG8XZqSL3bJy5j5bnU
cg1aiiDmKynB+hVSe92XixO92YSHrTBk
2qyWQFx4fIy8coqwTGVUtNRIE1t74Wt6
WUbLcynmKrxMCU6JZeZllZnUewoaixmX
minofTPWlQRDhQmLQu0fEi8BFTQpLXn9
MrE55ggwf3kW96IhJp8vmNqYQ+FHelWu
8eAVAzzAcUF8cjit+RvzBIm5rlTJc8H+
tEGhU/vADPN4PrNQlpDzMlJdiaJ4/EKo
uDJE9nQ+ORLugWbOjaNEqcVdzi0J+jLN
E54v2K/eVDkejE7zM8V7BxGk52MYvZSy
roLUarW0xPZkphPKfXkMkSd70xX+ERkL
tuhruo7s0syKx4vPzDSoPSam4ii9oJKd
nWZ6OcbULbJN7V60CmG1K5KMPbBfQDr6
VLLmjBIQj3dSnugqYVsJDj+T8qI5w6Bd
s3EW0kzCoYNo+/Q1BmXJTWExJojCJSdO
pVnCmbA94fXbYfPBc+kJW/JJeg1UqiDG
qyWFLOk4rfC3m5u/vP/725v/+ul24kbe
J86DU1vuLcGFpAgYDZxWiPOcNQxlYF0S
9yEctdimqV3nE10KWIOa3J47PjVRRASI
kjjDZCMUsns4fS9RH8gqhdltzhOQEcCO
s37PLVG1HAiL5DjQvDx7IpWMNanCTLNW
lmb3tyLyQBBiGgGEdBC211TEWgtkBTS4
4R1ROXGkGdY3KttPbADTK4XcP9cp9b4v
F6wJaWBJDAK1IS/wuSV6eGzLbUNSG94O
5lXgK+tMwljYlJ6i926ROWqH91vIgCuw
vYbISOKEBnhOMrlYcSatjiBRcTCdJSoO
pu4duRFs1ItjaovnFukaFGU+pzKcNTa2
95QlPiP6IajhpEGpe3JHuXDsgYyP8MEs
DIqibWrk6Pd0CkKf5hpeAkRTwIMF28wB
Tg1p4kSgLySAVsSVTNanHcMnkg0GV/lY
RDY+jAmhsfN+qPaqmecdFiH0vBJdA6Ls
ZyKJJhQSUcU8wEluC/wYforzeXEHhLKl
N4VSDP/rw90jImjypHTo5nZ6WUTQ8wp0
FYaq4LANvqgxdp143BL7Kw1vDGsSakw7
EkjkzCxEYoQtLfcE0oAQWh6bXO+EX8TP
8wp0DYImeOLQIxhBczM4ZWYLgZO8Ma0w
w7C9USGyHkd6KLMmE4X8eAjaoxWZnvei
Sxiafim18CIxKXu/8D3kcGUM2kWbS3E6
ALKZZDaebh/lJHM5DqWHvTJqRBofyIV4
U0tgelHRrgpOEUYLFBPgGXofdA9lUinO
j9By5R6ltGMhS1Q+IIiBSya/86DQGvsY
El7aw8sRtl5UvutC1cBiNVOwzZPV5Zqf
xkZ5w56Hsq9Mk7mJwMh8kYKatEFRFhbG
PabWi+KpLOHsDuGy2jKJkqXSKwaYDw3Z
bIU9IDmmtWXPCOzfScQtBVprJcE6w2Wa
QtbImnYrEaqGh9L2+MQJDndlsqPBerU4
bYLg7jTYfHUCqzyAWdbMhgc4me4bq7Oc
tGAlwxjRqWnkcz/MQXz+4fbTd2+/x529
+/s+SR4e2DLzG3vZa8xUVK2zt2fszWzw
D+hRQaI74lW1pCGSsKTotN8sS2t4hoyN
vKGIDCDK+OAFT+xD1XdcZHaZVXGiybMJ
0gIVgbwWbKM+5QUrU7nKYYX2MOPrk/eu
sZWsmNvzSIE2H49FYB/3vBSLPd6FohQ8
LjoCm3wVZyx2KuPOZ2yVfJVM2BA5jhP3
zyfGw9Na7rmG0ciMHys+nBd5wGYkXB9C
oAQoLWV7/kgS55RYsrParDN1K5lA8PYt
O/68fiGybZxZ3dYddjYHkjAqpcBOpGw8
wuOa2jMK8tchUumdcXoQ7lExrffkqAkG
D9FkAgTkun0DKNx6rjKEcTXu7MmDIqR0
vOlDEH0F6CJAPk0yXBYiBYEnly0gCK0c
2ZwwwMvexRBkIXZJO4JQ8sOXyFb8ZuSI
s0EhMnmLERsyHtPD1+7d1lOIeS7JrsFM
aHFsmbDGRSzZ7mfwExnKnJnRg68d3fSw
sRxVSjnDiyqjgmYvS+GlPbwsAue5pLkG
Os9OqcD+MYS76q2J0MlR0dN8tkWnan09
hZ1nk+WXAPPCAeYvJA++RJuvUa6ros2L
SPVL6HnR0PMiMv0Sh144Dt0kVSOr6iTW
ItauSFOpxWXNGaTG7r1D0nESKpombv00
aNyyDYamsrMLl8kVhGmuu5Dzy3IVfzq2
L2Ax8h9b4eMBEzAvYBHhXIWRvsDgOL1+
EyydCL/09k7O3NJnqsBabrAQEn6TnGRx
m86f3/3447tnkObBoS1X0SjG3CpnsEmZ
MrlKHKAIhaQZzfAOafMSFhhe4/JRZbdL
cUcjQmdhFJyyiV2kQ0zD9KHJvkVmqvSS
u11IDcTlByQKPqMwgd+wCIWToKyonah2
Z4HhxSHjS7GmOR7FxyXg25qw27ZtXsMS
SqrOkl+4Lb0U93pLUyOtWPRy4ggVM1bm
exiKv5Nb2OsUGfNVZtvY+HHcrPucenlw
YsvLXckKh0NmJJpru08t1AbzyI3YiW26
un1pmRMyJTxTQJ8IFe9XBmc3IEQCaDvz
+usXJuI8vRcmPnG4gBbUzC2vZJOxdmIF
3XNZ2TWYWVLSGnqTO/3cq3xd8OQhNC2M
tyyGo1GX3waaiXaBq0oq1N9GBc0YncCY
pGq8A3iXJ0FzYaXOBWEzurcTc6GOctj0
ynDVcGzZhclNWCK7Vn5mzluQ4bo6zYMM
iprN10NO4mQzzynUPJs01+Bm5JJWjgQz
XcT0w+zR5sT9ShabIdwA7G1WTwQ9XFsI
l5AtN7F36Y6HnJAXzSy3drlPyz6fZeg8
n3auAU/DGdMaM23OLe7zJBqeNjK9mzAj
YbYdPRP82cD1KlwwE6al9wOip3bWrEmg
+CyfhM8z2ts1AMoNV4HlM+XSFxLeyDXi
iQJDTOpcLnjdBaBFucCZDAzKnkcZFECz
9+XKFKZkj56P8fOXR4AvjqWVwSEgEw8j
sDQ27s8W5gCkkAxVEWrsAlO8f2HSiivS
ZOKJGxBMqxE+xVNGoZYlKL2YYFeFo+rL
shH9B2JemzmqhKvtYDohXFz/juVlwQk7
tNECx1jGxNTJ5e1inTzeI0i9nMKugVdY
FzZdV+MoNszKPBVMVuqM5xEXT4bH7bFp
qcoCLFO7KdVUBkVXeBsUbW5O1Tm9HKPr
JuFKnkVLmrfHlMiQrTzl2WDcSS0kp0WG
V3Scsb/aTF2tHsPiTZhQ0L7Lla3bgBoW
miLs+bFp/ebTz48F1f+8yYozqsX5Qc08
JdHD9j/+61e4kH/553/5x6//4fe/u/rb
7YfvPv7t7c2PP33++9v849Xth7ufv//+
9ttbCPGr728/3H6++eru5o6yvdt6ZEf3
sHBiGyoXPDEyFMGPSYn8U31vltDjIEZy
yoBdAmuPrNl15WBuzi3DLSoveGKBiyxx
E+TpQZgTjgu42x+ywElY31jCpVutj2Jw
Z0xkUwb+r1r4nB1ahOVDS8wTIaCrPGnV
9KKHdngPz/ecEZ8LR0Pp3mg3PrgOgIIQ
OEmW1Bas6PKZSbo2cmMBz7lRvUR50VM7
nz2DgxjcpOVGoOvlp2c1aB5I/Z9l0vgk
NTpzErjHQJ7dpL3UmZ3VqCXhTklSKxeu
ZH12k/Zyh3Y+o4bfWentc4sVtyicw6i9
3LldzE2rz2zVjv3X/7MM3Jl9tldxfBd1
4MozW7tXcoIje3NnO8JyPxxcuSzmyQmm
+vQE8WhU3FEylnIX8oWbH0HSUVX15UDc
3+XAIaQMLeorp+j2LVDtLp9fjOGaOU1h
R0ThWvU13rCGs5zZ8V08z1PHsYsYE2wB
V3gEzuG7Mwz8EPIpitMfrDwyYVJajB32
tXHoaFXUdaYTC9zdzNxhEksFaPucli7S
rOM9oZClSeyWrqlyOxtclkQG2bV6Gq+L
Vk97JcAt7fOLntrBTTyjV2cB7wCvh+ns
iZWK3EiNDINNIaOqbe2h2XWCQoTcyLlU
guUXPbOR7dmvOsJfLNqRRXupMxvdpr3c
uY1s1V7u1Ma1a7/Nz/1i4g5M3Ks4vrGt
3Ss5wnEN3/kOUPLDMNQU4afH81BMAjxt
hNDCsqcGaSdZ6JTZCvaK1SolSZ0JBmGP
TJUsDyU2KbsWU4gGk5YMMu17WjkhmWKq
xuSA2AhTi9qpEUrpG0d8GQ6XBnPKzBTP
E3uDjiiXu6g/fPxwn5jYJ9CDY1veIKNs
WvNlpHio5/li46QFGQkzeTt3bPiWVqu3
TZFnPbfecIMnxwouqebMnovXL87OJijT
4Xi/EtfSNTa2NNxWON5P+uzCZCdJhEFv
uWSLy31KyXejpea1K7U4d9QrKV2FnTDm
jOhbmVbEmQpFcCEl9t0rNRtb9rUw4Vrb
GLOKnZlXc3j8x79IttvIu4KephwSvEOr
C4T3zy3dw1NcXpVINl5hC2AzVo2m4WJP
STcys0ghbfJmsOM+qcYG70gS2dx7LnFt
gP3kS5IqKWVfv4BrZxsnaeb8Qn88Cu4o
w86lRhVa4Lx/bqmuQFTuZQvauJyWtcCJ
HAk2JYUqMXFLitMAbB0Yr0GNc6ipJJxD
HBRPe5FApsXL3uK9CKcHDMwvAKgxOXl5
JXdr9JkQ77KH6sJTxE+SJmnH1sQinIpE
0NBalT5zPSKgSt8p3LnAHFcXAfXM4lwD
qRwf5iAabC5Xpd0vjokMe4UEqxX+aqmy
Q0WFPa1JnVl/ajwcEFaLL9QIzUdp4O6l
U0B6ZqmuglJAgS+QVqfdnhylSH5tti1w
lUGre3aBZ5KhwVni0KvqpKfDASk8yUci
TS2extEzC3UNkpJ4pXIJHtwY6TxrHptG
MjwgmMxuJbfraeNTkRTWv3EMvY0Jpd6W
H+JEOJQWofSX+7hfBFatkcUIguVkcKoT
XVKGpjIrASMtRfYsZAu1JHZk9SUkvoVl
RGDt/kb3IuMSrF5YtGsgluGGOJsc4BUm
Z0oqScUlaysIaY0u7A7y+57gzzgDrjKw
QQEWx+Tukk/ZBJYAFgD2wvJdBbYB31Yj
N0koG33utyniX0jJ4iyFcTvVGSn0aPUz
XhkLDwq2JD3glfcyh6RlrH1W+ab6VL7M
tR/grlWY3AJhwjYvr7XV7Ow6OTGjUpqP
bfaccABWmrNEcpf8jpyw86Zx4Z9w9ZRM
oMHtnOT9pQM3wm5xIJd7xp3ity+lbrDP
nOsuJfo41a/mJFgC2CfSw4NbTgvDIJeQ
cdqeQEyzV8u+R3yBG5KVJnlzXpjzq5pC
I/GkyYS20mIkQ6HaEGDbPUydVvE4I0Rg
DUpYD83t0UDOGcVZCmslpTJ7H07MOSZA
sBtZaTC2kue8MNknC4f4oMK6fSuU4GGx
xG2b8NL5cHUDnEnRhP/jQkoZYcpxTWI4
WYRNo1tYSK9xfvkeHuPJzDCQgtua2Gsg
95lhxEJC+PDF2duIuefUcLMIby5U5+hO
U2oYTx/cOhiDTM9qABlX4JUnhV1t/aXj
bMu1IfYht1sKv57x3y/YFdD6Qsnh0YB1
MTu8gKu/nKO4DLKeyA/zf+S8a0SQuj2O
PZUgHgxZpYaeH+6Sda6eRWw9t0jXoOuL
pYiHA9jlFPECpJ5brKtA9WVyxMMh6qkk
8QKgnlusayCVWWJTdnsaPeJ4nyUuMJeB
IQ/84bovTRycLMhIHzLV0McD1cU88QGo
/rZkxIUA1vARLq4lKFJIdUpHwAGG4gIW
YaSjtj2Z4sJei8ruNPym1sZE2Bge5Eqr
tgSwlxbvGrDtvSlCdlAgaue/7cniwAne
xA3HXFe7N1tMOwJj0qZs03hYu5wtPsDa
S0t4Fe7C7TMiLzQd4HvvTcEy4+eTUz9X
Cn5Hvjg3X6rim1EtjhrLxr4jFS91flmE
3n1STrHNAhWrrfoXHvqx2fZ7IGZYESd9
JZ6GU4pMVq7MBR5wjiXNu+dzKKwWtVbI
v7Vj4zHTl0peqhbwxPQlE43brRNrBLiK
ujQ88OpErE402yN/ZQYKD39m3Jgz22rL
ccL43d3dzee3X3/9b7/fK8XD01pM/pOJ
kfl/VyfDL9RrNuJnJcus4b3rzt3GMMGK
0N5wu8wW+5UFOnHi5OGwOiPI0Rxv4aHq
48xiJ3/q4EvMNYEJ4k6ohebE55MrrB4A
MhmcZbhDywljErVX+FqR263vbTDwmUX4
BrGXBBlvB1ku3hCmKdhM3InPcS1wqSJf
2OCqQ1AeVq+yq4WOIS7RpszsCKd5BPpx
ThU9PLHlOBagBpWhmnqSvgszs9VDE3dE
cTfLNoe4TxmRu7IEem1k+Pd2PjxlKZOl
hTzRWuFvv35h0po8CBOfeSRbOR/BRXct
cVHBcUnn+aS5BjZpklWjKcIYbo6LV+na
EreGKeEOQU8+ZA/+jeEraYlZssdvLLmX
pAcETunjCb6tMcAhOgWdT1MSlwVP50Dh
JjglzQ8ddGP2NlSEuDh+eC0Q5Y6+CMtw
omFouecIblAdFDtrF+XUMuxF9EW0PJco
V+ClcoVia9Bd6CX/tV1rY7pBcdDcw1l2
KSVzjhCm1BSTt78NCpfZY1LcjSdlsudU
F9HyXKJcg5dM3oWMqxJOk5Y8Oz+VoiWf
MCNQMoBv1kvoPeKTTG5kmHB/oEfEy1Ic
KPGS55dTeHkuea5BTISYarSnyiUKzhDl
GSNEFVnZIxYQKqa8PdRkRoqJReXSjWnf
y4CIGd2swC+03u/QliDzl1MJF4bPhnhT
nS8+w/DGgtgTwYkPsNVMTzcfse3/Rl+I
m4iqwf7DQrXeODoggFqYGqwcQc2LcEcI
einJrkJTrTxl4J0wkzB3jFoztoyGVkSb
5LJDY7npiGa2sNiTB4VTXKwHLK3Lt+VF
PL2UYFdhKztWCpsgBJ5unBML5Klg1U4K
YhqWUHY0KmUulcP1ibKh312MEcFVs69p
nGTr5CdL4LpfuCXpoXATvOyjZQrMJys3
pSZ22CwmjSC7kLmYRRuc1aktUrkRjjE1
m0pjitu1VgTmFo+OZbiOrJr7lcHtKBK4
cplUAyNsP+mjSb5pKaTmMJvhQFTyTkd8
mE+lGP7wP/+4V46Hp3WigipeIsnw0ANX
h0/mF1DBsFTgc0VLezoforYE1yLTFe4c
H4XVeGGoxCUwiHpfvxx12gLXXE99Ny73
P/H4SBAb0jHFx7MJkguAEmSoVnmIy5W0
LML2I4GPW1jYnRSSed0Eh5ffCIjYro8c
PcElJq5phDfh1xXJwQpYha3AtaURLG0v
4Xd1hDIqz9a8JTKUnE4m/J5BiIeHtdyw
UpjZU58AgYWYtBGxD3evGEBNWCjdVg59
Y9eVl1GtkdhHtScP/dqAoSmSvSVwdmwA
QcKbC+4AlTi/OGF4MdxgToxhxLId9/Y+
n3ldA5MweZkzD5WNBoVLx4UTmByfsWjc
WWT7skVsAWe1jGXVyD27g+Kk9ibz8LQd
3xu21Lvxl2BzMdNwKeCEmkIV2fDJB46C
Lak0J3EvTnGd98WjeAMyvxQO/SWdBDse
cHo74LJgnVZuGUfPJNlVSCpmhQsHoLah
SpmCmYxPuNmLxWktsiMeNePOVmX3sLXS
/cMBodR3b+Ow+mCUlVNoei5RrsJTnzZn
HSGURjqlecAFXzCuylErpuQM2t6DzzJA
boULWnHrdVA85Vj2la950/nlNJ6ey+6u
QlQYV8aEvooORuV+S25jlZvtI4kN+m17
vwIfXbYbmvKvKWIZD1EnZ0CmRYRxKfZc
lVG4FKDmyvQcAkVSf9SUHkbH2aRapUHB
KgLSHbrKxwaKD6COgBsZFFL7evJJsNN6
8iMUvZBoVyEq207YmFKZrrR7RIXvGg1O
VIFxzmF7bAoZwozTkHO9T5m68AdEVPeK
eoMn566XEPVSYl2FrhCpMkFfmF+3PKeO
FF/gMGMlpW7cNog6wXRN0HiINxNRcxkU
XEmagEunEzK/LIPrfunWfCRdros+SN9z
O682WBA6RW2ZhSfh7FMDGLPdU3Um4RGy
BCQuMTbg9XaUZdECQV8r1nClPguGC8Nj
1wLQt5B+lIuQX70xlj6VUMwH7SV7rIqg
rQJkKnQhlgXC2C7JP/zum6/3SvLwwJaT
EBAlt1dH9WD6zTSb2mJh+t7LgttaAtM1
opiGWDeST3/ufOnXpYiSgAlKuzXCDBSJ
kyA7zyo5pAbheA/HpkOu8BvCqezgc8ix
ply4F6w2X6q4qJBQWTxOWuBFRYnzrJNx
e3WF+5sUz1zboZDsdWHyOGWunk+90MfO
pgAE8Cm6EYxt7FNgpVN1xJrc72WeJmUl
yYkPGp9NkIcHtphhIKglBCmJ28FZTugU
DhmGFa5QgQugFnfUXETFYZuL2GHizaai
LZCZMzncRbmwC+DViRLRQJlUEh96aE0p
Io7I8Blzk5P53eeQ5BqQFBLI5d7FIK3T
jVEpK94PoaxBp0PWPc0LpA8gUUDk8HAd
FSX75N7sQ5yCyKW8wsVAEtqYFZrJycjQ
2IeSr0vjckmoJasxcV+DLuCWNYDGMCjW
OihKTotZcnF3Z6IXWgbKc0lzBVRKaMoS
WkwARW7ymNJ+iFIKZdvY4cnW/816ibCH
2X4mF0n7IIOCpXl37ixOmJhTYHkuYa6B
y6iZiwgi9/lA+nFiPApcjhTgdsIjSpxl
2MGkQjJUXCynUUOTUAcFTIB99QaF2B5e
ljHzXAJdh5oAS2YdSDGV5lUbAiFyryvJ
jxr73rdPdcN7yokDiVD4NChkRi9wUzXc
jy1xCTXXJAwuh6Ca1YQro6CTc0XerW6B
TqlvbGpZdySEDOEmNJEPToWbPGqoGbMb
GNxLB9NallH0UtJdgagcBTUaEphZS2Wa
4BbjirDC/kTayh20vQTRwD4J/EqAQR0U
T6XPoE1KK1PR5QBPLyXWddjKQcLWyMRQ
ycgxD8AkDUIW7gblzTu2QuGX4RqBo7U6
X08bE1knUihuRHp4OUbW/aIt0o7S8xGW
/mlBDV4QnoEAU5Gc5Hq5s95gjdnhWX1T
233iD0YyFAWIkHihbcdZ0kPWCMAmH06d
nI3iRTY8j7VyGeYI5jgBsaY0Q+osesAS
YZkZfwcI41Rv5x/+45+/2SvJw+NalmRO
LbFtniMwZe4KFDa548sITVn/2lMYLY2j
xlVSuqexwjum0BAvBTb7D4GrsHZzCpfU
ZPgbnmjinFZklCCnM/H7BVmBixAO/JDK
MablKaZMtyWxvbMy816nvB9c4JJ9OMUa
B723w2hSPBSVixURz+SJzwhOOdtgDOra
3EF69dYWjmKeBZmd6hOGhvsJcX+ppk48
dC5BHh7XCb5ASJqMu0aisrlRV1qKBkTw
daF1Y5tu93Qru1ASMJNcvLnzBTI934w0
OjYEy30mIVUXIz70M0Ks3kgDVICgmdxP
ZzSta0AS2FWMhNiBc6M2x6IJ8mNDCMvw
lOeORFEiDZYo57mL8+yNCJE6EQT2/LZP
4yxi5GJW4UIoCctJOu8sXI1IBgQ8eXRd
gZs1cqKz7RwQDZa8jYVWVmRQjMziblon
I/aq+zJInkmSq2BSqg8p9JYx9nDOrWCm
QlZKJbbVsqNjKHJrFB4orlBFqJMHxUmy
bjGRkHvAjEf8FFSeS5yrwDKocqEXW0cs
thrvMY6cuJ6nj0zhbFLNeB2VoWXkwh9c
jcr9VMtocEleAXYfxGlbSJ+FXkbMcxna
dZjJVelWebxsF7hnbUwcPIORTuRBkR28
nNTOqlDySOrl2EcIBoTNiTEMdoy31RnD
jnBzVc7gUhia2dGXSjZf9iT3+5ugWEpC
zuZLfnZEmj6PCG2vfHa0DIqipESmZC12
Nk5/Qo+B9EKiXQWqjEhKYAa+RIBnm0fo
S4n4P6xLCnyfHbEn3oZ95bkE6wW0ERE1
+4KiWbCdvegIUS8l13XoCu2pgT14ML4l
2ZwdSp7TjRLhTmnY1k30JrPzJjGPjCAm
pTouuPbUfBNPU/rLIrg+g2gtHTdYw087
EG0g33RWlspOMWhIYDqIqfLsrc6zH1zo
GcNrrUzZtx0sC3hqcAKA28wpi2pdtHDX
MuMoVpPzCAMRLEk5snZvWP0+lOVflpP7
1oVwKtPwx//8p19Kx68S5sGJLZvfyKS7
RDItOvWZxyGNm2BIdp9ZJ9m+ZE2ExVlm
EVNEzNTzDJXdoYIYAb8zjlH0nty/qXTk
rFQsiGbSlqbAKZHzSZEnpWwxgWrqCRAl
iTWtKrw3kv/d+0ccKCrOYxRZTNszzFKT
V2eo3KZ5ZnMi6RWCJ9JIhhGGWXpk6sDj
7fFudYtFqAjbmKUWPcV98iwaeXBgy2mG
Qk4oep5sMbpnnrK+dIBlFURC21v+aOID
WVTgT6dpvrK0BoUv+N3cXzDEWBKOQh+s
Kzc58jO2CwhzNLCtOZ2isH4OUa5BSkia
W51byQ3eSp73kkauxOM0YWZ2EPHMnqHQ
wn5CuFIwDjqTrI2GlH1MRaQvTC/eGL4M
lEsJhgtBJQQtquz241bidN82rA2+KLeB
QBiatrcRRcS1KQIr+WRYmSZ8xwNLnR79
qWRxCizPJckVcBnJLA+VxNtwZgyP2xu7
Jl9bYm0rFVbRdtKfMLeo+H3shyudJWNA
tKSNe1DM6EP4i2h5NrVcgZc4ZJ/MZi8Y
jvx+9J5TSbUA6BCNpK3ZoTflmlzYQo2E
CwsVnUPNASGTFNy4dLzEh5dFyDyXPFeB
ZpXEjBHjei4LnrxuPHFwQYNxw4ekWneE
lzDYHHuofKqk9IUe44GmSOdSzVOb5iJm
rskbXCrUdIhDTNyS+vbRKdbkitCYgRB4
47aD05prhnHrGRpvYdokOyB8hhk3+1O/
BJ+XEuoKKMW3QBuzRHZpwX7MIbJBqBC0
ccUKvmNH+rYxhUsHi7yjw8adcwfupKxx
Me68mLKuQFVa6SZk/Kt0bR914NJ7SpxH
45DRnmlQwfUyNNJMLkgdNRC1ntfiRvKH
lyNU3S/bmvJxdzVg8Gk5LblpDYqTZef7
coKBdTNu46i+s/0+fwsVo9DVRFLYPrId
8EyT24ro2jR0fgyS5aSMCAkqgCsYwBA7
ARcECk+E0BoSI5rMYfQKeycwTA66iwmG
b/7j97sleXBgi5KESbSCcBR+Vebw34Qd
OWoh+TRnT5g+3xySWuaDzP3DDHB7+pbR
AG1xEiljlEU7tY9Oaxec9KTBQUD8l7k5
ioTQZxMj10QiPIFq4dvTCTniTI27IYpB
ZzOs3pvI1kSAJ6lIPO2wb6WShsqaDXfS
IpSpfUQJ0qNQW+WuPDbRvX5b27zBTXre
FoqpcI9KjJz4zblwRfkZRXl0XosqCfgS
MtCnqjGwKuvGVbXPxxWFdYRWb9PJN4mM
WpHF3lbYA5ofwtGSGkNUzs0Ib2cAYXZO
LLzk+YVnJdBHRASauekPQj3VVP0cEl2B
ltFHjoqSmNEIklROhpBkIMosN8ea9mWL
rKnBouI31r51eEi8NA9BgYue+TOfyF+E
y6XkwoUAMwanaWQjWSBP4zwLCk9FSJYJ
j43cLDu8Wi7FI/LiqbE6BaHjISbX+1En
+z4PVb+RJcw8kyxXoSYip8aFLLQWJTy4
sWQtU9pZxDduabfqpfLnDQEQtDOF3noz
IGjCHWwPwkSwfgI0z6WXa2ATOkO2g8ok
K/vZp24hVlrYChalCEPSbaip19w/wHwn
Od20xJFBU/IjzfQVRKdB81wSXQOb0bNw
kV3UOYTZ0sL/EYaXjTwBnM3crJxirSWy
u3FZXis6Jmjm6h4tXvTRyyFqrkkeXApB
YVR57HA/uaClTq0xXBHB3oRs1mC0N3bN
T235ZP7jtrOWg/aWzQEhVBwzA4/CM35L
UeeF5LoOTTkcCDzl0G4rdr9zMgH9M5tS
Iv4KO8ayC5kYEclWrpiYuBQGRFNRLxhx
GSrdXZW6AKeXUthVESn3KQtTC85QMlvi
GFM0poFDaJ6n3KGxkdMysOaNy73TqNBa
fDWsxL4atky0/MfIukm4UMJ7SQL7wpWG
R7Llml55KlvzWbNkUD6WPZdL340Mx1ZI
Cc/lOVOugcmkzEKccmfvRjbcyQFzEk8R
2HdgrP/aCPPA3U0ZjlSr+E0DGGNN9yO/
3DfPm0OAGnH4ZPaHjYvHrWLv39191nB3
c8dLu9spzMMjW7bAhVkdaFEkFd/9lAs1
i3JORRr7jXYQFZFQnk2hibytfVjUGk8h
Mb8BGbcRdgioeuLPpameZ9Da6LMUOCFW
cW/HnLjPKc3i0w+kIq9Q0naCHSMHLvcs
pElPKcz0cKRNV+aiAbERarVjIQTMOokE
E30JCTpJM1X4GqTPyexZGcDsJlzqzKpQ
Pe2QU4nGNZycWtJyXGR5XtU8OLETqhmU
u2pZiq91ZsatFQ56gVclzny8nVehclxG
2VQIs19rTxsFafiPJQEOpQ0gSfb1zMQK
2ol9Uqkk5jIScQXLx73VzyrKNZAJVFSS
ncI3oRLeQyY7FTI3mpba0o7mosiZYi5+
hmso0PE0KGSSPQmXDo306W2SJp5Czadp
hovjJs0EpxO4Rdq4kPB++iFRh6xCd0mJ
soeICDqIW4etV07dDYqbUhzyJ5HiMz0J
necT6SrwjEzVkeHN2DJo90wLDeEr3PRA
SyO48B1UC0lxFZprTpW7xwfFznkhvHcX
JS90L6LnOVV0DX4aDUeCnmZxWsUZQGFy
gXBcik2w28YVNkUq0ER2IHI5OxwiGxNC
AZv1QUXxWTuJomcU6QocjdydWBGVNC6G
bQ+xJ7wjLn/ikJhuVNCpf5v72SucW26f
tVEjz6R6791yjn0JQ+Mv5RMuH4bW0Lxn
PiTuBGpzIhC6lRuXw3JMYgdbbuaqt0qy
Iv6G2fQOB6ZsrpoFi3huEUkvJth1EamR
gy8650L0KatuOfEIQs8RirKlM5XtBVNf
PhvYn4/L0CZlUFSV1mm66CQ1X819hKkX
VNk18Ooj3BAjv5Nc5xO8loynAeI1ZgGh
0TvglRVYPF+ZuSk+aGPCq8Dy3Ev2BLZu
FW3VNAsSjgjLAvo4fc/SZzpssGexxQCV
zJKfkG2FxSYvvTJf9MgVViZ8S8ZPZ/Zn
bM8jkQUrs6RLtG+5O07kwCfjDucQtQxg
jrk5kzbY+q5bnBTkjINntTLXyt5mOV7L
8+0Pnz5++Ii7u/323fu3f3r/8du/vI07
pXp4doumuJRQWb9uynXO82CwcWiN/Ue4
3MQlsptnYaKrKSxVjRFR1EydC3OiAgtB
ipQRZpx8cjJM/I4k0Y1AMA6DcAu2L325
gEDZOZsjCU8E6laWvaaSmWLw/bUk020z
thZH1prZb1925JU4OBW4yaCyN9/6HAyb
7CFokvAQcQcwwBqqN9XH0pvqK9XU94Jx
XV9sfVj2AkI9OrvFzBKtSmaFjd2fc+mU
dI8IfJQfNA17glboKY2FCK14nfa7kHu+
WWylcPxxAKHSp3wQKj7zyqnB4+CcERlD
oi2sKTyLWNdAauRC7sr1Epngnx8KbbC8
iSzPKdVct++NCAgFQmbfhCYOIMuYmMqK
BJMR1knLfUPaIqQ+TUW8FKhGpiJIhyb6
eB8TdwyQ95H7JKJj7fb6aas+fEyitNRp
qQYEVanJ08Cdc5eT8yeA9fxiXQWt0fcN
5kwj0lKbNzpH/ETIqbJ4Q6q5bXHrm3Zd
OJbRCp0mREr3ExUDomutXWGr34G/LKPr
+SW7El/xbomsyQaQ7W4qW864bRwGJhnJ
x3TjFuCp2oenqgYuHClVah4TYLP7wdL9
4B62LqLrBcS6Bl/xobC2wo2xuekcsuI3
wEKTipfbe3QHDy+ThLTpyi2Y05rD8eAV
gZ6nDftaJq7hXMJX+6V0xIvFr2zWT7C6
hZ7xRPUDByFxerEYtNfXWG5tKGRfsZLt
AwcA37KMCrRHLcDHOHtx8a6CXCgvyfG4
h4vr8easEzvEayBThybGLDuIyyKZCoEQ
VnNmS3QaFHKbV8yvp7ZXfzmG3IsLeSX6
wsDUlti9qw/89yQ74oxqDAlKvIvioQEq
GClzGzVEncZE3xhdxDotZMC9LALwTiFD
Ew5FKvVRcaBL/qmU2VeojcvZo7elL5fe
GxSVrTEFca/c9/WLJqh/SAqXl7zb2/v6
jUwxqQWOaKS+pQtRtQKhGUBye9cAttp5
MMi83MM6YZH9msUOOKiaXFlkqa305t1f
b2Z5vv348+fdMj04ueWR5GbOLKcsC8R5
YRcEUBxWYLcLTfyOQLevFodxhhmYZnCq
tCZsVVQj9+YI01UsmT0Sag94lat4lLVt
LVnZLnZ+ofoqRretgLvlgTkibmEtraRY
dOZ/KplDbsYFQkn29CWyIpDJPZCdnXDC
W4V73sjNXzjP1wYwxm6DQ7fILGn3TSuh
AsMSl0TEBU6WMyjpwbktMyi1RujgnsQA
lLhnSedSW3KbV5bddjRSiGtk7Rywnbub
Oxdi8P2lHJDMI9Rk4Q07VUDzAt70wkJz
CjBB3tgQ6zGtx/MLdR2aGvmcslNC5hBn
SWRfi2cCA9x28PlG5zfk2A1kV7U3go8H
pvN+0CkHcxJJD1rYXgRLcdyFiy4bU1E2
8ddRucmuUir3S5MMYXvvML2sQJIzrg8b
FUm1G9zOu6Me6yzj6LlFugJJRdwvD9w7
UnKsU+M/R9i4UFrYKs4Z4s1KCutvjPsA
yqTGioNCaezUqt3KEK5OYenZ1XQVmnIh
CBuOID3g/gyn3kfIIUXuBSdR4Q7qlli4
x69yXss05kHxFJrRe4cf/l5G03OLdR2e
IojGYSfS1eW+5trJHTKwlEsKgK1qe0hc
Gu6azGoRll6sryAZEFH9YGLtqVZPKB5D
aky/2Lr2MqEqrHAsZIto7JGZGp0kNTjA
JVkJJNLaUd9JLAHCtIuvSyqDwqs9EW/f
9XUEr5cW7xqoLbhoduhDtWxKFPgKTXqy
zPgh9uFEw2ao5UyHr9AtTFLmQZFWetA6
BfWyGLReXHlXgC6MJ+xwbCQTFzg8M3cE
ngzy9AJM+M66oz6LsDVJzJZDg3/cDduI
oIvz9dkrcfa+6eUYdvcI+XEVQODAHlYB
YAUPZAzRwso2shSSsfkEh2xmACTKJHCM
k9yVGsyttbWRAW1HrRYXmbPPNitsRJga
x4H6TEoLR3tCHKJY64Mq0AH3maNzyUJz
SLHRPCXg21RXdLjJXqEent3yQEAxznrA
yGT2QNzzqcFPRmhE0jwSNm0v0RLy+f4I
krmDsUz8WyTITCZcvaH4pa9frBm+ExU2
dBquMPHLsnImJHI0aE8+Tj6dRbAk56nQ
ltoQWC4b5JSYUTTvDOA240lbYUQjs7rc
bkySpe1ZYimRfVXcImGdWgNCzeSs8Aad
0oagaOreMo64Pf7jRfjSt/c1rldTs8Rb
O84Yn0dxn57jcuqCW95w+Oyk4HvMPYzw
n/nTRpeM/Wo7yJrcZab1sPsF7eyiM8Zr
lXxQI9Rki0/cAVXdpSraG4+1QV9TqQb1
XUgZn0WqazCWjjAOvbDnGM9emYMg2Bd4
PuyIYhZmB7caedo5wUTuIFyqDQuynVQt
mYZHf9wpjL5HYBFyf73t7TKgi7cqzOAj
cOMg5n2hHT6yJI7ncbyv7ADdInywcQBc
LDjp7nCQ21xZl2RMOrOT8HsBIa8AYDhP
OPjM/UqGj0u5r+q1StbOSJrKIttVmSMn
kQbM8Bq05DEhuHYuvSUhw3E8BcGXUOQ1
IMxWisgw1/B9c0kokNeHtRx8MZaNw7TT
UJiwFYeXEZJMyxjGw+C+LbqbbJbMTkHw
BaS6CoQr28EpvBQQjLeZuZb7pTyDCUxJ
bKDfnmqGqY8kdog0IiENisJqwQG3k8r0
KOMId39zt9tlIDgzS0gMTqJkEJn958xV
n0DnrPCgw0aimW7pk69lZl9HrqmlQUG4
p8qtdtz1rMYx7l5eyKsgmC1S1pIxPw41
fiN4PL0uyKVbXC64ax0Hbr8w7xXFWArs
o7YDInAKjyTMUYMF0H0BLV6BvwiGcuMK
UHJP94YvJ5TPRg8LblblAMiOEJh7Bvik
xCzsohoUf6WTf012Wpz86wiAd0r4aWXB
5fm0suBif1pZsMxSvDCdbCeoaOg9Q7RK
qmKv+s4NrFxohlBPSF6cZEfauZDuXEh8
ThbA3v9ZIWI25BXFs5dGmCCJnSddame1
SO5kNckk/oCCcwDqV/pX5VeKRetEenBw
y+vRcdZAXzhXod6v/84xRcgRuF2Ezaw7
tqPnItDMSlJdK1MZn7zOpBznjpYhcFd7
oXdaaqy9ikB2CESVRn6u+Mvtq88iziYw
d7BxNBIIdJblyRXLXNHJZpuAN7mSa/Js
Nmdj4zbsQJjdzsCIEKhxE4APADHQ8ivr
w19c6AOBjlD46zvMREkO+vDHxcuFZvGa
U4owRggwOLSov9zI+jziPTjE5ZlqPG+Z
RNqIWUhZNtH3kbmMPGEctc554+TtxGkM
hGeBGygUU+kFUdabAQ65MtYi+cfrF3Aq
nq/FS51fyBrFlm5pVYqPQuZLGOEVuMpB
uojHj3EIrmt2nVQ6Oxh3niXZwWvBFZIl
840K5zNHRVXfnUSc6gkMOQWqpzvkLgar
sURnKWkV78K9O1M2in2sjS0cnJEteyq5
gGxYL83Qe7LJ10GRtTkHDfdSe0Qb4ilo
PbNQV4FrFKhpItc7112zK2LqZgUYBq/P
14j/dkzlJS8scVA/pqqSx0RX3xzjzKnu
BSc7BajnFuoKSA1OJRSZKJOUUm3zPK0G
7t0JbAHKUvOe5ZKR1PVWuIyJfvCgkCp9
FHiSauyELsuIem4LvAZTC5MEWSOXPcTE
Xhq7brAsiDA5aVBM9q1/rdzhVOCNmb/j
mJhqLkw8jfHID8aDuQSw63vhLhfDhgTD
UgquOEYxnWNYqBWCz8B1dMwpb45h3XLR
KrNfWoIMCrXeFzTNvjmyHOHshWW7CnPh
A+PpU1xnrT6w/yZfQ+cR2yIEaqz67cwb
cxYlMRI0CrKz0owHuYCWU4pcyFV+jL+X
FvYaLG45ECPNpctNJh2LocY5M+9QW2k7
Rm+5Es5p4BAxx5zvtzCMhsSWOx1n9s59
f1mE4j0idq9tkib7AfBm8XGRwGgIDyRc
yaOKsIb79k7ObWolQRE3JtR7vtWcEx0x
TnE1AM+uNT5cxdbgGmSYlzBps9VKMDCO
mKQRJkugAnFivK7WGzBKdG+0kpTFRI83
3y2VBXS3TA+ObrlEj0gXj13hYEGtds9I
w+WLpJoKHKXdMS2EKwscPSLRdZymqxse
IxxGIemytBF8rejb169TpyzBZ7V/RtIu
yJZ9DraORXe3VIUr2zSR3Li0E/XaKjDV
guMWbTj+yatCCEwChEoiIG6K3o67Xios
mcuSpZ8MnrYsmkl/nVy6A5ji3mcRpwED
xALU2siVjxkBCIDON35fRFMPzm45fwwD
zRERIHaDiKeFA+RYTdKguwI3egczMnxt
gKuxIptZg/CzaTEou2ZhHNgCMoI3ldnv
Oa3Ek76XQbjkJGX4UMU3rKwjRt4v1V/H
VFgQxQOonOoM3Ig3QyF5Filq8ueq7Ni6
RbaimHDn7NNqUgcFVenDQD1ZypbNk7D6
671vFwFW4Uxm506swUfqpno7FzVlvNCG
c7nEdmhVPs4IZDmvqT0ZMCK0Suwd6J6z
iDID7TG0XkCyK8AVzxlUFlaSu7dhLmFr
SF7OKS8SsnFX6d6gtpv1osZR30HBtZeX
uXrOE1JWToHrJfR1BbzC5S0kRfBtTaXN
9YEYG+BQyCwPTcs74JVvrVyCyEb22mxQ
eGU3aFfQOr+cRNhLiHYFxjoVG4JTGEtm
zGZ5RO4udFpMM24N2bMWOjs/b4RLTCAa
FGOjk0TBEjuBiXgd6Bhjy29tb7tQHMsp
fCIhnsKW5qItJJIDV4YoZzjVdszNN6Y5
AtwymBDNkgeFWyioO1JOyxL63OYR2l5e
xmuiWnjJeCiZU8ms2szMFyXzWwq3dmVY
1x1RLatS7q4J60OjIm/qDW8TsVpajGpf
QIlXYTBLrY3daRKcvG0C4Qq1M3JVMNvf
NjJOTfloqjDeX6nIuffkDwjDAFxvvjBv
rPGXRRjeKehoeIyeipXDPuGp9J9W/Ui3
Qo4kHDIpOpYEnXF1xlVMsNiaO8Udq37Z
OFSUuWbPdowFiZpl7zcHaJSp25y0+Wxy
J9EwfscApjp1QjTpMS8XLMdr8nozS5M5
WnOcnXpcF9CF0s9vlubRsS2HRObUfBoL
k1NzqJu4E8jEnPwi5W3Oc6/iuo9G8h6y
0vnzEospiXLwD6QFHWHKK/alg9a3N0Rv
YyVzENCPrOCclDomNHl2iVbyO2U+R9yI
cMJnzsGycRgz4bRbmLsaW/eyWEL0obAd
GgrPDL534tSJpZlmKnC9G67N3fMR9taa
s5fMKkrXP14zp8cd94WbbBc2YD6/RA/P
7URPjUTO6DH/lEKc99UG7qbIsNipMZO0
DVntGsiNAEwqRzLxHN9vdNLQrAo3b1dW
HF6/RFv2wQ8fd5peeFJ0TWEDmyoCSXJI
n1+qK3CU7A34+RzZ8aZV79PHNDJShBOk
jFx2NDWycYYrjVNtdVAcrT70Ea03Z9Ox
X8TR041vl0PSSDBTLoxHPMblehOUBhIZ
w4VXTnGRJWh77zGn7DPjPVrxmMeE0mm/
7iTS7MNmi1B6ZqGuAlPApiHEQLCRhTOY
MzMYiSc0e+WNO3927ANR4S9oOAciqg0K
phr7dq7oyxL9ZRlNzy3UVXjaEqLpjPeo
kdwSk9NrjTsjBFAIH6fF7VPvTDyzCwa2
HRGMmo0JpzqZtQlNTmPpuWW6Bk3JmRuZ
I6lsHJ2Zc2NFnCqwKsm1WPcwI+OOud+l
dJr7QfF0imN0Imb0cvsBnkZd2712QWxt
5FRVsm/6Cvgp5QSnnYvRpJCtfs+4bMSD
guvAOdRgsYwJrD7STgKtx62oE32aO8ZH
KHthUa9CXE7LBfzHQZyaOZQ/tbU1Nj5x
Y3E1p8beAbmwCswSNyazbNT4VVKn30xu
7PzlGHIvLeF14SzXbqmxqsplIjoXcRus
eC6c+uFK6h2MFWSeqkBbIyF+GhN9+9Tl
bOuyLMPvFvnirL6//fTju2m6M1y1gr/4
f1bG+mcNAnkqWuZle5Yfvmo7scuWND+Z
vlX0zhmdE/cc30gAaHLgW9rBcM7h6D76
UUud0sM4FC7L5eLrDDM+RLk2u1OYSm/V
y95iwSmBZALzx7k32qWDPAWkdvf57Q8f
f/60R46Hp7VohPE9lXx6UEpn2O8ZJzKO
NOkDXqobWyp6VEx2a1YEVRFaRacVxqUV
vH3keihJHE19/YKspFadGsbLlKsgHa3W
wnwAQoljhornESN+G84qkkeRWaNFMRaO
URIHOKCl94X12jJp6msjOQW+Y7M6chYw
C3S6MJHV2Re5YYOrEClCS3SKX7+p7fU4
6RsHuL1BrsmIxl2wAqe4HbeHP5MqHhzV
YumcZDBwmRLiIAaTkz/Eok1DSARUU05o
7WlvgtdrKcEIcWagdvdXEw2AexGZCcoB
xFi4GrzrIj7sY7JkICU1i4eOC2wxzyPH
NdCoXGDBhyojsKgxz0uiEaAax/Eqh9lr
3jHE7hRd8O1bxUG44RkSG7PVOjeBW22n
gPFp0uFy0OisSrwgYKFx8e1UjBHokLcs
BY7D7urlrzUw2VFhqadFlQMiI3PekxjN
Z4OXgfEcYlwDjYmLj4TbXxRBh8z0ppFL
7zirrkla3dORJIU7LoWPEtP+g0KjQCVY
S2ulN3q3tIyNZ9HGNejoIzVOaAsrj5/p
SmTWGGvAM2nkltgRSyLWhT0tiuiF6wR0
UGy0Fmajig/bSWw8hxzXBY65kOUF10Sm
l5mJlhE70EwsczXWjqaFyHYIdnADJMkH
XwYFR0CAr0zKpTxiCu87NfMiVsZfShBc
MKIsClUEEnAPsNncHsgOcHY1cnNVZElz
Yw8DHpSGN7LoFrzOPQwD4ibuQSZNbdMC
xiPgvIBM14WXWcmW1gChOP85UxuFe1Ng
ehGXRNPt+53hapFcD/jZyLOng2JoZ/PT
ifq9wKU9gtBLKOkqNA3k+yBxuHL/5BRq
ZuV2rJQi1VX3zKmS8glWiUM5cHJl2EiT
m3SmSDPZcqS5WaR4zl2GnLp38pj+aeZe
18cixaOjsJrB+flDPkVMyhHSWEi3pVDm
OQUUWdbCw0NKFy07BmiYSFLvNQKAV7E+
hUKGGbakWafMHMDsqk6A4Tkg9XYs0p7g
jEnsmulPHmUPfsCT+vb79x//tkeKh2e1
IEWuScjcloMw03z9hwsRwjO2Xiv/Oe9I
xxaObiBUUS5Sib2spDmw2Z9LGuhMjLD2
t4apu9gPrTKNh8ebizJb0Ri5zewsEiRl
s/o4G6zaMllWbGy6LQQBJ3KYgovKoiYp
GRhu7thEhziTbg6J9AiZnWSXFE9c0l1J
65NGsKu51kcCRGB3pddqmeDF7lfmKY8Z
gZ9DhEdntZyJpTvDAdPAfZEOaTGQOCcl
diHgMjc2DU2usJBHgHPnZOpJHnCzBA6t
rD4TAs9qBCFm5xYlNef88gZYWVjzARaR
llXkePzheTRxDSJCRxGE4JmiwtrU/MXN
qzxrPGEx7WkviOy8hxvFDb2JRCuDAiLb
pVii7CP+bIE8hYhPcwaXwkSO2tcWhbS+
9Crn1A/7vWhhWSjOzdqOriBEqwnRGDxm
xI8kGxwTFsW826uQNvFpzgD/0k6A5BmE
ugImER0x3UZ+x+SNTHOjV0I8Gcnnyp2u
O2j0uTub++EV1r6GMCpQSnJysiWRJkRQ
J1Dz+WW6Cjch9lB9no/XlKemEOhnJMGg
UaIMULY3E7Axm82huTQWtHVU4Gz5hJ5m
NjeeAtFzaOoaGFWY2sg194FpnnnMjHtH
mlb4L0Kh7Jgya3g8VGpwkn5//xFhdGoq
6HulxLOSRyD6i6mCi8WYEBsAgOs0iZtp
7t2KCE4CfNzKNVgh7eABjcBNPBM1Zciz
0zyMCKcqvq2z1HqYgufiiAU4Pb941wSg
mYnxwEpZt7pdX0uBapHhOQJg2ea8OVcr
LNMAtxvCF4lSRkVWc561BekKC9oLyHp2
8a4LTrVCc6U28oG2eVCpBjjBXvZit2jY
1pbQK3GZa6qAUGx76BntETHWOuXlgniN
wfsSxm6Vb5nEC3HKfV4e4cpBXj4bRZPY
YO2LwA6Fe7VJJd/0SfDMJgatDU9rj1+4
zRVWHL+y5cJW8WOD+82nnx+LrP95o/BL
fIyfvWiWPG7uEfwf//UrXMu//PO//OPX
//D7313d/BWCu7r9cPfz99/ffnuLT776
/vbD7eebr+5u7ijcu40ndXTlCye1OffC
nmU4WfB8uG6hL6NNAvVmOw9J3fISDdTy
UUX2LRXl7H1rnJS/9EkhAqYfVgDasFHH
YxY7HqmgBVExDDzeHP6L1p41h7dGIonK
PrZmC3Z++aS4pSSLM7qwHsEJ2cuf1cGl
P89T1eOeBkuSTUhgVn0QJ7KDOoTWxPuL
FsZMFg8qXpMilus2SS3odCKXPqaxzZTH
SF8M1SpDddGzGt5UXfi0xjVWlz2okc3V
sbv5xXKtsFwvd2yDG7GXPLhR7dk5zuy+
l4uMB4+auUrLT5u5hAVGZg4bZ2Xzckdt
YLMXeczNcm5aez6PA7cV30PaRJiwbZWU
3mKtJO4lIUMWrnC68nWSnM4MjUWIKmPw
IbghstiXEpWQr97IdYosDjdrCfYlLFD1
vfv1hrxfFeLhYS1zCnGwjt28lemn1Dvy
Iit6Rq4LsuvaDkpNMpRktpQGFq17kr2Q
3lFalBJDshF4cblq4ZEQM5cOQmsbO8lz
Ko7Ux9O0zyJEp69uJXK0man+JSGSMRz6
WkvN6ls5Ox7WxCVwXJdETobtM9HC4XY2
HHhrSvOqvZDKAg+wcFgXD/EI27UgJn2Q
onYuyprhOACvPPepbYG77VmkeHhaC2IU
7hAFnnHymZsCpipmw1dV+mKe7btKa6Kd
Vu4Vzrk3GrBo2sSfmsiO/gEkGHPSx3qY
EvPlKjkUS6VoZkr1PBJcg4hSkkaWt3At
WUqZFJH7JTlHBEtbocbbOn+mWU4t1GnI
EJa5d7ENiIlmJd5P7hU5BYgHrE+XgkTh
UtJEq47nikv3+tkrOfGyM4twonZPnBID
S56l9IFcGxMT4TaESYj4MJ4CxDMIcQ0k
wiiw9EjaLvx9TxaeIgcPIq6TCpS3d6WT
1o0smtoS9+w4Kd2YmKj3UqxckrQMiOeQ
4gpIDIjf8SxV6B3XAItOw9Bke7FM9AqJ
Rn+HNsLuMIrDe8O2t6hpUGjkXvlZkDG1
U7h4BkGuQcbojXc1MlWWE+vcHmbgsoy0
ENBKGMKNbXfTplG8fTI2k0ZY0UGDRQZd
kxDxYVwCxl8sFl8OI2uMsPI58tmCouSp
Vz0y01JybBzd2zpsOU39lAAbnn0npbOh
D4iREKI8yFOXMPL88lwVQSJubMoOBQ5c
zvutOBpdEFwmmNlQd3SsRyX7ITzW6Atj
x4RKsqDPXquvPzpCygvIcgVoxpLVcmPG
37g2evZfEZAwQkn4bZLL9k6daRSlxUbK
U+eEHBEwEwfyJt3kNN4CYG6Vp87ipPwe
pFkPR2a57abi3BJiCza1PlNe36Wdc+Su
8EQCSjIb+O/LeF5TE/xOyG3Jgi7mpwWu
YaEx4L4HGOZ8n+o6SEn/7ebmL+///vbm
v366/dRP4jelp1cf2+FtPGMRCf6JD18J
GdbhfUqvImX4+dlgRLUaNWplFQk/SsIk
2E/oKqsh5VRfxYWOjiy27Arm2lg8FM9V
SXKzgHOpeDZgX7hhwOshXKFXVQNAm5s8
F3YnLZ6b4icz8yVMh5Eru7zwqR3exnNW
LZsiuCHsOltQz2tlAi5XZghAe20ZCc4F
bHKsPl5G9o6HRRQvpKfnM2+QBGtPpJUj
g1UvWj6veVtqJxjbwHVXjA4EC42Z8xPn
MG4veHBnNG8anPSO5GzGkcDnN28veW7n
NHDKxYuN9PYQicnzG7iX1NSBPbhf7TAY
29pdzJ17Lec4sG/3ao5wYEfv3Gdo/dC0
4RjvG67UDk+QSeDAFK9C4/IxVduOVjXh
PH5jPjQmbmPtjyHcmQQLCevWmLha3d9H
UrGUJOLnAnMWJ3zlzz/cfvru7fd4y3d/
P8uBHVz/M7p6sG+lMBukwiXUXTxs8itw
/WpnNFh3XGT9U1ICcq8F3rG2lzqtwC7I
RkobxUPwnE8XuXKChcTunpAm2vzIGWH2
zdKgLux3PoUTUOlaGps1m+fcXuq4Dq7/
+R4uGDAx35NgjVOpuf86RGUx1Iyv4xzX
nle4ztDtaOT5gptSy6m49ezHNbbxWvKD
v5iv0+brRc5rZAP2Igc2sgl7kQMb2Yj9
qhP7xZ6dsmcvf3TjmraXP7txrdwZz45U
wH5gHEV4VBfnepMn58fVJ5mDBojb2WB5
oieQzZtcjEJqqDBvOY+NK25MlfynnSF8
a5tDIDFRhm7i+WcR2a+spJi0qOVitQzR
hNTS3J2bWt+2q1BrI9lM5vrwcDzq4CwV
f37344/vFrob1kvx4KyWmuQboK1xl1Dk
prqJkk8sQ/JAkQZDQb6Z7X3yVslMjTu1
vkEyXLOtgQRFRiqSRhKSVy/CnOvcR4YP
SRlEqshMvGnANslnk6AqfhNZEauTtiyJ
UPGB0pLjxy3JTFULc5K8g4y7Vp0Fapsa
cm+2AR2BFzEW60QyeDqSsWtMOU22YO9e
nQgtTp2ovCc4B1eJpNE4u+ZLLP0szyHC
o6NabBdzcrZCpE85Tss0YqkkBsrQzWql
bFTC3quER5XbVdguxgKI9xjhyTVvaYLn
k+sIVE+5lbn9L5N8mT4vp8Ylw8wBs6Ke
SYjr8JBDfiIGDbSUSTg/NUS3QrecDaDs
cNvO2BVI8cdHiRvDgjlD3Yh42Je54CiJ
iBbrSURcIEq8BCbCzqUKlyNWilRymyUJ
/ZXGKpdvptnGq9ffCe8Dz4jbsxMgNg4K
i8AYn/wT8wgCiHMKGc8iyRXYGLw3M7PB
E7a372YjN1pgvM0vcqFm2D62As8UBjVA
72HBvWtmQGiMnVrQ+rN/ChnPIMJV2CiR
zgdrc9xCHuZhXIGO4sx9sEzSntb4wFkV
4CtbcDluPyg44nTSg1nN3HN4Ah7Pooyr
AkZcSIvQu9pgHRC1T9bQSENJi8tVigj3
dkyRwbhDEZ3spVYfWxkQIbXUuZUaH9oi
Pv46yeFFwkeceBZqSDKxNs92sq9ejMO5
uJOiG+esu3aSBxfub5gYMW1MqIRm2Oy6
ktxkAScvIdEVkEnKGqnaGLFHrsSc0l7c
l9R5j2LUPTvCzIxryBqkaVw7NiZoCoGi
C5SrZBdA8/zyXIWfmcRLFTpKAgvyHk2j
ggYZQ2NhVNgvtn0VEfch55QCybBooXRM
+DSG3dM0UqzLseVmiU4C7P/1lHV5Mor0
TMn9q5UFj1N97Lcfvvv4t7c3P/70+e9v
84+/Lbm86R43PHPD3eM2Oa4qILyae9wm
x7Hu8dz6uNh2PYgkR7vLc+vk67jLc2vl
67jL8+rlr7cBDyLWgW/4vNr6Cm/4vIr7
Cm/4hX1dDa9XqoPd5At7u69ZkoPd5Iv7
u69ZlsPd5ot7vK9ZmsPd5mvyeV+zYEe+
49fk9b5mGb+SO74nnfK/HrL2B4xT2kKz
xG2csS2tHuhscKWGnFOqxWKsel/zDtWk
CJmiLJlt76wlJz47IiS2FDgs4J0LWq16
k0tkTTaVAcpqxfLMIcZJdxLC+5ZwHDGb
j3M45p+epPzh44f7+GazLA8PbIk9LJUk
MbPUxd0jOc8d+Gz3bpHz9MnaxmYinyUv
VjQql3EI3skXH6q2KNVITMAW3oUZklcn
Sc1tliQXrF63VpRclq3i+iUds1A/pxwt
V5aTOW1R8wLrBOUIUZeoVkjTFmRqljFu
3oUyFa7SzNuLotyoylIe59ezdUZ44Yrk
JgFPDgyGLBACvDoh5s6g7SvJr8npQHoC
ExgtlilzkXTcKP2Mcjw6seV+IggqtEit
o3pMxhX/zKXFiWw+pKPZQUeN2yxWUytJ
m3SqTZJABsVXLGhNI9D5cZJqblhQ8b6O
BuCBzZNI8nzvFzujSq6AyUgyzRxh/zIn
9OrcMAJLK04hk2pre/ptrWrlSm3odmX7
+6AgmaMvDy69BT731d/LQPm0NezCUBkk
Uzst0F5EjrzNUElqcIn4h1ii7NhnjQcj
Zo5KRGtc8TEoVnoXR7i2kHwYg+vSTgDm
+eS5AjK5rCZyIA7+CRcRx65VznVcUmOr
Zaacn5NDbETULN6VrLH7FKWdAM2zyXIV
bB7PPU69m075ztkVRCSc/tzeFB98M0sD
ckbcvYyJm0SMB2PLpsWT0Hk+5VwBnooD
NUN46VuIVOcRh2DezAmPlLvLtzHId2nC
qDaSjsM65FrLoOCp7Z7WWNmxvIScv9gX
dul4k7vuC4xrZNtfDTJ5RXRlYiBHpTRF
RLpjsXzg2iDLAb4Vfm0eFETTQ5N1YpP1
AoJeTK5r4k+f4cO/Vs2+C7JrGcdYGIAy
I7SrK5dMatB98ZUSQcugWCo5zn3W+LAs
QOmlhLoKVSPEVmhDAtn7yGLh2lpCrZFj
hGbcvbqje17gHVZAN5wu6GzxdV8Doio0
cs4OAUAXEfViyvolgfu6ErjM1H/J4L68
KHdncHcJ8ksK9/WkcPcI8ksO93XlcHcp
5Zck7utL4p4bLr9kcS+cxT03bH5J414w
jXt26PySx71wHvfcAPolkftCidyzx51f
Mrkvk8k9exz6JZX7AqncsyPrl1zuy+Ry
FwULUbog8cjkR3KFNchPmYLhwpplqmQk
adLy2vRcucY045tatdLaZD5xzWRo9Aej
xu20RQrJ0RjD5rZMiz5jE+yAxlwZtb5+
M1wo0dA5XhnA0w4j+q7wShqieOiAHmUb
3t3d3Xx++/XX//b7PSI8OKoTXpIA6+BI
QVFylNlLSlUqFEq5SyjFuh1LjSmGxGxi
S7Dr1S9MSVgUGqJVZUbw9cvQaTXvGTYn
zxdqSeOioeSYjhk2n0WGNZXEDAH32sF2
LqshBC0uyVJh9u7z8AUWmVvxuC2dq803
L8HEs1oz6clgwaW5PwS9rgCbkAzXXOlK
vH4DCyNVXICeMcJnFSLNwC6BfUmIyRD7
nUkVj07rBCNRIlO6k6Yj+J9xkuEFIhi4
ow1e0J4AtNQUSfze6Cd7ggjKiXc20oXh
L2kjACUs0qyM+NDzaPAoeyqtNtLtHRfH
nsegrsJELXA7GrNxjDNtygq5E4t/UGik
hu1erLRUFFEJmeRrJEfnkKDYz0Ws+/hW
ToHi0xzCxWBR4D2SO8w9D+veGFeMWtQW
oMywFtDUPfR96itA4ATjncj8m8cExlq4
v747rUXySWA8gxzXQGPkIhkY/Izr4WaU
Ovs3tWayv5H6re7QRl/kgNCLCd8ArZRB
obEAc2aTGj28XMTFc0hxDTKylYMqoTTx
EPpU18wZD5k1xB7ceCN71BFvjcjUEmRZ
2XIwKDRmNotMfmqxchIaz2FWV4EjI9mU
SPAtOdaZdhqghq8rMxuIN7a3/5C0GiAS
BcFGqNV/6YDgSHLw69CXCHMF8BI4/mIq
4HI4CQ1hOTp5kwdT8dOuGwYeAmRr3oS1
fU9KSAEK3lKDrwMrwTz+kDhZcBQ0ri7a
/vcxUJ5fpqvCyYr/0fuJdFHSfeUEIWbG
12qCUczQ8u1aGrk6WoXLkRDd9Pa2AUEz
+mKqa/cPeQeeIjjCzQsIdQ2EwuhyEzmM
YrYS7D4JW+D/EOZwzUF3VDcFNxxr4KJ6
blmvoyKoFo+LS19IVeIihJ5dpjmSPByo
CDysKS0bX5x4gJ5mhMNcqzKJNMcMXa2K
wAVXC53dvg6PwSZ8K6bXW67JZQqLixAX
z0xlTgyX9Pptb8OdTE5Rm9exZRx7AqBW
1vqF3U/LCYM//M8/7pHiwVktSLEWFkcM
wKZQlTYl74QE4PDgsuDf2Ai7FUCFhdxq
Ji0WaKHSpUhAAS4awO8rEPAYNPFhjk8y
GxDydUbkDINHw4Og/EzyowLk5vWruLCk
s4sv1cauDvxxm8G+ZYHUU62isISbOwwU
AgrsLoIjhUeVjk9i7TRzNw6kCffKBjCo
2XM8IsUzPnjEr+y6sgsGTrolgwE7mXPd
Jbzjs1osTkoK7N6Qyk0KbL9z6xAVUo8k
aW+4vm2tBG96azWQGVba4PD0VieoH6x1
YbKXsxBMp79+IdKbmVsJGGi/0WvjyjTY
kRTYYrOw6e95LOgKHFQOk5TCZBq/bd63
ac1K5NkXMbOyvVWWmzyBE0n4GOF5SGPC
INsI3aj1rVR4Mk8j4WJ+4OxYyFxaYs9N
LFCOqnNrj+KzRFsRBI4Yt7zs6GDP7B5i
mocZh2xj4iGMJ7GGRVreAKK0E5B4BkH+
OiiGQjnz05TYONX7AhABCKwgnjPuRS5t
sz4CD0OC7cFTy7mxUYERCuGI2NM8FNgy
MD6/CNdBIyJJTdmUsToeqzK5phq974Nn
n6Vs7FefMgqsXyFgbGrwfD2YHhMcU3ik
jBEwdQofz2FX10SKeJbgimZ8TzDEFZMk
mzD5WrxMCRHsWC8WMxxUmHU2CpRB8THH
Mu+jxoeyDI5rIv+z4yTCOO5UYoqVYf79
FK1FWA84Pb52vu1I5sCswo9iIwL+hpWo
o6KkzI0CyXfvHUHk+cX562hpsKZATEEU
YinkXhPR5B2wJVMQxWR79w7cHKvwd/BQ
wVr7svgR0VIszzN7wpT/MVaeXZZrI0om
zjkPVDMnJV2jmOuP1EoEEfBt96hmhlJm
Zs8zPCsrNihoGrPm014xk0XAPLtAmZ+G
LUN4F4CO3FO7vL4a9hTepiJyyryy2XsB
2kFFjbLOErf7s6EUFZqIGOHXMtPll8bu
2Vy4PK42RN8DmFuzefUfPqzd6rJjBoDE
FIu33y0mCf7wu2++3iXGg7Na9oEMkqT8
SqT6zInWzL4NjZkr+trGHoLJCVJrTPWk
wsvpVa3QGqCG+VcOBA4BmrBt91MjTV0/
uY8Sj37g7sJyKte6U4hURKZLYTvdGV3M
9DR4YiVyYJlNN1NlUmFxM/ctSvGJoF0z
0XiGKy4CgYn27gHEJ0rXHk5XwcM8gGnF
czbrIT6svdTMQ4UOCqV4JgEentRiAwhc
Gfg76iKsdbKlyZjgLghgGnRxo+c6Td8K
POPI5Z24094cyYFLNhBwWDDJCOBIBoe5
YuUddNewbg0eYkmkXInhmDTkmQzpKjxk
oxtnBRlvPvR8RBi6RB6YjHgp2famOvZZ
MgSKhksB5GYbFBCrR4y9OVKCnATEpazA
JSDR+F1K4iVvvfKxe7bcILSkq+NjVbJx
hXwfHGD8HFMogV0nMigiIiL3xHm3pfis
nALFMwhyDSwy1qjwIoWTszI3ueKq2TBV
2LFAGpFtc1hdt1NApFVKYVtLklAHRUYY
0fsUD92cZWQ8hxTXYCOMJyINYUdAhvhm
MXJwlqlz//e6vRYZfD95Yw+dsF95UGxU
IPs8r56dRWERG89iU9egY0L8jVic2TOY
t3mgLmWp+BwaLNDIHdiYjTQSrGIxC2dt
UGiEyNKDJuZFaFwR/18kcGyZgQfQkonP
OLdDIp5UNqZKDfA4w3beHljt2JIFvB2b
SKbOuQFhsvmkKwKwPl5HTDyGyfMLdQ1i
sgTMrmUINqQyj9rBMFY2RJJCAkHEDi2t
OcMkceyEXkOOg+KlJZ1tLT5cwssLiHMN
dJqG6l1SpOhh5N7Dygy9gt/SOKulhLfN
cWWuTK+KdTqBEgcFTxjWh4ikLYLnJSTK
7HjUEpnG0RP9ysL2K4HNxYMWvfuNc0DF
x0vgv+DfrG3PuQr79vjbSFcaesq1mHD0
hw3RzPOOAKMSeo0nTUjCBnSFjoZC0xMA
Jqea6/7wH//8zR4ZHp7VolYiGOHIQOLw
gNlED2HsJlZLJOiFMm0HTlZBcY/kPWHJ
oOdJirATjK0LMLs1pgGE2BMcxKH2+I9/
EerJ6Z9UHdAqnZCwwP37LDKt0DhpbJuD
yWUedTnWhC3mRA6kiNhkrobAKbZGRq2A
x6K2HbyUcJJTjvDHKox3p0mDNwRrBXEG
ukhk7Hz9lrZUux9Sr044gGsHFBl7enNv
ijqLEA+PaplpIFRFTEkWJJmz6BwGscZH
oBqCHN1eCYmJ47WBtbycpsnmZnh4M3kp
swLQB5Ag58tm5yc4Vhqed5blaWBwfycL
ITtN6xp4RDTZEAdGWj9ECmFOwiqtHowF
oszsnvjmnh54OrWwAUZShMDasAjpxJxw
A2lP8ZmcwsjFjMEFUDKT4Rf/CKWEf1Om
JGyJZrXAncW/Bdk4meWWmUNYiZN8TRp9
4kFRMkzJ6f4ST6HiGaS4Chcr/FF612r0
d2SKKJNweIdfh91gtma7OiYvSdI2x6o6
KiwCAZk8btlfUucMW0TGcwhyDTaWyO7W
QiJJPFUzYUTfRGCZdRyOsO9haVbmBTls
QrbLMig6FjzSD4LEZ3oKIM9hV9dAJJkh
yGuW2SlQJyIJTq/4ggr8c3A6360BJFSS
IAtHB6F/SqNGkL13OWTrXehpMYRckxO4
BFI24CMckupZnTIRbhuTM8m3HHiucXs8
SUpUPLZkabcw0Q2OB5Tivlrg3NmjP35Y
ztVzBJvnl+4qBDVAp8C+IobkaEGcwxJo
KsmuEF2aaNjOJslVByVqInEwq6I2KoR2
Xe0WlytVlhD0AjJdFWgWCLAxnBfa4rlT
vbBFllF+E9jP7RoLcxWVlBRG1qIy5dnH
A9OMR/tepPhkEUvPL9JUFV4P5FUbDvYE
40sCMCi8Ue48iHOmHZdcE7ch5KQt7pi1
DJWDt6SrhgQt+iwJF03B+4XdwpOTAegj
GOHsuJqDzwThMw9COQ8Xad5SIG3uifzB
H//zn/ak149Oa7FLHT4RWUG1wRbmMBc1
W6tF2UYJLcphB9cku8RYWDP2kE4tM5GY
TfIYg6oCaUdgxy82rZdxM+svbxzQSHdd
SLmiuNVzSZJ8LdEyK9ApNznBNwnbypbP
SD6kPPFNctYA1gNBKaQcdwSebBhxru7A
3TUTdSrumysYYAToS5Q2gIk92kICR4hl
zGKNhEiieqo5fa8QD89qkeAuQV1LYNmf
1FZ5nuly2pzUJEYymcc9q7uA1wlBZyFZ
cJyamzk8AzskPs7CdWGvX46RZC5weOAW
8g6czwWixLE3n5ViLBbOJcoV+JgYx3Bw
DhfENRXz1E4lJx00MgPea9lOnhVV4Tkh
7uT4ZhMbFSA1u7dWvF/rGp/JKYBcyh9c
AiIlcp0Iv1W4WKumuUkEBpVpLEQz7Bva
0VSAh8mbf6w2BLVZRwVJBG3qHQUuyzpV
I5ZB8hzSXAWTCCu5Mq+QoC3Nrc5cNQJP
KIpwzL1uDy4hyYoQTOADw3eFfRgUJu8X
AfKuyGd7AibPIcY1QBkVh1v7bkThuOC0
4ZLJAW624ZCXbWxZ7xn7yIUbjbG0hdq7
ZAeEydzSgxwzacVOgOQ5BLkmjCx0LX3W
ChJvLc04ibgjeghPIt4dfQTRMZgQDH2P
5uQSI+Ik7QkBsvTVGbW0JZxckRu4SFRp
MQqHpLk+pN2vExGuAFFIk+T1dQ9liHcp
cEGksXZS07CQyaU5dH+yb1zzl0XIvIBg
V6FnJS8mu0IirOLEUQA3SGkijSlzlbx9
8FKNFDDC5QjGDVF5UPSMcSq59/NZRM8L
SHRVxCnCqnTNXC7MueU54gS2OpsocDTV
um3zz+Qow5BXeMsh4v5rr5ANCKXmWdk4
sTty7+USlp5drOK8ZUYO/L5A9MSEdCKH
ekE0AU8pdwNMs6IxI2SBwmrbwQib+YBo
I3+0kJ6kOxmNrV4khoKZqCNQ/STzkZPa
VznhMzz8ch05A5JJLKapVTk1Jv3Nf/x+
lxgPTmsRR1uJECIbfyDJMI3Y8voi+y8L
E0MqO3bm4ZahhZAhy2LcucgrA5oWXzhs
mSxOI8yc4CH0LEJ1EnHfqkvlJF+guwhk
RDqVnN0pRwn/P3tv2yPJcWTp/hX9gGbB
318+avfOAAK0s4u7g/thvxBcDbkiRhIH
IjXa+ff3POYR1V2Zkd3BiMqs8kG3xM6u
fsmKdHO3Y2Zudk6HFSR0GFn4xpu4SSDs
hZzCzYo499oD0hnP1BGlgtNPaFpGiLd1
pFulK9eUD/G2gSbchJxFmMHF+gGYgyLW
GTFMkq9K8jeZK1tjO7yHEa/WaktDpkf0
DyBRit2vldmYqAtBeF5QKDioBWR7hFtQ
OXXFr761wZbrlYEqflYO5KjbzkCczjzF
Jz41aUXCU4FeJXjGJX3Hj93Hpe5BRpq3
GGzvcqkcuvIsk5f4T/aHBcYd7wiic720
AumTPu3ikebDRjbh2vaMIuktYNwqHDwA
GiET4F4/IJZHHrpcXLrQG0GJQ2I4Hmte
X24uFamjuYpSXje24zmxkWG4ddZLG/MW
MN7BjvugkRyDUSAKsG1NmWjnQhhEv6B/
43j9J5JEG5OPd0l5V54VGi1SRblxYFG5
hY13sOMedCSUzEHQrVOXvX57bXzOsmrJ
bozXneiXjYluzWySfMLEPCs8MgW80GqR
l93Axnt41T3oqG2UaK4M3FFCFG4JB0VG
WXEMgOSDDCL+CVaSQotwjBSKyvff2MDl
hODYe34mZurw6G6A445KwANwEplRpGEK
Zb022tjh4MlCMGc8A4qETqBk114KsAoz
Rd+sijkjSAof6nO0sw2S97fnPrzMOps6
iyi66FwuczB6R6qm3bXQBAnH+3xggGr6
wFn7Aom+WVPJOMZml3iCeus1XN7forvy
SnSAA7QjEInWtQcvFsZqlQ3qMEEycqa/
INIpT3Gd7oK4lAfmw87W3EpJqV/GLew8
bFK90TCi/fTRpu7K5covsKQ6IvXWTGaF
f5fUt3jf4jMvNyoFNXTGCtDIPJ5hJm28
kKHeYHjej7IdN2EO5i+aoZABe/9et5a8
kv/WIf9nYpMos1d01jYm3P/03c+/RPfz
9z/zZD8ft+PlYm0dTHyxsj9lDhTo8ppj
2txQp9GcoDcdJy5IdYyTcU2qdxodF5Hb
GBxw9t23GWRKY+6rFaNplHbIb7PcmUeC
9brY81o2LFCicVHVYeTenDwQlju5+0J3
VXdhKSwW4Zqi2UIFSFnL4YNYTe4UhiHH
AIb5VaF3rx0+KVe4xJ3Ar+ZitY1gzREO
oa74JA8Dt64+RR56Vfex4dVqbXfgkVpC
htpTttvC0ezjsKoibt4kn+mJ1SdnPFeZ
T4L7edxmKc+E72sw/cwAjzmGFR5zHHQO
is4Vf1DFAh83GH9f6yjugEU/urWgoGxc
Ni8JRJV3RTo4IqBnxNsnWJsbrDIEUu65
gW06UMx+GTXwQ0Cl5JvA+LJK8EBodCHb
eKTxb3ufnrl/lXooNPH6A1/9wQ6f5WoM
Hqrc9V5McPVJsZEepdGoZfc5WtBbAHkf
W+6ASFQOFUKiMMksV1vGabmLgh6U0deM
jY+nlIp7qQYgPBdw1HOi5NXkyDZI3sWO
u2DSVXpfa8vUYAOEzQtngRH3+gqXS6ln
REc8nKOInI4KYpgTJ0GJjw5WX9WbUHmf
Q7kDLKOj7lRQQIm1P2uvV1SCYoKyAEbJ
E1JdOt9yrLKcvEJuRoswI1hGdFiW3KPH
7RTyszWBR+aTnhJ8Z+4ceYpV1dIRtniG
QDoMPeEYs/N4JzRKKewinul6nhQ08WHP
0iNtEzEfYtM9+WVUkBL1py1mU+oapysl
fZWQni0pnqB5hm4i6swHjwyCi3VS7ERW
YFXtUgqwAZ2PMOguFPWm0Sa/oZQegtFl
sp3RV58rhTtFpida7qgD+iaoDnRjCo/j
nCjaaApZirG+byLoKZsqYDYbRh/ch7Za
VKv/0qh4U1krmXZlKNdTJh9OXIUw/yO/
7qjQWXXHk2OiFKUfylx13K+86T//9W+f
2mn88E+yuJZGnr4h3NaeCyr/9N9/oyf5
3T/+7r/+9r/8/h8+/OGPf/3pLz/pTX/8
w3d/+vZ//+mnP/zrt/7Dj3/5+W8//PDj
H36UJX/zw49/+fGX73+zrtzhhbv6KBsr
d5gmWdu4ICvCR17YG6i2wW3aUrTRhp1L
J2CPoXJRWZWlEACld7B23UfTZ6DZqNXr
m7rjuy55p08b4QHU2g0WeCE6SRQDqBkm
wGv3sLl04SmhZ4/eWUafrpR3sHKXH+V1
dt039YkLkWzk7XKwIS7NBfqGUK21grae
43Zt39rhq1PsKTWq/zbCFt/Bob2ft/OV
7lVI7Ua7R3h9d2fZ1H86h7cQ3wSuURXw
Kenoi6rKK3u8t16+O/q8zE1mlVWUpvg7
uLy3Xro7Ob0xCtmNEA01AtzDoD9/da/3
xis4b5R3HQL/p/OAjwn53tVCzhv/vatl
nDwYfMRaehc+WT5f19W0tX1R3qiOwxIh
93b+xgC3D9AjGnVNk8NbZRtioijJ7QHT
vv0gu+bSEeqib7KnQqFW3KBbrx6tUNhz
Yol1w5++u/JGKt09i1h3IyGNMaYMr3cb
DZrXl+nff/fv369W/vanv/2yUanab82L
JduwpsL0TiE/wWDcnjXmaE5mxkkmpin5
+ESaUfflQNs2E01WfKSVn5u8Fj1cuzNY
MjwrzKdAw72WynjyMrzNqEff1Y7CXGaI
GjXh7Db5MgIsKbIlginQ8z9z99VWvBwY
9+D+OFUYns6xdUMqOt/WF5G7AFmOUe9P
y/AE9cZk/XGQZPGhGJaN3N3F5PAyioFb
v7MlL9ds897Ojh0Xa4z8oow2JrZpiUDU
0Mt16Bdn2BQct/KpIYIWBmtaKAwetBpd
R4t5AmNm2AfWViX6ubwcXu4wFMUCCReT
wHf1rnuw0nEjpzgsckVX4srhB/+JFhw2
ahrGjl3CruyocFzLf+tctmQEEzNi5Wgp
r9nUcQiUbsLlxYX6gwHzum1pZNA6mR76
6NS5x/NnJI8y3YhJj8jxnxMwtd95cB9s
eXL0tzDzjsbcgZoKLUNWZgmLDEMx9cM3
8al6uROfOl5FEVG2wP84560C5Oid8tCc
c5oTN7t1uaBz2D/5YSvYZZ8bIHpH0+6B
UcRFHRNNcrF6iSuBmKKj7JXFUW1N0Z3g
94NowZWO7pmeZ7QyTQejcUgmL45XX91G
0ns63q9556Pzzs/dq39NQt+jWXckoY8y
6teM9KEZ6aPM+jU9fXB6es6we2vkvTFW
nF2nD7Ck17pqGNZSvpKhZJSTLqMRDQRv
NlDLYFXbmN3frpD7p1wY5ZFDSToCvT6P
bu4oiYe7XC9cf5ZXvHb1UPIFH4sRbBYT
IYM5qmQdLmik/AYs3bpcKD6a5E3srbZP
mOrebOnkB+TmO8ifjFL3NRtNfO2RokrQ
99ZmsYsZVJ71fRgQ84xu7buXQc8VuTab
PCofhXjfbt0uP8frXbCisBW16bTlGFlI
Cw20EQXTPar4aPd9FnqNKYWUldt6GGHe
w467o59zLdKjLUdAQ7YzYbzX9nR7GyTm
8nWDbkC5GTQyfOuuDRhf3de97eLd09vx
zjq7sgvXzq2k8sru7o2X7j4Ob8khQ2KS
R6lepcKXX9/jvfW+mzi2O9T7MJfze0ig
947Wcd6o7z0t4sQh4COW8VcUNJrLEfmd
2kvYaA4bNLCpMtkcqROStX/TnlrplKQ7
H6+Efu4KyFUgiLHnXLwfrMzyq1rrRJ05
VLSC3335UVa2LtrBHppNn4LVLbBm6A9D
z5+/cA+nq1MXa7ZZSU616V9n7vVo/FyC
d9RFas8lIs4VTpSSa+7Ik/BWJXZvWUBC
mYIWztA7ouHv35Y+Db6vvmgZtfIhPI28
hpvPnmu9uzGbzlbgztV7KAg3K8jcpWuP
pU780UYFGc0guW6dJrgr63FdJxT5UKsB
8SOt1mbL2nvSXpEdi/GPv/9KYxpyKMvB
TMoO0hOj4KkUJMiqvN/n+wxP2/Jq0bbl
1hQNCGdScnSsDHIhRNh8jh1rykmfIdtO
Vnh2Rduq6onHVXvwil5brYH+/CmMOeSd
9RLWF8XOrGpGB6sglOQ36NBe92h+GTRD
VkovW0a6G9a7nZiMFq0rDclA3fFIrzio
Y7wSGmuSnxMxkzUHobI6Dma+hZi3L9Yf
gZkQhUBRiM54DdW/PhETKU8NWUF77Vzs
zAmaWqJVzUC/7LcQ857G/DJmuhJJsJsi
eVzyejQv25AOT5Zc9kvNCZluKUou5bVb
gHlHU+6CzBstTFd9R4erstEGTpty9dY9
ilBzgmamz3A5mfAn30LMux7Nr4nmYxPN
nVfnb5F1pq9Z55ms82GW/ZqCPjYFfZRh
v+ajj89Hz9lWLnPYEuffnwvjIb20bIwK
VOgJhWMUaYnXvF1Q6Jy4RtPbFh/sLGoX
tBhN0o2B37jhZW9Nn/uKbCrT13Izof6K
5oj464rhu5fu8qO82qWCUvQkH9p0LAvM
XcU+vxwn2ifcOGQFmDtXLiHB7Ts8z0JN
IcI7WDhFVdrjuaCYmq8lcg5vOaocsVcU
QGLq2Xxo1AIK4aucEnIbe69iElSkTOln
Pe1Q+HrrZbv8IK94n4qQbQ8ZblM/JJUj
pMAoCdVQI9f8O4fyw5Pcv6cvHGEX49x5
63W7o4u7LtC9uovb2w0xnZNzPhgZbFe+
nJVlvrqPe+uVu5eX26oZvaqXe+OFu4+f
W/hwYnbycQitpx5e39G99Z6bOJo71Ocw
nde7DO3aK7u9d7WKk8Z572kNpw36HrGI
vXyybGhAfLKmL9u/cqBwqMMyVBJu3Nqh
COh8RYHcFPkWzT8YJXiD0k1H/cimzU8U
KpDt8D14n/tKfgUnXaEDMej8BjdD1RF+
jVVntQ66htDkz1sxQu6Sw+c5X+J2xeJX
GPNiyTaLUUinQGXeUhMSrnJ/LtExm4zf
TNBzhuExtop6ZHcFifq2WBOWg+RCT40K
9QTWRPZgFVnVEeHT+RbQMMwKIRgILfWu
5qzcybU4hBk3WPss4g2JO28kqZKgOS23
4BllMC9r6jmjPzanbtuipFioPuv7MEth
j1W4CvA5J4F1RXfj3ZcW8/gwyyWsiZFX
1IOqPldsvd/bkJcrtulkU8o6NdXwoq2K
5ASr2RV5kCJUUk5zQm6j5IRQfZEf6tGu
7LyNvVovetZ2yRuxwbuzZeofZ9MHg09r
1jyiA0mT+r1d7A68vKEddzXWdbi4otPO
Ea+Zy4dozD0z4mUYbC+52bzH6AnYRszb
1+mPwcxcdDi8D1p09MMX0PQxVAbMSnU+
CBZOiB57oqtmWrm9ylvUSUFTZ8Mu0asd
TXu5iZv3M+ou5Lxg8Q5PWmP9qfakQlxI
XxSJHpesCsTHiXY2RVR62zArdHa7KYZX
gBYJ087dBM87WnMPfLrMU9UqW2qH9fbM
wVQgcjBS4a4NceKWVe+baiy0DoTOjOmk
AKociyVLtVvP6DDwJoje0+t+TTsfnHbu
vDX/moO+J9PuzEEfZNuvCelDE9JHWfVr
dvrY7PTX2TV/+Df9fdYpZK27VfjDB88s
7Aemud3zb1mb5sv2tJ65hqdp2Tvagbdt
S1kb51i1V0Jqa+SkGFi/H4NM4lCxPE4q
0RQHO20x292hjicLEZgCJ5RFbdzsvDvb
6lz6VUOwm3Gd/KGyhxaalj5dj77pNP78
y7d//Olvfz1tx4vV2s5Sg0zospLJjO/N
q9iT/KXCVldNe0dH6zi1ShwKGKS8xUXz
X0KjCkUrlo0FyfD3b0kjY11OaTR5ZbmX
qpQvC1cVEcqud7NlVGZYY2vGJhq3Q1+Y
YxVItR44eOvMW9AZZhYDLWX93XxcqLUx
UVcQCA3ImYwuOn2v2nyrlTmNGXytd4XK
VzIZW62NUngjbYX93GlP1o2y7qtZ8XK5
tuuAgTvUrtQFxWS39P16vVuEng9ijXKs
VX+hL8zFcFmHj7TIniujyKrfHlo/E5gx
1RKf9ZNLounX1Uy4q59J2vM9/eoX8HEh
vle4q8SkKbGRl/VrvsLts85Q9r1qzcOx
3vyFtkXZSYbeQqkRhYZJITJYo59rNngJ
2e4NkHxZWXgsTMr3lZYrQXaG36as1Vwd
UA6z78Sz7UypSCFV00EMcEV3PXuZFSiD
wnIq9NUENvXVbay8j0n3oKVDra0qveTi
hdsR+i7QImeQSnZQWuVPTbvp9CPEHR0s
0T3YkPiEeFnq4NUK41TqY9wAzDuZchdk
1hQ5JZQK5DKyX2PYiOOFYo3qUTumaD5O
f0f227ZUcMXIFScETa4oPh7MURzdxs27
udpdyJljQUoRkV5ms5a6gcJb6h2ChRpN
xfI4cJJfC1m4ail9CDnOCJzGUslI2FJt
2cDN8NnKwYNTTc+ZMVqvoMC2r2OotVN1
7jFiWabej1+IlmSeOyuW4P9+UgSt0aRY
9JI/vlwj6GOMuxNNQ1HiUrnCDEpS8mKQ
Si1OCShjePVMDdcHZbH06Cv79Eylzwmn
Iw5wLYy2hZq34PRBdt0FrU27DjkaSns6
wGF10VR9cxEsIgfljstJGSUBBH/c6YCo
s0JrsPuWkga0Br8FrUcMa0QJixUrYrIf
wO3azLTUcxq2eWnZpLOi5LP2Ikv2GwxJ
Qlg5SS6m4cLsqzWajrAOWhUKXwi//Y/f
/r///Lvf/n4nQ1JEy0UJbpA7iOO4psy4
ujKoFJu5uPfviaMtls9hQKzdelelfLUL
Z6vsu0F29Uft1m9/+NNPfz9txsv1uhX8
aoVjsG0m0Hu+zZRLIITK1oBUXsDqrzEl
zS60tyhL0raKxTZ4Srm1GriSbY5i4/s3
JQQZPHoNg5+6jMQm99ShxhDG+R6vi3+v
Z83e4NmVU5Op+o1rFcVtyOU4xS7FwLY/
Ga9H6r4UaGUvc9JfdSgV72YdbdvBOQxf
K2BA70NBmVbCz+BrkzNRmmA1eJesSyqR
bSs8QafRtHvuaMeLBds0I0EaF8/eBuPW
6zEaCyynQQyciZGjZ5KKYlXQzF0+OkY2
spOUFwkzoQvRPutThEMlf3ImGRv6MJo6
hR0sEYqh4Z623AGUCn5SEog55sJCtEek
JKDzzG8E5sW45Dt6KD3aY9q7QfYTWOZJ
kXK0JQyctK6oTZx8WVh4LFIGAZUCVoCr
N0oM6/1kkivkQMvFKnc+fCZpTOJSRdio
bxnypDjZlvBwrE6/CZL3MuUemPR088i1
diQUdAD19k+9eb2l8uJKI1k8A5SuaIek
xL11g3NyUqAM4zimAZRkabeA8l623AGV
QJmOXtT/UtQXH+KTshLFrFlnVYe5pNFS
fTB4ZQq45eaYmlVQPCtQRuuQX0056rjb
OHkvU+5BytqqwhFa0YSKva9Iaa3UiIA0
fOHxnBJrGZ0/fHO+TQqUfqxKW/oztpBS
2fhn6gQPzi+ba0JF+VmbbF5JSenL1P7T
oVUIl44fUGZYFBDDdIdIuU0MzIia3i6F
nJV/3NCjvkLNR5l1V6IpwyrwCYp5iteX
8UN4ovZTYZKIOZR6CkD1jZMeL+dMM2Bs
cwLoIJdTSGFWXTjmrgD0UWbdA6YxCOoU
BQXhJiNy+UlH1MIYM7jsOsx6FE17o+DU
S01h8QVToumYGlzsGn3ZRNMDdg0xPlfc
s6Mtunxag8+k/C+MWmKmN9/K8+4W/3lp
2nMuV6SKveLbtdcWoXGGlVKinehEr62w
unWBQS5I5VTzwSU0bRiUxuX+9egT+OCS
njVt9Uur8jGTF2tASNv7Dc2Q7/9dT3LO
gpfrtGVCdpYLMlNUaqiDmT6Up1oQhHeB
jr7i6+W5/HUTDs26k4Q1JXdvSVtkzlc5
UbVb17DFJvTuDJhHc7A53SGGh4CzYiNu
h7V8JdzBgNr33G1Guzq/1rIaybCC10j9
AJV2pljsBJZYkIGWn204v3SsWeib9b4k
95q8Up+01PNyKsxK6DNURWd+CjpWnYS1
LbPZ/X4Cs1KhizwqU7+u5b2CAS/Xabsi
62nI9KGjQBA+dmPC2ROcXSvHeKL3ADHC
mmqKTEVEuzfSN4q4byZ0FI3NEPEI3tfZ
Mf3SQtpWm5ZM6xSZWg/X912v4EP3oGBi
yE/AHGkqCPX5movGdj1cV3KJUsDxDoMG
k50+azVfMwqxE6JgH/jdR4bZByvAFg6+
rBQ8CAnpJqErz8uT9txXyYgQdHoTGn8Q
jpcTYi6x8DG5QHPLdp0SChXWsWClWZAd
gg23baLha5txBx7KSPjahuOlQ2RxpkP+
Uh640MZST8Ch8iolGL5juJKqXfjNiIdN
W3F1p8Kam4D46jbcBYkhOPO5vTGctUJi
x8dqEzihlm8nJk1kxKCwpnSEm2qy9v4J
IbFYRW5kFXkkjpuQ+OrudA8o9kBTbUK2
JXjTlhlDfcoVtQG4jIIPtJ3oYtc/V0rh
cTfm1OcERXTa1sB0lOquIDF8Ltt/VJ5Y
AiSuRXGq4pHnIEdnh9PSXQ/CAyHA8URR
7jTqLNIfnFpfzuR88OhdG4OYBvBULbbg
8c4m3ZM50tpIc6ysmdYG95ToY43M9zE/
dIyMZHGxlR7bnOS0YR+dFCbpQVxOJwrP
mzB5b1vuQkzhlq9N+BiSaSqtlbioSKXC
c1x8LseaYesTnE+FltyMKFNeyGUmRExE
FFdumdS3EfNXmjO1sFqPMmWQQ0+fGDQq
Ibyg+q1RJ69Wa9xJNd8g3esFQjwtLxcl
aR0ckjWL50quFzrmjs9Mu9TRHmfwHbqL
cbtFm482TAYFqNtN4G6zs2yEHe+eIO/A
6WLKlItWUPDU+lVd4E/fbfWpH7HkxXrd
OJo2PqK/VRF/XYdwm6dPhLswuZJ0vE/d
unv0ZrUIKmO3i6FI53vnQkYRWlMQPYEl
GcL5YMNfVh/3YxTMV6jPFd1mLnHuZ0kY
CFD4rJ373VsFghCLUku9i+vPQ2E+KM/U
aY1GdWOcbIeJMCtMTjE7QXLrgzMe9vOa
AjJ4Dvf0/p1s7DbwNY6kvjDpuhIEHKbg
qJd+xyN5sVybhqwYszLCUUoaM/oMeSjA
ja5xF5LSqUsPRA/QdOsQr4+ewwgtkeKt
JDDu1c1wF4lYnEmvLeN7JmdWabVxiiqL
53b1jq51D0h6lxXGNvp+XI4rk4GLimwp
xtaqBJ/7tsMnMmvHojLaGt9skOdPiJJy
nWY9qAE/+WEF2DhEJbcw84Ix8ZGo6Yk4
lQHLYaTs3HqxpXwq0kQA8U9UwnisHPtN
f4r4V1pAPLu5riHthLjZinNrJa+MeuQG
aN7JlDtg0yO3QRGdkQOGC9ZQ1tXaaHpm
iMfHE7CZIZ/pLZRqmVKcEzZhfF21mJOd
yU3QvJMhd8AmhZ9kN1GMzRoNp03SBviA
BKVkgyWfITPotMM2RbI6/l1Z7KS4OZrs
kjXEFp9vwea9vOse4IwhKfnwNq2Nh12L
NsEuvyCyVdZ5kMlgQDA9YbQm5F77vOll
8vbo5FWGnwtkXmDl56dnH4ubOkM1Q/xN
S7Jra0jkYQqhXFXkr0s9w2VAn1CBpkK7
uSRbmDlx00a68qA+bCVtIOeDDLsLRSsV
qhZqaNoFz5z9zgZ6KvLmBX7gEydWZ74a
ORjfaMwwzgij3ebdlxOrr/wWkj7IsHtQ
NdCc44s8pYCV3qT2RDsJBfims+zKuQY8
r0TUJe8o8pc46B0mBNVo4QZNUnyqcbly
haoHrAqt7WJB7XoZp4VPbBoUTcYLzpFs
ZB+Dnj9ci92NuyyfuSfrRlXa1ok1/SPK
AvbAdPEcvvpUhKGkiNsDIetI47Q+UKEE
a07ocQY28DGR6ZfBTBsB6yUwCl70AVv2
G4SXf//++3/90398+/3//bcf/zqIoU/Z
8mLNtjssu5AiCOfpw1M0901+UsCrbSB3
a2gXz53PRN5SQ4oNHQFDI4Be+yeXToN1
KjOMmSQbqvI4SaZNRrwkn8MthTWvpXp3
cxYXM+09Akl3q4ZbsnwFeEALQngmFYbx
stuFi965Hb9X8bImorNciUUmmOzBEpOg
Csd6qMp3N/QW3505g19mKIyOVl/ZByE8
sZSBGfPqrst/r2zQy3Xb7p+NVhAypQbF
Umt9QX+jxkKfSoAy+njAq3hIea+eWG+n
2LkPz8EVu6LgnOG+nSEw8kNfYLGovjL7
NoV6wSkfCHKFfYMz5pUtugs9C3fdnVtr
BuIXvmj6U3I01nmoi45Huj6TtRVBjfyS
Uqk8J3yOdYnLuKZdtmzj58s6wxsgaFHI
o/C3UEKC1WuxZ+rVnrVDfZCOn0+lKjnp
m+WoHKa0HidF0PrCoNYdvQmgd7XnlyE0
sNICz9IzLFr5mVUaxi70ARskcuEEhAqN
g4dSKOcOteCkEOrHvBp0ux9ftiH0ribd
BaIlwHFjDWFVa74U1psNzisqyrmVfLy5
j0aTjtSg3ii1sbdnxFBr7fvoc/1NBL2r
QfdgaOyR5tFUE12bCxGrU+LSGUBU3pKh
TDh+RiusqrjchtjKqO/Oh6FuLMvCHOA3
ITR9rqbwBmjaGWl2ej45abdQNiTPfDNV
yxZggjoR7FIkwrZ6K7QB8qxgardTYZCD
u7qZjT7SsjtSU4fcI8GtkLXBSLtU7vVX
+Gc0/ungnxgjUyrUlZ/juChMTQur1utP
4//Hl2tYfaRx9yGs3HBPCRJql2L78E18
yspYm9HP5JA2+IR+ZSGp0gJcI+qSipzi
pBgb46C2sAsZyEI3Qfa4eb17JrgovfUP
0PV87MiOXF5dkEznWipdlgHBrxtF/NZ0
uptS7NrTGIkzkunW9bzZRpd0es9Muwi4
S+XdKxqwdjUOLzEENakE451+/1559BrJ
HY9Up46CREeTTUlgkXn9NRPCL3/88a//
8u0P+mTf/ccZQ14u13bbEWETzgS5c7/G
wXQLKrFRZkM/mT9xd0obTIg2YdFHb0DU
zpJZY/DonaUZzHgdNkXmA1C4zXwKepDv
ZkQ2CWRUClKuY99Vrwa9agUwMrOryE+O
aYhGUMzNbkL5qJ1iRWjG9q5AOnLjmIan
qIHjqW/hSpYrf//OdtQcAg08n/wIawVC
n1WhB3eECYVjhYD3s+zV6m3P2odKiAs9
qw5zX8fQTNyxyw/TmBROJKp6W9Ra4hAI
WLRxFDe15gM3sj20Cczqw6CSDnb1uLwk
ZXyKjJRklBiTqXLczZQ7ILMzeFPNZIm1
XcNdxcCoWBPM6I/KmauYijCgawoYGJ2K
k2KmHMzSFxhH0L4FmC8LDo+ETCWHsXRA
TQBJE9CHYpKPsaAxhxJuP3dBKlSh/6U5
7QY/7osnxMxsy6XkczC2jQhuCzXvZcnP
4uYIc2DRz9GjaFKfW8dCse4KxkG7zPna
3AkTgmZJ2nErm9dgUN9EyftYch9OMuWS
kCpWiJLTeikKrYJTKogiYI3wPhyOZOUX
ArkOw/xpCCVOCJQtuWcajORuo+S9TPk1
tXxwavmZqsHXPPO92XRXnvkQi35NOt86
6XyAmb9moG+QgR6xa4HFYTGs7NU/6L0E
AGbtlNMHONgvWDRS0boJKQWXOsz1hlag
46LF1+JiC72QduVGq6fwpHTtin5K5AhF
OZoj9C0r7S/2XOjVcW0h4II+5/075SFr
pFRskXAwoHU2guBytyGiW8I4/+e7P//5
u9OWvFixTUPSSAt3bS46rDl9rC30qq3o
YLASOp5QVInQMynLLXAzpiEil6CqL/rN
3vMUl6TehWG9Gj71xNbI6wZBYYBrLTUO
sIl23M+sRbGPQqJCNHzjfEbuxTO0xAqQ
AgM8i04rfqWQz8SmyC8d13Fg0Ft7BGls
4rex31yjKxjlX7SQJnC+QifTA0zZmneF
SWOFaf5Atxgmy3zdvPuaZ/RizbYPaa9e
fwvxPj3qasymdacHVEiogLWFw5TwjbMO
U6A8eF/8VNKXEQwNyF6GCWwp1+o+2lJf
jXkmCzkge2qx+BvagK90LncAp/6Y0LSX
AMepNt83/kn4SVNZ03m2hotTqg250fGk
iMHD5DlmDieEzm6t10NLhIHMm9C5oZXz
MPBE9i3AMKXAm17pZ/Hz3GERIzrOIOvh
g+n1cI4R2t5r00dOc6JndbZgdSjmLC9b
gHk3Y+6ATE9DiN5bByfrdOY1X8GOySio
jGrSH8ZMZ3ourdAyF20CdU7M9GWMRg5x
KH1VbmLm/Q7nDtT0WYmhx+F63EbNz+WH
JkeSYPsLRWnscSWVLNBtsMN5Ytw5UVPe
xX1izdFTtIma9zuce3CTmV8H4ZenXl/X
SlfVqaTES2EXaZzDhxOeAD22zr6DFXJS
3PRGKEb7mn72QyrvCjY/P/z7YAhl2bXP
jLrIkzEvVaLO3Au8yaWgU3/4kAqa9YAA
Jdfovc2JoHEQHuc4UMNvZZwPM+sOMIWn
IQUfXVeSUgs3pE0+RhkGrfVND+3KqQpR
SNypRf0gM1rEk+fD0moUDEIPa8KuRsFw
DaWPO7C7YJUxCT0c3CgFRdLlxMYiA8ei
TVm1OfrxEys/zwgjrDsMZcyJq4ss9nJi
V1nsS1w9YllkzYZdk5dtP6nUc3f3kljD
M+2nY6I/ioXxvyuTfjh8yxKwvvZqcgS1
bRl/Nm16dDCj8qGtu7N//uvfPjXU+PGN
D0+oc/fsCkBvtF4jU/+n//4bPcvv/vF3
//W3/+X3//Dh7z/+5V9++vu33//53375
j2/znz/8+Jef//bDDz/+4UdZ8Tc//PiX
H3/5/jc/f/8zxv354KJdf4iNRTumqAAN
o0vGxZhoZR8cBjCPNDSpGvKZe5cs9SeP
OmNUDNo68uVvtmKe0XF5FUevSm4bjEvH
thlBDJSGNWoXM0Llxkg2N8IVIaiQiNqu
3cHmiuUn7rhdhBta4WJPb7hgl5/g1XYY
YRN8qklpD0VBEx3W9wID9B+CiX2LUH57
j8WnDKs2unT6LEgYvOUmu58v05ZA8iZx
pYUYXLmDL7P86T+VN1MM4FIqwv3QmzCg
vro3e6M1u6M/c70nmLCjYvdi5dlX9WZv
tWB39Gc1MfdVI91fPRi9ymv7szfbZo+L
zl7doV3Hrf+pfNtlpBZf2be9h+WbN2x7
F6s3cwx3rwWszwPBDYGYF+uX2svDG1D5
4OKTIXNES18xnoMhNHAFSit+KVbUjial
Rt2PPa/n23l8w1O26BDtWSju9wXB0d1l
0a4+xCtGc1Ap5OQTw9B1TNA49LE61CaB
hqwNutobm260KAautrO8AP7zzdaMipYO
TdLPkJO9at6QKkcL3WBfot1ThxDoFO++
9lqFTHvPaXpKWnwYK31JiTmvt1uxy4/w
iq5NNhB6djvytNSb6EngYzeFeClVLgF2
LhlqJDSuRZhadDByfss1m9udfTEG/urQ
NhzaW63azC7trdZsbqf2Zqs2s1v7dWHu
Vw935eHexQLO6+zexfLN7PfutoAhf5x5
WpL79OnYE/n/y8txq9zoI9G6l29QV+kt
copGIcXbfFh7WfRhmAt3aFueoLOPOtI9
M82aeylhDFPUItei7xFN3mmCZpbBaxva
6OEaF/y4K2pQQ1fjOVn8SFQ1zPyXn/7y
XJE4ZcyLRdtuCNW/hhjbIRhSfB/WjMyF
e/1u8062OKarNqachKvsdtrvc1oaCEtG
uVY7yOcYZ1CHqe5Ta9Yxe4o2SnVUxZjV
3aCzf11zaqk6HXxVrije0IWhsTdpVWvO
SBQu1gyokOdUZWtGFd1xISeFMb1T88zF
AhE7nPqFPksvqD+3MEPfSinO+kAHH+SI
ybJ8nwKqXmHjKBtalq9tzotV25ZvgvIm
RYjJQ33mVu4QP6KplYgqTxDxQs+smICe
U+2XQY9J+2kJ3tS86UCbwJzobph/df3j
i7dJ+iSoSsizbzDJvba73YGdppfne2h0
FyZrXeV8Rl843Ql3STxxXLap0ffdaTfF
EQwVq/nAc+FpHR2+y1TiFnhe8Cq/AXxC
9UnPbTFG3LoYFDWBqpibGx2h5xn41BsL
LqsJWORW5oTPvASOy+rcRM+72nMPfgrS
EPPqGfoGGT+tAEpmQR+oUtzKFMHxA9oS
smMczr4qIc4HoIOUdT2gRmK1iZ/3Nege
BIVmoynRl4PV0vuFL0c5hkJcb+KHPpyg
2XVo9LmAtFvTqcyTIuhCObPY01ZpGz/v
63H3ICgUDMI2PVTXmqdFL6z4Jg8Z5U8M
SE9EuLilhOBsSE27J8+JoBtcOZcA+vm2
7DfJRfVPO6RHekbZYTmrAtCq3/W9muTP
CUrsoNMKMTcDrTEMpZ9ZwXRdnW0wfaht
dwFrFe7J+hEKToj/FmCFX5DZU4oQ6YTQ
muvd+CYLbiD4WRNTb6mLM/688fMVrD7W
tHsgVn+rIDvbc0hKPlYFR88/otW4YvYT
CsMtNuFBbR2Zrlpmhdj8MVZiJHATYl/T
uKm9NC5F9Eu4LbJeQ6293eIbRCs2O9RF
yWRWgRGFwSG3rmeu0dSjDxd7GVHW9kaC
nML4ghU+MYHlWxTaT+CSR7E3LuKlS7E3
V7k26FOVRvQvFh+o6p8y5sWi3Sj2yuEG
oZ2OVAyr1ioaFj7oSLlIVaKdQFgXYVlx
woGSlrCD7DVRimAQNU5BHllfmLPazJcC
UxeiFkr5ftiSunxlewpUMxGR18JpJ92o
9gboiTrUDc/iNkK/JBMoGFYMi/zaCWnE
Drudzx1xoTyIOYQIriZoAQIttTPMn46x
RL/odBVLcnLqTdhEc7C1K9/fnpfrdqPe
G9DQKzngY9tzvTeVqhCgVh2jfLyaRLlX
EVpiGzOkWRck9YJXeeDm4ZadwKDPhV7/
/LOgNCmlV+YGX1drG8qlr+1wd6DnW5V7
J4PPzXLvNXx+vvbwEACtMhuBJzTMocS1
+MCzlkaVlij1DLl9KjqdVD+YN5sVPj8N
dN1N9LyvOffg55tVe2dD0M1y7waA3tmk
uyD0jQq+s2HodsV3A0Pv7HX3oGiuLerP
Kxrdzj9XfEuC/FruGlaleqLi6xRBd4au
KLb4OTF0q+D7EkJ/XW3hMfloUwZT4FpT
DDMokI3NSAmqpwgs6BNknCn45oJYaYiU
znKaFFCHXO9afKlbgPpY4+4CV9Ogba76
yP/iWvAt1PCqzq7OPkrBx3XBSZe4HihO
O2VEjRNiqx9p6Vr5tZcrcH2weXcBrZFm
ow4EM4Lc5wK0rjPMiTpCU8ra4pm6b2cE
VNmwXLRL0yLtEjUtWLYJtKfsm3xfTRmK
9ov9xsdm6Ujr8ksDl4LwrJxzssr9Nrt2
Li7oMVstKG0twILYtG/0ABb9Sj8dP7/k
SzXEWHsOOQ8ywoJWSmnE3nLXM5DWDa4G
10Zzi+U2T1kfAeVh4hpvRfOLWsR3P//8
/S/f/va3/+P3J+14sVybdixwBsTgYxAI
9uUawoQr9Ps5QsnsT/QQmpMaMicIkVgp
PMItWxuTIKDtDLeqYbT4rna0Ft8coMuJ
IQ7+0+sO39eyY8UcgnKZpyNxvH0TU6jc
C0FToC1+YXoN8r9MPnjlPH1ESwcOZHyi
ZF8h4iuDh9mIC8ej0fecK1tFgdkEDrdX
40jXi//4QitdMA2VEoNWr1y337+iNV8u
2WbtvtSuMNQ4QXtviyaFz8paZQW6XGLV
sTohHKNvwSVA07vAKmvHslUFFijAwb3Q
ZjBmdKbdJQAahPeGEwxK6VhyK1GU7l8X
e1/Nwe4ASlMnjUheOli18nr/3bryjERi
qVdoao9HusU6+6tXWAgkz4mUfqxXHB52
hL2bSPmy4vBIrAxRAVkOaIlwhgcxv9be
l9hkWe2GRHfN8Xy0BNLRWmxwcdDtzAeV
Y4ev18d2ubaJlPex4x6sRE/PFClzVY68
XKMhEUc+xbHMOftjZd1v0pO+XUlKSbL8
tFB3ZqSEMlcQaUN24+cbOHk3W34ZKUMk
G/VBCNAiHQxLJUFJA90pvadRAjl+6eJp
WcgtIWEa4qxIOXI0b/DghsjjJk7eyb3u
QMqAZF3pJtzEZMRiSaEmps2xFzlgX080
70ZnU9Qp2sjuEG2YDynHFvdLMTdsppSf
rxg8FDQVZer8MA6Omt9SpQTXqG84bQlu
wU6gpiKe2hVh9aRvUvucoDl0CpxcWf/0
h0UY3jgNrjD0MSbeg6dysr4pnSgM8bZn
RBVO6IFTyHLKlHOPFYMWOaFaK44gKV2T
m4iTImqq5oX1kj++bGDqw2y7A19Trcph
fMoByZ5VfRiyyRY4aDrLCX3HwwfYFBIC
fPqZSv2k+BqaVYhyGeGvtR1dAexpu+LN
LuyaXOuXt6ZRYGuiIdBwbF+bQjijtMUr
1uXGer0apJ8bThqXkEFIZwq4ieotiBus
ZDRu12hnDh359JjyBI45eDNrGTdUQ7oe
BgrHtALaLnWjmWwx5H/7X/900pAXy7Xp
fGsy8R8GKLTPfFnSGWss86lgg4pPPkxw
4vSm1QhUPK0wixqFzQllRWqeyYoJLJmC
BR85G5wmGmLhuB3N5QFeEV+s4eFOxtT7
B/qf6ZXtbTtYIp4SVBa9B73SS/eF5xR3
nUmdInoDD59J+SjhKDpIaEW7RRmxR29q
K3JR+jGBq/Wj8FLacpOsFyLApv2uKKW4
Uq6n+V/tSF6s1nY8FPQ0tRe0SumwGOco
8pc60xzwNfdjfYDjcDu4tDqzckkJkWVK
MSt61EeHmAXaqziBIUO1kY7lSFL44vNF
jziP/JmyvU5P1v2MuQcomX3oOnVZgCjE
VMj2jX9SHMfMWqG1CG7nSyGnX3eVnbgh
i8H0bV1bx/hng0pU6XhyPyoNzu78N7Fy
s8bwELQUgulsopiFJHRdtb5NJjHQNMD0
ZzjRlSDv03G0LVQfa8hzYmV0TOg/lQUi
0Cy/iZX3MeYetKRvS/lJpgEh5rqOvMai
HEr/LqH0F09EsNquWS628vYg55xoed1v
vQmWdzqTe+DSiPPQFEgIGIS8ghzBitxs
R3YB8x7Cy66PW5gebJ6Lm1jWgvyEeKn4
j/pWNYTg55toeSdr7sLLQh2eDFgL63pa
W/s8V536o5pbdIg1Hh/mjo6xsxorXdZp
0szSj7Z5ZW1bJb+whZ27ygePwVHM2YNi
IvlaPeSqgNgY+WUomPzTtRPDaL7JzKkz
rR9r485gSiStYeQnwXro7WUTSR9j2j2o
GnOiXyHTYtr8mmTRxxWMBC4F6DCO56DR
Z6We8si1hp6XutB0qOo/winObQtVH3Ra
9yAsrCk6sjlEQV6NS0LKMHen7y/Qkavd
eKZGxHBEUgYQXPIrU+B0ABvGlJeNFD7p
i7yNsKct2/KVZXUsLyxbPGQ0FGwoImwo
E43+TXTdueJu1jUW16KgjngNqWbBhz8e
AwfHpL6ibAeL3UhnaIIpype0Pe3bTOCF
i6FqWJzZKDpU7gdr5XK5uA1Cz2HG//YP
//zbc2a8Wq7tlhSkvomvwFyX16YUrXpy
NramPZDDsWvScUI9tyuIx3lazkJdbAkQ
RYVprW3xzL87SzbOpn+iKN/G8UR1WTma
DQkhmX6rc/O8JYugS7BZ0fyOMM1vHsgI
8Qm3lJXRs7RERl7/FKocRUxM6tcTA2mu
gJ6MvMEyXpc2MT11ZaqRdt8ZCG/iR0NG
C+EDDa+hpqTcovpe73ciLxdr+9pTgRLk
NTCOwTa3XGkT8uo4Z6sYnRlfCaVThFKy
JHTs3pfF5Xv45E1+nGaUd29G6EUWM+qX
dp2tJ29oCsjFRGhO72fHPQDJCFyr5E9e
+Upc7lSUtmgDyLfKI+NxTxzHwEW4vKp2
hXxQmBQhrwdENwFyq67wKIjsEN3KmBEZ
ScHlgmuRS84K2Veg4FfPNOAqOUFFxIXG
4EScEiFTHuzl1nkrE/rbIHkfa+6CSbRA
O+xxRknY0kpeDhlKgVdB/xhZ2BMwiZv2
MM7pzcqkKOlH3SAku0yh6nITKe9kzB1Y
qfOR6SPhXtPLF/eVmMhm56v+UwpV4wmO
P1qVam9UJRyUeHNipXY1R1Mv/uPLNlze
yZi7ALNnerLkKeT8FV9+WHpAXO1FT1q5
QWsnxlUC6bRXTIVPzUPyY0LAXCYly9Kj
UbYQc0+d4FHoaQSZ+i05xup1Nlf05OTa
SEvs7mgJaERUdFVnBcwBwrnepgRP32wU
jsk7/G1zaRs9H2PZfUgau3cxeYbGi/WD
L0hKTcgrxUC3p56ZLzNt7FJoCuRuZ1Ik
TebKVsum1DaR9EGG3YOqjNLroGrpK9tv
SUGtTqQjFQI1hVOg2qsyF/qIUs7oSEyK
qkNnKA31qOXlGlVPG7aGflWN18G7vCFt
gQoPkS2h7HaJCA8sDEygYM1rkV4ZJFTO
ncKDgyfj+JEtnnvDGluEBn9M9yauonxX
qhQSUiHv3xmPJmqdD7OolhP7QuWPyQtE
8FCy36gu/M9//OeTprxYr+27bm4kfQ+B
MoBO6VLsUwhVPCzysSR/kNyaLrNeID7W
hrL+k7q2LujRwG2aCalpv39LylJrse95
mJ1RLh2XXLgvSrfLtqcNSeHcxiqEo4kG
6e0xJZ97YNigVR3ftYuag2TsuQg+hBOz
Zy7LiCVz0cZIxRCOUAgMOsBIJ/SewdnS
Tr4YUr9sA0nl6oBQLuvLLTaT1zDjy8Xa
LvZlxUAR4c2iTVXD0nkbIxSLuSHawf3H
mYtOV2JuEJ4XPWRLyw5T8JC6dhCSTDPY
sfe1bFtoRbaeBIahBZpeGXyxSYd7edY9
IEk8q7NYIbvo2lfLsL2+Q9IjBtmyKd06
o+RSdKaV+/QYCjNqc2LkOma/6AbEmxC5
WVF4CEgq1q0yoE+IUtMPsISf2ULPBlWc
vPKJATJPIwb306k4Og2mhEg/+A3TmAjM
n8HI+1hyH0pSkm29NDkN+CPXGrzxlVXH
jIpMIB9yhrNa39zXFnKCn7LPiZNxaVgb
9DTLyyZU3suaO8BSC6xf63cU3ZTSAwLy
Sv4U8cBIDKFCpVv2hICoDmQOFA1LABr7
nGCplXBWLLDZTn3lb+PlndzsHsQMVmnz
TBM3e69nxCwoxNIuLyA9DpjZN+6/SWuQ
lU1zAmb8WK9d7juv8XJXreAxCSZ1v6yD
oqjV92d/6wE7HVGdXpr/zsgnwWkSmP5u
ekMZdk7wHMRfcdAo6Ktb+PkYy+7DUl+7
UX1qpRXLrmVbDIFaWoUgl0DgMJLWrIwz
1hqoDNfBdjIfksrrfmJaOA63kPRRdv0y
qqK5Iz8bE1UEKBeWarwLsMIziRKB2BMM
89zyIMfEFVMMk0KqSy8ObLqRgp63q3bP
VQ+1fNoLu0YqBvDzKSnM/UY+mmrT08kd
M9bXQ1jrAXAx10wfEGz1x4u2RnSkR/M5
t9wVSMfxaPp2BFAdDcQ6BcK2IcLXxicY
L14L521kGVaxerO775/+v//nMyX4fca8
WLHNEnyLNo+N5hm3k+t0KAMsCm24F/Gx
H2cNg7dEOyaQJLkyrlbICWjIxq+UOINq
wNaQAwI03cEZGoxs7V5WLIlOS/0Jyq43
1H5RrJUHoTO/m3LXmo3mCrd/gXqllBMC
hCHKFzHfzYx2MKbYyMPqt7NJN8UZphp8
HaSg485TX+F6Sa5RzGgNJc6b7MWvYMmL
9do8jgrP9G6RxgGEcOqH8NSbLIpUCzFc
yoNI8Tjfrc654tpGaVjvaeCjPIaJDkec
RAPu+7fkEJQBpPhU2Vr8IGbyTVmEzmSE
h+uOnnUHTPpaSmX6T4cyNNrf/ROoDgGg
zmSKgvVzbAl0MQWGD6scA/pLs8KkncYh
Rgwf8E2U3KopPAgnYYE3AkwaFeqKk07Y
xgSCnIeSjNKPt8Erd4UdOSkTQh24xTmB
cqm+r9zOt4DyTobcA5VKWYox00QaiuDh
/2b97dx8QyZUv5uOlYe+aU8Qz7tEzmuj
wetN9XxgyYbHfjX5T162wPJO1twBlzRx
IqleKqHqOtIAfaox/cNSU92JRngm7O1s
Z7zQULqdEC2LnUQB/1B2HcW+TbS8l4fd
lVZyowlRCc0kNI0+OYiFFNYqyywll3No
qVONqjNTEkZKMylapiF6vwhwJNuS12i5
p1zwqAwTvgsdmIgE0qLtSq6Cow0KQSkS
HY9nobeOEJj3bnWDOWHzeujoCjYfZNE9
EIrmHOJziW2XnwtAQekJGmMKkWqjjHj8
6tMbKafr0KDWwbM4H4DKjZnbLaPRdrxc
AeiD7LoLTBPD7QpBGUtoa4jrIXGoGRZN
ZRzpxBWL53Yit5wZUlO+kudE00VBvpgm
gAthO/c8bVemuq/6p2WUC7ui4ECjbIIT
8wZFTfGx4yLrUBBdB7GDh9mG+xUB40GJ
svKUa7RZNQdfXKgfA145/s4tbPW1z3B3
1p1xDumlfnwZ05ZaW31CiANudWf+8//8
/XlTvliwzSNKm0L1SQgHt1T7kGipUMLR
EtfipaRzxaFUi94mwV/SF563SO3CN6jf
kDucYSIpRSshVLvaXnhMYAeNKIHCkZdv
lWvPWzEl0nc5ODmGxqzChhm9K1nurilJ
Mcr4NT5NiKMrsIuZhrzD8In2js50Q3ow
jvEjHX8HkXghtWVh3r+XrdFaTbyVtRxU
Pf6JfahMokLN7X29Vap9hbN4uV7bVydJ
f80nvWQErZYwiLBJDtkUA+UaT7Tzkaco
I/ICHEHyaBpC10KZaEgl2jzD+zekj8Fm
dY16Zgjcpo4MYg0IIiq3vtVg+yo+9cvw
6KHuJ9OXofWOqx2ZXXD6R7Kn/nIIJ1r5
Yi9woMSs/evH/dGE+Lj2rRVzrIMBfhse
t6oIjwFIa2r3DsGjosy0hA/xSYDpoZuB
7lZZyymERH+j9F6IoxTEryHsdBBJrPrR
ln4MM26j5H2MuQcnia3RJ/K07zhlJh/a
U+6tGHEeE23n2Kb1LglRVo9iiyyYJkVK
XBaXYK0tB/IWUt7pWO7BSgartdDoH6GE
u7KzKS9hNjcnmM8TFKknNAP1Dgm9FFhu
RrfxhGA5xOXGFIYbA3+baHk3F/tFvLRB
XMgvC8M/7bmxqzZm73NA5xPdnUPnsj31
LnulkqpvueQ4bzq5SJy2wT+zvFzj5Z46
wYOSS/l8GqSVTcq5LhSBY4Sz9FjRrw8C
1RPlH3lxb+raMm4brRgTYucQL3fJL1Lm
dRM6H2PXfTBqkteMGWVZOPTldla/pb/F
DF9Hz7qcQdKKXgvqDVlZS5o15+xjvloP
/umPUZXvfQtWH3R6d0Esd0BYofum12cd
K8Tyau1QGTCFciIdjXScoClIx0OaNB21
sgJNqsvl1BbAHrGqbx9NqA3i7Bs9/47R
XF8cXf0tODIT/qLeSGcqhfjMvZl+qfda
Na26sewlU0aPxxjk0W6h3C/b6TsgYfc8
Ve+RshRU1CgcV0Ty/n1yhzjtea5+QFfm
SMBPiau2kPOiyvCn737+Jbqfv/+ZZ/v5
pDEvlmzbmox5Kg5ukMjQ9rTM1gfU7fW0
OtpcqZ7g0qQpLDLtAedHHzMsPrWce4Vy
vc8ROQWfV2rbMAaUsoUkoJTrssF1xehV
bRmzg5ZLrlMGu5HPVFyfvGuFCayswlY4
9BppGuku92OIuvBR6QGY1a/JdAjicjLr
oDRR6hpmEINsiipXU7pFaA7iB5hCOZsy
63Wb5qva8sWCbd9mQzPdQhVyxlKe67hp
/IExTnUlqCekkKB+D2S/OeTuF0siWibg
8YwkTXGfzUIstmT3UzeSk2FIQ/ih17Jx
Ofa6pvwyXjqKubnpHzN18IyXVRmM/lmi
BQojHDuXmeY+mGyIqOlOmBgwlXU9+9gU
y23AfFloeDRkBgaAfOkt0NPwPBzI3TW1
noLat5LK40EtddxMOKUUWT/7ORGzjsan
kZEOidltyLyjNb8MmhG5oiijyRNWSNue
UbMqEHWwUKPFcvyaRYfeodgDo0lZo5/Z
EDPV9Hwwx/juNmLe0ZRfxkwG8HVg6Ajq
po48jmWQd9aJobIQSz1YK1qu3yJMtzmE
rm/rXZkUNFsNK2i2Gm6D5j2t+WXYVAZY
XbJUM+ixVlmcjt5WkhM0DtxjFEPfhCcl
141hVvazouU2MWxarL/kmTXXbdj0n6se
PDzpbJVigWJQB09XWLu/6JFVrtkDx8md
oc+scusUrQJGznbZPyGEQiS6ynA4q01f
I+jjDLsjA5WjzZBfaKELLJkLmOIfWyrZ
p5z0AQ4e2lFNCN0JSKEgip0L9DkBVXHi
6oHLqLlcA+rjLPtlbG0F2pHElbTccnqW
5vUIChcFUOjchxPECdmjpej5FiEtc4MT
Qmts/vnENiN8u4bWg3blVnOxIhQm7YOP
n5bocXnpokRP87vOIlQn0d8wbUJdTuvc
SpfjLfW500huXAFwo9chnaIm6lAhZyiR
oWNIo9Mo4ySg8agFprL374uRhOTJu2lz
oaqgF/xwhAeRRldXrls4//DHv/70l5/0
6X78w3d/+vZ//+mnP/zrt/6kVS+WbtMR
N3lHOqy5ikOLc6nfdeMrj3AqBGWZxykU
0M6COM1z+6oFWIbvkeBT+K3tlObQCYiD
o2U16lCVqfTCKsQUZDkuJu9vU7RxuiOs
rQw2bF6mwc8YiIhldNfSEugJc7lE55BC
n0pcc7hHF22AAIbDL9VHzIRwYUhM19BH
MYEH7mM0pxtZFmx5cscmkChnqB2r6CFc
9+jewaSXK7fpe32vCM9V/UlYFfP4Vw6O
Ka8Yh7mcEzK9jCoJ10sMxSMBYQ9GH3Yv
QTkyc+0TWDTa5Zle+scXBZUoSsXIB8kI
KzzC9e4BVKewtDCMVl1mzVfuMKb/Kr1l
+rtJZ/gYooanROUefSaKkIq6nruRpoNU
nUQbbgk21bG8bELqy0rEW4FqUmzeo5Gb
waSx9AsqXg9KYUtk4lVZ0oleJI/wVpJl
C+Ba5sTUoRqtBQojXy03IPXuNt0Dqo5k
oyUGU4h249JgltF4YF4/VhRfj7fZe8Yx
7KDW2nXm85yg6se6jOkl529i6t1tugdV
AxX7BvVg/lRlW/+UXqQUGGYKpybSMkPc
qSls9CXXMCmqWl9ZCy/6yoZP6+kmwt7f
E+/BWDkS+3u+KFSFo3XNWtFES3ar3eRx
/HHlUEXYDk5hPIGwfNqsVacUb2zjwajE
bSFs+VxB4o3ANjIbHmq2Hnq7KRxVCerD
TkGtZwDjxB1r7jCyRE+5KS7NodNBbbCD
6v3SIpo3kPbRtt0HuiQk9PTrmMphrhEy
7lnWzrh4OFZO6EdE5cdJy9MVJ/uFUW4+
2C023OaX2X6/hbqPtu8eALa0BPkVDpSA
sX6olC8ydxjZwxzozw28KVVmFB7t2Ky3
7VMCcLQLneXwjuLiNeSeM6+826UxQ/vk
NmDY/OX5RfeDiYsSQt12zR5N9Ng9/eap
rASkEUYY+LSDNQnGEzEzEVzVQnQS3eQW
FZ9Sun3jUMMUzjmNpu5Fg9SYygLV10aX
bah+owz1p++/+/fvV0N++9PffjlrzItF
27amnkT5pg5rRpTuWaGy5KZH7SRDaBec
OK5cL1Fuluvixm6oa8m6er6KrJcCkgns
KUzBD4fSxtC4EeloKZm1R0iZYGWrcfTV
bdoC04kITiD6e6N/tLfGNbtn4r+serKZ
ttFsNHTIB5yIjmsQCjTF3jBz9LQQnXPt
HzJiBW0Kdcow2JAWk4bBhhSU22lxfaeL
2sf8iGP6cuG2h8gJ6BKNR91F5mqRIFSI
A6+G4vmqMx7OoSqKB80uBPWubpTRMyEy
EqTQ2OcZbmDz6AIufgytWnCglJ97dWIX
7jUfcEh3oGgKlP5lSX6d3KJ5klvojTqg
p82pHhujGSiqMIkqXKJfP9Q2J4r6QUOX
FiXvWyh60ar2FjiKCpAn92IAua4Zq+Ik
7T2kgVyG0v7ElStaGjTYMK0qXJ4URvNg
BKhLn1q/CaL3NukeGPUKjELqcq6R1rR1
DCOkWjM9/c01AvYTwqMoX3Z2BgRXqwzM
bDiqDHWpMvWPL9s4em+r7kBS/ZEyaO5s
QqX9NT/1goxTDoqPiryvYPT4MXVC0eht
lF7YORTrJ4TRMJQgF9KHIT21CaN3NugO
IK2Bh+mV/u9aFsfiitABbS40pnrIx5k/
8bkpMSPJ0Fvsc+LoNZvrJYz69Nm+tDdJ
TWvJ0K4Wh2ZIWC5cERDJOrLJxaBY9Tj5
g7wTJUo5dyi1k0H2hIhahiqpdZF8/DHK
MkZwdo2vj7b1Hqyt2XFDGgNv2tdBZOJk
eU+OX249neHa1mYr0BwqBSaHmhRq/cDY
0Vvq4ybQPtq8e0C3+J5x5JG6gVx5fyrB
24RHhbp5IWk9kbwWhnsqHV81tT4p6g4f
N7SAXLCi8BXqnrDup/X+UBjqelnv1ym9
MG6i37DgOowJ4MbguTw0vWrdtxgWJpdu
t3tyz5gmH2TyXSof6A4xU1mpOY+pK1M6
ZjmS6dO9fy99DcB05iIOiJRAtEueHR1s
4aQ5L9Zt2xPLYspGfPJJAfGa9XjPP7HR
Dog/T8AuVauMbq9gNrtBvCa3UAsiUNpC
mdBgApPmPJrAs+Vty4sPGZnHUhuNCP16
/PweVlXQUhv9Y621W91rkWE6SJHkIVvr
a094Igj00aE2zcuJWdcevXaIMIghsIUi
PyUFetUx56foeYbZHLq+ePK2TEbaAHPQ
jhRGBYayFUf0fZ3+Z816sXbbV+kJvrKA
PHBnOuxDeILJJcPAhXLCSfZtuhrhDZbn
CIjXDCfitOHoJlEI7ssMYVM2j0t3h2W1
g+4WZjBUz7S+dFs/xqRfhlPPcF/g2YKi
1T4m53VQHZd0MrKOVnXH727oXetBObEc
VQhjSGlCPI2DunChiYjpFqB+uWHtEZDa
EIQJ1rqkZPMjn7OR9za6TNkOJ7hGHSRh
uXHgS0ixz4qp3ttETliouQc39yam3t+w
O1DVdYW9GbJex2RkXxsR0RSJresfJ9da
PAGqHumOrHenV8604aZEVaR59OTQDn18
2UbV+xt2D67SAxhy612OJOZVeCZV5upl
0GQtE/2E8EyiHRz6S/LWIYkwIbCOLv/g
bSbHjVB+C1cfYNQdiSpDONCTQvUS0yL8
5Rq87EhdykenfuLKNbksCCqo5Cq5yWFO
ZL0Wlr4C1l/dlvYIjC00pSSFwfqJ9uZ1
6JV8RC7Uw2JjLe7HSdNahiRZXqEwuzct
xjJqCNG+se4OncVrjH24jXfArdZcpxeR
vjoExBcb01dm3cSxRmVEJ2TeihU9a+7G
G17CnGgbl5bnYWHUKjfR9uEm3gO8icAp
NkWxcqE1LT46cKEXOeH6zXJG8C0qYkNa
BUri7v2kuOvbUBkvLy59+nIVtIXC54z9
8l7ATPvyXsB2wMt7AU/fDnM08pH1Bpce
0+w0mSYEGAIHujy5WhFejcLipqD3nKAG
zq7AvOYIEcYgQKeiLF8SGP4IE3jsbmGW
tqgZvS9KEpWCnqIX+UMjA/1cV1v4/EXP
LoNerts2BZAnFwoIFWWP7Gt/SvBYhyDH
HnHT5wpSPnANIgdRe2lDkjxZyBWQaU+K
yyewZ+gju20ruTsfQ+sYoEhpzK5d3wa8
tjUFok1OsCMoReC+Zc1CzxmjzVA+xTqO
JzKovmXTFajxnAIyJZFkFPERmphBGZkh
KcoAOh1YMwzq+FGHWg6ovqqW5EaqMVFw
S5E93d+iF+t2g6KLLsXqTFF1JaPVX5Gn
7XjKCL/hCbpvU9LRMVe2AFvxeC5ElUsi
u6ozmNOocWCHtkVbXlwWksntJORkmolc
39ueOwDUo0fugjUoxRI/jtIFLgWgcI8Q
r59oUdTj9eC5MfLkx2FOBFUoZDGwH+Lk
LpebEHq7o+1hIIrj7TI63f2j984o0gqs
IoFWY52kE8OvTA3A1Eg2RXf4nCAal7T/
48X6FoTe15p7QJRJ5VSLVlqhrgLbZ44J
gnP9Ps/s84kGiYiETunaxaH1XuqkGDoa
R2JfPtQtBL2zQXdgKMrVhdYFBORCTs/s
AyalkavSa4RRThSa8EbBVGyUsvRSpkRR
7jE/cbml3EbRO9t0D45mvQuiO5F2K7eW
HXztgWIK7VeZ6vEJQgk4S7VBILmlXXlS
HB0paCwvaENGW5MPm5i6v1/tYfhKx7gw
QqGRTvLS6s4wivxNjcYJlOsZdomU5D2Y
/SX2QgZoSoBdPHIOHz3yJcI+1rZ70Ja5
DqSLtOzcBIVnyUBFx1Uup+mQQw9zoqKU
tQoFrYeScvOTpqzBOmLcGK8nAdzC2web
dw/2escdVEa5upasv4UECgaJ8vFQQ4ez
asrJmqq8ArUKj8iU2FsGvVMZTE7LyxX0
njAvl0arJbVjZFYYDj4anLD2pXVNCddu
R+UBdXxvwHAMsWvr4MJrWNNZxUFgJlU/
hDROtFOkAiFmUwKYetB2GYrT3AsKN3yL
MnibgdhJ5ltJ/9MQ6B3lVQIWxNya29fP
Fs8a9WLlbtWc6DRT0INvDivHdICdtGOQ
kLuO74mAGb6DiKA9TXJCfnu2yLdO8grO
iMTev1Hl23A3jFFYkcKqxPI5ySLTrtCm
Qsv7ALOGoDCu94DosrzfjdZTZ9MbpXje
Y5VBolpM84N+C2nd4yRODB/W5GKNiPTo
+NuTMdGOZEdgMmiKq7oSjISYo1rG8A6U
VA3dL5T59vaInzXp5cJto2s3sQa55yQX
UteTqnzWJxSptAEr0ynH1SAruqYounhU
BeJ4NJp3WsM5xCn0G7ItuZlUv7RAimCk
E4EE1MHCdbX/Lt53D6QyUucU2gU9nIPw
/Jv4lFCIjN4lstse86lLHDfqFNy1eqTW
/ayYaveqEHNYZGx10W1U/XI324NwVYfR
WAwTvUfrfCzBtkOYjt7idCKNhfvAO67n
Q0xDI3I+UDVuHxcXAV5r5NyE1PvbdA+o
+p5TstoiSRiXFksXm8xaY1H8Cq4elYsc
p7Wh2C2Xq03Slz6h+VDVzPY0qv/tJqje
36j7YNVZiirbye0qC1uMKidcGTJXPtuF
sceS1m/6kwIkJJydwNvH0ldG+PlwtQ1G
0za4TJeXbWh9gAveA67IqXgBaYCtMHCo
n5i8QqgD6uCyKMsdP6xZO0s5O3e8PfpZ
09U4Bq9GbCBntp2w1l/bovYQlIUjOyc5
bJ+t/XRFWf2vo4jdUUULZ6R0HLmxT9bd
vJh4Opj1eWGptReB1xbOPtzAu/JYLbPi
nMZiV6pPK+TC5NVhk00wpZxooeglwR/l
kZglkpoUcn1Zuias5k0X0SbuPtzIOyG4
wW8Yai1Rx/05sU2Vgfeae+1yRScS2+xa
kHfQsyDaXOYE4NyMgUIv6ePLNQCfM7Ev
pV0Y1PgJX9r9QqPQKUQKQabOqHRsUsbL
3cRglPZF0N0XR128T535yJCiO6i6M6A+
lVqKwL4ov8pDDS525mGy4nISrj6Dn05D
l2UMbnkjQM2eO73UtLQhty90tsXrm55f
bc7LZduemVW0UHvMg1fxedYjwC1QrJ07
KO09IY/VUBbocvjcBI/9okBarqTLiyTb
LVMgr44EYDum2pnj4atUcD0uQkTnYvw8
5+lr2DQ1qg2RqkOv8JluNhT7UKqCwcx1
WghL7xZqEAW9QtS7nWKq4/3EGfFg7Zks
Q3LZZU9WfMsUqCoDnW0CN1wH6YTg49Ph
DlvCanUYebqEnrFXatDcA4x7sYTbsxwJ
zQmFABVxsrbyxmQA0fihYOprxyNlRU89
MwNEMJlDGbM7mQMbUZ2u3s1A8pSycRSm
bJ2Ay0tQnIxunIIYoVdK9/fBeyDVe86Q
IhodUQWu6UN8qparwa+cjUvuXHqLTHim
gqkAErqhSTE1jKYo+9nfRNTbXW4Pw9Sq
dKj0wBBdghl6vRQPmXqH/E0w7oITF7EC
a33H4KP2VM51Vkw1SF0t+hlEva9Nd2Hq
lsacESfSfdYarI/BrgEOH9IWGAB3jDc7
P0ZK58NUP/BzMakRLmyh6J0NugdHHbob
vcN9yvTmksZUemJyIKksMZ6Rr9NJVDZn
Uj5xof+ZDkSvm/+3EfTO5tyDoQwttwy9
dBsNEaO5oeBsfQwIrOuYnuhHLMj8ohgc
UhhEVzNi6JKTLdWULQzVXt3ZrvYwPO09
kWuknmL2dZ1Vr4GSZ9V3SE7R7/FpOu8h
NaCCgeadG+OuM8LpIlE39FVHD/k1nj7W
vvuwlUlj/WHXs0IU0xQBC2dlkAgHNRMB
p3snUHDXQ/gWwuAxmxBc4yIZOi7sxssl
uj7YvDuQVuFwhFkWRJVR+4omCLDReJaz
7FyOdztx79scXEAB8umhfDUf1rYlvV82
7CbWHjCuVuqHH//65++eA+5e9RP/Qas6
vur6hi/tCtVzRzkcUYJ8QxMg+h5jcr3I
g4cYn8v9SjQTmnK0QMNDfYKUODLmrAxY
wNv6SBD0WKWB5RnO41YncMwLo6cfys7V
TnFWEqeIhltPlGau9a9ktZ9/+faPP/3t
r4fNeL1Wm2bUEYmNExPgYqtri2lvEV5M
JhWY9jmRrcpLcL/HrFIepImhw2Gt/d11
cPVbM8BrhiZxaUf0A0K4pHBN0UsuLfjr
bsRXsWHN+i7yZnDP97yNoMymhpyYH0e/
aDmJXMcyi1cYpESz4/hBpAlZmKAAEfHn
cRCL8TgVYnIuat6/i82xPDeURitsmgax
cnfvtboxX9cDX8WCFwu1eQghQRzJZy1l
1Yf0WdsrKg6qoQd/tHfJ3oluYGFtoYJh
FSl0ezsVfEXVpfs6gf1gJVvHMcIYGWr0
DhCrB+31uzjRnViYGqNR8uUdZrOl90xu
L5qOCrQ8NZ4Q1vCE9Tbe4emFLm1SKKR5
QY8+JjAsVNsEwpcFhYdBYQjVjByUJhQj
nh/ELbIqvUKcQGUq8Zhu1RIc1di9dgTH
upTBZj8jGrZkRa3RQpj8TTi8gyX3AGIi
d+yklHLuLSzarJEp8wSJsB5Tu+5Me29l
QLWWHoCPMCcghm7SgFRePv0xPswteLyD
RXcAJAzNLQmsSoYjS7vLLEr9x8l9oJFL
E8Pxk4maQuQmmEuyMEZSJ8RI7epoJl3E
5JbJ6Q2cfHU77s0ajUyFOETJxUcJKhRU
af1hfLHKpieYWqqNtQYPbbgrkyIlyomW
atTRcGJ50xVY+s8VBB6XQmYIJluil6Iy
RrjgZuCGxVUfON4tn5hoawqFldzo9Hu+
W5gUNTOdGUsGsvAqXoLm/S26K6EchwiH
GDNqcgM/K8QtOOAeGSU+PnbqUd1oqRhX
rrOwb0YAJRRfp07H3MAVZN7fnnvSy4Du
OuSP+nV7lsZQnKtT21NDRxWvfEJuKhVy
nRgKc8R9DFXPB581+7YYVL/sW9h51KAl
DgtSB/tg8zX2ZUal4VOD0h3SC6rIPfaU
t+/EeBwP+0dtMRpb+xLE+KZ911toxqZ0
gu8swKKmLKhBFW4DbEkwQPtgRJzBo47z
/h2uAAisqN0GFPXVqD5GoVrAkTWdU3c9
Gv5HbdRvf/jTT38/bsWrxdowo9fy2h9q
Z9FZvZZ9PHVz7jwzUH9cJjl3uncBFj6u
lX2SIgngHF24LBSYgkp/uJOs2JxQiB5N
zwJXyM+gtPXluvTzCiY0zU8CG5zX9uAS
JFRBcUjpcr0hLn0HjC354OnK9LWfaIdv
xqOeYXt2sKYNAzqTuq+Qjvop9FTTWKs+
OK4SHlYJNMGEzmF3+kDXvNqvcQQvV2rT
kzLNoB8KrPV3ln5aT2+tVxrC3L4e8kRe
Cd8SIUHRT/rA0R6spm5jboUGwTqDCfO4
mlwcKayMH77xT6jiIUmqpJziyjUbzquc
wh1w6J1RpUEZppf1jtlryxXuLTp/IZ7g
I3M1et9oRpBTJWadFA/D6CGoo+44Wgi2
4fBleeBBgOgC7FOCRcoDTPuuiUdQ8om4
W0kNxuUzg6DMoMUkUNS7VytBz4iJ0Bc8
G5IGmxuQ+Opm3AOKgFWOHkG1ohRz6WnI
imkSLKElObqCDh9Gan1NO8IrY1GEUyZF
RT9oDYcJ9UW+BYuvfxT3AKPC/szAaUUH
iMHj5U65NNDRywsycnniLsSHZJp+zD8w
e98mhcYwqOsXn0p35S1kvMNh3IGNlVpE
Cj5yahDeWZIMhSQtaQ+iDeHPdAfgcJRa
8SyxtWmx0Q+Z1mxdzU/L6bwGx8/m/49K
HKGFC55xL8U8Nde1XwC2hexKjcDZGQEn
l6yVpCk2DoNuYEaUROJ3rc8pntjAyHtb
cw9cBiUa1AHgfqurYCLFc3oXIW6UP/TH
4VKZB+wbejbX67xwGZJNC1GMpss5+S24
vPvh3JdSQgcXK+cn5DIOp4fcnGYQLr6g
5TtD74fDheeccp8xXcyIm4365DicOJVN
1Dxoz7qYU+YLz5VziI5eVs5bpDYdGtJl
/fpsfjh05Ma9NDGTohoQ1y301cg8Qdmu
X3TrSb50n//81799aqDxQ6uSonZUNj/f
TQR2JNn/9N9/oyf53T/+7r/+9r/8/h8+
fP/vstKHH//y899++OHHP/yoL37zw49/
+fGX73/z8/c/Y8mfD63S1WNvrNLRCJDm
by6UdCaa3srobKIrkWKoNoIimv4rlgnq
xcggiXZOzyE/dp0CnQNIG3jPmOC1Zzi2
m9Z6sC+o+DUOxtL81WgPjKaKmv3W5crm
OmmBkUZoXC8zygj96IPX6fLBX3NDtV6b
thLhbrIm+ohWVkRHVuex5I1JgFvbKdKp
3UJDxjOh1vXoZZrXOVme89U97XBPD1yp
yR3UQ1dqZhf12IWa1Uldh5Vf/dUX/dVb
LdrUruvtFm1eL3aHNXvup/LNfdpQVXt+
2VClYxJQt3KlVusqvlq30bCqY8ZoW625
6KGXlipl+aHmRtcmA+v+WBurfQPl9ui3
uyaXlWo3Riq6nyleocqDNPAEJbgaRhmV
01bDyPL1MZrgwvvYw8al8Z+++2JT3BeN
eLlQm0MCOSF1pW3cCxJNY9mbjYgXlE60
M8/cb+Sqs9CK9UVrQ1idplS5VKrJXJDF
GViFaT15rqGO3S/QdzXKjNyLu+o2VJdf
wYRQszfaielddDcaVUtLMUPOj1p2rn5F
woKsCX2IkfviE3yUdN+AnSlXhtbtwby2
BqeesfM4Q1tjhu7vefLRWuG0qpxAhI2d
39LNfg0LXizUlgGbl+tviM17eAuXo9MU
dQQZjmmPJGg7XCwNTTEfMvaCQ18Hl01K
KKxUxZZ8AvSz378F6cS0/ptaP1GEtEHc
YkM8JcIcRc8ZY9Z38qpfhMalFzignaxw
KiGCvJKKaiOgW+65/A/618ehkRvXgEiz
tpc8g8nCTwiNURtfT95bubRpRM3oBlJe
kC09CCtjhPsoEt0Ndqx1/kqJn1N6EFFa
bsd4YpduACicq0eVA73QOClaNmsx3rCp
kSXfBM/Xt+ou+GQORK7WVaWN9HQs50uZ
hQJZ2NFyUHJ0gik2GmV5Nl7wWtqc8Bmi
Xcr2aH3wwWgJtxH0Dmb8MoaSw+phFKko
5HSozSzTHTk7MC8iRJ9PSGXruHu6KpXc
Oka68qQgGrNxFW853GyueBNE72DTfRlm
tCksRv0q3XOD1QMRgBAKw3ZKOxkiOw6j
+qwN1hdo1xUktTlhFLJrQqNqwZ0p3m5A
52dvkh+WcdaO4IVcqmuZfviRcmYfS6pK
9n1i350B0QrTr45m095pNpE2I4hWUwtf
LKqvyjZw3t2muzC0FkZjZcNQA806K4jG
BnVXU/pBc+PxIRCID0nCs0kvtDonhsK8
Z12SfXTXWYB3haH3t+ielDQ1uY4KM2aA
9nPNSTNNr3DVFTfEzQ/CKVV1hbvaGC35
5ZBOiKYhGxHU6nfHVPYVgh60aFwNigU/
2rNdcg7gapvVjzOE+K95NRBzMn2irGCY
0pJRMvveeuPEV9MK2+hr3Sxz+yfiMkqN
NDv3hJzpZjvB37///l//9B/ffv9//+3H
v46V+DVF7r2rdvUpXvFqQKYvrRnRiq/a
5SOfpV5eBHxGjLx31Xi/1J1CFK9D1yps
gG+6bjFphbjwVHoEAc4rbrbCMCgslo55
tKGpYby08Eol9NP2XkNprzWELTvOLubu
bnWMPWjNLj7EK261AFUanC3aVpnvYN8v
yjoIfCY9Wdm7aHq/ZnO48lsuBB359Kar
dk+/5roAjMulEJWDGNPNK/u1rR6EqT3b
COiyTJIUAQimlVSkOzi2t1u4+7m23BDa
jVnZh4IdX17ds73hot3Fty0omiF9RAfG
ycENvsPX9W1vt24zR21f7E2Y2tE9KIR7
J4s4bzz3XhZw4uDuzktYxpopxEJ2Y2nS
goH4pSdEKNX5Etgo4bUc4dKhhFhrKNaT
FIYYD0wl3LC1pq8Tqgd7TzHTkQ1JToSl
a/qoX3mxaL/88ce//su3P+g9v/uPu6zX
y+e/d3iHv/M0f0AssVE7vdEIWHHPCfoJ
Cve3RlHuvVYZ1V6TfowwZr7i3tK26pGG
Cgd3nR85RBBoQBhVY9KC7T+bUX4yNqj9
THWpvNXOevH0r7mxqnaWVxQhK8D1NzZW
9h6mLmFsyxuqxbc2ltxiaSgrFJj5bk2l
3H2tpvZaW4HvV791y2+9xWrN7LneZHdN
67veZLUm9l5fjFq/OrJtR/bmCzevT3vz
pZvVvd1v4eQYxmqZnMvHG+9IE+bLvpRQ
S41QNnFhu80PjLamVqElaICVJy8dSLHn
hgNqdPS2fkwvYfhFLvpz7NX7oCNdxywE
9F0xtdZigcBohs4UH58JZXy0zgBfEX2h
JBGa3P8NDsT/892f//zddd/CfitertWG
GSPCFKVEV0qQtZ6b6bPtzKzfQEntOJ+s
1ynhTHgfo6+DJinlGBDopA4Z65aDfnc2
DNGkbMsYqdJX+UPQC03pIVQdXKpa97Fi
zi2hUIBiYUl+s3/eNwgKadqqDMUNzlS9
sTftJq0y3DLHJ1oqW4MbmYimrqlgpdQQ
mgpVDgAG6Q2X9+6MmPtoxTXZxNyZbsl0
VGqLQ1CV3L0O4tVabbZV19wKus+REn0b
HURIQMUA6z5apPGgYq0d6Qr7D2qXqeng
VZeXzVURSeip9SI/MIMVa1rp1jPKUN/4
J/jjs8JnWbI6Y3K9izvdAYoobQW02OXe
E6pPA8lwrpGOWTlA+dozoCgsdL2aPC0K
yL1NiooWYS9mDGOhNlFxg//wEbiIGEOL
zGTnrIDWrQNJpgHvvb7WMdKJOnFhptQF
qSklHEQ6eVJkrK6uAnvVtVuoeAcr7sFF
VyqqvsItra+wEZVoxsYhOkBWNOhfnlKJ
TkJdLocUolZByazIaMy/T1otixyQCrwB
jXew4z5w9J3u0C7faRJq68wRQtY1e4hE
mr48cRrZrt2nrrdpUO5Nio4VLfrn45hu
ouM93OqepFH4yG2q79yih3VsJZJMMlri
EyQGJ9QPGGzSvo0kHnBLWAP5jPhIg8Fi
SOOr3sLHLzMdPgIqZThba/0F+tz7ejgd
o1/ot3Mt30/wysbAwD7SSAlO/TInUAqB
VqCE6mQDKO9vzl25ZB6xqmLriFbJkkx2
SG/kDRV41n5iMEUZTkbGS28TeguTAqYN
0A9janNvoeX9jbk3q5StU3Y5KfFb5oyC
p4CZkPmOGPb40VQk5akuZEKswWk+I2rm
kJ819oIO5xZqHjXoYr/xv1Gsri8mjF6p
qv9h5z3HrRb1H//yLz/9/dvv//xvv/zH
t/nPv6qwfOgjHthys33EY1bcdXXwXj7i
MStO9RHvfRY3W6rnsONkH/Le5/FdfMh7
n8h38SHveya/3OU7h1Hn/bz3Panv7/Pe
99C+v8/7xvFtdO/WpnN9xjeOcN+xHef6
jG8e475jS872Kd88yn3HtpztU76nOPcd
m3XiD/yeIt13bOH38YGfOaPsp4/F+QvC
KJ+by4U7a5rwbqgW55R6SC7Xblcqa38J
3bPdJQ/bH10nRzZIhHUbVnK9dadf5ftv
XFkfLLjsW6K9pczA1KcPYOSoHbbM9YXm
4GKLXHKE+tVfN18uhv7LT395TmtOmPPF
qm3dmqXUUE7ILreKEObaVoBIvIcbNfdQ
4nE+zQwHMgPdPUMCNjSotbuSPj5EqcFt
DIm8O2v6bryLWgtjdOsVNb8QWtJG7T5V
mXWDGf4VTck0OOKYUQdtQxTEOpZbSdHD
YAr9bCnLJTRDGDQTuepppz1ObZJoVsk+
0p2bB4Gmq40hdZQwkp5kBm6+MC6G+2gZ
ComuE67M0J3ItdKmEO9oyKsVuzGiAJNw
yXT1rF2GdqcdewtMwOg4HVS8HVTWOndI
TzqcUTNO9cQYSPY+Nmc+eIbrzzR4XLXO
thtbta5a+byCmmnV+tWSwp097JcA05t2
qUs1V9gQe1smFVzQmS569KpT6dJx4mlZ
02Wm/JorfN42KWIG17352Br6Jz9oVXA9
fAY9X/aEPRY/dT4h/Ia3iGEU19M6HInW
QzUak4S89fFeBe3jpiehk1AbyE+LoN1t
W1d/4m+h6d1M+2U8ddpetdQemg5tqsgj
hyc6n7X9Ums0V7hqAerxnqLcmi/dR9k4
2vTLjIjaFhkHZ/MpOjK3EPVextyHqSXA
WlzoutVyl7wOjLEhUyuMPnSPiNVrspjN
iKo1mXYBahfryy1Uvafn/WIiKnvFZvPF
3ct+a/tmCDjcUirTnL6fgdXqkX4rldn5
WGfNRGWt50mHkm/g6Ge7xB6ckqJ/GAJt
sx2hjroGTDrciqRiD9FkPI4jqg9GPi9f
pTA7m97mhIia9KyLXZPNslwh6KOMugNM
O8MmgadLCoh6XpJTxa0ZxQ5qSTGcgFJE
mLQrtLGE2370Wk+IpcUSmjCYQxXO9y0s
fZBZ98GqbClH0n014QA/wl/OKsS7iJrV
UI9Xj5hALCFaSbG3XCeF1GKkD6OHPira
3cLTB3rgrzXed1bjpZ7/tcj71uZ8nSLv
GVt+rfK+pyrvCUt+LfO+vzLvSSf7tc77
vuu8d8bQr4XeNyz03hdTv1Z6H1vpvTuu
fi31PrrUe2ds/VrrfbNa771T06/F3rco
9t4ZUL9We9+o2nt3aP1a7n2Lcu+WWWVI
MyM5/idWbV6/fsEorKMod6doCEJNeZFN
seZCNaIjU6Vj1UdVCbkgbcqEH/alnagq
wbQjfI46q/qvmkBQDspyfNJvZz3FhsbS
uzOrVtrOaV7YxkyJW24wKXNoY4mvKRy/
+/nn73/59re//R+/P2HBi6XazGEEczXi
FZPi3vwx5tU5kr9GIbvpfB6njybBdSnL
iHnxWHLknQJW1Oap4PcENoSxGq8SBqeq
1gwjyrugYN2Noy33uxhReZ/CV6OcLvpW
20akEOiDsF67DGH1hTvT42qDDpAeTnHw
ceCEhkrbSU/nFTon06XOXU6Kj61st9YZ
/KsxtT95rQufSuuKs0VdG06hRFTi7nQQ
L5ZqUyJdBnaKbnzR3+BG4ZvlhsT30J0C
4RBd7MchUtGTq1GIiEZ6sN2s51KMW3pI
nEIqR+/fhtRV9Oh68R9fYKVUUEQ5Tbjv
/H2MuAMPow4J/G1ef4dCQlxyCZ3e0kKQ
+7DE6vBBhBQuyCs3Y5uy6tiMgGhCdc7K
8m5IMWzi4cvawaMQkXtNeBllb9nr+RzK
jRZvZXRY+Y7nlY70Q0llJdpzfVI8JJPi
/NX8/PMmGr6+CffgYZSH046SK01IVixw
mGIWVFJY6DVwt3c4LFWmpafQSfeuF7u9
nxANc8CRwuBthzH4W2h4h2O4Aw+R1Oho
Igf+7nptkoSQqMwKw/SST/BRkxompVZW
Ssqx1EkBUS7KrsVasci02y3BJiDewY57
UsRceyiwUiuViOljjqgjKCs7PapL7kyO
6CtOIXhceltO43yYaC1BQhZbH28f4woU
P5v8Pyxj1EKnBgrWAhntcpyUYGS0kLkI
s1zjaMKY2A6ZnrFWm3XWzAiQMpyVobpt
+JBMuOAKIu9u0F3Zo/ZZERqgWR2or37T
nsj8MxXX2GP34RxpvMvVO2roWYc+TwqX
oTm7vbTkV+5rEy7vf0D3ZJI6xlpufGFO
0S0HFIXxwm1Z0U5QTnkik0Tu3Mn555gR
4JkUOPtIQPTSP75cAee9Lao1owzeuv5e
rXWbRl6hrKO3tofcuaFeOr5M2EEb0tq9
cj7WIrQQ0utj64z6rk1VdPjtyRQVFeUo
vHdOYQKnm/1HkRU/+uIS7i9lJIC88rd6
3VS7GPG//a9/OmHEi6XaMGLTAtNN63sq
Jec1r9ReiwmUr8oK20GBFetVpAjfIjo5
8vR2X8mtdA3Oy1UoiMh+BgsqVlwtqDX5
EJ/sbqjA395jvG6+fA3rKbSqyBsS/qRt
0n+osX10XIXzQEvHtk5l0e5CeCHLAsd0
jjjJxDdohco/J29lHW0RuQZnmaZCrTSB
Q63W5iOnZHWdqpg+PHV6CORbUYms8T6H
73KltrvwHCKNegas7Ue7QLfOkYaGI1qe
x4o6RjDf9dmDdg+iAUHblcdqKO7Q0ZkS
ugcTGDDWvkqN6Zf9wzfpKduGj9qcQsub
ddVzvnMHAHrqOMqGdEBSqm6NaTwGR8DG
BS37iRCVrRDppU4KUn2ok+JfKjaCoH1u
7SqK5m9D4GY14K4guAyPyNTkeKjWKBhd
LElkWpucqXxFQWLicF1HaJrJFGukPazY
Rd2EQOitl8zV4Uy5+96Ewte34g4wZGhH
B67VPMTi8odvAt0cuPkSGy0Y7VzGKL/p
kVKt9F/FOeFQofoHEzGya3/EOLfh8PVN
uAcQY1MemH2WO8jJstpxehzKYlHelmwx
+DMFVvmaTGTTE/riwc8JikhYfuJTk4LS
bVy8g0PdkxpqHU1Qserfl7reG/fqUXZx
Jg5vSnFHkVHf2vSsXO2CR5semRAZKVGv
Da7LjdA1LO7J9e+dJuZesqBQVlU2V9zH
NJGD1NFCCzLI8VMZ+jjbIVBHyNaTNCE8
otC8ClIpnrgGx7vbcgdOorhZCoLUOj06
wx/SU9MXjjSdFmJthlMwqd2UrW1LEEys
MydO0hOxmhJJ0WuUvLstd2WQyjZaaUxd
BjoARjwr8+pUo2Cts9PjGbjUoyXfevR6
imxNEROipR74Wa06bqaQ9zamzoQcKF1O
LSB9u60T9/+3d3Y7chxHFn4VPQA1yP+f
S3vXBgx4vRcW9lbQChKWWEECLMnP7/NF
Vs2Q09VksaqrZ8oY2dAMRbKnKiMzTkRG
xDk0xYXOfaeCFB3Pyc9Sb27V+yZY1SNv
r0rS1hJ8VTRVeomyYh2PluiEzKU1WmNP
4GcVvM24qW/b8Go1eI8qdKHweq3r8b/+
9M0f9hjx45VaFsml4ZsmGlk7Pg33OKgn
gmBA2J52iP1Vgq+gbIcWymQj83pffaS8
bsGR93oGE8ZqhATjVsc6dfQurKxTnkYG
d7XpcacJnYAZ2XPfquKN5YGP5Ap1GA83
iOKduTVDRxKH0TMVj7p9QCsyQ+Rbht2g
ZZdGk0flOtdmNskwz1CrUp5mUwE+JOvZ
iRb8FCGFJdxBsXk95iBeLNZijwdhZWBT
hVjn8hTxpDdZ4owOa983jNX0HJRaZcip
/Ri+DagOiMSSP0MJOXEMpmGsZG1X8lqe
dkCftdddumQJuY0vXQGIziNEHYR9jnQz
ppm8x+svY/nSGMTK23pXp350+WVFv4IO
KsdjTueEmJgglJhn6qxqvIyJSxcCx6Ni
5wa7ZZ2I5uVZJ6IXx8iua/ouxI268aNO
7Jjn6L0QPnXjGjgjJlbrkpua5RTMx6uo
eIARV+AildxEJlCt2Dh1sOYAjZaSPcSH
IdnZfhRzCx3Y9VY5OSkqVvd0qSO/cg0S
b2/C9aDI4Kp+R2dktmGKSjW17E1ecEd0
yoWCYqPE7Dn+4LSYKE8yO9Niw1WLmHiE
L12Bip7LOtxpVg7QlNLNOQazcS22AKld
39p5U2j01BsmjrMOdrfboFNiYh83Fn2M
JutXZREWVyT/d8gbqTNy1dIaiaKfRx5D
h6CwVBKRsj1YZcpRfkgfJzfuR0PvGTES
YiK7HLOUo8TlzPF4i66Ay1b0Z5occacf
c65leSFcYwyrVcgitjNoRR11+foQ9N7e
TQ2P54PLkO3Ra7XRsY8ItHKLS+h5uHHX
AWnUb8Nzp88IrYTZug6I8EnHmMu8Pdll
KdZ/VQiP+2mRVHj05IL1q74EpsefVy6Z
YsFhOHBt2QXT65EY3kikFBN7m6CYBdcb
ROOx2H5g5QyCPjllPYA8/fAjCqq52W2d
3pMTOOA+XbeOFK6PLuVYUiHoS02J3CV5
4WTBv//5mz0e9+OVWp44lzduJA9K5FNK
8x2CUQAXiFp6o0d+c6IZFTgHWO24RXJh
jCtXHwmf5dajds4JTJhtNsvnYcNs6Wa3
NlI5r6KdH+sxNoSPStkjc1mKRJYPodJ3
rrgzbAbUDx976pK8nwLeGppCpW1Z5kiA
Oq/YooVcI0MpCr305qlViiLtBG7VR7t6
ZTYGU0a7evUu58r/m3xWuNrXuvMcPlur
xcgnVZuIityqhTnJjISe2mc10udTtrMG
BBqSS6UAmfRZZXhS5UIFioIKppyixkw1
z0DRhiSnL/DndrqcSi69XmUN2GnENXAo
T4pXU3iDz6NsOu7PsW2BmznAw7aDNkAB
Ha1Y2q9E9SPNPh8gRmdjDXkMkU5Up4uQ
uHhdcDgoxg63Wqo6uMxd9enq1TOw7H1U
hq84bDt1QNRRLPoJOXsmQ/05IXGsyjQ/
cQ0Ob2+/NYCIiYLcGlfl0T9ysASGT6lB
CRNb5LJ4+71rUxpLU1/mwnJQQZ0QE0Oy
3TcdRP0qXMPEAw7iClTsvgZZuuQUG710
kx1TTWwCJQU41s2jVvRdyd10yNZdmcqR
JwRFObMPrOibRaeLoHiAFVdliTpu8qaK
QpqC1DEgT0xSoOH02mOulrDjXkcxXRzA
iyZCduWksGjs5wrmw8gX4xIqrsn7j88a
eSA9jT4rB4hTohxzgbwhyKQ6TlM/5NZo
ld5WhikrNwgtpHMiZLXCuDZ2ch/8MyqV
Y2T5OWQebtw16Nlyj/J/2oYtUJwcPlcP
GkuAxSxDybM9m4yw5xf2SAxMwZ8WOo3W
lllug85elqDz+NO6BkWDMr+U6NFK/rG/
jmnlJADUb8N93bfHs0HYrA9xOpfKLAdZ
2hlxNDZTjZlMGgdDxAWOHm5SWK9yU2yj
HKQtDxcw2BMK2aSCotJmtk+7JEgpOIgi
OeHbx0V8VLITUHNSlKWM01ywvABhklA2
oFFzBvoWn7KVpHG3NNTwfU6wCQuukncN
Rqrl+4K//c9/7rg+v1ir5bJ0Bt+1rxjA
8zM5ViRxkaOkflnyRpK6aUozNircmR5p
fTvMGE2mRnbNSkLPgKS5W4oJaZ8dSWe1
I/kv3xxd/Fqq1I8yJIzNRJj0YfUr4jDE
nxVCbMS3GEEaq68N4JiD7b0K7uoOpjqZ
i7iqcdpbHF3odMXK+SbuJbTFTuBh/eBE
GKfRJpzlaz1U/NSFtXLHWPDZOi2exEKQ
Zpx0FJ4nYgFtrJhaUoLTsWXfeOkzNUMT
PCn0aolCiEGMg2Qn8cQW4J8BJJM3fJ+O
on5lPosBJsXlysblUF1YkMm7jSFXIGMD
+uCoT/ABhpk0shYvb5wD3VY17NAK0R5B
zYsCZzGJjTPCotI0c6jBjqJ+Fa4C49Kd
wR2gMWXl8UJC55Vr1ubn80idRHln4K7d
7aAXMMaQhFNApMTbcpwRGXUi5jmtmsp1
WDzAjGuAMcuOJLfaYXCPzwEOImoMrlNk
9nummptvqMYxiSsENoqBE+Ji5o5jsiLc
UVeA8QAbroBGGiRd7z5Sd6rzbA+CAjbZ
rKha/nRb186s9aMkyyfy1BJaPyswKsie
G2DboAVbRsUjrLgCFyNhh7YUZktPGWP1
mXtThphb3jGh5ZXkRzSji1f8lAbf5wmR
0YdmOYa3MWw/9uMlMq64BbgDSAYm7Hqr
8FUxVDC3DVAbpm3VMw0N/+uOBDJr4wRv
XPWhp3hOmCzWf/6gL3n+soiUx5t1XTZZ
YRiEOcIIsucEgrwhwaNN+bntCGGNwlVx
Fe1BufqzZpPd8F4vMPS6rRh9AZzHm3RN
epmtM4uSBvzZcWbpSXAtK10JmaboPTc9
3gY2FVRRJ63RpZPCqNxvNaMab5Z+1ZaR
9GizGoGcC7X70gr3qcuo6uihh2qWSYQS
JwXvzgHTBkx6/rhRJmRS8I76NMLlRouf
9T55mdJXomltqTM4XyM4nucsq3mzznhG
adaGHyMM24vXBd/8/a97LPjxQi13+vQc
lYYwid1p6/x66gvogl0XggBOPnIHfCoe
Y+IBphF5JjcU7TokvVCJd5oOXr8Btfnm
0La00U2Dd6WfJnhBZw3HGNAFBgZqQZe5
XhmwDIG5ZoH347i6jpz8LSwBKaEhuv30
CXCJcXrOmfBnEIfBaBDhtIUqJp/CpxYY
GWb7NTuAMp88nF4lINiYrl287rTf86Va
5CAUaFXfKtmDfNx80cOh9ZFR9hZJMrdX
Jx3ol7UV9ASKqPLYWVExlqsMz2r7nMCG
6Yn9Iw32j4x38R7Ss+CtLfsIH7oGBRV6
waAcU4Hqrj0yhAr/MG9AYGfPlauzVjua
oz0TQu2cMGhSRRCOjZl1JR9XkXDpiuBw
LEwJEnP9bmxQos00PNQgi9Y9CCTli3dg
obV7dOPm6kg3lVOCofKxGQxr+gQY3t6G
a+AQ6xF0EJDGPnp0XawIMhcOsIfrdXvv
TtZboruVOZDB8q4zAuIQAUWa6p3F69ch
8QArrgFFBjdTogkDKZdZZCDYREhWBkSC
0XeISBbP/5oNNrtsafQJMVFhw8w4qG/9
NUw8wJuuyg0j7ZGcSGQhHrWyWfdSOD2e
Z910FuuD7AbLT+wFCpc4EQ6cDxR79TP/
Z+ceZwkRV6T6x4Mj7GtJHsNGN+rc4RG7
74pea2WeLO3Q4olIHFqnnKzZh+zH+bCR
4bWZZbCObPoCGw+35hqY7Fn7DpGAypz8
FLDSxFOSrEmg0vz2FldfU7D6Zg5lXMuf
EyaZWXvkdKl+ESSPN+eqJBLlD6SxIXqE
zmpKImUGNCGaDOr2jE8yDq0gKrVIE+1Z
U0guEGfaOmuqv4DLrdb0ebKf/evJnO6Z
o4UQWAFk99Z/cUXhJQGXI8yBAXSKfRAS
pKQFt4tSy02HMz4UZzPv+uiOsxJulvnB
oESDZpLRhhP4Wr2ADYh00zCavpgqtS1y
yWgcFX850vzTd7/+Ft2vP/zK8/26x5Qf
rdiCLeUoqGPFLNyscrqznn0tXeEP+szI
RMXtneeKwhg/6vLnlrvagzFUpFeHmDu4
dgJL2oSi6WSNOpZd9ITADCqRQJVJFxrq
bmTGGKNMpqAjIH24aEX9/BT9uA6tpUxM
hJHyVcqJsWa6Xrezn1N5bdkrlhpseDjX
2iKOwPcoKNVzvH7vGmx4gDZgvnDX4x9i
hIy8GgmK3ubyiu42RrxYreVegYr2Krzn
CF/HSXsQUKCz2EhylHNuv99JQHkBaOUa
mvFpJhp+MwrNzvzuOeh3BvXZGNGi3vju
axRC9VqFC7Asr5ouyT5u51U/B5DMxhVr
wKiOCm+b26gUkMCthCJ9gCJ583WrfGiO
dPiiMTqkQs6IkESC5lcVoz3jyXI9fAIt
P74wuB9eIpfsGkTovTAlN8+D8NcCUqaQ
PtSojbmdC13vCxsPE5YT8cIpEbO7Zcvq
d/w19DzErJ/HT4S2a1H20IwfTU7kXXiI
JgReUlNgFPQHLRDdzmuXW/NFWYjsG2M4
KYI2G/JhXIm30nG5hqBHGHIdhnKNEej/
V8Zk7f8T5SQbMbXiu1DU+x33QLR9UUPz
DS8QQj4pisK9YRUSY0+xL9dQ9Chv+9lE
U7ZiyLP61L1s96g7EXCypdQoT7mnapk8
g7UQ+7oscK5nzTRlqT5fs9vEyxJufvLm
4I4pp4ciFAUf5YRdudRjj3mgFyT2oMAm
7pMwoL8LPUNm8aaL2tMhaNKzPrYT9CXE
vIdBV4AnlUsfeLJEV3Oekk/FpzlW/Qcl
XDHsgE6FVV07QptKOO19OSl2DrJQKIt5
rWImvcDOO5h0HYzSlqpU0VcEDx8rY/R4
FDlKU4Db2As7qptFQB3tmrAPdYozQmiJ
fb5917d+ET/3WDS1YcGo4/WuzfaESucZ
t6+MOn5eZurnstj5bnsjekPkPucea0Us
0X4cBa/ucfR0RV/60m/+8fuHdhr/CJo6
vDCMJCjI7kQfI0P/239/pQf5y5//8h9/
+ONf//Tu+//7xy8//6IPff/9dz99+78/
/fL9/3/r373/+dfff/zx/ffvZcmvfnz/
8/vffvhqXrjN6/bsRRaWbXOzd4JsubVQ
a4xcotrPY6xAB6hV64FfYDxaXLmvrZah
LRUqXQWEQ/Xl186EnxVst8DU7iU0bN9z
Jg5ekeVAC2eQKyfo4Hwylk068S+9w+LS
+Qeoc/V5Dnryjn7lSy/cxZvcZtcN0B2X
9jFWinGDZSsR2zIYyPW+9ysXznTpIbLU
R3mkikN68ZU70M15xHay0UbKp1d3ez9n
OdS/pafzKBbrsAb2c+/1CE/3wqt3nK+j
PpMRVYyesZl0e1/3skt3kLebKD7hCY2x
mXiMN962G7u7F953p43rLmPef0vXd3yQ
95oW8rwR3ytaxTOHf3dYRrz408ox9/zB
sn50j5GYzKVTNzCce2Wo19Pc53P1nba4
D5gxUDgNEcgNiC9tH0Cjddg7BntZ4WzV
AD0Wclc9Nq3fKS6Qo7fdUoIfunDObmW5
L4UeDf2R0iGvuaip//DdP3+YDf3tL7//
dnkrtd6gz1ZtwaDBVdm7FYhmUxk9uCC/
aynX6LhG6wx/bD1BzeN+KnVdE7jisbiQ
CtyEVaFIOYU1k6kvQ/nJW+mgyCsC0ZGa
JgUsXw82JQBWawqUsNPiFSNchxToquCk
9ToP9lKFYVhUkFr8LsrpXARTAi0T36zJ
ml2ivhfItEYHo1zwCe4Yuz15aPlDhmIb
B3EdXim4i9nzqcNaV46168X6LbdIdB8J
g+R5Azf/GFaORaZNCOVCGLaHpq/UFItn
aJQKks0JJSNCFdC1xsDxGe6O05BInzwu
YSTXx5m6JnMnmcv3BbWbm57SFQgaFD3H
Uii+UEV8YjpxJUH7hmqKMpSNspyjdQaV
1qDPVhwd/GjFPyGCVlsxtCctenLtOoI+
q6ffF0MdNLZEPfKyBZHwcqXvaHMrPqU/
+X5BgBxwD2dFUTf1CExtH9cw9EBjfh5F
nVLNoKhI8Uq1yvd0PLXU6HGYDIpO1vZ5
Uk/5L3E0a22Dh+iMKEqC+XQ6sxG/L0Pn
ceZcB54FJQ6oBLhcUB47C6o0YV7vDknY
ukNQJQTlKLKqnsDEc88JnjoYxhMehp5K
Inq8Ap4HHs+3BPQlEtBPldPfstFXaNqV
2ei97PqWmr5ganonI7/lqffPU3eZdu3t
eaMX0VXlia7KOd+oBDGFWLmVrB/rUNYq
Qwe5oeZZY6oexZaFttFrtZySinUUBLxJ
+aK2k3BI3eH5m9yw7KCkMSNwRFCpt47D
UD2jJhdgUVKwtraWmCrnRpCb5TmeGLNe
bNlMGZu+U0zpFpdta/M5axQD5BR0oVc/
fhzjMQG9StegLVlbrkGvIiEhrJCy4mZe
wco9e5Xb7ThnPHStBjrJo6JfE3moCJ+R
CstxQ3y0dukgja5ZAAVctNy/pC/xoLN6
nJezqiijH4Xo1od4ey+3tmfibH5O1ugK
aLg90zYxoaqburmXXbdjHN3Yc9olcYiz
xRiHOs2tPd1LL95hvs5npQqxQpWdIKn2
t3d1L3xgTxzSbWqIOJvfOzi+e0VreOZg
71Ut44kjvzus4xfcYwSd6cQFWTMRiMVy
ns599Cl3Z5IKIc1MlKX5VLh17i7mPSUg
zw1356cUjzq4PZmS/kyBImc0L85wDVnb
E8e9KRgEqNJ9QBegtmaDs5+otoe9d1LP
VmzxQlnBZdEz5cx/6tMNf0WMWX7dm8ZN
2D4ZW1A1iS5R1YMmzUoo3Fxn71uOAssz
DMZGIfpkyJhNtyDDYhURnZBfRCfuSDOG
TDskpI9Bf2SRWNTLERF4lBJ8LVMFmfIM
Isz0BCvWidulRmt3LWiPFNA4D04tvTe+
z+UC+90ZtGGS1dcRnBvURFQFQoNZTsvq
E695qCGfr9iibw2V8MaZVrZCyDzdEZsS
RWe2GSqfHYzbBdIBpuXHwwwyCVCzmXwl
w8ELcPb6bNnj46w6lDj+IaHH2pBvhIQj
frot7QZn8rMwednIFBTFEkPJJ2e4emsw
jt7NIR7M2wJhGNJw3/2cMElgSaeEJUuD
MW0ZJ68X1u+AlAtNTAMrZQIFcsxl9IpK
144RdZih9fFU+dNgGjsfVlat4iMVvnGL
LmHlgaZcgZZwIVQlDUkIKX/r8nLj0fZi
ayn6fB3zQp9hPSdahsGGNpGimSUX0fI4
U67ByyvdS142iIHZULQJaNrZcio9lXnE
oJRZKvcrvs+c6ucDTHho5nPJiNI1xDz0
ZL6llndNLVcWyN/yzFdi1TV55r1s+pZ0
3jPpvJNV3zLQO2egu+ya02TH7GXRx9tv
+fiPrJq5mw8kDzH2eKuR/UkKXGlqRMug
yX+nMXCS6S/rtfvI8Hjp6+tZSo+U/Xao
oGH7Kl/CKRS/6MJ7/dJ9/Co3qxt4Y5Fs
vXrSQ2cE6RC7+iLYs1F9v3bhwgMzTzmH
rCiUr69g1ZQoydcwa9NuOJUvRwbne9XZ
aVAaG2ktos09wZQbktBoZaklPDiyLzk5
Gt976S+9aBfvcTtWCOU3ChxikPtpOGgz
UdbRRTejw/i3AJTLZzQ86IhSTfNwEKf2
4st2jHNzVxjNb+3a1vY5nMy56QOUDSu/
dlkflQcZ/C2d20uv2zHubfFi6Jbu7UWX
7SAHd2WE7LYO7qX324njt03dCyfzdwcH
c69qCU8Z2b2eFTxvmHeHNezlg1VjLvqD
Jf2on6tUOfsWawo6dXlZ3o9ZOm3VAvNU
Q/B01veL+ivo8AX0HNuO8iqVW+jWqBZB
sBzHoxkNYIzNI0V3Cj1V79qsp+qtSuyE
poP9AhFT2PM/WS2Pi9cTX2DNZ0t2rVqe
anWFweLe2lzIKa343JLwP8Hfv+3eaZhT
0UMk4GeitRc3zMl0u86TPr8yFfr6rSno
fdRTTaM6l3VMWoRxkJ5m9+kulr3GjHT7
6mcVuZC2XCrPhTv4oJVGg2ZWqnZo5bIB
Ak+qSGf70ZR3RbO8FCQ4GUi3J0OXXtGO
QLnVpZ7MV2fKasMwoRmngGxnhBE1Bvlm
5Vcw4qdPdwjuteWzFVs8l3UovHeHNGOO
s/yU7KejxElFd2wHeYurDSz3Vsft3foe
9BPloZSdJ3nxUM5gy+Ta7GT1rY0p+yp/
lgn0eK+yoPR3Sx/7ecS8IhJ3OaC1o+c8
aPsmj4JnE8DkkyJmyn5uYklj/OAKYl6v
lt8HM5/Nh00zKKHrVFfFQeg7bmODmPgH
kIbskZ+sc5iN0eaEkCmceFTNKDZGtwyZ
x1lzBWheKsLNjjYy2cBiZ7SZt4MmYbF+
jNLVTJdAPidoykPhXVu2RkE7nIuYeZwx
16AmqW7zrbsos7k2C1Y7pRNKJJocpTK5
bZqN8QG2s4AuvXI+BUJp6ko6IWq22mfU
1LfXUfNAP/uWad4901xZFX9LO1+Radel
nXey7FsOeu8c9E6GfUtI75yQfpld/wVQ
SwECFAAUAAAACAAAAC5d4Eo9hXsPAAAb
OwAAKgAAAAAAAAAAAAAAgAEAAAAAYXJ0
aWZhY3RzL3JwNF92NF9iNC9wcmltYXJ5
X3N0YXRpc3RpY3MuY3N2UEsBAhQAFAAA
AAgAAAAuXR6b1KH2ywAAmdIBACsAAAAA
AAAAAAAAAIABww8AAGFydGlmYWN0cy9y
cDRfdjRfYjJfcnYxNS9zZXNzaW9uX2xv
c3Nlcy5jc3ZQSwECFAAUAAAACAAAAC5d
R49h1UABAACTBQAAIAAAAAAAAAAAAAAA
gAEC3AAAYXJ0aWZhY3RzL3JwNF92NF9i
NC9jb3ZlcmFnZS5jc3ZQSwECFAAUAAAA
CAAAAC5d93ATipwWAACtWAAAIgAAAAAA
AAAAAAAAgAGA3QAAYXJ0aWZhY3RzL3Jw
NF92NF9iNC9yb2J1c3RuZXNzLmNzdlBL
AQIUABQAAAAIAAAALl3NChvlngEAAFsE
AAA2AAAAAAAAAAAAAACAAVz0AABhcnRp
ZmFjdHMvcnA0X3JvYnVzdG5lc3NfcHVi
bGljX3YxL3JlZmVyZW5jZV9xbGlrZS5j
c3ZQSwECFAAUAAAACAAAAC5dTx1VOTkH
AAD7MQAAOgAAAAAAAAAAAAAAgAFO9gAA
YXJ0aWZhY3RzL3JwNF9yb2J1c3RuZXNz
X3B1YmxpY192MS9yZWZlcmVuY2VfY29u
dHJhc3RzLmNzdlBLAQIUABQAAAAIAAAA
Ll0d7v4R0QsAAG1RAAA4AAAAAAAAAAAA
AACAAd/9AABhcnRpZmFjdHMvcnA0X3Jv
YnVzdG5lc3NfcHVibGljX3YxL3NlY29u
ZGFyeV9tZXRyaWNzLmNzdlBLAQIUABQA
AAAIAAAALl0kXYA9pAIAAKAIAAA3AAAA
AAAAAAAAAACAAQYKAQBhcnRpZmFjdHMv
cnA0X3JvYnVzdG5lc3NfcHVibGljX3Yx
L3BpdF82MF9jb250cmFzdHMuY3N2UEsB
AhQAFAAAAAgAAAAuXeOurferAgAAqggA
ADgAAAAAAAAAAAAAAIAB/wwBAGFydGlm
YWN0cy9ycDRfcm9idXN0bmVzc19wdWJs
aWNfdjEvcGl0XzMwMF9jb250cmFzdHMu
Y3N2UEsBAhQAFAAAAAgAAAAuXaP6aDgt
BgAAlS4AAEgAAAAAAAAAAAAAAIABABAB
AGFydGlmYWN0cy9ycDRfcm9idXN0bmVz
c19wdWJsaWNfdjEvcGxhY2Vib19sb2df
cmlkZ2VfaGFycV9ydjE1X2RyYXdzLmNz
dlBLAQIUABQAAAAIAAAALl0LRvg5KAYA
AJUuAABIAAAAAAAAAAAAAACAAZMWAQBh
cnRpZmFjdHMvcnA0X3JvYnVzdG5lc3Nf
cHVibGljX3YxL3BsYWNlYm9fbG9nX3Jp
ZGdlX2hhcnFfcnYzMF9kcmF3cy5jc3ZQ
SwECFAAUAAAACAAAAC5dSacvRhkJAAAk
LQAAMwAAAAAAAAAAAAAAgAEhHQEAYXJ0
aWZhY3RzL3JwNF92NF9iNC92b2xhdGls
aXR5X3JlZ2ltZV9zZWNvbmRhcnkuY3N2
UEsBAhQAFAAAAAgAAAAuXS4ZaCnyIAEA
K6MZACUAAAAAAAAAAAAAAIABiyYBAGFy
dGlmYWN0cy9ycDRfdjRfYjJfcnYxNS9z
dW1tYXJ5Lmpzb25QSwECFAAUAAAACAAA
AC5dwDRtcVnrAAAWkAcAKAAAAAAAAAAA
AAAAgAHARwIAYXJ0aWZhY3RzL3JwNF92
NF9iNC9yZWdpbWVfc2Vjb25kYXJ5LmNz
dlBLBQYAAAAADgAOAEIFAABfMwMAAAA=
'''.split())  # wrapped at 32 characters; joined before decoding
# Validate the encoding and the complete archive before reading any input.
raw_bundle = base64.b64decode(BUNDLE_BASE64, validate=True)
assert hashlib.sha256(raw_bundle).hexdigest() == BUNDLE_SHA256
public_archive = zipfile.ZipFile(io.BytesIO(raw_bundle))
assert len(public_archive.namelist()) == len(set(public_archive.namelist())) == 14
assert set(public_archive.namelist()) == set(PUBLIC_HASHES)
# Every read checks the file fingerprint, including subsequent reads.
def fetch_bytes(path):
    raw = public_archive.read(path)
    assert hashlib.sha256(raw).hexdigest() == PUBLIC_HASHES[path], path
    return raw
for path in PUBLIC_HASHES:
    fetch_bytes(path)
manifest = pd.DataFrame([{'File':p.rsplit('/',1)[-1], 'Integrity':'Verified'} for p in PUBLIC_HASHES])
display(HTML('<h3>Inputs ready: 13 CSV files and one JSON file verified</h3>'))
display(manifest.style.hide(axis='index'))
hash_detail=''.join('<p><b>'+html.escape(p)+'</b><br><code>'+sha+'</code></p>' for p,sha in PUBLIC_HASHES.items())
display(HTML('<details><summary>Input fingerprints and source reference</summary><p>'+PUBLIC_SOURCE_COMMIT+'</p>'+hash_detail+'</details>'))
# Reject an empty table rather than continuing with missing results.
def read_public_csv(path):
    frame = pd.read_csv(io.BytesIO(fetch_bytes(path)))
    assert not frame.empty, path
    return frame
statistics = read_public_csv('artifacts/rp4_v4_b4/primary_statistics.csv')
losses = read_public_csv('artifacts/rp4_v4_b2_rv15/session_losses.csv')
summary = json.loads(fetch_bytes('artifacts/rp4_v4_b2_rv15/summary.json'))
# Labels are separate from the source identifiers used in every calculation.
families = {'log_ridge_harq':'Linear','lightgbm_qlike':'Trees'}

# Record that this step completed.
EXECUTED_CELLS.append('step_01_verify_inputs')


File,Integrity
primary_statistics.csv,Verified
session_losses.csv,Verified
coverage.csv,Verified
robustness.csv,Verified
reference_qlike.csv,Verified
reference_contrasts.csv,Verified
secondary_metrics.csv,Verified
pit_60_contrasts.csv,Verified
pit_300_contrasts.csv,Verified
placebo_log_ridge_harq_rv15_draws.csv,Verified


### Reading

These are the aggregate inputs described in Section 3.10 and used in Table 7. All input fingerprints must match before the first result is calculated.


## Step 2: The four primary contrasts

Verify the session counts, unique dates and finite losses. The calculation uses the already aggregated session losses; applying QLIKE to averaged targets and forecasts would change the estimand. The original checks require agreement within 1e-12 QLIKE and 1e-10 percentage points.


In [2]:
# @title Step 2: The four primary contrasts
# Use math.fsum for accurate summation and Matplotlib for the figures.
import math

import matplotlib
import matplotlib.pyplot as plt


# Compare paired session losses with one published contrast.
def compare_saved(frame, row):
    """Reconcile one published contrast with paired session losses."""
    # Split the expanded and baseline information sets within each model family.
    expanded, baseline = row.contrast.split("_over_")
    base = frame[f"loss__{row.family}__{baseline}"]
    extra = frame[f"loss__{row.family}__{expanded}"]
    # Reject nonfinite losses and duplicate dates before taking means.
    if not all(math.isfinite(x) for x in [*base, *extra]):
        raise ValueError("Nonfinite losses.")
    if frame.session_date.nunique() != len(frame):
        raise ValueError("Duplicate sessions.")
    # Weight sessions equally; a positive difference favors the expanded model.
    mean_base = math.fsum(base) / len(base)
    delta = math.fsum(base - extra) / len(base)
    # Express the difference as a percentage of the mean baseline loss.
    percent = 100 * delta / mean_base
    # Check the count, mean, difference and percentage against explicit tolerances.
    assert len(frame) == row.N_sessions
    assert abs(mean_base - row.baseline_loss) <= 1e-12
    assert abs(delta - row.estimate) <= 1e-12
    assert abs(percent - row.qlike_reduction_percent) <= 1e-10
    # Return the computed values and their differences from the published values.
    return {"family": row.family, "contrast": row.contrast,
            "sessions": len(frame), "delta_QLIKE": delta,
            "reduction_percent": percent, "delta_error": delta - row.estimate,
            "check": "PASS: aggregates"}


# Select RV15 in the development window.
selected = statistics.loc[
    (statistics.horizon_minutes == 15) & (statistics.window == "primary")
]
# Recompute both contrasts in both model families.
checks = pd.DataFrame(compare_saved(losses, row)
                      for row in selected.itertuples())
assert len(checks) == 4
print("Four RV15 contrasts: arithmetic checks passed")
display(checks)

# Sort session dates before drawing the cumulative series.
ordered = losses.sort_values("session_date")
dates = pd.to_datetime(ordered.session_date)
# Plot paired losses and cumulative QLIKE differences in separate panels.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")
for family, label, color in [
    ("log_ridge_harq", "Linear", "#0072B2"),
    ("lightgbm_qlike", "Trees", "#D55E00"),
]:
    # Use the same dates for B1 and B2; each point is one session.
    base = ordered[f"loss__{family}__B1"]
    extra = ordered[f"loss__{family}__B2"]
    axes[0].scatter(base, extra, s=18, alpha=0.5, color=color, label=label)
    axes[1].plot(dates, (base - extra).cumsum(), color=color, label=label)
# Draw the equal-loss diagonal; points below it favor B2.
limit = max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])
axes[0].plot([0, limit], [0, limit], "k--", linewidth=1, label="Equal loss")
axes[0].set(xlabel="B1: session QLIKE", ylabel="B2: session QLIKE",
            title="419 sessions: points below the diagonal favor B2",
            xlim=(0, limit), ylim=(0, limit), aspect="equal")
# Draw zero as the boundary between positive and negative cumulative differences.
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(xlabel="Session date", ylabel="Cumulative QLIKE difference (B1 − B2)",
            title="Cumulative difference; positive favors B2")
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.2)
axes[1].tick_params(axis="x", rotation=30)
plt.show()
print("Matplotlib:", matplotlib.__version__)

# Check the reported percentages rounded to three decimal places.
expected_rounded = {('log_ridge_harq','B1_over_B0'):0.880,
                    ('log_ridge_harq','B2_over_B1'):0.623,
                    ('lightgbm_qlike','B1_over_B0'):1.170,
                    ('lightgbm_qlike','B2_over_B1'):-0.115}
# Match each percentage to its model family and contrast.
for row in checks.to_dict('records'):
    assert round(row['reduction_percent'],3) == expected_rounded[(row['family'],row['contrast'])]
print('PASS: 0.880 / 0.623 / 1.170 / −0.115 %; primary_statistics.csv checks passed')

# Record completion after all checks and figures finish.

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_02_the_four_primary_contrasts')


Four RV15 contrasts: arithmetic checks passed


,family,contrast,sessions,delta_QLIKE,reduction_percent,delta_error,check
0,log_ridge_harq,B1_over_B0,419,0.001616,0.880055,2.580401e-17,PASS: aggregates
1,log_ridge_harq,B2_over_B1,419,0.001134,0.622794,5.746272e-17,PASS: aggregates
2,lightgbm_qlike,B1_over_B0,419,0.002197,1.169969,3.989864e-17,PASS: aggregates
3,lightgbm_qlike,B2_over_B1,419,-0.000213,-0.114739,-6.155558e-17,PASS: aggregates


Matplotlib: 3.11.2
PASS: 0.880 / 0.623 / 1.170 / −0.115 %; primary_statistics.csv checks passed


### Reading

Option state reduces loss by 0.880% in Linear and 1.170% in Trees. The mixed flow block adds 0.623% in Linear and −0.115% in Trees, so the increment depends on the family. These are the four contrasts in Table 7; the cumulative panel represents Figure 8.


## Step 3: The final window

Verify the eight RV15 rows spanning both windows, then read their saved intervals and decision states. H2 is tested formally only after H1 rejects. Missing formal H2 probabilities remain missing; the report also gives nominal diagnostics.


In [3]:
# @title Step 3: The final window
# Read the stored inference columns.
inference_columns = ['window','family','contrast','N_sessions','qlike_reduction_percent','ci_low','ci_high','p_for_decision','hypothesis_status']
rv15_inference = statistics.loc[statistics.horizon_minutes.eq(15), inference_columns].copy()
assert len(rv15_inference) == 8
# Leave unavailable formal p-values marked as not tested.
display(rv15_inference.style.format({'qlike_reduction_percent':'{:+.3f}', 'ci_low':'{:+.6f}', 'ci_high':'{:+.6f}', 'p_for_decision':'{:.4f}'}, na_rep='Not tested'))
# Check the H1-to-H2 testing sequence in the final window.
final_rows = rv15_inference[rv15_inference.window.eq('confirmation')]
assert set(final_rows[final_rows.contrast.eq('B1_over_B0')].hypothesis_status) == {'NOT_REJECTED'}
assert set(final_rows[final_rows.contrast.eq('B2_over_B1')].hypothesis_status) == {'NOT_TESTED'}

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_03_the_final_window')


,window,family,contrast,N_sessions,qlike_reduction_percent,ci_low,ci_high,p_for_decision,hypothesis_status
8,primary,log_ridge_harq,B1_over_B0,419,+0.880,+0.000253,+0.003454,0.0390,REJECTED
9,primary,log_ridge_harq,B2_over_B1,419,+0.623,+0.000335,+0.001932,0.0032,REJECTED
10,primary,lightgbm_qlike,B1_over_B0,419,+1.170,+0.000611,+0.004103,0.0135,REJECTED
11,primary,lightgbm_qlike,B2_over_B1,419,-0.115,-0.002044,+0.001104,0.6280,NOT_REJECTED
12,confirmation,log_ridge_harq,B1_over_B0,25,+0.174,-0.002209,+0.002878,0.3908,NOT_REJECTED
13,confirmation,log_ridge_harq,B2_over_B1,25,+1.997,-0.001109,+0.014742,Not tested,NOT_TESTED
14,confirmation,lightgbm_qlike,B1_over_B0,25,+3.163,+0.001644,+0.016807,0.0568,NOT_REJECTED
15,confirmation,lightgbm_qlike,B2_over_B1,25,-0.992,-0.009072,+0.002195,Not tested,NOT_TESTED


### Reading

Neither family rejects H1 in the final window, so H2 remains untested under the registered sequence. The Trees H1 probability is 0.0568. Table 13 reports this result; its positive point estimates do not confirm the full sequence.


## Step 4: Reference models and complementary errors

Verify reference losses and matched contrasts by horizon, reference and sample counts. Each matched difference must equal reference QLIKE minus model QLIKE within 1e-12. Both reference masks are retained.

Read all 12 RV15 development combinations of family, information set and error metric. MAE is mean absolute error in realised variance; RMSE is root mean squared error. The charts multiply both by one million, with separate vertical scales.


In [4]:
# @title Step 4: Reference models and complementary errors
# Read all rows in the public reference tables.
reference_qlike = read_public_csv('artifacts/rp4_robustness_public_v1/reference_qlike.csv')
reference_contrasts = read_public_csv('artifacts/rp4_robustness_public_v1/reference_contrasts.csv')
secondary_metrics = read_public_csv('artifacts/rp4_robustness_public_v1/secondary_metrics.csv')
assert [len(reference_qlike),len(reference_contrasts),len(secondary_metrics)] == [4,24,72]
assert set(reference_contrasts.role) == {'POST_PRIMARY_NOMINAL_ROBUSTNESS'}
assert set(secondary_metrics.role) == {'DESCRIPTIVE_NO_TEST'}
# Match each contrast to its reference horizon and sample.
matched = reference_contrasts.merge(reference_qlike[['horizon','reference','N_origins','N_sessions','estimate']], on=['horizon','reference'], suffixes=('','_reference'), validate='many_to_one')
assert (matched.N_sessions == matched.N_sessions_reference).all()
assert (matched.N_origins == matched.N_origins_reference).all()
assert ((matched.reference_qlike - matched.estimate_reference).abs() < 1e-12).all()
assert ((reference_contrasts.reference_qlike-reference_contrasts.model_qlike-reference_contrasts.estimate).abs() < 1e-12).all()
# Display all reference levels and their coverage.
display(reference_qlike[['horizon','reference','N_origins','N_excluded','N_sessions','estimate','ci_low','ci_high']].style.format({'estimate':'{:.6f}','ci_low':'{:.6f}','ci_high':'{:.6f}'}))
# Use each reference's matched sample; lower bars mean lower QLIKE.
fig, axes = plt.subplots(1,2,figsize=(13,4.8),layout='constrained')
for ax, reference in zip(axes, ['seasonal_persistence','HAR_dwm']):
    part=reference_contrasts[(reference_contrasts.horizon==15)&(reference_contrasts.reference==reference)]
    assert len(part)==6
    labels=['Reference']+[families[r.family]+' '+r.set for r in part.itertuples()]
    values=[part.reference_qlike.iloc[0]]+part.model_qlike.tolist()
    ax.bar(labels,values,color=['#777777']+['#0072B2' if f=='log_ridge_harq' else '#D55E00' for f in part.family])
    ax.set(title=f'{reference}: {int(part.N_sessions.iloc[0])} sessions',ylabel='Mean QLIKE (lower is better)')
    ax.tick_params(axis='x',rotation=55)
    ax.grid(axis='y',alpha=.2)
plt.show()
# Display all 24 reference contrasts, grouped by horizon.
for horizon in sorted(reference_contrasts.horizon.unique()):
    print(f'Reference contrasts, RV{horizon}; difference = reference − model:')
    cols=['reference','family','set','N_sessions','reference_qlike','model_qlike','estimate','ci_low','ci_high','p_raw']
    display(reference_contrasts.loc[reference_contrasts.horizon==horizon,cols].style.format({c:'{:.6f}' for c in ['reference_qlike','model_qlike','estimate','ci_low','ci_high','p_raw']}))

# Read the Trees B1 and B2 values on the seasonal-persistence sample.
seasonal_lgb=reference_contrasts[(reference_contrasts.horizon==15)&(reference_contrasts.reference=='seasonal_persistence')&(reference_contrasts.family=='lightgbm_qlike')].set_index('set')

# Record completion after all checks pass.

# Select all RV15 development-window combinations of family, set and metric.
metric_rows=secondary_metrics[(secondary_metrics.horizon==15)&(secondary_metrics.window=='primary')].copy()
assert len(metric_rows)==12
# Scale variance errors by one million for display.
metric_rows['error × 10⁶']=metric_rows.estimate*1e6
metric_rows['CI low × 10⁶']=metric_rows.ci_low*1e6
metric_rows['CI high × 10⁶']=metric_rows.ci_high*1e6
display(metric_rows[['family','set','metric','N_sessions','error × 10⁶','CI low × 10⁶','CI high × 10⁶']].style.format({c:'{:.4f}' for c in ['error × 10⁶','CI low × 10⁶','CI high × 10⁶']}))
# Plot MAE and RMSE separately because their scales differ.
fig,axes=plt.subplots(1,2,figsize=(12,4),layout='constrained')
for ax,metric in zip(axes,['MAE','RMSE']):
    table=metric_rows[metric_rows.metric==metric].pivot(index='set',columns='family',values='error × 10⁶').rename(columns=families)
    table[['Linear','Trees']].plot.bar(ax=ax,rot=0,color=['#0072B2','#D55E00'])
    ax.set(title=f'RV15: {metric}',ylabel='Variance error × 10⁶ (lower is better)',xlabel='Information set')
    ax.grid(axis='y',alpha=.2)
plt.show()
# Calculate the B2/B1 percentage change for each family and metric.
for family,label in families.items():
    for metric in ['MAE','RMSE']:
        values=metric_rows[(metric_rows.family==family)&(metric_rows.metric==metric)].set_index('set').estimate
        change=100*(values['B1']-values['B2'])/values['B1']
        print(f'{label}, {metric}: B1={values["B1"]:.9g}; B2={values["B2"]:.9g}; B2/B1 descriptive reduction={change:+.3f} %.')

# Record completion after all checks pass.

# Record that this step completed.
EXECUTED_CELLS.append('step_04_reference_models_and_complementary_errors')


,horizon,reference,N_origins,N_excluded,N_sessions,estimate,ci_low,ci_high
0,15,seasonal_persistence,159054,1778,417,0.557917,0.513928,0.605794
1,15,HAR_dwm,160832,0,419,0.408061,0.365068,0.477471
2,30,seasonal_persistence,158959,1873,417,0.406483,0.369537,0.446299
3,30,HAR_dwm,160832,0,419,0.339623,0.301285,0.399460


Reference contrasts, RV15; difference = reference − model:


,reference,family,set,N_sessions,reference_qlike,model_qlike,estimate,ci_low,ci_high,p_raw
0,seasonal_persistence,log_ridge_harq,B0,417,0.557917,0.182054,0.375863,0.338188,0.415956,0.000100
1,seasonal_persistence,log_ridge_harq,B1,417,0.557917,0.180591,0.377326,0.339412,0.417769,0.000100
2,seasonal_persistence,log_ridge_harq,B2,417,0.557917,0.179471,0.378446,0.340552,0.418661,0.000100
3,seasonal_persistence,lightgbm_qlike,B0,417,0.557917,0.184349,0.373568,0.336114,0.413360,0.000100
4,seasonal_persistence,lightgbm_qlike,B1,417,0.557917,0.182467,0.375450,0.337845,0.415463,0.000100
5,seasonal_persistence,lightgbm_qlike,B2,417,0.557917,0.181889,0.376028,0.338149,0.416185,0.000100
6,HAR_dwm,log_ridge_harq,B0,419,0.408061,0.183660,0.224400,0.186418,0.283476,0.000100
7,HAR_dwm,log_ridge_harq,B1,419,0.408061,0.182044,0.226017,0.187504,0.286188,0.000100
8,HAR_dwm,log_ridge_harq,B2,419,0.408061,0.180910,0.227150,0.188542,0.287244,0.000100
9,HAR_dwm,lightgbm_qlike,B0,419,0.408061,0.187750,0.220310,0.185181,0.274063,0.000100


Reference contrasts, RV30; difference = reference − model:


,reference,family,set,N_sessions,reference_qlike,model_qlike,estimate,ci_low,ci_high,p_raw
12,seasonal_persistence,log_ridge_harq,B0,417,0.406483,0.146321,0.260162,0.230430,0.292198,0.000100
13,seasonal_persistence,log_ridge_harq,B1,417,0.406483,0.143900,0.262583,0.232566,0.295027,0.000100
14,seasonal_persistence,log_ridge_harq,B2,417,0.406483,0.143093,0.263390,0.233252,0.295677,0.000100
15,seasonal_persistence,lightgbm_qlike,B0,417,0.406483,0.146775,0.259708,0.229752,0.291660,0.000100
16,seasonal_persistence,lightgbm_qlike,B1,417,0.406483,0.143887,0.262596,0.232787,0.294519,0.000100
17,seasonal_persistence,lightgbm_qlike,B2,417,0.406483,0.143522,0.262961,0.233176,0.294791,0.000100
18,HAR_dwm,log_ridge_harq,B0,419,0.339623,0.147307,0.192316,0.160141,0.240612,0.000100
19,HAR_dwm,log_ridge_harq,B1,419,0.339623,0.144760,0.194863,0.161913,0.244792,0.000100
20,HAR_dwm,log_ridge_harq,B2,419,0.339623,0.143957,0.195666,0.162883,0.245438,0.000100
21,HAR_dwm,lightgbm_qlike,B0,419,0.339623,0.149317,0.190306,0.160043,0.234768,0.000100


,family,set,metric,N_sessions,error × 10⁶,CI low × 10⁶,CI high × 10⁶
0,log_ridge_harq,B0,MAE,419,4.6812,3.5029,6.5898
1,log_ridge_harq,B0,RMSE,419,33.6418,8.5500,56.7060
2,log_ridge_harq,B1,MAE,419,4.7196,3.5023,6.6947
3,log_ridge_harq,B1,RMSE,419,33.5516,8.5570,56.5531
4,log_ridge_harq,B2,MAE,419,4.6911,3.4978,6.6293
5,log_ridge_harq,B2,RMSE,419,33.5190,8.5172,56.5167
6,lightgbm_qlike,B0,MAE,419,4.7374,3.4727,6.8170
7,lightgbm_qlike,B0,RMSE,419,34.7724,8.8382,58.6048
8,lightgbm_qlike,B1,MAE,419,4.7705,3.4666,6.9217
9,lightgbm_qlike,B1,RMSE,419,34.8570,8.8279,58.7401


Linear, MAE: B1=4.71957487e-06; B2=4.69114355e-06; B2/B1 descriptive reduction=+0.602 %.
Linear, RMSE: B1=3.35516459e-05; B2=3.35190183e-05; B2/B1 descriptive reduction=+0.097 %.
Trees, MAE: B1=4.77047347e-06; B2=4.78031957e-06; B2/B1 descriptive reduction=-0.206 %.
Trees, RMSE: B1=3.48570093e-05; B2=3.48762896e-05; B2/B1 descriptive reduction=-0.055 %.


### Reading

HAR d/w/m has mean QLIKE 0.408061 at 15 minutes. Seasonal persistence uses 417 sessions, whereas HAR uses 419; each comparison belongs to its own mask. Table 11 and Figure 12 provide this context for the nested contrasts in Table 7.

Linear B2 slightly reduces MAE and RMSE relative to B1; Trees B2 raises both. These are descriptive complementary errors, with saved intervals rather than additional primary tests. Table 12 gives the original units; Figure 13 uses the scaled display.


## Step 5: Claims ledger

Each report value below is transcribed from the PDF. The comparison column distinguishes paired-loss calculations from saved statistics read from verified files. Tolerances are absolute: 0.0005 percentage points for three-decimal reductions, 1e-6 QLIKE, and exact equality for counts and decision states. Probabilities use 0.00005 to match four-decimal printing. Other rounded percentages use half their printed unit, shown on each row. Complementary errors use their printed precision in realised-variance units. Any mismatch stops the notebook.

The checks cover Tables 7, 11, 12 and 13.


In [5]:
# @title Claims ledger
# Fixed values transcribed from the report; never replace them with source-file values.
REPORT_CLAIMS = [{'report_location': 'Table 7', 'claim': 'Linear H1: estimate', 'report_literal': '+0.001616', 'report_value': 0.001616, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: ci_low', 'report_literal': '+0.000253', 'report_value': 0.000253, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: ci_high', 'report_literal': '+0.003454', 'report_value': 0.003454, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: percent_reduction_mean', 'report_literal': '+0.880', 'report_value': 0.88, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: p_raw', 'report_literal': '0.0390', 'report_value': 0.039, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Linear H2: estimate', 'report_literal': '+0.001134', 'report_value': 0.001134, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: ci_low', 'report_literal': '+0.000335', 'report_value': 0.000335, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: ci_high', 'report_literal': '+0.001932', 'report_value': 0.001932, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: percent_reduction_mean', 'report_literal': '+0.623', 'report_value': 0.623, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: p_raw', 'report_literal': '0.0032', 'report_value': 0.0032, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Linear H2: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Trees H1: estimate', 'report_literal': '+0.002197', 'report_value': 0.002197, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: ci_low', 'report_literal': '+0.000611', 'report_value': 0.000611, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: ci_high', 'report_literal': '+0.004103', 'report_value': 0.004103, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: percent_reduction_mean', 'report_literal': '+1.170', 'report_value': 1.17, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0135', 'report_value': 0.0135, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H1: decision', 'report_literal': 'Reject null', 'report_value': 'Reject null', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 7', 'claim': 'Trees H2: estimate', 'report_literal': '-0.000213', 'report_value': -0.000213, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: ci_low', 'report_literal': '-0.002044', 'report_value': -0.002044, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: ci_high', 'report_literal': '+0.001104', 'report_value': 0.001104, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: percent_reduction_mean', 'report_literal': '-0.115', 'report_value': -0.115, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: p_raw', 'report_literal': '0.6280', 'report_value': 0.628, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 7', 'claim': 'Trees H2: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Linear H1: estimate', 'report_literal': '+0.000416', 'report_value': 0.000416, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: ci_low', 'report_literal': '-0.002209', 'report_value': -0.002209, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: ci_high', 'report_literal': '+0.002878', 'report_value': 0.002878, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: percent_reduction_mean', 'report_literal': '+0.174', 'report_value': 0.174, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: p_raw', 'report_literal': '0.3908', 'report_value': 0.3908, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H1: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Linear H2: estimate', 'report_literal': '+0.004767', 'report_value': 0.004767, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: ci_low', 'report_literal': '-0.001109', 'report_value': -0.001109, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: ci_high', 'report_literal': '+0.014742', 'report_value': 0.014742, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: percent_reduction_mean', 'report_literal': '+1.997', 'report_value': 1.997, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: p_raw', 'report_literal': '0.1758', 'report_value': 0.1758, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Linear H2: decision', 'report_literal': 'Gate closed', 'report_value': 'Gate closed', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'log_ridge_harq', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Trees H1: estimate', 'report_literal': '+0.007482', 'report_value': 0.007482, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: ci_low', 'report_literal': '+0.001644', 'report_value': 0.001644, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: ci_high', 'report_literal': '+0.016807', 'report_value': 0.016807, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: percent_reduction_mean', 'report_literal': '+3.163', 'report_value': 3.163, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: p_raw', 'report_literal': '0.0568', 'report_value': 0.0568, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H1: decision', 'report_literal': 'Do not reject', 'report_value': 'Do not reject', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B1_over_B0'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 13', 'claim': 'Trees H2: estimate', 'report_literal': '-0.002272', 'report_value': -0.002272, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: ci_low', 'report_literal': '-0.009072', 'report_value': -0.009072, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_low', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: ci_high', 'report_literal': '+0.002195', 'report_value': 0.002195, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'ci_high', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: percent_reduction_mean', 'report_literal': '-0.992', 'report_value': -0.992, 'unit': 'percentage points', 'tolerance': 0.0005, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'percent_reduction_mean', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: p_raw', 'report_literal': '0.7554', 'report_value': 0.7554, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 13', 'claim': 'Trees H2: decision', 'report_literal': 'Gate closed', 'report_value': 'Gate closed', 'unit': 'status', 'tolerance': 0, 'source': 'artifacts/rp4_v4_b4/primary_statistics.csv', 'filter': {'horizon_minutes': 15, 'window': 'confirmation', 'family': 'lightgbm_qlike', 'contrast': 'B2_over_B1'}, 'column': 'hypothesis_status', 'operation': 'status_label'}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: N_sessions', 'report_literal': '417', 'report_value': 417.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: N_origins', 'report_literal': '159,054', 'report_value': 159054.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: reference_qlike', 'report_literal': '0.557917', 'report_value': 0.557917, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'reference_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: model_qlike', 'report_literal': '0.182054', 'report_value': 0.182054, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'model_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'Seasonal persistence: p_raw', 'report_literal': '0.0001', 'report_value': 0.0001, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'seasonal_persistence', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: N_sessions', 'report_literal': '419', 'report_value': 419.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_sessions', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: N_origins', 'report_literal': '160,832', 'report_value': 160832.0, 'unit': 'count', 'tolerance': 0, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'N_origins', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: reference_qlike', 'report_literal': '0.408061', 'report_value': 0.408061, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'reference_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: model_qlike', 'report_literal': '0.183660', 'report_value': 0.18366, 'unit': 'QLIKE', 'tolerance': 1e-06, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'model_qlike', 'operation': None}, {'report_location': 'Table 11', 'claim': 'HAR d/w/m: p_raw', 'report_literal': '0.0001', 'report_value': 0.0001, 'unit': 'probability', 'tolerance': 5e-05, 'source': 'artifacts/rp4_robustness_public_v1/reference_contrasts.csv', 'filter': {'horizon': 15, 'reference': 'HAR_dwm', 'family': 'log_ridge_harq', 'set': 'B0'}, 'column': 'p_raw', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B0: MAE', 'report_literal': '4.68118e-6', 'report_value': 4.68118e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B0', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B0: RMSE', 'report_literal': '3.36418e-5', 'report_value': 3.36418e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B0', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B1: MAE', 'report_literal': '4.71957e-6', 'report_value': 4.71957e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B1', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B1: RMSE', 'report_literal': '3.35516e-5', 'report_value': 3.35516e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B1', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B2: MAE', 'report_literal': '4.69114e-6', 'report_value': 4.69114e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B2', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Linear B2: RMSE', 'report_literal': '3.35190e-5', 'report_value': 3.3519e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'log_ridge_harq', 'set': 'B2', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B0: MAE', 'report_literal': '4.73740e-6', 'report_value': 4.7374e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B0', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B0: RMSE', 'report_literal': '3.47724e-5', 'report_value': 3.47724e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B0', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B1: MAE', 'report_literal': '4.77047e-6', 'report_value': 4.77047e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B1', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B1: RMSE', 'report_literal': '3.48570e-5', 'report_value': 3.4857e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B1', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B2: MAE', 'report_literal': '4.78032e-6', 'report_value': 4.78032e-06, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B2', 'metric': 'MAE'}, 'column': 'estimate', 'operation': None}, {'report_location': 'Table 12', 'claim': 'Trees B2: RMSE', 'report_literal': '3.48763e-5', 'report_value': 3.48763e-05, 'unit': 'variance', 'tolerance': 5e-11, 'source': 'artifacts/rp4_robustness_public_v1/secondary_metrics.csv', 'filter': {'horizon': 15, 'window': 'primary', 'family': 'lightgbm_qlike', 'set': 'B2', 'metric': 'RMSE'}, 'column': 'estimate', 'operation': None}]

def build_claims_ledger(report_claims, read_public_csv, fetch_bytes, checks, quick=False):
    """Compare fixed report values with verified files and the calculated primary contrasts."""
    import html
    import json
    import math
    import numpy as np
    import pandas as pd
    from IPython.display import HTML, display

    primary_path = 'artifacts/rp4_v4_b4/primary_statistics.csv'
    summary_path = 'artifacts/rp4_v4_b2_rv15/summary.json'
    primary = read_public_csv(primary_path)
    cache = {primary_path: primary}

    def read_rows(source):
        if source not in cache:
            path, _, key = source.partition('#')
            if key:
                value = json.loads(fetch_bytes(path))
                for name in key.split('.'):
                    value = value[name]
                cache[source] = pd.DataFrame(value)
            else:
                cache[source] = read_public_csv(path)
        return cache[source]

    def select(frame, filters):
        chosen = frame
        for name, value in filters.items():
            chosen = chosen.loc[chosen[name].eq(value)]
        return chosen

    def one(frame, filters):
        chosen = select(frame, filters)
        assert len(chosen) == 1, f'Expected one result for {filters}; found {len(chosen)}'
        return chosen.iloc[0]

    def primary_row(horizon, family='log_ridge_harq', contrast='B2_over_B1'):
        return one(primary, {'horizon_minutes': horizon, 'window': 'primary',
                             'family': family, 'contrast': contrast})

    # Check the separately published segment table against its saved summary.
    if not quick:
        segment_rows = read_rows('artifacts/rp4_v4_b4/regime_secondary.csv')
        summary_rows = read_rows(summary_path + '#regime_secondary')
        for row in select(segment_rows, {'horizon_minutes': 15, 'window': 'primary',
                                         'statistic': 'mean'}).itertuples():
            if row.subset not in ('first_hour', 'last_hour'):
                continue
            saved = one(summary_rows, {'subset': row.subset, 'family': row.family,
                                       'contrast': row.contrast, 'statistic': 'mean'})
            for name in ('estimate', 'ci_low', 'ci_high', 'p_raw', 'p_holm'):
                assert abs(float(saved[name]) - float(getattr(row, name))) <= 1e-12

    results = []
    selected_claims = [c for c in report_claims
                       if c['source'] != 'NOT_IN_12_CSV'
                       and (not quick or c['report_location'] in
                            ('Table 7', 'Table 11', 'Table 12', 'Table 13'))]
    assert selected_claims, 'The claims list is empty.'
    for claim in selected_claims:
        filters = dict(claim['filter'])
        operation = claim.get('operation')
        source = claim['source']
        column = claim['column']
        basis = 'Saved statistic verified'
        if claim['report_location'] == 'Table 7' and column in ('estimate', 'percent_reduction_mean'):
            row = one(checks, {'family': filters['family'], 'contrast': filters['contrast']})
            actual = float(row['delta_QLIKE' if column == 'estimate' else 'reduction_percent'])
            basis = 'Paired losses recalculated'
        elif operation and operation.startswith('placebo_'):
            horizon = filters['horizon']
            draws = select(read_rows(source), filters)['delta'].to_numpy(dtype=float)
            assert len(draws) == 50 and np.isfinite(draws).all()
            observed = float(primary_row(horizon).estimate)
            operation_name = operation[len('placebo_'):]
            actual = {
                'count': len(draws), 'observed': observed, 'mean': math.fsum(draws) / len(draws),
                'quantile_025': float(np.quantile(draws, .025)),
                'quantile_975': float(np.quantile(draws, .975)),
                'exceedances': int((draws > observed).sum()),
                'ascending_rank': 1 + int((draws < observed).sum()),
                'empirical_p': (1 + int((draws >= observed).sum())) / (len(draws) + 1),
                'retained_percent': 100 * (math.fsum(draws) / len(draws)) / observed,
            }[operation_name]
            basis = 'Saved permutation draws summarised'
        elif claim['report_location'] == 'Table 17' and operation:
            row = one(read_rows(source), filters)
            state_filter = dict(filters, contrast='B1_over_B0')
            state = one(read_rows(source), state_filter)
            baseline = float(primary_row(15, contrast='B1_over_B0').baseline_loss)
            if filters['contrast'] == 'B2_over_B1':
                baseline -= float(state.estimate)
            assert baseline > 0
            actual = 100 * float(row.estimate) / baseline
            basis = 'Saved cutoff difference rescaled'
        elif claim['report_location'] == 'Table 3':
            all_assets = filters.get('asset') == 'All'
            if all_assets:
                filters.pop('asset')
            rows = select(read_rows(source), filters)
            assert len(rows) == (6 if all_assets else 1)
            if column:
                actual = float(rows[column].sum())
            elif source.endswith('#empty_window_secondary.census'):
                actual = 100 * rows.N_empty_origins.sum() / rows.N_origins.sum()
            else:
                actual = 100 * rows.eligible_rows.sum() / rows.scheduled_rows.sum()
            basis = 'Saved census counts aggregated'
        else:
            row = one(read_rows(source), filters)
            actual = row[column]
            if operation == 'status_label':
                actual = {'REJECTED': 'Reject null', 'NOT_REJECTED': 'Do not reject',
                          'NOT_TESTED': 'Gate closed'}[actual]
        expected = claim['report_value']
        tolerance = claim['tolerance']
        if claim['unit'] == 'status':
            matched = actual == expected
            computed = str(actual)
            allowed = 'Exact state'
        else:
            actual = float(actual)
            assert math.isfinite(actual) and math.isfinite(float(expected))
            matched = abs(actual - expected) <= tolerance
            computed = f'{actual:.10g}'
            allowed = f'{tolerance:g} {claim["unit"]}'
        assert matched, f'{claim["report_location"]}, {claim["claim"]}: {actual} differs from {expected}'
        results.append({'Report location': claim['report_location'], 'Claim': claim['claim'],
                        'Report value': claim['report_literal'], 'Recomputed here': computed,
                        'Tolerance': allowed, 'Basis': basis, 'Match': '✔'})
    ledger = pd.DataFrame(results)
    assert len(ledger) == len(selected_claims) and ledger['Match'].eq('✔').all()
    # Group the complete ledger into expandable tables for a compact reading view.
    parts = [f'<p><strong>Claims ledger: {len(ledger)} of {len(ledger)} matched ✔</strong></p>']
    for location, group in ledger.groupby('Report location', sort=False):
        parts.append('<details><summary>' + html.escape(location) + f': {len(group)} matched</summary>'
                     + group.drop(columns='Report location').to_html(index=False, escape=True)
                     + '</details>')
    display(HTML(''.join(parts)))
    return ledger

# Stop on the first mismatch before allowing the receipt to run.
claims_ledger = build_claims_ledger(REPORT_CLAIMS, read_public_csv, fetch_bytes, checks, quick=True)

# Record that this step completed.
EXECUTED_CELLS.append('step_05_claims_ledger')


Claim,Report value,Recomputed here,Tolerance,Basis,Match
Linear H1: estimate,+0.001616,0.001616311703,1e-06 QLIKE,Paired losses recalculated,✔
Linear H1: ci_low,+0.000253,0.0002529862672,1e-06 QLIKE,Saved statistic verified,✔
Linear H1: ci_high,+0.003454,0.003453534377,1e-06 QLIKE,Saved statistic verified,✔
Linear H1: percent_reduction_mean,+0.880,0.8800546831,0.0005 percentage points,Paired losses recalculated,✔
Linear H1: p_raw,0.0390,0.039,5e-05 probability,Saved statistic verified,✔
Linear H1: decision,Reject null,Reject null,Exact state,Saved statistic verified,✔
Linear H2: estimate,+0.001134,0.001133759659,1e-06 QLIKE,Paired losses recalculated,✔
Linear H2: ci_low,+0.000335,0.0003350369703,1e-06 QLIKE,Saved statistic verified,✔
Linear H2: ci_high,+0.001932,0.001932161587,1e-06 QLIKE,Saved statistic verified,✔
Linear H2: percent_reduction_mean,+0.623,0.6227940964,0.0005 percentage points,Paired losses recalculated,✔


### Reading

Every checked row must show ✔. A saved probability is verified against the report, while the four Table 7 means and reductions are recalculated from paired session losses. The ledger therefore states which kind of evidence supports each row.


## What this notebook establishes

The headline values in Tables 7, 11, 12 and 13 agree with the published aggregates within the stated tolerances. The input checks fail if any byte changes; the ledger stops if a checked value differs from the report.

## What it cannot establish

Model fitting, predictor construction and point-in-time filters run on licensed minute-level data that cannot be redistributed. Their outputs enter this notebook as saved aggregates. Section 3.10 explains what is retained and how access works. The historical final window does not confirm the full sequence in Table 13.

## Receipt

The final cell checks the exact step order and records the input fingerprints, numerical contrasts and software versions. A completed run shows PASS_PUBLIC_SAVED_RESULTS. It records the verification supporting Table 7 and the Claims ledger.


In [6]:
# @title Receipt
receipt = {'status':'PASS_PUBLIC_SAVED_RESULTS', 'source_commit':SOURCE_COMMIT,
    'public_source_commit':PUBLIC_SOURCE_COMMIT,
    'bundle_sha256':BUNDLE_SHA256, 'input_sha256':PUBLIC_HASHES,
    'sessions':len(losses), 'contrasts_recalculated':checks.to_dict('records'),
    'public_reference_rows':len(reference_qlike),
    'public_reference_contrast_rows':len(reference_contrasts),
    'secondary_metric_rows':len(secondary_metrics), 'new_fits':0,
    'restricted_files_read':0, 'widgets':0,
    'scope':'Public saved-results arithmetic and static figures; no training or new inference',
    'runtime':{'python':sys.version.split()[0], 'pandas':pd.__version__, 'matplotlib':matplotlib.__version__}}

# Require every preceding step, including the ledger, to have completed.
EXECUTED_CELLS.append('step_06_receipt')
assert EXECUTED_CELLS == ['step_01_verify_inputs', 'step_02_the_four_primary_contrasts', 'step_03_the_final_window', 'step_04_reference_models_and_complementary_errors', 'step_05_claims_ledger', 'step_06_receipt']
receipt['executed_code_cells']=len(EXECUTED_CELLS)
receipt['executed_code_cell_ids']=EXECUTED_CELLS.copy()
receipt_json=json.dumps(receipt,indent=2,ensure_ascii=False)
Path('PUBLIC_EXECUTION_RECEIPT.json').write_text(receipt_json,encoding='utf-8')
# Present a short result; retain the complete machine-readable record below.
display(HTML('<h3>Verification complete</h3><p><b>PASS_PUBLIC_SAVED_RESULTS</b></p><p>'+str(len(EXECUTED_CELLS))+' code cells completed; '+str(len(claims_ledger))+' report values matched.</p><details><summary>Complete execution receipt</summary><pre>'+html.escape(receipt_json)+'</pre></details>'))

